---
title: Provenance Razor Control Specification
version: 0.3.0-alpha
supersedes: 0.2.0-alpha
status: alpha
confidence-classification: provisional
artifact-class: research-specification
license: MIT
owner: Kevin R. Barrow

evidence-basis:
  - mathematical-formalization
  - executable-semantics
  - conformance-vectors
  - cross-runtime-verification
  - randomized-property-and-totality-testing

maturity:
  control-architecture: stable-draft
  canonical-encoding: provisional        # PRC-LP-0
  transport-schemas: provisional         # Protobuf and FlatBuffers, both compiled and round-trip tested
  scoring-profile: experimental          # coverage-v0.3.0-alpha
  reference-runtime: python
  conformance-vectors: generated-provisional
  cross-runtime-verification: python-and-javascript
  dag-decomposition: pending
  independent-verification: pending

version-policy:
  major: incompatible normative change
  minor: normative extension
  patch: clarification or correction
  pre-1.0: incompatible normative changes increment the minor version (SemVer 2.0.0 item 4)
---

## Provenance Razor Control Specification
#### v0.3.0 Alpha

The Provenance Razor Control Specification is an evolving research and engineering framework for deterministic, lineage-aware execution control. Everything in this document is provisional. Version status reflects the quantity and quality of supporting evidence, and nothing here is frozen.

Consistent with the Provenance Razor itself, the specification treats its own assertions as hypotheses. Its objective is not certainty. It is a deterministic framework for judging whether an assertion has enough evidence, verified provenance, verified membership, observation and reconciliation to justify confidence proportional to the consequences of error. Each result carries only the authority its provenance warrants.

**Changes** are summarised in §6. Every normative change is backed by conformance vectors in `tests/`.

---

### 1. Key Domains, Canonical Encoding, Signatures and Validity

#### 1.1 Key domains

$$\text{Domain}(K) \in \{ K_{\text{profile}}, K_{\text{approval}}, K_{\text{decision}}, K_{\text{receipt}}, K_{\text{provenance}}, K_{\text{membership}} \}$$

1. $K_{\text{profile}}$ signs control profiles, scoring profiles and trust-state objects.
2. $K_{\text{approval}}$ signs manual approvals ($p$).
3. $K_{\text{decision}}$ signs decision capabilities ($d$).
4. $K_{\text{receipt}}$ signs receipts ($r$).
5. $K_{\text{provenance}}$ signs provenance attestations ($\pi$, §2.4).
6. $K_{\text{membership}}$ signs family manifests ($\mu$, §2.6). New in 0.3.0.

**Domain-scoped resolution.** $\text{Verify}_{K_{\text{dom}}}$ means three separate steps:
1. resolve the key identifier *only* among keys registered in domain `dom`;
2. verify the signature;
3. check epoch and policy authorisation.

A key registered in another domain is treated as not found. The same rule applies to every object type.

#### 1.2 Canonical encoding and hashing

$$\mathcal{H}(z) \equiv \text{SHA256}\left( \text{PRC-LP-0}(z) \right)$$

- $z$ is the structured canonical object, encoded exactly once.
- PRC-LP-0 is the canonical, length-prefixed tag-length-value encoding (A-01). Decoders MUST reject every non-canonical byte string.
- Integers are int64, and every integer this specification defines is non-negative. Reason-code masks therefore use bits 0–62; **bit 63 is reserved.**
- Timestamps are int64 microseconds since the Unix epoch, UTC.

#### 1.3 Detached signatures

$$\sigma_z = \text{Sign}_{K_{\text{domain}(z)}}\left( \mathcal{H}(z) \right)$$

A signature is never a field of the object it signs.

#### 1.4 Transport encodings

Protobuf and FlatBuffers (`schema/`) are transport and storage formats only. Their bytes MUST NOT be hashed, signed or compared. A verifier MUST:

1. parse the transport message;
2. reject unknown fields;
3. validate presence, lengths and enumerations;
4. convert to the canonical object;
5. apply §1.5;
6. compute $\mathcal{H}$ over PRC-LP-0;
7. verify the detached signature.

#### 1.5 Canonical object validity and total verification

**CanonicalObjectValid(z, schema)** runs after decoding and before any signature or semantic predicate. It requires:

- exactly the schema's fields, with none missing and none extra;
- every field of its declared type, with a boolean never accepted as an integer;
- non-negative integers within int64;
- a supported `schema_ver`;
- 32-byte lowercase-hex digests;
- canonical lowercase UUIDv7 identifiers;
- key identifiers of the form `kid:<domain>:<16 hex>`;
- enumeration values from the defined set.

Schemas are defined for $d$, $p$, $r$, $\pi$ and $\mu$. Evidence members have an open schema: `evidence_id`, `parent_id`, `producer`, `value_bits` (16 hex digits) and optional `state` are required to be well formed, and claim fields may be added.

**V-1 (total verification).** For every input $z$:

$$\text{Verify}(z) \in \{\text{ALLOW}, \text{MANUAL\_REQUIRED}, \text{DENY}\}$$

Verify MUST NOT raise or terminate abnormally. A malformed object yields a structured rejection: `DECODE_INVALID`, `OBJECT_SCHEMA_INVALID` or `APPROVAL_SCHEMA_INVALID`. Structurally invalid families or members are excluded and audited. Any residual internal fault is converted at the verifier boundary into DENY with `SAFE_STATE_LOCK | VERIFIER_FAULT`.

---

### 2. Evidence, Provenance, Membership and Deterministic Math

#### 2.1 Equivalence classes and the scoring profile

Evidence is partitioned into lineage families $\bar{F}_1, \dots, \bar{F}_m$. The members of a family are exactly those admitted by §2.6.

- $F_k^*$ is the family's unique structural root among its admitted members.
- A family with zero or several roots, a cycle, or a duplicate identifier is malformed. It is excluded and audited.
- The family value is:

$$V(\bar{F}_k) = V(F_k^*) \cdot A(\bar{F}_k), \qquad V(F_k^*) \in [0,1]$$

$M(x,a)$, $C(x,a)$, $T(x,a)$ and $T_{\text{manual}}(x,a)$ are defined by the active **scoring profile**. The scoring profile is bound by $d.\text{scoring\_profile\_hash}$ (§2.7). A scoring profile MUST:
- be deterministic;
- obey §2.3;
- take dependency atoms only from §2.4 and family members only from §2.6.

Positive mass SHOULD be monotone. The reference candidate is `coverage-v0.3.0-alpha`.

$$F_{\text{eligible}}(x,a) = \emptyset \implies M(x,a) = 0.0 \implies C(x,a) = 0.0$$

#### 2.2 Optional member states

$$\text{State}(F_i) \in \{\text{VALID}, \text{UNAVAILABLE\_OR\_STALE}, \text{DEGRADED}, \text{CONTRADICTORY}, \text{INVALID}\}$$

* **UNAVAILABLE_OR_STALE:** no positive support. $A$ is unaffected.
* **DEGRADED:** $A(\bar{F}_k) = \kappa_{\text{degraded}}$.
* **CONTRADICTORY:** no positive support, and **no effect on $A$.** Its effect on authorization is governed only by §2.5.
* **INVALID:** the family is invalidated and a security audit event is emitted.

$A(\bar{F}_k)$ is the minimum factor over contributing members, or 1.0 if none contribute. A TAMPERED provenance attestation (§2.4) makes its member INVALID.

#### 2.3 Numerical comparison

$$\text{AuthorizationCheck}(C, T) \iff (C(x,a) \ge T(x,a)) = \text{TRUE}$$

- **E-1:** every binary64 input MUST be finite before any profile logic runs. Otherwise the result is DENY with Safe State Lock.
- **N-1:** binary64 with every intermediate rounded separately; no fused multiply-add, reassociation or extended precision. Transcendental functions MUST be profile-specified algorithms, such as `prc-exp-v1`.
- No epsilon tolerances. `-0.0` normalises to `+0.0`. Subnormals are preserved. A NaN or ±Inf forces $C = 0.0$ and **State 2 (Safe State Lock)**, which always denies.

#### 2.4 Verified provenance (P-1)

Self-declared dependencies MUST NOT be used. An atom `TYPE:id` is admissible only if it arrives in a provenance attestation that satisfies CanonicalObjectValid and:

$$\text{DependencyAdmissible}(\pi, e) \iff \begin{cases} \text{Verify}_{K_{\text{provenance}}}(\mathcal{H}(\pi), \sigma_\pi) \land \pi.\text{domain} = \text{"PROVENANCE-RAZOR-PROVENANCE-V0.3.0-Alpha"} \\ \land\; \pi.\text{attester\_id} = \text{Owner}(\pi.\text{provenance\_key\_id}) \land \pi.\text{evidence\_digest} = \mathcal{H}(\text{EvidenceCore}(e)) \\ \land\; \text{EpochAuthorized} \land \neg\text{Revoked}(\pi) \land \pi.\tau_{\text{issued}} \le \tau_{\text{eval}} \le \pi.\tau_{\text{expiry}} \land \pi.\text{attester\_id} \ne e.\text{producer} \\ \land\; \forall t: \text{AttesterAuthorized}(\text{type}(t)) \land \text{Registered}(\text{Resolve}(t)) \land \forall \theta: |\text{atoms of type } \theta| \le \text{Cap}(\theta) \end{cases}$$

- The atom registry, the alias table, the caps and the independence rule belong to the **trust state** (§2.7).
- Outcomes:
  - **ADMITTED:** the resolved atoms are used.
  - **UNATTESTED** or **INADMISSIBLE:** the lineage is unknown.
  - **TAMPERED:** the member is INVALID.

#### 2.5 Adverse disposition (A-1)

A contradicting member is **relevant** only if it meets both conditions:
- its `conflict_type` is recognised by the trust state;
- its `contradicts` field equals the request's $\langle$action, assertion, target, evidence_snapshot_id$\rangle$.

Irrelevant contradictions are audited and have no effect. For each relevant contradicting member $e$ admitted by §2.6:

$$\text{ContradictionDisposition}(e)= \begin{cases} \text{DEFEAT}, & \text{Provenance}(e)=\text{ADMITTED}\\ \text{CONTAIN}, & \text{Provenance}(e)\in\{\text{UNATTESTED},\text{INADMISSIBLE}\}\\ \text{LOCK}, & \text{Provenance}(e)=\text{TAMPERED} \end{cases}$$

$$\text{AdverseDisposition}(x,a) = \max_{\prec}\{\text{ContradictionDisposition}(e)\} \cup \{\text{CLEAR}\}, \qquad \text{CLEAR} \prec \text{CONTAIN} \prec \text{DEFEAT} \prec \text{LOCK}$$

A numeric Safe State Lock also yields LOCK. Contradictions never change $A$, $V$, $M$, $C$ or $T$. They affect only which authorization modes are eligible:

$$\text{AutomatedAllow}(x,a) \iff C \ge T \land \text{AdverseDisposition} = \text{CLEAR}$$

$$\text{ManualAllow}(x,a) \iff C \ge T_{\text{manual}} \land \text{AdverseDisposition} \in \{\text{CLEAR}, \text{CONTAIN}\} \land \text{ExternalApprovalValid}(p, d)$$

| Disposition | Outcome |
|---|---|
| LOCK | DENY, with Safe State Lock |
| DEFEAT | DENY. An ordinary manual approval cannot override it; any future override needs a separately named, higher-authority exception object. |
| CONTAIN | MANUAL_REQUIRED if $C \ge T_{\text{manual}}$, otherwise DENY |
| CLEAR | ALLOW if $C \ge T$, otherwise DENY |

**Containment obligations.**
- Relevant adverse claims collapse under the canonical contradiction digest $\mathcal{H}(\langle\text{conflict\_type}, \text{contradicts}, \text{claim}\rangle)$. Repeated submissions of the same claim create one obligation.
- Each occurrence is audited with its ingress channel.
- `containment_digest` is $\mathcal{H}$ of the sorted CONTAIN digests, or NULL_CONTAINMENT_DIGEST when the disposition is CLEAR.

$$\text{NULL\_CONTAINMENT\_DIGEST} = \mathcal{H}\left(\text{"PROVENANCE-RAZOR-NO-CONTAINMENT-V0.3.0-Alpha"}\right)$$

#### 2.6 Family membership (M-1)

Membership is asserted by a **family manifest** $\mu$ signed in $K_{\text{membership}}$ by a membership authority, typically the ingestion gateway. It contains:
- domain, schema_ver, manifest_id, membership_key_id, membership_trust_epoch;
- authority_id and family_id;
- ingress_channel;
- members, each as $\langle$evidence_id, $\mathcal{H}(\text{EvidenceCore}(e))$, parent_id$\rangle$, sorted and unique;
- $\tau_{\text{issued}}$ and $\tau_{\text{expiry}}$.

Evaluating a family's manifest gives one of three outcomes:

- **TAMPERED** — the family is invalidated and a security event emitted. Causes:
  - a malformed envelope or payload, or a wrong domain string;
  - a key from another domain, or a bad signature;
  - an authority_id that is not the key's owner, or a family_id that doesn't match (a replayed manifest);
  - a non-canonical member list;
  - a listed member altered after the manifest was issued.
- **UNVERIFIED** — the family is excluded from support, and its contradictions have no effect. Causes:
  - no manifest, or a manifest signed by a key that isn't registered;
  - an unauthorised epoch, a revoked manifest, or a manifest that isn't fresh;
  - the authority's `membership_scopes` don't cover the family_id;
  - a listed member withheld from the submission.
- **VERIFIED** — members are classified individually:
  - **LISTED:** the digest and parent match the manifest.
  - **QUARANTINED:** the member is not listed, or it reuses a listed evidence_id without matching the manifest. It is excluded entirely and audited only: it can neither invalidate, attenuate, contain nor defeat the family it names.

Structurally invalid members (§1.5) are quarantined before membership is evaluated.

#### 2.7 Trust context and binding

Each rule is committed by exactly one object:

| Object | Commits | Bound as |
|---|---|---|
| Control profile | ManualOnly, $\Delta\tau_{\max}$, reusable authority classes | `control_profile_hash` |
| Scoring profile | Ω, $T_{\text{base}}$, $T_{\text{base,manual}}$, β, κ values, gate, Γ, material types, component ids | `scoring_profile_hash` |
| Trust state | key registry for all six domains (owners, epochs, authority classes, authorised atom types, membership scopes), revocations of attestations and manifests, atom registry and aliases, per-type caps, attester independence, conflict types, clock authorities | `trust_state_hash` |

The verifier MUST confirm all three:
- $d.\text{trust\_state\_hash} = \mathcal{H}(\text{TrustState}_{\text{active}})$;
- $d.\text{scoring\_profile\_hash} = \mathcal{H}(\text{ScoringProfile}_{\text{active}})$;
- $d.\text{control\_profile\_hash} = \mathcal{H}(\text{ControlProfile}_{\text{active}})$.

Two evaluators that use different registries, alias authorities, revocation sets or attester authorisations therefore produce different decision bindings.

---

### 3. Canonical Objects and Bidirectional Binding

#### 3.1 External manual approval ($p$)

$p$ carries:
- domain `"PROVENANCE-RAZOR-APPROVAL-V0.3.0-Alpha"`, schema_ver, approval_id, approval_key_id, approval_trust_epoch;
- authority_class and authority_scope;
- $x$, $a$, target, actor, req_id, capability_id, state_ver;
- control_profile_hash, scoring_profile_hash, trust_state_hash;
- adverse_disposition, containment_digest;
- $\tau_{\text{issued}}$, $\tau_{\text{expiry}}$, and decision = ALLOW.

$$\text{ExternalApprovalValid}(x,a,d,p,\sigma_p) \iff \begin{cases} \text{CanonicalObjectValid}(p, \text{approval}) \land \text{Verify}_{K_{\text{approval}}}(\mathcal{H}(p), \sigma_p) \land \text{domain ok} \\ \land\; \text{ApprovalKeyAuthorized} \land \text{ScopePermits}(p.\text{authority\_scope}, x, p.\text{target}) \\ \land\; p.x = x \land p.a = a \land p.\text{state\_ver} = \text{CurrentStateVersion} \land p.\text{control\_profile\_hash} = \text{Active} \\ \land\; p.\text{scoring\_profile\_hash} = d.\text{scoring\_profile\_hash} \land p.\text{trust\_state\_hash} = d.\text{trust\_state\_hash} \\ \land\; p.\text{target} = d.\text{target} = \text{CurrentTarget} \land p.\text{actor} = d.\text{actor} \land p.\text{req\_id} = d.\text{req\_id} \land p.\text{capability\_id} = d.\text{capability\_id} \\ \land\; p.\text{adverse\_disposition} = d.\text{adverse\_disposition} \land p.\text{containment\_digest} = d.\text{containment\_digest} \\ \land\; p.\tau_{\text{issued}} \le d.\tau_{\text{eval}} \land \tau_{\text{exec}} + U \le p.\tau_{\text{expiry}} \land \text{ApprovalUnconsumedOrReusableAsClassified}(p) \end{cases}$$

The approver acknowledges exactly the adverse claims the decision carries. An approval of a clean request cannot be reused once containment appears, and an approval of one set of claims cannot cover another.

#### 3.2 Null digests

$$\text{NULL\_APPROVAL\_DIGEST} = \mathcal{H}\left( \text{"PROVENANCE-RAZOR-NO-APPROVAL-V0.3.0-Alpha"} \right)$$

#### 3.3 Decision capability ($d$)

$d$ carries:
- domain `"PROVENANCE-RAZOR-DECISION-V0.3.0-Alpha"`, schema_ver, profile_version, lineage_schema_version;
- capability_id, req_id, evidence_snapshot_id, actor, auth_mode ∈ {AUTOMATED, MANUAL};
- $x$, $a$, target, state_ver;
- **control_profile_hash, scoring_profile_hash, trust_state_hash**, evidence_hash, lineage_hash, approval_hash;
- **adverse_disposition ∈ {CLEAR, CONTAIN}, containment_digest**;
- decision_key_id, decision_trust_epoch, **eval_clock_id**;
- $\tau_{\text{eval}}$, $\tau_{\text{expiry}}$, decision = ALLOW, and reason_codes (a non-negative bitfield).

A decision is never issued for DEFEAT or LOCK.

#### 3.4 Bidirectional authorization mode validation

$$\text{AuthorizationModeValid} \iff \begin{cases} d.\text{auth\_mode} = \text{AUTOMATED} \land x \notin \text{ManualOnly} \land d.\text{adverse\_disposition} = \text{CLEAR} \land d.\text{approval\_hash} = \text{NULL\_APPROVAL\_DIGEST} \\ \lor\; d.\text{auth\_mode} = \text{MANUAL} \land (x \in \text{ManualOnly} \lor d.\text{adverse\_disposition} = \text{CONTAIN}) \land d.\text{approval\_hash} = \mathcal{H}(p) \land \text{ExternalApprovalValid} \end{cases}$$

Consistency: $d.\text{adverse\_disposition} = \text{CLEAR} \iff d.\text{containment\_digest} = \text{NULL\_CONTAINMENT\_DIGEST}$.

#### 3.5 Decision verification predicate and clock authority (C-1)

The evaluator's clock is $d.\text{eval\_clock\_id}$. The executor's clock is $\langle\text{clock\_id}, \tau_{\text{exec}}, u_{\text{exec}}\rangle$. Both MUST be registered clock authorities in the trust state, and $u_{\text{exec}}$ MUST NOT exceed the executor authority's maximum; otherwise the result is `CLOCK_UNTRUSTED`. Every time comparison uses the combined worst-case uncertainty:

$$U = u_{\text{eval}}^{\max} + u_{\text{exec}}$$

$$\text{DecisionBindingValid} \iff \begin{cases} \text{CanonicalDecodeValid}(d) \land \text{CanonicalObjectValid}(d, \text{decision}) \land \text{Verify}_{K_{\text{decision}}}(\mathcal{H}(d), \sigma_d) \land \text{domain ok} \\ \land\; d.x = x \land d.a = a \land d.\text{target} = \text{CurrentTarget} \land d.\text{state\_ver} = \text{CurrentStateVersion} \\ \land\; d.\text{control\_profile\_hash}, d.\text{scoring\_profile\_hash}, d.\text{trust\_state\_hash} \text{ match the active objects (§2.7)} \\ \land\; d.\text{evidence\_hash} = \mathcal{H}(F_{\text{eligible}}) \land d.\text{lineage\_hash} = \mathcal{H}(D_{x,a}) \\ \land\; d.\text{adverse\_disposition}, d.\text{containment\_digest} \text{ match the current evaluation} \land d.\text{decision} = \text{ALLOW} \\ \land\; \text{ClocksTrusted} \land d.\tau_{\text{eval}} + U \le \tau_{\text{exec}} \land \tau_{\text{exec}} + U \le d.\tau_{\text{expiry}} \land (\tau_{\text{exec}} - d.\tau_{\text{eval}}) + U \le \Delta\tau_{\max} \\ \land\; \text{DecisionKeyAuthorized} \land \text{CapabilityAvailable} \land \text{AuthorizationModeValid} \end{cases}$$

Verifiers report every failing reason bit. On a decoding or schema failure, only that single bit is reported.

---

### 4. Transaction Boundaries, State Preconditions and Signed Receipts

#### 4.1 Local targets (compare-and-swap)

The transaction proceeds in this order:

1. BEGIN, then lock the target.
2. Verify CapabilityAvailable.
3. Verify CurrentStateVersion = $d$.state_ver.
4. MarkCapabilityConsumed.
5. ApplyMutation.
6. AdvanceStateVersion.
7. WriteReceipt.
8. COMMIT.

If a verification fails, the transaction rolls back and a REJECTED/PRECONDITION receipt is written in a separate transaction.

#### 4.2 Distributed targets

This section is unchanged from 0.2.0. The target enforces the state precondition. Outcomes are COMMITTED, REJECTED (PRECONDITION, TARGET or PREPARE), ROLLED_BACK (DISPATCH) or UNRESOLVED (RECONCILIATION, with reconciliation_required). A reserved capability is never released.

#### 4.3 Execution receipt ($r$)

$r$ carries:
- domain `"PROVENANCE-RAZOR-RECEIPT-V0.3.0-Alpha"`, schema_ver, receipt_id;
- capability_id, req_id, decision_digest = $\mathcal{H}(d)$, $x$, target, **control_profile_hash**;
- expected_state_ver = $d$.state_ver, observed_pre_state_ver, observed_post_state_ver (null if unknown);
- execution_status, failure_phase, reason_code, reconciliation_required;
- pre_state_digest, post_state_digest ($\mathcal{H}(\text{State}_{\text{post}})$ if known, else NULL_STATE_DIGEST);
- $\tau_{\text{dispatch}}$, $\tau_{\text{finalized}}$, receipt_key_id, receipt_trust_epoch.

The scoring profile and trust state are committed through `decision_digest`.

$$\text{NULL\_STATE\_DIGEST} = \mathcal{H}\left( \text{"PROVENANCE-RAZOR-NO-STATE-V0.3.0-Alpha"} \right)$$

---

### 5. Conformance Status

| Deliverable | Status | Location / harness |
|---|---|---|
| Normative control specification | **ALPHA**; control architecture **STABLE DRAFT** | this document |
| Canonical encoding PRC-LP-0 | **PROVISIONAL** | A-01 |
| Transport schemas | **PROVISIONAL**: Protobuf and FlatBuffers compiled, round-trip tested, strict | `schema/`, `tools/transport_roundtrip.py` |
| Scoring profile `coverage-v0.3.0-alpha` | **EXPERIMENTAL** | `tests/vectors/scoring_coverage_v0.3.0-alpha.json` |
| Superseded profiles | `jaccard-v0.1.0-alpha`, `coverage-v0.2.0-alpha` (exploit reproduces by design), `coverage-v0.2.1-alpha` | `tests/vectors/` |
| Golden, numerical, lineage, authorization, transaction vectors | **GENERATED, PROVISIONAL** | `tests/` |
| Randomised property and totality tests | P1–P6, P9–P11 hold | `tools/property_tests.py` |
| Cross-runtime conformance | **PYTHON + JAVASCRIPT**; Go / Rust / C++ pending | `tools/check_vectors.py`, `tools/xcheck.mjs` |
| Independent external verification | **PENDING** | — |

**Exploit statement.** For `coverage-v0.3.0-alpha` there is no known open exploit **under the declared assumptions**:
- the trust state (key owners, authorisations, revocations);
- the atom registry and alias authority;
- the family-membership authorities;
- the clock authorities;
- the calibration.

Each of these is an assumption, not a result. The open questions are listed in `docs/CONFORMANCE_ASSUMPTIONS.md` (Q-1 to Q-6).

---

### 6. Changes

#### 6.1 From 0.2.0-alpha to 0.3.0-alpha

| Change | Section | Closes |
|---|---|---|
| CanonicalObjectValid for $d$, $p$, $r$, $\pi$, $\mu$ and evidence members; V-1 total verification | 1.5 | validly signed malformed decisions were accepted or raised exceptions |
| Domain-scoped key resolution for every object type | 1.1 | asymmetric cross-domain diagnostics |
| Reason masks non-negative; bit 63 reserved | 1.2, 3.3 | signed/unsigned mismatch |
| Contradictions no longer change $A$; A-1 disposition CLEAR / CONTAIN / DEFEAT / LOCK; MANUAL_REQUIRED outcome | 2.2, 2.5 | Q-2 (unverified contradictions suppressing support) |
| Containment obligations deduplicated; approval must acknowledge `containment_digest` | 2.5, 3.1 | review spam; stale approvals |
| $K_{\text{membership}}$ and signed family manifests; quarantine of unlisted members | 1.1, 2.6 | Q-3 (injection into another source's family) |
| Trust state as a committed object; `control_profile_hash`, `scoring_profile_hash`, `trust_state_hash` on $d$ and $p$ | 2.7, 3 | scoring profile and registry were not bound |
| C-1 clock authorities and combined uncertainty $U$ | 3.5 | G-10 |
| Receipt `profile_hash` renamed `control_profile_hash` | 4.3 | naming ambiguity |
| FlatBuffers compiled and round-trip tested | 1.4 | transport completeness |
| Exploit statement qualified by its assumptions | 5 | overclaim |
| Domain strings move to `V0.3.0-Alpha` | all | cross-version replay |

#### 6.2 From 0.1.0-alpha to 0.2.0-alpha (summary)

- `K_provenance` and P-1 verified provenance.
- D-1 authenticated defeaters.
- Approvals bound to target, actor, req_id and capability_id (G-01 to G-03).
- Expected vs observed receipt state.
- Single encoding layer and detached signatures (G-04).
- E-1 and N-1.
- Scoring delegated to profiles (G-05).
- Transport schemas.
- The "SPECIFICATION FROZEN" status removed.


---
# Conformance suite — specification 0.2.0-alpha

The first cell is the revised normative text (0.2.0-alpha). Its §6 lists every change from 0.1.0-alpha and the vectors behind each one. The table below records how the conformance harness itself changed from the original draft cells:

| Earlier draft | Now |
|---|---|
| `uuid4()` and `datetime.now()`, so every run produced new hashes | Fixed base time (2026-01-01T00:00:00Z), deterministic UUIDv7, deterministic Ed25519 test keys. Regeneration is byte-identical and checked. |
| Validators re-hashed with the same function that wrote the value, so they always passed | Every case **declares its intended outcome from the spec**. The generator refuses to write if the reference model disagrees. A separate **JavaScript implementation** re-derives the results. `MANIFEST.json` freezes the bytes. |
| Sorted-key JSON as "canonical" encoding | `PRC-LP-0` length-prefixed TLV encoding with a strict decoder and must-reject vectors (A-01) |
| Mock hashes and no signatures | Real `H(profile)`, `H(F_eligible)`, `H(D)`, and real Ed25519 signatures over `H(d)`, `H(p)`, `H(r)` in separate key domains |
| 2 auth cases, string reason code | 38 `DecisionBindingValid` cases (freshness, skew, key domains, tampering, decoding, bindings, bidirectional mode, approvals, 3 spec-gap cases) with a provisional bitfield registry |
| Kahan sum over NaN/Inf, result `nan` | §2.3 cases: exact-threshold, 1-ulp, −0.0 normalisation, subnormals, quiet/signalling/negative NaN, ±Inf. The Safe State Lock is asserted. Kahan cases are informative. |
| 2 YAML examples, never loaded | 10 lineage cases (§2.1–2.2) and 10 CAS/reconciliation cases (§4) with signed receipts, all recomputed by the checker |

Every interpretive choice is written down as **A-xx** in `docs/CONFORMANCE_ASSUMPTIONS.md`, and every spec gap found as **G-xx**. Both are provisional until the normative text adopts or replaces them.

## 1. Setup

In [1]:
%pip install -q cryptography pyyaml grpcio-tools protobuf

In [2]:
import os, sys, subprocess

REPO_URL = "https://github.com/krbarrow/provenance-razor"
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
REPO = os.path.join(BASE, "provenance-razor")

if not os.path.isdir(os.path.join(REPO, ".git")):
    r = subprocess.run(["git", "clone", "-q", REPO_URL, REPO], capture_output=True, text=True)
    if r.returncode != 0:
        print("clone failed, working in a plain directory:", r.stderr.strip()[:200])
        os.makedirs(REPO, exist_ok=True)
os.chdir(REPO)  # absolute path: re-running this cell never nests directories
for d in ("reference", "tools", "docs", "tests", "spec", "schema/protobuf", "schema/flatbuffers"):
    os.makedirs(d, exist_ok=True)
print("working directory:", os.getcwd())

# Optional: flatc for the FlatBuffers round trip (best effort; the step is skipped if unavailable)
import shutil, urllib.request, zipfile
if not (os.environ.get("FLATC") or shutil.which("flatc")):
    try:
        url = "https://github.com/google/flatbuffers/releases/download/v24.3.25/Linux.flatc.binary.clang++-15.zip"
        zp = os.path.join(BASE, "flatc.zip"); urllib.request.urlretrieve(url, zp)
        zipfile.ZipFile(zp).extract("flatc", os.path.join(BASE, "flatc_bin"))
        os.chmod(os.path.join(BASE, "flatc_bin", "flatc"), 0o755)
        os.environ["FLATC"] = os.path.join(BASE, "flatc_bin", "flatc")
    except Exception as ex:
        print("flatc not available, FlatBuffers round trip will be skipped:", type(ex).__name__)
print("flatc:", os.environ.get("FLATC") or shutil.which("flatc"))

## 2. Source files

These cells write the repository files. They are the same files that are committed, so you can also skip these cells and use the checked-in copies.

### Reference model — `reference/prc_ref.py`

In [3]:
%%writefile reference/prc_ref.py
"""
prc_ref — Python reference model for the Provenance Razor Control Specification v0.3.0-alpha.

This module is the executable interpretation used to generate and check conformance
vectors. Where the specification is silent or ambiguous, the choice made here is
recorded in docs/CONFORMANCE_ASSUMPTIONS.md with an identifier (A-xx). Those choices
are provisional and must be ratified (or replaced) in the normative text.

Nothing in this module reads the wall clock or a random source. Every output is a
pure function of its inputs, so vectors regenerate byte-for-byte.
"""
from __future__ import annotations

import fnmatch
import hashlib
import math
import struct
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

from cryptography.hazmat.primitives.asymmetric.ed25519 import (
    Ed25519PrivateKey,
    Ed25519PublicKey,
)
from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import serialization

SPEC_VERSION = "0.3.0-alpha"
ENCODING_ID = "PRC-LP-0"

DOMAIN_APPROVAL = "PROVENANCE-RAZOR-APPROVAL-V0.3.0-Alpha"
DOMAIN_DECISION = "PROVENANCE-RAZOR-DECISION-V0.3.0-Alpha"
DOMAIN_RECEIPT = "PROVENANCE-RAZOR-RECEIPT-V0.3.0-Alpha"
DOMAIN_PROVENANCE = "PROVENANCE-RAZOR-PROVENANCE-V0.3.0-Alpha"   # §2.4 P-1 (new in 0.2.0)
DOMAIN_MEMBERSHIP = "PROVENANCE-RAZOR-MEMBERSHIP-V0.3.0-Alpha"   # §2.6 M-1 (new in 0.3.0)
NO_CONTAINMENT_LABEL = "PROVENANCE-RAZOR-NO-CONTAINMENT-V0.3.0-Alpha"
NO_APPROVAL_LABEL = "PROVENANCE-RAZOR-NO-APPROVAL-V0.3.0-Alpha"
NO_STATE_LABEL = "PROVENANCE-RAZOR-NO-STATE-V0.3.0-Alpha"

KEY_DOMAINS = ("profile", "approval", "decision", "receipt", "provenance", "membership")

# ---------------------------------------------------------------------------
# 1. Canonical encoding  (A-01: PRC-LP-0 tag/length/value encoding)
# ---------------------------------------------------------------------------
#   value   := tag:u8 || len:u64be || payload[len]
#   0x00 null     len=0
#   0x01 false    len=0
#   0x02 true     len=0
#   0x10 int      len=8   signed two's complement, big-endian (int64 range only)
#   0x11 f64      len=8   IEEE 754 binary64 bit pattern, big-endian (no normalisation)
#   0x20 string   UTF-8 bytes (must be valid UTF-8)
#   0x21 bytes    raw bytes
#   0x30 list     concatenation of encoded items
#   0x31 map      concatenation of (encoded string key, encoded value) pairs,
#                 keys strictly ascending by UTF-8 byte order, no duplicates
T_NULL, T_FALSE, T_TRUE = 0x00, 0x01, 0x02
T_INT, T_F64 = 0x10, 0x11
T_STR, T_BYTES = 0x20, 0x21
T_LIST, T_MAP = 0x30, 0x31

INT64_MIN, INT64_MAX = -(2**63), 2**63 - 1


class F64:
    """Explicit binary64 wrapper so a float's exact bit pattern (incl. NaN payloads) survives."""

    __slots__ = ("bits",)

    def __init__(self, bits: int):
        self.bits = bits & 0xFFFFFFFFFFFFFFFF

    @classmethod
    def of(cls, x: float) -> "F64":
        return cls(struct.unpack(">Q", struct.pack(">d", x))[0])

    def __eq__(self, other):
        return isinstance(other, F64) and other.bits == self.bits

    def __repr__(self):
        return f"F64(0x{self.bits:016x})"


class CanonicalDecodeError(ValueError):
    pass


def _frame(tag: int, payload: bytes) -> bytes:
    return bytes([tag]) + struct.pack(">Q", len(payload)) + payload


def encode(v: Any) -> bytes:
    if v is None:
        return _frame(T_NULL, b"")
    if v is True:
        return _frame(T_TRUE, b"")
    if v is False:
        return _frame(T_FALSE, b"")
    if isinstance(v, int):
        if not INT64_MIN <= v <= INT64_MAX:
            raise ValueError(f"int out of int64 range: {v}")
        return _frame(T_INT, struct.pack(">q", v))
    if isinstance(v, F64):
        return _frame(T_F64, struct.pack(">Q", v.bits))
    if isinstance(v, float):
        return _frame(T_F64, struct.pack(">d", v))
    if isinstance(v, str):
        return _frame(T_STR, v.encode("utf-8"))
    if isinstance(v, (bytes, bytearray)):
        return _frame(T_BYTES, bytes(v))
    if isinstance(v, (list, tuple)):
        return _frame(T_LIST, b"".join(encode(i) for i in v))
    if isinstance(v, dict):
        for k in v:
            if not isinstance(k, str):
                raise ValueError("map keys must be strings")
        keys = sorted(v.keys(), key=lambda s: s.encode("utf-8"))
        return _frame(T_MAP, b"".join(encode(k) + encode(v[k]) for k in keys))
    raise TypeError(f"unencodable type: {type(v).__name__}")


def _decode_at(buf: bytes, pos: int) -> Tuple[Any, int]:
    if pos + 9 > len(buf):
        raise CanonicalDecodeError("truncated header")
    tag = buf[pos]
    (ln,) = struct.unpack(">Q", buf[pos + 1 : pos + 9])
    start, end = pos + 9, pos + 9 + ln
    if end > len(buf):
        raise CanonicalDecodeError("length exceeds buffer")
    body = buf[start:end]
    if tag in (T_NULL, T_FALSE, T_TRUE):
        if ln != 0:
            raise CanonicalDecodeError("non-zero length for null/bool")
        return {T_NULL: None, T_FALSE: False, T_TRUE: True}[tag], end
    if tag == T_INT:
        if ln != 8:
            raise CanonicalDecodeError("int length != 8")
        return struct.unpack(">q", body)[0], end
    if tag == T_F64:
        if ln != 8:
            raise CanonicalDecodeError("f64 length != 8")
        return F64(struct.unpack(">Q", body)[0]), end
    if tag == T_STR:
        try:
            return body.decode("utf-8", errors="strict"), end
        except UnicodeDecodeError as e:
            raise CanonicalDecodeError("invalid UTF-8") from e
    if tag == T_BYTES:
        return bytes(body), end
    if tag == T_LIST:
        items, p = [], start
        while p < end:
            item, p = _decode_at(buf[:end], p)
            items.append(item)
        return items, end
    if tag == T_MAP:
        out, p, prev = {}, start, None
        while p < end:
            k, p = _decode_at(buf[:end], p)
            if not isinstance(k, str):
                raise CanonicalDecodeError("map key is not a string")
            kb = k.encode("utf-8")
            if prev is not None and kb <= prev:
                raise CanonicalDecodeError("map keys not strictly ascending (unsorted or duplicate)")
            prev = kb
            if p >= end:
                raise CanonicalDecodeError("map key without value")
            val, p = _decode_at(buf[:end], p)
            out[k] = val
        return out, end
    raise CanonicalDecodeError(f"unknown tag 0x{tag:02x}")


def decode(buf: bytes) -> Any:
    """Strict decoder: rejects anything that is not the unique canonical encoding."""
    v, end = _decode_at(buf, 0)
    if end != len(buf):
        raise CanonicalDecodeError("trailing bytes")
    if encode(v) != buf:  # belt and braces: round-trip must be exact
        raise CanonicalDecodeError("round-trip mismatch")
    return v


def H(v: Any) -> str:
    """H(z) = SHA256(CanonicalEncode(z)), returned as lowercase hex (A-02: single encoding layer)."""
    return hashlib.sha256(encode(v)).hexdigest()


def H_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()


NULL_APPROVAL_DIGEST = H(NO_APPROVAL_LABEL)
NULL_STATE_DIGEST = H(NO_STATE_LABEL)
NULL_CONTAINMENT_DIGEST = H(NO_CONTAINMENT_LABEL)

# ---------------------------------------------------------------------------
# Deterministic identifiers and test keys
# ---------------------------------------------------------------------------


def uuid7(unix_ms: int, label: str) -> str:
    """RFC 9562 UUIDv7 with the 74 'random' bits derived from SHA256(label) (deterministic)."""
    r = int.from_bytes(hashlib.sha256(("PRC-UUID7|" + label).encode()).digest()[:10], "big")
    rand_a = (r >> 68) & 0xFFF
    rand_b = r & ((1 << 62) - 1)
    val = (unix_ms & ((1 << 48) - 1)) << 80
    val |= 0x7 << 76
    val |= rand_a << 64
    val |= 0b10 << 62
    val |= rand_b
    h = f"{val:032x}"
    return f"{h[0:8]}-{h[8:12]}-{h[12:16]}-{h[16:20]}-{h[20:32]}"


@dataclass
class TestKey:
    domain: str
    label: str
    seed: bytes
    private: Ed25519PrivateKey = field(repr=False)
    public_hex: str = ""
    key_id: str = ""

    @classmethod
    def derive(cls, domain: str, label: str) -> "TestKey":
        assert domain in KEY_DOMAINS
        seed = hashlib.sha256(f"PRC-TEST-KEY|{domain}|{label}".encode()).digest()
        priv = Ed25519PrivateKey.from_private_bytes(seed)
        pub = priv.public_key().public_bytes(
            serialization.Encoding.Raw, serialization.PublicFormat.Raw
        )
        kid = f"kid:{domain}:{hashlib.sha256(pub).hexdigest()[:16]}"
        return cls(domain, label, seed, priv, pub.hex(), kid)

    def sign_digest(self, digest_hex: str) -> str:
        # Ed25519 is deterministic (RFC 8032), so signatures are stable golden values.
        return self.private.sign(bytes.fromhex(digest_hex)).hex()


def ed25519_verify(pub_hex: str, digest_hex: str, sig_hex: str) -> bool:
    try:
        Ed25519PublicKey.from_public_bytes(bytes.fromhex(pub_hex)).verify(
            bytes.fromhex(sig_hex), bytes.fromhex(digest_hex)
        )
        return True
    except (InvalidSignature, ValueError):
        return False


# ---------------------------------------------------------------------------
# Reason-code registry (A-03: provisional bit assignments)
# ---------------------------------------------------------------------------
REASON_BITS = [
    "DECODE_INVALID",              # 0
    "SIGNATURE_INVALID",           # 1
    "DOMAIN_MISMATCH",             # 2
    "ACTION_MISMATCH",             # 3
    "ASSERTION_MISMATCH",          # 4
    "TARGET_MISMATCH",             # 5
    "STATE_VERSION_MISMATCH",      # 6
    "PROFILE_HASH_MISMATCH",       # 7
    "EVIDENCE_HASH_MISMATCH",      # 8
    "LINEAGE_HASH_MISMATCH",       # 9
    "DECISION_NOT_ALLOW",          # 10
    "NOT_YET_VALID",               # 11
    "EXPIRED_CAPABILITY",          # 12
    "EVAL_SKEW_EXCEEDED",          # 13
    "DECISION_KEY_UNAUTHORIZED",   # 14
    "CAPABILITY_UNAVAILABLE",      # 15
    "AUTH_MODE_INVALID",           # 16
    "APPROVAL_SIGNATURE_INVALID",  # 17
    "APPROVAL_DOMAIN_MISMATCH",    # 18
    "APPROVAL_KEY_UNAUTHORIZED",   # 19
    "APPROVAL_SCOPE_DENIED",       # 20
    "APPROVAL_BINDING_MISMATCH",   # 21
    "APPROVAL_EXPIRED",            # 22
    "APPROVAL_CONSUMED",           # 23
    "APPROVAL_HASH_MISMATCH",      # 24
    "SAFE_STATE_LOCK",             # 25
    "THRESHOLD_NOT_MET",           # 26
    "FAMILY_INVALIDATED",          # 27
    "NO_ELIGIBLE_EVIDENCE",        # 28
    "CAS_PRECONDITION_FAILED",     # 29
    "TARGET_REJECTED",             # 30
    "DISPATCH_TIMEOUT",            # 31
    "RECONCILIATION_UNRESOLVED",   # 32
    "LINEAGE_CLASS_MALFORMED",     # 33  (scoring profile)
    "CONSEQUENCE_CLASS_UNDEFINED", # 34  (scoring profile)
    "NONFINITE_INPUT",             # 35  (engine guard, E-1)
    "APPROVAL_SUBJECT_MISMATCH",   # 36  p.target/actor/req_id/capability_id != d (closes G-01, G-03)
    "APPROVAL_NOT_YET_VALID",      # 37  p.tau_issued > d.tau_eval (closes G-02)
    "DEFEATER_PRESENT",            # 38  authenticated contradicting evidence (categorical veto)
    "PROVENANCE_TAMPERED",         # 39  attestation signature/digest failure -> family invalidated
    "OBJECT_SCHEMA_INVALID",       # 40  CanonicalObjectValid failed for d                       (0.3.0)
    "APPROVAL_SCHEMA_INVALID",     # 41  CanonicalObjectValid failed for p                       (0.3.0)
    "TRUST_STATE_MISMATCH",        # 42  d.trust_state_hash != evaluated trust state              (0.3.0)
    "SCORING_PROFILE_MISMATCH",    # 43  d.scoring_profile_hash != active scoring profile         (0.3.0)
    "ADVERSE_DISPOSITION_MISMATCH",# 44  d.adverse_disposition / containment_digest != current    (0.3.0)
    "ADVERSE_CONTAINMENT",         # 45  unresolved adverse claim: automatic ALLOW prohibited     (0.3.0)
    "CLOCK_UNTRUSTED",             # 46  unregistered clock or uncertainty above its authority    (0.3.0)
    "MEMBERSHIP_UNVERIFIED",       # 47  family has no admissible manifest -> excluded            (0.3.0)
    "MEMBERSHIP_TAMPERED",         # 48  manifest forged / replayed / member altered              (0.3.0)
    "VERIFIER_FAULT",              # 49  a verifier raised; converted to DENY at the boundary     (0.3.0)
]
assert len(REASON_BITS) <= 63, "bit 63 is reserved: reason masks are non-negative int64"
RC = {name: 1 << i for i, name in enumerate(REASON_BITS)}


def reason_names(bits: int) -> List[str]:
    return [n for i, n in enumerate(REASON_BITS) if bits >> i & 1]


# ---------------------------------------------------------------------------
# Trust registry
# ---------------------------------------------------------------------------


def registry_entry(key: TestKey, epochs: List[str], authority_classes: Optional[List[str]] = None) -> dict:
    e = {"domain": key.domain, "public_key": key.public_hex, "epochs": sorted(epochs)}
    if authority_classes is not None:
        e["authority_classes"] = sorted(authority_classes)
    return e


def key_authorized(registry: dict, key_id: str, epoch: str, domain: str, authority_class: Optional[str] = None) -> bool:
    e = registry.get(key_id)
    if e is None or e["domain"] != domain or epoch not in e["epochs"]:
        return False
    if authority_class is not None and authority_class not in e.get("authority_classes", []):
        return False
    return True


# ---------------------------------------------------------------------------
# 1.1 Domain-scoped key resolution (0.3.0: identical for every object type)
# ---------------------------------------------------------------------------


def domain_key(registry: dict, key_id, domain: str) -> Optional[dict]:
    """Resolve key_id ONLY within `domain`. A key registered in another domain is not found."""
    e = registry.get(key_id) if isinstance(key_id, str) else None
    return e if isinstance(e, dict) and e.get("domain") == domain else None


def verify_in_domain(registry: dict, key_id, domain: str, digest_hex: str, sig_hex) -> bool:
    e = domain_key(registry, key_id, domain)
    return e is not None and isinstance(sig_hex, str) and ed25519_verify(e["public_key"], digest_hex, sig_hex)


# ---------------------------------------------------------------------------
# C-1 Clock authority (closes G-10)
# ---------------------------------------------------------------------------


def clock_bounds(trust_state: dict, eval_clock_id, exec_clock) -> Tuple[bool, int]:
    """Returns (trusted, U) with U = u_eval + u_exec, the combined worst-case uncertainty in µs.
    The evaluator's clock contributes its authority's registered maximum; the executor reports
    its current uncertainty, which must not exceed its authority's maximum."""
    clocks = trust_state.get("clock_authorities", {})
    ev = clocks.get(eval_clock_id) if isinstance(eval_clock_id, str) else None
    ex_id = exec_clock.get("clock_id") if isinstance(exec_clock, dict) else None
    ex = clocks.get(ex_id) if isinstance(ex_id, str) else None
    u_exec = exec_clock.get("uncertainty_us") if isinstance(exec_clock, dict) else None
    trusted = (ev is not None and ex is not None and isinstance(u_exec, int) and not isinstance(u_exec, bool)
               and 0 <= u_exec <= ex["max_uncertainty_us"])
    u_eval = ev["max_uncertainty_us"] if ev else 0
    return trusted, u_eval + (u_exec if isinstance(u_exec, int) and u_exec >= 0 else 0)


# ---------------------------------------------------------------------------
# 3.1 ExternalApprovalValid (0.3.0)
# ---------------------------------------------------------------------------


def external_approval_valid(ctx: dict, p_bytes_hex: str, sig_p, d: Optional[dict] = None,
                            U: int = 0) -> Tuple[bool, int, Optional[dict]]:
    import prc_schema as SC
    try:
        raw = bytes.fromhex(p_bytes_hex)
        p = decode(raw)
    except (CanonicalDecodeError, ValueError, TypeError):
        return False, RC["APPROVAL_SCHEMA_INVALID"], None
    if SC.validate(p, "approval"):
        return False, RC["APPROVAL_SCHEMA_INVALID"], None
    d = d or {}
    bits = 0
    reg = ctx["trust_state"]["key_registry"]
    kid = p["approval_key_id"]
    if not verify_in_domain(reg, kid, "approval", H_bytes(raw), sig_p):
        bits |= RC["APPROVAL_SIGNATURE_INVALID"]
    if p["domain"] != DOMAIN_APPROVAL:
        bits |= RC["APPROVAL_DOMAIN_MISMATCH"]
    if not key_authorized(reg, kid, p["approval_trust_epoch"], "approval", p["authority_class"]):
        bits |= RC["APPROVAL_KEY_UNAUTHORIZED"]
    if not fnmatch.fnmatchcase(f"{ctx['x']['action']}:{p['target']}", p["authority_scope"]):
        bits |= RC["APPROVAL_SCOPE_DENIED"]
    if (p["x"] != ctx["x"] or p["a"] != ctx["a"] or p["state_ver"] != ctx["current_state_ver"]
            or p["control_profile_hash"] != ctx["active_control_profile_hash"]
            or p["scoring_profile_hash"] != d.get("scoring_profile_hash")
            or p["trust_state_hash"] != d.get("trust_state_hash")):
        bits |= RC["APPROVAL_BINDING_MISMATCH"]
    if (p["target"] != ctx["current_target"] or p["target"] != d.get("target") or p["actor"] != d.get("actor")
            or p["req_id"] != d.get("req_id") or p["capability_id"] != d.get("capability_id")):
        bits |= RC["APPROVAL_SUBJECT_MISMATCH"]
    # the approver acknowledged exactly the adverse claims the decision carries (containment binding)
    if p["adverse_disposition"] != d.get("adverse_disposition") or p["containment_digest"] != d.get("containment_digest"):
        bits |= RC["ADVERSE_DISPOSITION_MISMATCH"]
    if not p["tau_issued"] <= d.get("tau_eval", -1):
        bits |= RC["APPROVAL_NOT_YET_VALID"]
    if not ctx["exec_clock"]["tau_exec"] + U <= p["tau_expiry"]:
        bits |= RC["APPROVAL_EXPIRED"]
    reusable = p["authority_class"] in ctx["profile"].get("reusable_authority_classes", [])
    if p["approval_id"] in ctx.get("consumed_approvals", []) and not reusable:
        bits |= RC["APPROVAL_CONSUMED"]
    return bits == 0, bits, p


# ---------------------------------------------------------------------------
# 3.4 / 3.5  AuthorizationModeValid and DecisionBindingValid (0.3.0)
# ---------------------------------------------------------------------------


def max_eval_skew_us(profile: dict, action: str) -> int:
    per = profile.get("max_eval_skew_us", {})
    return per.get(action, per["*"])


def _decision_binding_valid(ctx, d_bytes_hex, sig_d, p_bytes_hex=None, sig_p=None):
    import prc_schema as SC
    try:
        raw = bytes.fromhex(d_bytes_hex)
        d = decode(raw)
    except (CanonicalDecodeError, ValueError, TypeError):
        return False, RC["DECODE_INVALID"]
    if SC.validate(d, "decision"):
        return False, RC["OBJECT_SCHEMA_INVALID"]  # nothing semantic is evaluated on a malformed object

    bits = 0
    ts = ctx["trust_state"]
    reg = ts["key_registry"]
    if not verify_in_domain(reg, d["decision_key_id"], "decision", H_bytes(raw), sig_d):
        bits |= RC["SIGNATURE_INVALID"]
    if d["domain"] != DOMAIN_DECISION:
        bits |= RC["DOMAIN_MISMATCH"]
    if d["x"] != ctx["x"]:
        bits |= RC["ACTION_MISMATCH"]
    if d["a"] != ctx["a"]:
        bits |= RC["ASSERTION_MISMATCH"]
    if d["target"] != ctx["current_target"]:
        bits |= RC["TARGET_MISMATCH"]
    if d["state_ver"] != ctx["current_state_ver"]:
        bits |= RC["STATE_VERSION_MISMATCH"]
    if d["control_profile_hash"] != ctx["active_control_profile_hash"]:
        bits |= RC["PROFILE_HASH_MISMATCH"]
    if d["scoring_profile_hash"] != ctx["active_scoring_profile_hash"]:
        bits |= RC["SCORING_PROFILE_MISMATCH"]
    if d["trust_state_hash"] != ctx["trust_state_hash"] or d["trust_state_hash"] != H(ts):
        bits |= RC["TRUST_STATE_MISMATCH"]
    if d["evidence_hash"] != ctx["evidence_hash"]:
        bits |= RC["EVIDENCE_HASH_MISMATCH"]
    if d["lineage_hash"] != ctx["lineage_hash"]:
        bits |= RC["LINEAGE_HASH_MISMATCH"]
    if d["adverse_disposition"] != ctx["adverse_disposition"] or d["containment_digest"] != ctx["containment_digest"]:
        bits |= RC["ADVERSE_DISPOSITION_MISMATCH"]
    if (d["adverse_disposition"] == "CLEAR") != (d["containment_digest"] == NULL_CONTAINMENT_DIGEST):
        bits |= RC["ADVERSE_DISPOSITION_MISMATCH"]
    if d["decision"] != "ALLOW":
        bits |= RC["DECISION_NOT_ALLOW"]

    trusted, U = clock_bounds(ts, d["eval_clock_id"], ctx["exec_clock"])
    if not trusted:
        bits |= RC["CLOCK_UNTRUSTED"]
    t_exec, t_eval, t_exp = ctx["exec_clock"]["tau_exec"], d["tau_eval"], d["tau_expiry"]
    if not t_eval + U <= t_exec:
        bits |= RC["NOT_YET_VALID"]
    if not t_exec + U <= t_exp:
        bits |= RC["EXPIRED_CAPABILITY"]
    if not (t_exec - t_eval) + U <= max_eval_skew_us(ctx["profile"], ctx["x"]["action"]):
        bits |= RC["EVAL_SKEW_EXCEEDED"]
    if not key_authorized(reg, d["decision_key_id"], d["decision_trust_epoch"], "decision"):
        bits |= RC["DECISION_KEY_UNAUTHORIZED"]
    if d["capability_id"] in ctx.get("consumed_capabilities", []):
        bits |= RC["CAPABILITY_UNAVAILABLE"]

    # §3.4 (0.3.0): MANUAL is required for ManualOnly actions AND for contained evidence
    manual_only = ctx["x"]["action"] in ctx["profile"].get("manual_only_actions", [])
    contained = d["adverse_disposition"] == "CONTAIN"
    mode_ok = False
    if d["auth_mode"] == "AUTOMATED":
        mode_ok = (not manual_only) and (not contained) and d["approval_hash"] == NULL_APPROVAL_DIGEST
        if contained:
            bits |= RC["ADVERSE_CONTAINMENT"]
    elif p_bytes_hex is not None:
        ok_p, pbits, _ = external_approval_valid(ctx, p_bytes_hex, sig_p, d, U)
        hash_ok = isinstance(p_bytes_hex, str) and d["approval_hash"] == _safe_hash_hex(p_bytes_hex)
        if not hash_ok:
            pbits |= RC["APPROVAL_HASH_MISMATCH"]
        bits |= pbits
        mode_ok = (manual_only or contained) and hash_ok and ok_p
    if not mode_ok:
        bits |= RC["AUTH_MODE_INVALID"]
    return bits == 0, bits


def _safe_hash_hex(h):
    try:
        return H_bytes(bytes.fromhex(h))
    except (ValueError, TypeError):
        return None


def decision_binding_valid(ctx: dict, d_bytes_hex: str, sig_d, p_bytes_hex=None, sig_p=None) -> Tuple[bool, int]:
    """Total verifier boundary (0.3.0 V-1): never raises; any internal fault is a DENY."""
    try:
        return _decision_binding_valid(ctx, d_bytes_hex, sig_d, p_bytes_hex, sig_p)
    except Exception:  # noqa: BLE001 — the boundary converts faults into a structured rejection
        return False, RC["VERIFIER_FAULT"] | RC["SAFE_STATE_LOCK"]


# ---------------------------------------------------------------------------
# 2.3 Numerical comparison  (A-04, A-05)
# ---------------------------------------------------------------------------

def f64_from_bits(bits: int) -> float:
    return struct.unpack("<d", struct.pack("<Q", bits))[0]


def bits_of(x: float) -> int:
    return struct.unpack("<Q", struct.pack("<d", x))[0]


def _is_exceptional_bits(bits: int) -> bool:
    return (bits >> 52) & 0x7FF == 0x7FF  # exponent all ones → Inf or NaN (checked on bits, no FP op)


def _norm_zero(x: float) -> float:
    return 0.0 if x == 0.0 else x


def authorization_check(c_bits: int, t_bits: int) -> dict:
    """AuthorizationCheck(C, T) with §2.3 exception handling. Returns result, effective C bits, lock flag."""
    if _is_exceptional_bits(c_bits) or _is_exceptional_bits(t_bits):
        return {"result": False, "c_effective_bits": 0, "safe_state_lock": True}
    c = _norm_zero(f64_from_bits(c_bits))
    t = _norm_zero(f64_from_bits(t_bits))
    return {"result": c >= t, "c_effective_bits": bits_of(c), "safe_state_lock": False}


def kahan_sum(input_bits: List[int]) -> dict:
    """Informative harness op: compensated sum; any exceptional input/intermediate → 0.0 + lock."""
    if any(_is_exceptional_bits(b) for b in input_bits):
        return {"result_bits": 0, "safe_state_lock": True}
    s, c = 0.0, 0.0
    for b in input_bits:
        y = f64_from_bits(b) - c
        t = s + y
        c = (t - s) - y
        s = t
        if not math.isfinite(s) or not math.isfinite(c):
            return {"result_bits": 0, "safe_state_lock": True}
    return {"result_bits": bits_of(_norm_zero(s)), "safe_state_lock": False}


# ---------------------------------------------------------------------------
# 2.1 / 2.2 Lineage consolidation  (A-08 .. A-11)
# ---------------------------------------------------------------------------
STATE_FACTOR = {"VALID": 1.0, "CONTRADICTORY": 0.0}


def evaluate_lineage(case: dict) -> dict:
    kappa = case["kappa_degraded"]
    classes_out, audit, eligible = {}, [], []
    for cls in case["classes"]:
        cid = cls["id"]
        states = [m["state"] for m in cls.get("optional_members", [])]
        if "INVALID" in states:
            classes_out[cid] = {"eligible": False, "invalidated": True, "A": None, "V": None}
            bad = [m["id"] for m in cls["optional_members"] if m["state"] == "INVALID"]
            audit.append({"event": "FAMILY_INVALIDATED", "class": cid, "members": bad})
            continue
        factors = []
        for s in states:
            if s == "UNAVAILABLE_OR_STALE":
                continue  # excluded from positive support, A unaffected
            factors.append(kappa if s == "DEGRADED" else STATE_FACTOR[s])
        A = min(factors) if factors else 1.0
        V = cls["representative"]["value"] * A
        classes_out[cid] = {"eligible": True, "invalidated": False, "A": A, "V": V}
        eligible.append(cid)
    pair = case.get("pairwise_independence", {})
    for cid in eligible:
        if len(eligible) == 1:
            classes_out[cid]["I"] = 1.0
        else:
            classes_out[cid]["I"] = min(pair[f"{cid}|{j}"] for j in eligible if j != cid)
    out = {"classes": classes_out, "eligible": eligible, "audit_events": audit}
    if not eligible:
        out["M"], out["C"] = 0.0, 0.0
        out["reason_code"] = RC["NO_ELIGIBLE_EVIDENCE"] | (RC["FAMILY_INVALIDATED"] if audit else 0)
    else:
        out["M"], out["C"] = "UNDEFINED_BY_SPEC", "UNDEFINED_BY_SPEC"
        out["reason_code"] = RC["FAMILY_INVALIDATED"] if audit else 0
    return out


# ---------------------------------------------------------------------------
# 4. Transactions and receipts  (A-12 .. A-15)
# ---------------------------------------------------------------------------


def build_receipt(*, d: dict, d_hash: str, receipt_id: str, status: str, failure_phase: str,
                  reason: int, recon_required: bool, pre_state: dict, post_state: Optional[dict],
                  tau_dispatch: int, tau_finalized: int, key: TestKey, epoch: str) -> dict:
    """§4.3 as revised in 0.2.0 (R-1): the receipt separates what the decision *expected*
    from what the executor *observed*. `post_state` is None when the outcome is unknown
    (UNRESOLVED); then observed_post_state_ver is null and post_state_digest is NULL_STATE_DIGEST."""
    return {
        "domain": DOMAIN_RECEIPT,
        "schema_ver": SPEC_VERSION,
        "receipt_id": receipt_id,
        "capability_id": d["capability_id"],
        "req_id": d["req_id"],
        "decision_digest": d_hash,
        "x": d["x"],
        "target": d["target"],
        "control_profile_hash": d["control_profile_hash"],
        "expected_state_ver": d["state_ver"],
        "observed_pre_state_ver": pre_state["version"],
        "observed_post_state_ver": post_state["version"] if post_state is not None else None,
        "execution_status": status,
        "failure_phase": failure_phase,
        "reason_code": reason,
        "reconciliation_required": recon_required,
        "pre_state_digest": H(pre_state),
        "post_state_digest": H(post_state) if post_state is not None else NULL_STATE_DIGEST,
        "tau_dispatch": tau_dispatch,
        "tau_finalized": tau_finalized,
        "receipt_key_id": key.key_id,
        "receipt_trust_epoch": epoch,
    }


def apply_mutation(state: dict, mutation: dict) -> dict:
    data = dict(state["data"])
    data.update(mutation)
    return {"version": state["version"] + 1, "data": data}


def _res(status, phase, reason, post, consumed, recon, trace):
    return {"status": status, "phase": phase, "reason": reason, "post_state": post,
            "consumed": consumed, "recon": recon, "trace": trace}


def execute_local(state: dict, consumed: List[str], d: dict, mutation: dict) -> dict:
    """§4.1 single-phase CAS. On abort the transaction rolls back and a REJECTED receipt is
    written in a separate transaction (A-12). The state is known to be unchanged."""
    trace = ["BEGIN", f"LOCK target state_ver={d['state_ver']}"]
    if d["capability_id"] in consumed:
        trace += ["CapabilityAvailable=false", "ABORT"]
        return _res("REJECTED", "PRECONDITION", RC["CAPABILITY_UNAVAILABLE"], state, list(consumed), False, trace)
    trace.append("CapabilityAvailable=true")
    if state["version"] != d["state_ver"]:
        trace += [f"CurrentStateVersion={state['version']} != {d['state_ver']}", "ABORT"]
        return _res("REJECTED", "PRECONDITION", RC["CAS_PRECONDITION_FAILED"], state, list(consumed), False, trace)
    post = apply_mutation(state, mutation)
    trace += ["MarkCapabilityConsumed", "ApplyMutation", f"AdvanceStateVersion -> {post['version']}",
              "WriteReceipt", "COMMIT"]
    return _res("COMMITTED", "NONE", 0, post, sorted(consumed + [d["capability_id"]]), False, trace)


def execute_distributed(state: dict, reserved: List[str], d: dict, mutation: dict, target: dict) -> dict:
    """§4.2 prepare/dispatch/reconcile. `state` is the target's actual state; the *target*
    enforces State-Precondition. `target` scripts the network/target:
       {"dispatch": DELIVERED|TIMEOUT|REJECT, "reconcile": APPLIED|NOT_APPLIED|UNKNOWN}"""
    trace = []
    if d["capability_id"] in reserved:
        trace += ["PREPARE: capability already reserved/consumed", "FINALIZE REJECTED"]
        return _res("REJECTED", "PREPARE", RC["CAPABILITY_UNAVAILABLE"], state, list(reserved), False, trace)
    consumed = sorted(reserved + [d["capability_id"]])  # A-13: once reserved, never released
    trace += ["PREPARE: reserve capability_id", "PERSIST PREPARED",
              f"DISPATCH Idempotency-Key={d['capability_id']} State-Precondition={d['state_ver']}"]
    mode = target["dispatch"]
    if mode == "DELIVERED":
        if state["version"] != d["state_ver"]:
            trace += [f"TARGET CAS FAILURE observed_version={state['version']}", "FINALIZE REJECTED"]
            return _res("REJECTED", "PRECONDITION", RC["CAS_PRECONDITION_FAILED"], state, consumed, False, trace)
        post = apply_mutation(state, mutation)
        trace += ["TARGET ACK", "FINALIZE COMMITTED"]
        return _res("COMMITTED", "NONE", 0, post, consumed, False, trace)
    if mode == "REJECT":
        trace += ["TARGET REJECTION", "FINALIZE REJECTED"]
        return _res("REJECTED", "TARGET", RC["TARGET_REJECTED"], state, consumed, False, trace)
    assert mode == "TIMEOUT"
    trace += ["NETWORK TIMEOUT", "STATE INDETERMINATE", "RECONCILE"]
    rec = target["reconcile"]
    if rec == "APPLIED":
        post = apply_mutation(state, mutation)
        trace += ["RECONCILED: applied", "FINALIZE COMMITTED"]
        return _res("COMMITTED", "NONE", RC["DISPATCH_TIMEOUT"], post, consumed, False, trace)
    if rec == "NOT_APPLIED":
        trace += ["RECONCILED: not applied", "FINALIZE ROLLED_BACK"]
        return _res("ROLLED_BACK", "DISPATCH", RC["DISPATCH_TIMEOUT"], state, consumed, False, trace)
    trace += ["UNRESOLVABLE", "FINALIZE UNRESOLVED"]
    return _res("UNRESOLVED", "RECONCILIATION", RC["DISPATCH_TIMEOUT"] | RC["RECONCILIATION_UNRESOLVED"],
                None, consumed, True, trace)


### EXPERIMENTAL scoring profile jaccard-v0.1.0-alpha — `reference/prc_scoring.py`

In [4]:
%%writefile reference/prc_scoring.py
"""
prc_scoring — PRC Scoring Profile `jaccard-v0.1.0-alpha`

    Status:          EXPERIMENTAL
    Normative role:  reference candidate (replaceable)

The control architecture requires *a* deterministic scoring profile. This module is the
first concrete candidate. It exists to generate evidence about the model, not to settle it.

Pure functions only: no signing, no CAS, no receipts, no clock, no randomness.
Every float is IEEE 754 binary64 and every operation is +, -, *, / (correctly rounded)
or an exact power-of-two scaling, so results are bit-reproducible across runtimes
provided the runtime does not contract a*b+c into an FMA. Transcendental functions
come from `prc_exp` below, never from the platform libm (see S-07).

Component ids (recorded in the profile and in every trace):
    canonical-representative : structural-root-v1
    independence-function    : material-dependency-jaccard-v1
    mass-function            : quality-weighted-independent-mass-v1
    confidence-function      : exponential-saturation-v1
    threshold-function       : consequence-exponential-v1
    exp-function             : prc-exp-v1
"""
from __future__ import annotations

import math
import re
import struct
from typing import Dict, List, Optional, Tuple

import prc_ref as R

PROFILE_ID = "jaccard-v0.1.0-alpha"
COMPONENTS = {
    "canonical_representative": "structural-root-v1",
    "independence_function": "material-dependency-jaccard-v1",
    "mass_function": "quality-weighted-independent-mass-v1",
    "confidence_function": "exponential-saturation-v1",
    "threshold_function": "consequence-exponential-v1",
    "exp_function": "prc-exp-v1",
    "summation_order": "representative-evidence-id-utf8-ascending",
}

DEPENDENCY_TYPES = (
    "SOURCE", "DATABASE", "COLLECTOR", "RUNTIME", "CLOCK", "NETWORK_AS", "TRUST_ROOT", "API_ENDPOINT",
)
ATOM_RE = re.compile(r"^([A-Z_]+):(\S+)$")


class ProfileError(ValueError):
    pass


# ---------------------------------------------------------------------------
# binary64 helpers (bits in, bits out — no decimal round trips)
# ---------------------------------------------------------------------------

def f(bits_hex: str) -> float:
    return struct.unpack(">d", bytes.fromhex(bits_hex))[0]


def hx(x: float) -> str:
    return struct.pack(">d", x).hex()


def _finite(x: float) -> bool:
    return not (math.isnan(x) or math.isinf(x))


# ---------------------------------------------------------------------------
# prc-exp-v1 : deterministic exp using only basic IEEE operations  (S-07)
# ---------------------------------------------------------------------------
# Cody–Waite reduction x = k*ln2 + r, |r| <= ln2/2, then Horner evaluation of the
# degree-13 Taylor polynomial, then exact scaling by 2^k. Accuracy is ~1 ulp; it is not
# correctly rounded, but it is *the same* on every conforming runtime.
EXP_INV_LN2 = f("3ff71547652b82fe")
EXP_LN2_HI = f("3fe62e42fee00000")   # low 21 bits of the significand are zero -> k*LN2_HI is exact
EXP_LN2_LO = f("3dea39ef35793c76")
EXP_COEFF = [1.0 / math.factorial(n) for n in range(14)]   # c0..c13, each a correctly rounded quotient
EXP_COEFF_BITS = [hx(c) for c in EXP_COEFF]


def _pow2(k: int) -> float:
    """Exact 2^k built from its bit pattern, -1074 <= k <= 1023."""
    if k >= -1022:
        return struct.unpack(">d", struct.pack(">Q", (k + 1023) << 52))[0]
    return struct.unpack(">d", struct.pack(">Q", 1 << (k + 1074)))[0]


def prc_exp(x: float) -> float:
    if math.isnan(x):
        return x
    if x > 709.0:
        return math.inf
    if x < -745.0:
        return 0.0
    k = math.floor(x * EXP_INV_LN2 + 0.5)
    r = (x - k * EXP_LN2_HI) - k * EXP_LN2_LO
    p = EXP_COEFF[13]
    for n in range(12, -1, -1):
        p = p * r + EXP_COEFF[n]
    if k < -1022:  # two-step scaling so the only rounding happens in the final multiply
        return (p * _pow2(-1022)) * _pow2(k + 1022)
    if k > 1023:
        return (p * _pow2(1023)) * _pow2(k - 1023)
    return p * _pow2(k)


# ---------------------------------------------------------------------------
# Profile compilation
# ---------------------------------------------------------------------------

def compile_profile(p: dict) -> dict:
    """Validate a scoring profile and return a compiled copy with floats decoded.
    The profile's fingerprint is H(profile) over its declared (hex-bits) form."""
    if p.get("id") != PROFILE_ID:
        raise ProfileError("unsupported scoring profile id")
    for k, v in COMPONENTS.items():
        if p.get(k) != v:
            raise ProfileError(f"component mismatch: {k}")
    c = {"fingerprint": R.H(p)}
    for name in ("omega", "t_base", "beta", "kappa_unknown", "kappa_degraded"):
        v = f(p[name + "_bits"])
        if not _finite(v):
            raise ProfileError(f"{name} not finite")
        c[name] = v
    if not c["omega"] > 0.0:
        raise ProfileError("omega must be > 0")
    if not 0.5 <= c["t_base"] < 1.0:
        raise ProfileError("t_base must be in [0.5, 1.0)")
    if not c["beta"] >= 0.0:
        raise ProfileError("beta must be >= 0")
    for name in ("kappa_unknown", "kappa_degraded"):
        if not 0.0 <= c[name] <= 1.0:
            raise ProfileError(f"{name} must be in [0, 1]")
    mats = p["material_dependency_types"]
    if sorted(set(mats)) != sorted(mats) or any(t not in DEPENDENCY_TYPES for t in mats):
        raise ProfileError("material_dependency_types must be distinct known types")
    c["material"] = frozenset(mats)
    c["gamma"] = {}
    for action, gbits in p["consequence_gamma_bits"].items():
        g = f(gbits)
        if not (_finite(g) and 0.1 <= g <= 10.0):
            raise ProfileError(f"gamma for {action} outside [0.1, 10.0]")
        c["gamma"][action] = g
    return c


# ---------------------------------------------------------------------------
# Evidence normalisation and structural representative (S-01, S-02)
# ---------------------------------------------------------------------------

def dependency_atoms(evidence: dict) -> Optional[frozenset]:
    """All declared atoms (any type) as a set, or None if any atom is malformed/unknown-typed."""
    out = set()
    for a in evidence.get("dependencies", []):
        m = ATOM_RE.match(a)
        if not m or m.group(1) not in DEPENDENCY_TYPES:
            return None
        out.add(a)
    return frozenset(out)


def material_atoms(atoms: frozenset, profile: dict) -> frozenset:
    return frozenset(a for a in atoms if a.split(":", 1)[0] in profile["material"])


def normalise_evidence(e: dict) -> dict:
    """Canonical form used for hashing: dependencies sorted and de-duplicated."""
    n = dict(e)
    n["dependencies"] = sorted(set(e.get("dependencies", [])), key=lambda s: s.encode("utf-8"))
    return n


def canonical_digest(e: dict) -> str:
    return R.H(normalise_evidence(e))


def canonical_representative(family: dict) -> Tuple[Optional[dict], Dict[str, int], Optional[str]]:
    """structural-root-v1: exactly one member whose parent is absent or outside the family,
    every other member must reach it through in-family parents without cycles.
    Returns (root, depth-by-id, error). Digest order is never used for selection."""
    members = family["members"]
    ids = [m["evidence_id"] for m in members]
    if len(set(ids)) != len(ids):
        return None, {}, "duplicate evidence_id in family"
    by_id = {m["evidence_id"]: m for m in members}
    roots = [m for m in members if m.get("parent_id") is None or m.get("parent_id") not in by_id]
    if len(roots) != 1:
        return None, {}, f"expected exactly one root, found {len(roots)}"
    root = roots[0]
    depth: Dict[str, int] = {}
    for m in members:
        seen, cur, d = set(), m, 0
        while cur["evidence_id"] != root["evidence_id"]:
            if cur["evidence_id"] in seen:
                return None, {}, "cycle in parent links"
            seen.add(cur["evidence_id"])
            cur = by_id[cur["parent_id"]]
            d += 1
        depth[m["evidence_id"]] = d
    return root, depth, None


# ---------------------------------------------------------------------------
# Independence (S-03, S-04)
# ---------------------------------------------------------------------------

def pairwise_independence(dk: frozenset, dj: frozenset, profile: dict) -> Tuple[float, bool]:
    """material-dependency-jaccard-v1. Returns (I, unknown_fallback_used)."""
    if not dk or not dj:
        return profile["kappa_unknown"], True
    inter = len(dk & dj)
    union = len(dk | dj)
    return 1.0 - inter / union, False   # int/int -> correctly rounded binary64


# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------
STATE_FACTOR = {"VALID": 1.0, "CONTRADICTORY": 0.0}


def evaluate(request: dict, families: List[dict], profile_decl: dict) -> dict:
    """evaluate(request, evidence families, profile) -> DecisionTrace (plain dict, bits as hex)."""
    prof = compile_profile(profile_decl)
    reason = 0
    audit = []
    fam_out = {}
    eligible = []  # (rep_id, family_id, V, material atoms)

    for fam in sorted(families, key=lambda x: x["family_id"].encode("utf-8")):
        fid = fam["family_id"]
        root, depth, err = canonical_representative(fam)
        atoms_ok = all(dependency_atoms(m) is not None for m in fam["members"])
        if err or not atoms_ok:
            why = err or "malformed or unknown-typed dependency atom"
            fam_out[fid] = {"eligible": False, "status": "MALFORMED", "error": why}
            audit.append({"event": "LINEAGE_CLASS_MALFORMED", "family": fid, "detail": why})
            reason |= R.RC["LINEAGE_CLASS_MALFORMED"]
            continue
        optional = [m for m in fam["members"] if m["evidence_id"] != root["evidence_id"]]
        states = [m.get("state", "VALID") for m in optional]
        root_state = root.get("state", "VALID")
        if "INVALID" in states or root_state == "INVALID":
            fam_out[fid] = {"eligible": False, "status": "INVALIDATED",
                            "representative": root["evidence_id"]}
            audit.append({"event": "FAMILY_INVALIDATED", "family": fid})
            reason |= R.RC["FAMILY_INVALIDATED"]
            continue
        if root_state != "VALID":  # S-05: a non-VALID representative gives no positive support
            fam_out[fid] = {"eligible": False, "status": "REPRESENTATIVE_" + root_state,
                            "representative": root["evidence_id"]}
            continue
        factors = [prof["kappa_degraded"] if s == "DEGRADED" else STATE_FACTOR[s]
                   for s in states if s != "UNAVAILABLE_OR_STALE"]
        A = min(factors) if factors else 1.0
        V_rep = f(root["value_bits"])
        if _finite(V_rep) and not 0.0 <= V_rep <= 1.0:  # S-08: quality scores live in [0, 1]
            fam_out[fid] = {"eligible": False, "status": "MALFORMED", "error": "value outside [0, 1]"}
            audit.append({"event": "LINEAGE_CLASS_MALFORMED", "family": fid, "detail": "value outside [0, 1]"})
            reason |= R.RC["LINEAGE_CLASS_MALFORMED"]
            continue
        V = V_rep * A  # NaN/Inf propagate into M and are caught below (fault closed)
        mat = material_atoms(dependency_atoms(root), prof)
        fam_out[fid] = {
            "eligible": True, "status": "ELIGIBLE",
            "representative": root["evidence_id"],
            "representative_digest": canonical_digest(root),
            "lineage_depth": dict(sorted(depth.items())),
            "A_bits": hx(A), "V_bits": hx(V),
            "material_atoms": sorted(mat, key=lambda s: s.encode("utf-8")),
        }
        eligible.append((root["evidence_id"], fid, V, mat))

    eligible.sort(key=lambda t: t[0].encode("utf-8"))  # canonical summation order (S-06)
    pair_out, unknown_used = {}, False
    I_k = {}
    if len(eligible) == 1:
        I_k[eligible[0][1]] = 1.0
    for _, fk, _, dk in eligible:
        if len(eligible) == 1:
            break
        vals = []
        for _, fj, _, dj in eligible:
            if fj == fk:
                continue
            i, unk = pairwise_independence(dk, dj, prof)
            unknown_used |= unk
            pair_out[f"{fk}|{fj}"] = hx(i)
            vals.append(i)
        I_k[fk] = min(vals)

    M = 0.0
    for _, fid, V, _ in eligible:
        M = M + V * I_k[fid]
        fam_out[fid]["I_bits"] = hx(I_k[fid])
        fam_out[fid]["contribution_bits"] = hx(V * I_k[fid])

    if _finite(M):
        C = 1.0 - prc_exp(-(M / prof["omega"]))
    else:  # §2.3: a float exception anywhere in the pipeline must not become "very confident"
        C = math.nan
    action = request["action"]
    if action in prof["gamma"]:
        g = prof["gamma"][action]
        T = 1.0 - (1.0 - prof["t_base"]) * prc_exp(-(g * prof["beta"]))
    else:  # fault closed: an action without a consequence class can never be authorised
        g, T = None, math.nan
        reason |= R.RC["CONSEQUENCE_CLASS_UNDEFINED"]

    if not eligible:
        reason |= R.RC["NO_ELIGIBLE_EVIDENCE"]
        M, C = 0.0, 0.0
    chk = R.authorization_check(R.bits_of(C), R.bits_of(T))
    if chk["safe_state_lock"]:
        reason |= R.RC["SAFE_STATE_LOCK"]
    elif not chk["result"]:
        reason |= R.RC["THRESHOLD_NOT_MET"]
    # Invalidated/malformed families are audited and excluded; they can only lower C, never block alone.
    allow = chk["result"] and not reason & R.RC["CONSEQUENCE_CLASS_UNDEFINED"]
    return {
        "scoring_profile": PROFILE_ID,
        "profile_fingerprint": prof["fingerprint"],
        "action": action,
        "families": fam_out,
        "eligible_families": [t[1] for t in eligible],
        "pairwise_independence_bits": dict(sorted(pair_out.items())),
        "unknown_lineage_fallback_used": unknown_used,
        "gamma_bits": hx(g) if g is not None else None,
        "M_bits": hx(M),
        "C_bits": f"{chk['c_effective_bits']:016x}",
        "T_bits": hx(T),
        "safe_state_lock": chk["safe_state_lock"],
        "decision": "ALLOW" if allow else "DENY",
        "reason_bitfield": reason,
        "reason_codes": R.reason_names(reason),
        "audit_events": audit,
    }


### EXPERIMENTAL scoring profile coverage-v0.2.0-alpha — `reference/prc_coverage.py`

In [5]:
%%writefile reference/prc_coverage.py
"""
prc_coverage — PRC Scoring Profile `coverage-v0.2.0-alpha`

    Status:          EXPERIMENTAL
    Normative role:  reference candidate (replaceable); supersedes jaccard-v0.1.0-alpha's mass rule

Mass is set-union coverage over material dependency atoms instead of a pairwise
independence penalty:

    w_k        = V(F̄_k) / |D_k|                         per-atom share of family k
    best(d)    = max_{k : d ∈ D_k} w_k                   each atom credited once, at its best share
    M          = Σ_{d ∈ ∪ D_k} best(d)                   summed in ascending UTF-8 order of d

A family with no material atoms is mapped to the single synthetic sink `UNKNOWN:*`, with
its value multiplied by κ_unknown, and κ_unknown is forced to 0 when T(x,a) ≥ unknown_gate_t.

Properties (proved in docs/SCORING_PROFILE_coverage-v0.2.0-alpha.md, tested in
tools/property_tests.py and witnessed in the vectors):
    monotone      adding an eligible family never lowers M (bitwise, under round-to-nearest)
    idempotent    adding a family whose atoms and value duplicate an existing one leaves M unchanged
    bridge-safe   a family overlapping several others can only add its own best shares
    order-free    permuting families, members or atom lists leaves every output bit unchanged

KNOWN OPEN EXPLOIT: dependencies are self-declared (`dependency_source: self-declared-UNVERIFIED`).
A claimant can pad its own atom list with fabricated atoms and collect up to V of mass per
item. The vectors carry this as an expected-failure case until the verified-provenance
boundary is specified.

The same arithmetic rules as prc_scoring apply: binary64, + − × ÷ only, no FMA contraction,
exp from prc-exp-v1.
"""
from __future__ import annotations

import math
from typing import List

import prc_ref as R
import prc_scoring as S

PROFILE_ID = "coverage-v0.2.0-alpha"
UNKNOWN_ATOM = "UNKNOWN:*"
COMPONENTS = {
    "canonical_representative": "structural-root-v1",
    "mass_function": "material-dependency-coverage-v1",
    "unknown_lineage": "single-sink-consequence-gated-v1",
    "confidence_function": "exponential-saturation-v1",
    "threshold_function": "consequence-exponential-v1",
    "exp_function": "prc-exp-v1",
    "summation_order": "dependency-atom-utf8-ascending",
    "dependency_source": "self-declared-UNVERIFIED",
}
f, hx, prc_exp = S.f, S.hx, S.prc_exp


def compile_profile(p: dict) -> dict:
    if p.get("id") != PROFILE_ID:
        raise S.ProfileError("unsupported scoring profile id")
    for k, v in COMPONENTS.items():
        if p.get(k) != v:
            raise S.ProfileError(f"component mismatch: {k}")
    c = {"fingerprint": R.H(p)}
    for name in ("omega", "t_base", "beta", "kappa_unknown", "kappa_degraded", "unknown_gate_t"):
        c[name] = f(p[name + "_bits"])
    if not c["omega"] > 0.0:
        raise S.ProfileError("omega must be > 0")
    if not 0.5 <= c["t_base"] < 1.0:
        raise S.ProfileError("t_base must be in [0.5, 1.0)")
    if not c["beta"] >= 0.0:
        raise S.ProfileError("beta must be >= 0")
    for name in ("kappa_unknown", "kappa_degraded", "unknown_gate_t"):
        if not 0.0 <= c[name] <= 1.0:
            raise S.ProfileError(f"{name} must be in [0, 1]")
    mats = p["material_dependency_types"]
    if sorted(set(mats)) != sorted(mats) or any(t not in S.DEPENDENCY_TYPES for t in mats):
        raise S.ProfileError("material_dependency_types must be distinct known types")
    c["material"] = frozenset(mats)
    c["gamma"] = {}
    for action, gbits in p["consequence_gamma_bits"].items():
        g = f(gbits)
        if not 0.1 <= g <= 10.0:
            raise S.ProfileError(f"gamma for {action} outside [0.1, 10.0]")
        c["gamma"][action] = g
    return c


def _utf8(s: str) -> bytes:
    return s.encode("utf-8")


def evaluate(request: dict, families: List[dict], profile_decl: dict) -> dict:
    prof = compile_profile(profile_decl)
    reason, audit, fam_out, eligible = 0, [], {}, []

    action = request["action"]
    if action in prof["gamma"]:
        g = prof["gamma"][action]
        T = 1.0 - (1.0 - prof["t_base"]) * prc_exp(-(g * prof["beta"]))
    else:
        g, T = None, math.nan
        reason |= R.RC["CONSEQUENCE_CLASS_UNDEFINED"]
    # consequence gate: at or above the gate, unattributed evidence carries no weight
    kappa_u = 0.0 if (T == T and T >= prof["unknown_gate_t"]) else prof["kappa_unknown"]

    for fam in sorted(families, key=lambda x: _utf8(x["family_id"])):
        fid = fam["family_id"]
        root, depth, err = S.canonical_representative(fam)
        atoms_ok = all(S.dependency_atoms(m) is not None for m in fam["members"])
        if err or not atoms_ok:
            why = err or "malformed or unknown-typed dependency atom"
            fam_out[fid] = {"eligible": False, "status": "MALFORMED", "error": why}
            audit.append({"event": "LINEAGE_CLASS_MALFORMED", "family": fid, "detail": why})
            reason |= R.RC["LINEAGE_CLASS_MALFORMED"]
            continue
        optional = [m for m in fam["members"] if m["evidence_id"] != root["evidence_id"]]
        states = [m.get("state", "VALID") for m in optional]
        root_state = root.get("state", "VALID")
        if "INVALID" in states or root_state == "INVALID":
            fam_out[fid] = {"eligible": False, "status": "INVALIDATED", "representative": root["evidence_id"]}
            audit.append({"event": "FAMILY_INVALIDATED", "family": fid})
            reason |= R.RC["FAMILY_INVALIDATED"]
            continue
        if root_state != "VALID":
            fam_out[fid] = {"eligible": False, "status": "REPRESENTATIVE_" + root_state,
                            "representative": root["evidence_id"]}
            continue
        V_rep = f(root["value_bits"])  # finiteness already enforced by the engine guard
        if not 0.0 <= V_rep <= 1.0:
            fam_out[fid] = {"eligible": False, "status": "MALFORMED", "error": "value outside [0, 1]"}
            audit.append({"event": "LINEAGE_CLASS_MALFORMED", "family": fid, "detail": "value outside [0, 1]"})
            reason |= R.RC["LINEAGE_CLASS_MALFORMED"]
            continue
        factors = [prof["kappa_degraded"] if s == "DEGRADED" else S.STATE_FACTOR[s]
                   for s in states if s != "UNAVAILABLE_OR_STALE"]
        A = min(factors) if factors else 1.0
        V = V_rep * A
        mat = S.material_atoms(S.dependency_atoms(root), prof)
        unknown = not mat
        if unknown:
            V = V * kappa_u
            mat = frozenset([UNKNOWN_ATOM])
        w = V / len(mat)
        fam_out[fid] = {
            "eligible": True, "status": "ELIGIBLE",
            "representative": root["evidence_id"],
            "representative_digest": S.canonical_digest(root),
            "lineage_depth": dict(sorted(depth.items())),
            "A_bits": hx(A), "V_bits": hx(V), "unknown_lineage": unknown,
            "atoms": sorted(mat, key=_utf8), "per_atom_share_bits": hx(w),
        }
        eligible.append((fid, mat, w))

    best = {}
    owner = {}
    for fid, mat, w in eligible:  # families already in canonical order; ties keep the first owner
        for d in mat:
            if d not in best or w > best[d]:
                best[d], owner[d] = w, fid
    M = 0.0
    for d in sorted(best, key=_utf8):
        M = M + best[d]
    credited = {fid: 0 for fid, _, _ in eligible}
    for d in best:
        credited[owner[d]] += 1
    for fid in credited:
        fam_out[fid]["atoms_credited"] = credited[fid]

    C = 1.0 - prc_exp(-(M / prof["omega"]))
    if not eligible:
        reason |= R.RC["NO_ELIGIBLE_EVIDENCE"]
        M, C = 0.0, 0.0
    chk = R.authorization_check(R.bits_of(C), R.bits_of(T))
    if chk["safe_state_lock"]:
        reason |= R.RC["SAFE_STATE_LOCK"]
    elif not chk["result"]:
        reason |= R.RC["THRESHOLD_NOT_MET"]
    allow = chk["result"] and not reason & R.RC["CONSEQUENCE_CLASS_UNDEFINED"]
    return {
        "stage": "PROFILE",
        "scoring_profile": PROFILE_ID,
        "profile_fingerprint": prof["fingerprint"],
        "action": action,
        "families": fam_out,
        "eligible_families": [e[0] for e in eligible],
        "atom_credit_bits": {d: hx(best[d]) for d in sorted(best, key=_utf8)},
        "atom_owner": {d: owner[d] for d in sorted(owner, key=_utf8)},
        "kappa_unknown_effective_bits": hx(kappa_u),
        "gamma_bits": hx(g) if g is not None else None,
        "M_bits": hx(M),
        "C_bits": f"{chk['c_effective_bits']:016x}",
        "T_bits": hx(T),
        "safe_state_lock": chk["safe_state_lock"],
        "decision": "ALLOW" if allow else "DENY",
        "reason_bitfield": reason,
        "reason_codes": R.reason_names(reason),
        "audit_events": audit,
    }


### Engine: non-finite guard (E-1) and profile dispatch — `reference/prc_engine.py`

In [6]:
%%writefile reference/prc_engine.py
"""
prc_engine — base evaluation engine: input guard, then scoring-profile dispatch.

The guard runs *before* any profile code (proposed normative rule E-1):

    Every binary64 value that enters evaluation — every `*_bits` field of the scoring
    profile, every consequence Γ, and every `value_bits` of every evidence member
    (representative or optional) — MUST be finite. If any is NaN or ±Inf the engine
    returns DENY with SAFE_STATE_LOCK | NONFINITE_INPUT and does not invoke the profile.

This makes the fail-closed behaviour independent of how carefully a profile handles
exceptional values (jaccard-v0.1.0-alpha needed its own guard to stop exp(-Inf) = 0
from producing C = 1).
"""
from __future__ import annotations

import math
from typing import List

import prc_ref as R
import prc_scoring as S
import prc_coverage as COV
import prc_coverage_v021 as COV21
import prc_coverage_v030 as COV30

PROFILES = {S.PROFILE_ID: S.evaluate, COV.PROFILE_ID: COV.evaluate}
TRUST_AWARE = {COV21.PROFILE_ID: COV21.evaluate,   # profiles that take the provenance trust state
               COV30.PROFILE_ID: COV30.evaluate}


def _nonfinite(bits_hex: str) -> bool:
    x = S.f(bits_hex)
    return math.isnan(x) or math.isinf(x)


def nonfinite_inputs(families: List[dict], profile: dict) -> List[str]:
    """Return the location of every non-finite input, in a deterministic order."""
    bad = []
    for k in sorted(profile):
        if k.endswith("_bits") and isinstance(profile[k], str) and _nonfinite(profile[k]):
            bad.append(f"profile.{k}")
    for a in sorted(profile.get("consequence_gamma_bits", {})):
        if _nonfinite(profile["consequence_gamma_bits"][a]):
            bad.append(f"profile.consequence_gamma_bits.{a}")
    fams = [f for f in families if isinstance(f, dict)] if isinstance(families, list) else []
    for fam in sorted(fams, key=lambda x: str(x.get("family_id"))):
        mem = fam.get("members") if isinstance(fam.get("members"), list) else []
        for m in sorted(mem, key=lambda x: str(x.get("evidence_id")) if isinstance(x, dict) else ""):
            vb = m.get("value_bits") if isinstance(m, dict) else None
            if isinstance(vb, str) and len(vb) == 16 and all(ch in "0123456789abcdef" for ch in vb) and _nonfinite(vb):
                bad.append(f"{fam['family_id']}/{m['evidence_id']}.value_bits")
    return bad


def evaluate(request: dict, families: List[dict], profile: dict, trust: dict = None) -> dict:
    """Total evaluation boundary (V-1): for coverage-v0.3.0-alpha a fault inside the engine or
    profile is returned as DENY | SAFE_STATE_LOCK | VERIFIER_FAULT, never raised."""
    if profile.get("id") == COV30.PROFILE_ID:
        try:
            return _evaluate(request, families, profile, trust)
        except Exception as ex:  # noqa: BLE001
            reason = R.RC["SAFE_STATE_LOCK"] | R.RC["VERIFIER_FAULT"]
            return {"stage": "VERIFIER_FAULT", "scoring_profile": profile.get("id"), "fault": type(ex).__name__,
                    "M_bits": S.hx(0.0), "C_bits": S.hx(0.0), "T_bits": None, "safe_state_lock": True,
                    "decision": "DENY", "reason_bitfield": reason, "reason_codes": R.reason_names(reason)}
    return _evaluate(request, families, profile, trust)


def _evaluate(request: dict, families: List[dict], profile: dict, trust: dict = None) -> dict:
    bad = nonfinite_inputs(families, profile)
    if bad:
        reason = R.RC["SAFE_STATE_LOCK"] | R.RC["NONFINITE_INPUT"]
        return {
            "stage": "ENGINE_GUARD",
            "scoring_profile": profile.get("id"),
            "profile_fingerprint": R.H(profile),
            "action": request.get("action"),
            "nonfinite_inputs": bad,
            "M_bits": S.hx(0.0), "C_bits": S.hx(0.0), "T_bits": None,
            "safe_state_lock": True,
            "decision": "DENY",
            "reason_bitfield": reason,
            "reason_codes": R.reason_names(reason),
        }
    if profile.get("id") in TRUST_AWARE:
        return TRUST_AWARE[profile["id"]](request, families, profile, trust)
    fn = PROFILES.get(profile.get("id"))
    if fn is None:
        raise S.ProfileError(f"unknown scoring profile {profile.get('id')!r}")
    return fn(request, families, profile)


### P-1 provenance attestation and DependencyAdmissible — `reference/prc_provenance.py`

In [7]:
%%writefile reference/prc_provenance.py
"""
prc_provenance — P-1 verified provenance (proposed normative, spec 0.2.0-alpha §2.4).

A dependency atom is admissible only when it arrives in a provenance attestation that
passes DependencyAdmissible. Self-declared `dependencies` on evidence are never read.

Attestation payload (canonical PRC-LP-0 map, signed detached):

    domain                  "PROVENANCE-RAZOR-PROVENANCE-V0.2.0-Alpha"
    schema_ver              spec version
    attestation_id          UUIDv7
    provenance_key_id       KeyID in the K_provenance domain
    provenance_trust_epoch  EpochID
    attester_id             identity the key is registered to
    evidence_digest         H(EvidenceCore(e))  — the member without `provenance`/`dependencies`
    dependencies            ascending (UTF-8), duplicate-free list of "TYPE:id"
    tau_issued, tau_expiry  int64 µs UTC

Envelope carried on the evidence member:  {"payload": <attestation>, "signature": hex}
with signature = Ed25519(sk, H(payload)). The signature is never inside the signed bytes.

Outcomes of admissibility:

    ADMITTED      atoms usable (after alias resolution)
    UNATTESTED    no envelope                                  -> UNKNOWN:* sink
    INADMISSIBLE  well-formed but not acceptable (unknown attester, stale, revoked,
                  self-attested, unauthorised type, unregistered atom, cap exceeded,
                  non-canonical list)                          -> UNKNOWN:* sink
    TAMPERED      evidence of forgery or mis-binding (bad signature, digest mismatch,
                  wrong domain string, key from another domain, attester_id not the
                  key's registered owner, malformed envelope)  -> family INVALID (§2.2)
"""
from __future__ import annotations

from typing import Optional, Tuple

import prc_ref as R
import prc_scoring as S

REQUIRED = ("domain", "schema_ver", "attestation_id", "provenance_key_id", "provenance_trust_epoch",
            "attester_id", "evidence_digest", "dependencies", "tau_issued", "tau_expiry")


def evidence_core(member: dict) -> dict:
    return {k: v for k, v in member.items() if k not in ("provenance", "dependencies")}


def evidence_digest(member: dict) -> str:
    return R.H(evidence_core(member))


def make_attestation(member: dict, key: R.TestKey, attester_id: str, deps, *, epoch: str,
                     tau_issued: int, tau_expiry: int, unix_ms: int, label: str,
                     digest_override: Optional[str] = None, sort_deps: bool = True,
                     domain: str = R.DOMAIN_PROVENANCE) -> dict:
    dl = sorted(set(deps), key=lambda a: a.encode("utf-8")) if sort_deps else list(deps)
    payload = {
        "domain": domain,
        "schema_ver": R.SPEC_VERSION,
        "attestation_id": R.uuid7(unix_ms, "att|" + label),
        "provenance_key_id": key.key_id,
        "provenance_trust_epoch": epoch,
        "attester_id": attester_id,
        "evidence_digest": digest_override or evidence_digest(member),
        "dependencies": dl,
        "tau_issued": tau_issued,
        "tau_expiry": tau_expiry,
    }
    return {"payload": payload, "signature": key.sign_digest(R.H(payload))}


def _utf8(s):
    return s.encode("utf-8")


def admissibility(member: dict, trust: dict, prof: dict, tau_eval: int) -> Tuple[str, str, Optional[frozenset]]:
    """DependencyAdmissible(d, e, profile). Returns (outcome, detail, resolved atoms | None)."""
    env = member.get("provenance")
    if env is None:
        return "UNATTESTED", "no provenance attestation", None
    import prc_schema as SC
    if SC.envelope_errors(env):
        return "TAMPERED", "malformed attestation envelope", None
    p, sig = env["payload"], env["signature"]
    if SC.validate(p, "attestation"):
        return "TAMPERED", "attestation payload fails CanonicalObjectValid", None
    if p["domain"] != R.DOMAIN_PROVENANCE:
        return "TAMPERED", "wrong domain string", None
    reg = trust.get("key_registry", trust.get("registry", {}))   # 0.3.0 trust state | 0.2.x registry
    entry = reg.get(p["provenance_key_id"])
    if entry is None:
        return "INADMISSIBLE", "unknown attester key", None
    if entry["domain"] != "provenance":
        return "TAMPERED", f"signed with a {entry['domain']}-domain key", None
    if not R.ed25519_verify(entry["public_key"], R.H(p), sig):
        return "TAMPERED", "signature invalid", None
    if p["attester_id"] != entry.get("owner", entry.get("attester_id")):
        return "TAMPERED", "attester_id is not the key's registered owner", None
    if p["evidence_digest"] != evidence_digest(member):
        return "TAMPERED", "evidence_digest does not match this evidence", None
    if p["provenance_trust_epoch"] not in entry["epochs"]:
        return "INADMISSIBLE", "trust epoch not authorised", None
    if p["attestation_id"] in trust.get("revoked_attestation_ids", []):
        return "INADMISSIBLE", "attestation revoked", None
    if not p["tau_issued"] <= tau_eval <= p["tau_expiry"]:
        return "INADMISSIBLE", "attestation not fresh at tau_eval", None
    if prof["attester_independence_required"] and p["attester_id"] == member.get("producer"):
        return "INADMISSIBLE", "self-attested (attester is the evidence producer)", None
    deps = p["dependencies"]
    if deps != sorted(set(deps), key=_utf8):
        return "INADMISSIBLE", "dependency list not canonical (sorted, unique)", None
    resolved = set()
    for a in deps:
        m = S.ATOM_RE.match(a) if isinstance(a, str) else None
        if not m or m.group(1) not in S.DEPENDENCY_TYPES:
            return "INADMISSIBLE", f"malformed atom {a!r}", None
        if m.group(1) not in entry.get("authorized_atom_types", []):
            return "INADMISSIBLE", f"attester not authorised for {m.group(1)}", None
        c = prof["aliases"].get(a, a)
        if c not in prof["registered_atoms"]:
            return "INADMISSIBLE", f"unregistered atom {a}", None
        resolved.add(c)
    counts = {}
    for c in resolved:
        t = c.split(":", 1)[0]
        counts[t] = counts.get(t, 0) + 1
    for t, n in counts.items():
        if n > prof["max_atoms_per_type"].get(t, 0):
            return "INADMISSIBLE", f"{n} {t} atoms exceed cap {prof['max_atoms_per_type'].get(t, 0)}", None
    return "ADMITTED", "ok", frozenset(resolved)


### Scoring profile coverage-v0.2.1-alpha (superseded) — `reference/prc_coverage_v021.py`

In [8]:
%%writefile reference/prc_coverage_v021.py
"""
prc_coverage_v021 — PRC Scoring Profile `coverage-v0.2.1-alpha`

    Status:          EXPERIMENTAL
    Normative role:  reference candidate; supersedes coverage-v0.2.0-alpha

Identical mass rule to coverage-v0.2.0-alpha (material-dependency-coverage-v1 with the
single consequence-gated UNKNOWN:* sink). What changes is where atoms come from and how
contradiction is handled:

  P-1  D_k comes only from the representative's *admitted* provenance attestation
       (prc_provenance.admissibility), resolved through the profile's alias table against
       its closed atom registry and bounded by per-type caps. Self-declared
       `dependencies` are ignored. Unattested / inadmissible -> UNKNOWN:*.
       TAMPERED on any member -> the family is INVALID (§2.2), audited, PROVENANCE_TAMPERED.

  D-1  Defeater channel (authenticated-contradiction-veto-v1):
         D⁻(x,a) = { optional members in state CONTRADICTORY whose attestation is ADMITTED }
         Decision = ALLOW  requires  D⁻ = ∅   (categorical; never a negative mass term)
       An *unverified* contradiction keeps the §2.2 behaviour (A = 0 for its own family) and
       is audited as UNVERIFIED_CONTRADICTION.

Positive mass stays monotone (P1–P4). The decision is monotone for every addition except an
admitted defeater, which by design always yields DENY (P6).
"""
from __future__ import annotations

import math
from typing import List, Optional

import prc_ref as R
import prc_scoring as S
import prc_coverage as COV
import prc_provenance as PV

PROFILE_ID = "coverage-v0.2.1-alpha"
UNKNOWN_ATOM = COV.UNKNOWN_ATOM
COMPONENTS = {
    "canonical_representative": "structural-root-v1",
    "mass_function": "material-dependency-coverage-v1",
    "unknown_lineage": "single-sink-consequence-gated-v1",
    "confidence_function": "exponential-saturation-v1",
    "threshold_function": "consequence-exponential-v1",
    "exp_function": "prc-exp-v1",
    "summation_order": "dependency-atom-utf8-ascending",
    "dependency_source": "attested-P-1",
    "defeater_policy": "authenticated-contradiction-veto-v1",
}
f, hx, prc_exp = S.f, S.hx, S.prc_exp


def compile_profile(p: dict) -> dict:
    if p.get("id") != PROFILE_ID:
        raise S.ProfileError("unsupported scoring profile id")
    for k, v in COMPONENTS.items():
        if p.get(k) != v:
            raise S.ProfileError(f"component mismatch: {k}")
    base = dict(p, id=COV.PROFILE_ID, **COV.COMPONENTS)  # reuse numeric validation
    c = COV.compile_profile({k: base[k] for k in base if k in COV.COMPONENTS or k.endswith("_bits")
                             or k in ("id", "material_dependency_types", "consequence_gamma_bits")})
    c["fingerprint"] = R.H(p)
    reg = p["atom_registry"]
    if reg.get("mode") != "closed":
        raise S.ProfileError("coverage-v0.2.1 requires a closed atom registry")
    atoms = reg["atoms"]
    if atoms != sorted(set(atoms), key=lambda s: s.encode("utf-8")):
        raise S.ProfileError("atom registry must be sorted and unique")
    for a in atoms:
        m = S.ATOM_RE.match(a)
        if not m or m.group(1) not in S.DEPENDENCY_TYPES:
            raise S.ProfileError(f"bad registry atom {a}")
    c["registered_atoms"] = frozenset(atoms)
    c["aliases"] = dict(reg.get("aliases", {}))
    for src, dst in c["aliases"].items():
        if dst not in c["registered_atoms"] or src.split(":", 1)[0] != dst.split(":", 1)[0]:
            raise S.ProfileError(f"alias {src} -> {dst} must target a registered atom of the same type")
    c["max_atoms_per_type"] = {k: int(v) for k, v in p["max_atoms_per_type"].items()}
    c["attester_independence_required"] = bool(p["attester_independence_required"])
    return c


def _utf8(s: str) -> bytes:
    return s.encode("utf-8")


def evaluate(request: dict, families: List[dict], profile_decl: dict, trust: Optional[dict] = None) -> dict:
    prof = compile_profile(profile_decl)
    trust = trust or {"registry": {}, "revoked_attestation_ids": []}
    tau_eval = request["tau_eval"]
    reason, audit, fam_out, eligible, defeaters = 0, [], {}, [], []

    action = request["action"]
    if action in prof["gamma"]:
        g = prof["gamma"][action]
        T = 1.0 - (1.0 - prof["t_base"]) * prc_exp(-(g * prof["beta"]))
    else:
        g, T = None, math.nan
        reason |= R.RC["CONSEQUENCE_CLASS_UNDEFINED"]
    kappa_u = 0.0 if (T == T and T >= prof["unknown_gate_t"]) else prof["kappa_unknown"]

    for fam in sorted(families, key=lambda x: _utf8(x["family_id"])):
        fid = fam["family_id"]
        root, depth, err = S.canonical_representative(fam)
        if err:
            fam_out[fid] = {"eligible": False, "status": "MALFORMED", "error": err}
            audit.append({"event": "LINEAGE_CLASS_MALFORMED", "family": fid, "detail": err})
            reason |= R.RC["LINEAGE_CLASS_MALFORMED"]
            continue
        # provenance for every member (tamper anywhere invalidates the family)
        prov = {}
        for m in sorted(fam["members"], key=lambda x: _utf8(x["evidence_id"])):
            prov[m["evidence_id"]] = PV.admissibility(m, trust, prof, tau_eval)
        tampered = [(eid, d) for eid, (o, d, _) in prov.items() if o == "TAMPERED"]
        optional = [m for m in fam["members"] if m["evidence_id"] != root["evidence_id"]]
        states = [m.get("state", "VALID") for m in optional]
        root_state = root.get("state", "VALID")
        prov_out = {eid: {"outcome": o, "detail": d} for eid, (o, d, _) in prov.items()}
        if tampered:
            fam_out[fid] = {"eligible": False, "status": "INVALIDATED", "cause": "PROVENANCE_TAMPERED",
                            "representative": root["evidence_id"], "provenance": prov_out}
            audit.append({"event": "PROVENANCE_TAMPERED", "family": fid,
                          "members": [{"evidence_id": e, "detail": d} for e, d in tampered]})
            reason |= R.RC["PROVENANCE_TAMPERED"] | R.RC["FAMILY_INVALIDATED"]
            continue
        if "INVALID" in states or root_state == "INVALID":
            fam_out[fid] = {"eligible": False, "status": "INVALIDATED", "representative": root["evidence_id"],
                            "provenance": prov_out}
            audit.append({"event": "FAMILY_INVALIDATED", "family": fid})
            reason |= R.RC["FAMILY_INVALIDATED"]
            continue
        # defeater channel: authenticated contradictions veto, unverified ones only zero the family
        for m in sorted(optional, key=lambda x: _utf8(x["evidence_id"])):
            if m.get("state") == "CONTRADICTORY":
                if prov[m["evidence_id"]][0] == "ADMITTED":
                    defeaters.append({"family": fid, "evidence_id": m["evidence_id"],
                                      "attestation_id": m["provenance"]["payload"]["attestation_id"]})
                else:
                    audit.append({"event": "UNVERIFIED_CONTRADICTION", "family": fid, "evidence_id": m["evidence_id"]})
        if root_state != "VALID":
            fam_out[fid] = {"eligible": False, "status": "REPRESENTATIVE_" + root_state,
                            "representative": root["evidence_id"], "provenance": prov_out}
            continue
        V_rep = f(root["value_bits"])
        if not 0.0 <= V_rep <= 1.0:
            fam_out[fid] = {"eligible": False, "status": "MALFORMED", "error": "value outside [0, 1]"}
            audit.append({"event": "LINEAGE_CLASS_MALFORMED", "family": fid, "detail": "value outside [0, 1]"})
            reason |= R.RC["LINEAGE_CLASS_MALFORMED"]
            continue
        factors = [prof["kappa_degraded"] if s == "DEGRADED" else S.STATE_FACTOR[s]
                   for s in states if s != "UNAVAILABLE_OR_STALE"]
        A = min(factors) if factors else 1.0
        V = V_rep * A
        outcome, _, atoms = prov[root["evidence_id"]]
        mat = frozenset(a for a in atoms if a.split(":", 1)[0] in prof["material"]) if outcome == "ADMITTED" else frozenset()
        unknown = not mat
        if unknown:
            V = V * kappa_u
            mat = frozenset([UNKNOWN_ATOM])
        w = V / len(mat)
        fam_out[fid] = {
            "eligible": True, "status": "ELIGIBLE",
            "representative": root["evidence_id"],
            "representative_digest": PV.evidence_digest(root),
            "lineage_depth": dict(sorted(depth.items())),
            "provenance": prov_out,
            "A_bits": hx(A), "V_bits": hx(V), "unknown_lineage": unknown,
            "atoms": sorted(mat, key=_utf8), "per_atom_share_bits": hx(w),
        }
        eligible.append((fid, mat, w))

    best, owner = {}, {}
    for fid, mat, w in eligible:
        for d in mat:
            if d not in best or w > best[d]:
                best[d], owner[d] = w, fid
    M = 0.0
    for d in sorted(best, key=_utf8):
        M = M + best[d]
    C = 1.0 - prc_exp(-(M / prof["omega"]))
    if not eligible:
        reason |= R.RC["NO_ELIGIBLE_EVIDENCE"]
        M, C = 0.0, 0.0
    chk = R.authorization_check(R.bits_of(C), R.bits_of(T))
    if chk["safe_state_lock"]:
        reason |= R.RC["SAFE_STATE_LOCK"]
    elif not chk["result"]:
        reason |= R.RC["THRESHOLD_NOT_MET"]
    if defeaters:
        reason |= R.RC["DEFEATER_PRESENT"]
    allow = chk["result"] and not defeaters and not reason & R.RC["CONSEQUENCE_CLASS_UNDEFINED"]
    return {
        "stage": "PROFILE",
        "scoring_profile": PROFILE_ID,
        "profile_fingerprint": prof["fingerprint"],
        "action": action,
        "tau_eval": tau_eval,
        "families": fam_out,
        "eligible_families": [e[0] for e in eligible],
        "atom_credit_bits": {d: hx(best[d]) for d in sorted(best, key=_utf8)},
        "atom_owner": {d: owner[d] for d in sorted(owner, key=_utf8)},
        "defeaters": defeaters,
        "kappa_unknown_effective_bits": hx(kappa_u),
        "gamma_bits": hx(g) if g is not None else None,
        "M_bits": hx(M),
        "C_bits": f"{chk['c_effective_bits']:016x}",
        "T_bits": hx(T),
        "safe_state_lock": chk["safe_state_lock"],
        "decision": "ALLOW" if allow else "DENY",
        "reason_bitfield": reason,
        "reason_codes": R.reason_names(reason),
        "audit_events": audit,
    }


### CanonicalObjectValid — total schema validation (§1.5) — `reference/prc_schema.py`

In [9]:
%%writefile reference/prc_schema.py
"""
prc_schema — CanonicalObjectValid(z, schema)   (spec 0.3.0-alpha §1.5)

A total function: for ANY Python value it returns a (possibly empty) list of error strings
and never raises. It runs after PRC-LP-0 decoding and before any signature or semantic
predicate. An object is canonical-object-valid iff the list is empty.

Rules common to every schema:
  * the object is a map with exactly the schema's keys (no missing, no extra)
  * every field has its declared type; bool is never accepted as an int
  * integers declared `uint` are 0 <= v <= 2^63 - 1 (this is also the 63-bit reason mask:
    bit 63 is reserved and a value with it set is negative in int64, so it is rejected)
  * `schema_ver` must be a supported version
  * digests are 64 lowercase hex characters (32 bytes); UUIDs are canonical lowercase UUIDv7
"""
from __future__ import annotations

import re
from typing import Any, List

SUPPORTED_SCHEMA_VERSIONS = ("0.3.0-alpha",)
UINT_MAX = 2**63 - 1

RE_HEX32 = re.compile(r"^[0-9a-f]{64}$")
RE_UUID7 = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-7[0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$")
RE_SEMVER = re.compile(r"^(0|[1-9]\d*)\.(0|[1-9]\d*)\.(0|[1-9]\d*)(-[0-9A-Za-z.-]+)?$")
RE_KEYID = re.compile(r"^kid:[a-z]+:[0-9a-f]{16}$")
RE_ATOM = re.compile(r"^[A-Z_]+:\S+$")
RE_SIG = re.compile(r"^[0-9a-f]{128}$")


def _is_int(v):
    return isinstance(v, int) and not isinstance(v, bool)


def _check(kind: str, v: Any) -> str:
    """Return '' if v conforms to kind, else a short reason."""
    if kind == "str":
        return "" if isinstance(v, str) and v != "" else "non-empty string"
    if kind == "uint":
        return "" if _is_int(v) and 0 <= v <= UINT_MAX else "integer in [0, 2^63-1]"
    if kind == "hex32":
        return "" if isinstance(v, str) and RE_HEX32.match(v) else "32-byte lowercase hex digest"
    if kind == "uuid7":
        return "" if isinstance(v, str) and RE_UUID7.match(v) else "canonical UUIDv7"
    if kind == "semver":
        return "" if isinstance(v, str) and RE_SEMVER.match(v) else "SemVer string"
    if kind == "keyid":
        return "" if isinstance(v, str) and RE_KEYID.match(v) else "key id kid:<domain>:<16 hex>"
    if kind == "schema_ver":
        return "" if v in SUPPORTED_SCHEMA_VERSIONS else "supported schema_ver"
    if kind == "bool":
        return "" if isinstance(v, bool) else "boolean"
    if kind == "map":
        return "" if isinstance(v, dict) else "map"
    if kind == "actor":
        ok = (isinstance(v, dict) and set(v) == {"id", "type"} and isinstance(v.get("id"), str) and v["id"] != ""
              and v.get("type") in ("service", "user"))
        return "" if ok else "actor {id, type in service|user}"
    if kind == "action":
        ok = isinstance(v, dict) and isinstance(v.get("action"), str) and v["action"] != ""
        return "" if ok else "action map with non-empty 'action'"
    if kind == "atoms":
        ok = isinstance(v, list) and all(isinstance(a, str) and RE_ATOM.match(a) for a in v)
        return "" if ok else "list of TYPE:id atoms"
    if kind == "manifest_members":
        if not isinstance(v, list):
            return "list of members"
        for m in v:
            if not (isinstance(m, dict) and set(m) == {"evidence_id", "evidence_digest", "parent_id"}
                    and isinstance(m["evidence_id"], str) and m["evidence_id"] != ""
                    and isinstance(m["evidence_digest"], str) and RE_HEX32.match(m["evidence_digest"])
                    and (m["parent_id"] is None or (isinstance(m["parent_id"], str) and m["parent_id"] != ""))):
                return "members [{evidence_id, evidence_digest, parent_id}]"
        return ""
    if kind.startswith("enum:"):
        return "" if v in kind[5:].split("|") else f"one of {kind[5:]}"
    if kind.startswith("opt_"):
        return "" if v is None else _check(kind[4:], v)
    return f"unknown schema kind {kind}"


SCHEMAS = {
    "decision": {
        "domain": "str", "schema_ver": "schema_ver", "profile_version": "semver", "lineage_schema_version": "semver",
        "capability_id": "uuid7", "req_id": "uuid7", "evidence_snapshot_id": "uuid7", "actor": "actor",
        "auth_mode": "enum:AUTOMATED|MANUAL", "x": "action", "a": "map", "target": "str", "state_ver": "uint",
        "control_profile_hash": "hex32", "scoring_profile_hash": "hex32", "trust_state_hash": "hex32",
        "evidence_hash": "hex32", "lineage_hash": "hex32", "approval_hash": "hex32",
        "adverse_disposition": "enum:CLEAR|CONTAIN", "containment_digest": "hex32",
        "decision_key_id": "keyid", "decision_trust_epoch": "str", "eval_clock_id": "str",
        "tau_eval": "uint", "tau_expiry": "uint", "decision": "enum:ALLOW|DENY", "reason_codes": "uint",
    },
    "approval": {
        "domain": "str", "schema_ver": "schema_ver", "approval_id": "uuid7", "approval_key_id": "keyid",
        "approval_trust_epoch": "str", "authority_class": "str", "authority_scope": "str", "x": "action", "a": "map",
        "target": "str", "actor": "actor", "req_id": "uuid7", "capability_id": "uuid7", "state_ver": "uint",
        "control_profile_hash": "hex32", "scoring_profile_hash": "hex32", "trust_state_hash": "hex32",
        "adverse_disposition": "enum:CLEAR|CONTAIN", "containment_digest": "hex32",
        "tau_issued": "uint", "tau_expiry": "uint", "decision": "enum:ALLOW|DENY",
    },
    "receipt": {
        "domain": "str", "schema_ver": "schema_ver", "receipt_id": "uuid7", "capability_id": "uuid7", "req_id": "uuid7",
        "decision_digest": "hex32", "x": "action", "target": "str", "control_profile_hash": "hex32",
        "expected_state_ver": "uint", "observed_pre_state_ver": "uint", "observed_post_state_ver": "opt_uint",
        "execution_status": "enum:COMMITTED|ROLLED_BACK|REJECTED|INDETERMINATE|UNRESOLVED",
        "failure_phase": "enum:PREPARE|PRECONDITION|DISPATCH|TARGET|RECONCILIATION|NONE",
        "reason_code": "uint", "reconciliation_required": "bool", "pre_state_digest": "hex32",
        "post_state_digest": "hex32", "tau_dispatch": "uint", "tau_finalized": "uint",
        "receipt_key_id": "keyid", "receipt_trust_epoch": "str",
    },
    "attestation": {
        "domain": "str", "schema_ver": "schema_ver", "attestation_id": "uuid7", "provenance_key_id": "keyid",
        "provenance_trust_epoch": "str", "attester_id": "str", "evidence_digest": "hex32", "dependencies": "atoms",
        "tau_issued": "uint", "tau_expiry": "uint",
    },
    "manifest": {
        "domain": "str", "schema_ver": "schema_ver", "manifest_id": "uuid7", "membership_key_id": "keyid",
        "membership_trust_epoch": "str", "authority_id": "str", "family_id": "str", "ingress_channel": "str",
        "members": "manifest_members", "tau_issued": "uint", "tau_expiry": "uint",
    },
}

RE_BITS64 = re.compile(r"^[0-9a-f]{16}$")
MEMBER_STATES = ("VALID", "UNAVAILABLE_OR_STALE", "DEGRADED", "CONTRADICTORY", "INVALID")


def member_errors(m: Any) -> List[str]:
    """Structural validity of an evidence member (open schema: extra claim fields are allowed)."""
    if not isinstance(m, dict):
        return ["member: not a map"]
    errs = []
    if not (isinstance(m.get("evidence_id"), str) and m["evidence_id"]):
        errs.append("member.evidence_id: expected non-empty string")
    if "parent_id" not in m or not (m["parent_id"] is None or (isinstance(m["parent_id"], str) and m["parent_id"])):
        errs.append("member.parent_id: expected null or non-empty string")
    if not (isinstance(m.get("producer"), str) and m["producer"]):
        errs.append("member.producer: expected non-empty string")
    if not (isinstance(m.get("value_bits"), str) and RE_BITS64.match(m["value_bits"])):
        errs.append("member.value_bits: expected 16 lowercase hex digits")
    if "state" in m and m["state"] not in MEMBER_STATES:
        errs.append("member.state: unknown state")
    if "dependencies" in m and not isinstance(m["dependencies"], list):
        errs.append("member.dependencies: expected list")
    return errs


# Signed-envelope shape shared by attestations and manifests
def envelope_errors(env: Any) -> List[str]:
    if not isinstance(env, dict) or set(env) != {"payload", "signature"}:
        return ["envelope must be exactly {payload, signature}"]
    if not isinstance(env.get("signature"), str) or not RE_SIG.match(env["signature"]):
        return ["signature must be 64 bytes of lowercase hex"]
    return []


def validate(z: Any, schema: str) -> List[str]:
    """CanonicalObjectValid(z, schema): [] iff valid. Never raises."""
    try:
        spec = SCHEMAS[schema]
    except KeyError:
        return [f"unknown schema {schema!r}"]
    if not isinstance(z, dict):
        return [f"{schema}: not a map"]
    errs = []
    keys = set(z)
    for k in sorted(set(spec) - keys):
        errs.append(f"{schema}.{k}: missing")
    for k in sorted(keys - set(spec), key=lambda s: str(s)):
        errs.append(f"{schema}.{k}: unexpected field")
    for k in sorted(set(spec) & keys):
        why = _check(spec[k], z[k])
        if why:
            errs.append(f"{schema}.{k}: expected {why}")
    return errs


### M-1 family manifests: FamilyMembershipAdmissible (§2.6) — `reference/prc_membership.py`

In [10]:
%%writefile reference/prc_membership.py
"""
prc_membership — M-1 FamilyMembershipAdmissible (spec 0.3.0-alpha §2.6)

Family membership is asserted by a *family manifest* signed by a membership authority
(K_membership domain) — typically the ingestion gateway that received the evidence:

    domain                  "PROVENANCE-RAZOR-MEMBERSHIP-V0.3.0-Alpha"
    schema_ver, manifest_id (UUIDv7), membership_key_id, membership_trust_epoch
    authority_id            the key's registered owner
    family_id               the family this manifest governs
    ingress_channel         where the evidence entered (recorded on containment events)
    members                 [{evidence_id, evidence_digest = H(EvidenceCore(e)), parent_id}]
                            sorted by evidence_id (UTF-8), unique
    tau_issued, tau_expiry

Envelope on the family:  family["manifest"] = {"payload": ..., "signature": hex}

Family-level outcome:
    VERIFIED     manifest admissible; members are classified individually
    UNVERIFIED   no manifest / unknown key / epoch / revoked / stale / scope does not cover
                 family_id / listed member missing from the submission
                 -> family excluded from support; its contradictions are quarantined
    TAMPERED     malformed envelope or payload, wrong domain string, key from another domain,
                 bad signature, authority_id not the key's owner, manifest for another family
                 -> family INVALID, security event

Member-level outcome (only when VERIFIED):
    LISTED       evidence_digest and parent_id match the manifest
    QUARANTINED  not listed: excluded from the family entirely, audit only — it can neither
                 invalidate, attenuate, contain nor defeat the family it names
    ALTERED      listed but digest/parent differ -> family INVALID (MEMBERSHIP_TAMPERED)
"""
from __future__ import annotations

import fnmatch
from typing import Dict, Tuple

import prc_ref as R
import prc_schema as SC
import prc_provenance as PV


def make_manifest(family: dict, key: R.TestKey, authority_id: str, *, epoch: str, tau_issued: int,
                  tau_expiry: int, unix_ms: int, ingress_channel: str = "gateway:primary",
                  members=None, family_id: str = None, domain: str = R.DOMAIN_MEMBERSHIP, label: str = None) -> dict:
    if members is None:
        members = [{"evidence_id": m["evidence_id"], "evidence_digest": PV.evidence_digest(m),
                    "parent_id": m.get("parent_id")} for m in family["members"]]
    members = sorted(members, key=lambda m: m["evidence_id"].encode("utf-8"))
    payload = {
        "domain": domain,
        "schema_ver": R.SPEC_VERSION,
        "manifest_id": R.uuid7(unix_ms, "manifest|" + (label or family["family_id"] + "|" + R.H(members)[:12])),
        "membership_key_id": key.key_id,
        "membership_trust_epoch": epoch,
        "authority_id": authority_id,
        "family_id": family_id or family["family_id"],
        "ingress_channel": ingress_channel,
        "members": members,
        "tau_issued": tau_issued,
        "tau_expiry": tau_expiry,
    }
    return {"payload": payload, "signature": key.sign_digest(R.H(payload))}


def family_membership(family: dict, trust: dict, tau_eval: int):
    """Returns (family_outcome, detail, admitted_members, quarantined[(evidence_id, why)], ingress_channel).
    Never raises for malformed families or manifests."""
    env = family.get("manifest")
    if env is None:
        return "UNVERIFIED", "no family manifest", [], [], ""
    if SC.envelope_errors(env) or SC.validate(env.get("payload"), "manifest"):
        return "TAMPERED", "malformed manifest", [], [], ""
    p = env["payload"]
    if p["domain"] != R.DOMAIN_MEMBERSHIP:
        return "TAMPERED", "wrong domain string", [], [], ""
    reg = trust["key_registry"]
    entry = reg.get(p["membership_key_id"])
    if entry is None:
        return "UNVERIFIED", "unknown membership key", [], [], ""
    if entry.get("domain") != "membership":
        return "TAMPERED", f"signed with a {entry.get('domain')}-domain key", [], [], ""
    if not R.ed25519_verify(entry["public_key"], R.H(p), env["signature"]):
        return "TAMPERED", "signature invalid", [], [], ""
    if p["authority_id"] != entry.get("owner"):
        return "TAMPERED", "authority_id is not the key's registered owner", [], [], ""
    if p["family_id"] != family["family_id"]:
        return "TAMPERED", "manifest governs a different family (replayed)", [], [], ""
    ids = [m["evidence_id"] for m in p["members"]]
    if ids != sorted(set(ids), key=lambda s: s.encode("utf-8")):
        return "TAMPERED", "manifest members not canonical (sorted, unique)", [], [], ""
    if p["membership_trust_epoch"] not in entry["epochs"]:
        return "UNVERIFIED", "membership epoch not authorised", [], [], ""
    if p["manifest_id"] in trust.get("revoked_manifest_ids", []):
        return "UNVERIFIED", "manifest revoked", [], [], ""
    if not p["tau_issued"] <= tau_eval <= p["tau_expiry"]:
        return "UNVERIFIED", "manifest not fresh at tau_eval", [], [], ""
    if not any(fnmatch.fnmatchcase(family["family_id"], pat) for pat in entry.get("membership_scopes", [])):
        return "UNVERIFIED", "membership authority's scope does not cover this family", [], [], ""
    listed = {m["evidence_id"]: m for m in p["members"]}
    admitted, quarantined, altered = [], [], []
    by_id = {}
    for m in family["members"]:
        eid = m.get("evidence_id") if isinstance(m, dict) else None
        by_id.setdefault(eid if isinstance(eid, str) else None, []).append(m)
    for eid in sorted((k for k in by_id if k is not None), key=lambda s: s.encode("utf-8")):
        cands = by_id[eid]
        if eid not in listed:
            quarantined += [(eid, "not listed in the family manifest")] * len(cands)
            continue
        want = listed[eid]
        match = [m for m in cands if PV.evidence_digest(m) == want["evidence_digest"]
                 and m.get("parent_id") == want["parent_id"]]
        if match:
            admitted.append(match[0])
            # same-id impostors are quarantined, so an injected duplicate cannot trigger tampering
            quarantined += [(eid, "duplicate evidence_id not matching the manifest")] * (len(cands) - 1)
        else:
            altered.append(eid)
    quarantined += [(None, "member without evidence_id")] * len(by_id.get(None, []))
    missing = [eid for eid in listed if eid not in by_id]
    if altered:
        return "TAMPERED", f"listed member(s) altered after the manifest was issued: {', '.join(altered)}", \
            admitted, quarantined, p["ingress_channel"]
    if missing:
        return "UNVERIFIED", f"listed member(s) withheld: {', '.join(missing)}", [], quarantined, p["ingress_channel"]
    return "VERIFIED", "ok", admitted, quarantined, p["ingress_channel"]


### EXPERIMENTAL scoring profile coverage-v0.3.0-alpha (A-1 + M-1) — `reference/prc_coverage_v030.py`

In [11]:
%%writefile reference/prc_coverage_v030.py
"""
prc_coverage_v030 — PRC Scoring Profile `coverage-v0.3.0-alpha`

    Status:          EXPERIMENTAL
    Normative role:  reference candidate; supersedes coverage-v0.2.1-alpha

Positive mass is unchanged (material-dependency-coverage-v1 with the consequence-gated
UNKNOWN:* sink). What changes:

  M-1  FamilyMembershipAdmissible (prc_membership): only members listed in an admissible,
       signed family manifest take part. Anything else is QUARANTINED — audit only.
  A-1  AdverseDisposition ∈ {CLEAR, CONTAIN, DEFEAT, LOCK}, precedence LOCK > DEFEAT > CONTAIN > CLEAR.
       For each *relevant* contradicting member (recognised conflict_type, bound to this
       request's action, assertion, target and evidence snapshot):
           provenance ADMITTED                 -> DEFEAT
           provenance UNATTESTED | INADMISSIBLE -> CONTAIN
           provenance TAMPERED                  -> LOCK
       Contradictions NEVER change A, V, M, C or T (0.2.x set A = 0; that is removed).
  Outcome:
       LOCK    -> DENY (Safe State Lock)
       DEFEAT  -> DENY (authenticated defeater; not overridable by ordinary approval)
       CONTAIN -> MANUAL_REQUIRED if C >= T_manual, else DENY; automatic ALLOW prohibited
       CLEAR   -> ALLOW if C >= T, else DENY
  The trust state (key registry, revocations, atom registry and aliases, per-type caps,
  attester independence, conflict types, clock authorities) is an input bound by hash; the
  scoring profile holds only the arithmetic.
"""
from __future__ import annotations

import math
from typing import List

import prc_ref as R
import prc_scoring as S
import prc_provenance as PV
import prc_membership as PM

PROFILE_ID = "coverage-v0.3.0-alpha"
UNKNOWN_ATOM = "UNKNOWN:*"
COMPONENTS = {
    "canonical_representative": "structural-root-v1",
    "mass_function": "material-dependency-coverage-v1",
    "unknown_lineage": "single-sink-consequence-gated-v1",
    "confidence_function": "exponential-saturation-v1",
    "threshold_function": "consequence-exponential-v1",
    "exp_function": "prc-exp-v1",
    "summation_order": "dependency-atom-utf8-ascending",
    "dependency_source": "attested-P-1",
    "membership": "signed-family-manifest-M-1",
    "adverse_disposition": "contain-defeat-lock-A-1",
}
PRECEDENCE = {"CLEAR": 0, "CONTAIN": 1, "DEFEAT": 2, "LOCK": 3}
f, hx, prc_exp = S.f, S.hx, S.prc_exp


def _utf8(s):
    return s.encode("utf-8")


def compile_profile(p: dict) -> dict:
    if p.get("id") != PROFILE_ID:
        raise S.ProfileError("unsupported scoring profile id")
    for k, v in COMPONENTS.items():
        if p.get(k) != v:
            raise S.ProfileError(f"component mismatch: {k}")
    c = {"fingerprint": R.H(p)}
    for name in ("omega", "t_base", "t_base_manual", "beta", "kappa_unknown", "kappa_degraded", "unknown_gate_t"):
        c[name] = f(p[name + "_bits"])
    if not c["omega"] > 0.0 or not 0.5 <= c["t_base"] < 1.0 or not 0.5 <= c["t_base_manual"] < 1.0 or not c["beta"] >= 0.0:
        raise S.ProfileError("parameter out of range")
    for name in ("kappa_unknown", "kappa_degraded", "unknown_gate_t"):
        if not 0.0 <= c[name] <= 1.0:
            raise S.ProfileError(f"{name} must be in [0, 1]")
    mats = p["material_dependency_types"]
    if sorted(set(mats)) != sorted(mats) or any(t not in S.DEPENDENCY_TYPES for t in mats):
        raise S.ProfileError("material_dependency_types must be distinct known types")
    c["material"] = frozenset(mats)
    c["gamma"] = {}
    for action, gbits in p["consequence_gamma_bits"].items():
        g = f(gbits)
        if not 0.1 <= g <= 10.0:
            raise S.ProfileError(f"gamma for {action} outside [0.1, 10.0]")
        c["gamma"][action] = g
    return c


def trust_policy(ts: dict) -> dict:
    reg = ts["atom_registry"]
    return {"aliases": dict(reg.get("aliases", {})), "registered_atoms": frozenset(reg["atoms"]),
            "max_atoms_per_type": dict(ts["max_atoms_per_type"]),
            "attester_independence_required": bool(ts["attester_independence_required"])}


def contradiction_digest(m: dict) -> str:
    return R.H({"conflict_type": m.get("conflict_type"), "contradicts": m.get("contradicts"), "claim": m.get("claim")})


def evaluate(request: dict, families: List[dict], profile_decl: dict, trust: dict) -> dict:
    prof = compile_profile(profile_decl)
    pol = trust_policy(trust)
    tau_eval = request["tau_eval"]
    binding = {"action": request["action"], "assertion": request.get("assertion"),
               "target": request.get("target"), "evidence_snapshot_id": request.get("evidence_snapshot_id")}
    reason, audit, fam_out, eligible = 0, [], {}, []
    obligations = {}   # contradiction digest -> record
    disposition = "CLEAR"

    def raise_disp(level):
        nonlocal disposition
        if PRECEDENCE[level] > PRECEDENCE[disposition]:
            disposition = level

    action = request["action"]
    if action in prof["gamma"]:
        g = prof["gamma"][action]
        T = 1.0 - (1.0 - prof["t_base"]) * prc_exp(-(g * prof["beta"]))
        T_manual = 1.0 - (1.0 - prof["t_base_manual"]) * prc_exp(-(g * prof["beta"]))
    else:
        g, T, T_manual = None, math.nan, math.nan
        reason |= R.RC["CONSEQUENCE_CLASS_UNDEFINED"]
    kappa_u = 0.0 if (T == T and T >= prof["unknown_gate_t"]) else prof["kappa_unknown"]

    import prc_schema as SC
    well_formed = []
    for i, fam in enumerate(families):
        if not (isinstance(fam, dict) and isinstance(fam.get("family_id"), str) and fam["family_id"]
                and isinstance(fam.get("members"), list)):
            audit.append({"event": "FAMILY_MALFORMED", "index": i})
            reason |= R.RC["LINEAGE_CLASS_MALFORMED"]
            continue
        well_formed.append(fam)
    for fam in sorted(well_formed, key=lambda x: _utf8(x["family_id"])):
        fid = fam["family_id"]
        # structurally invalid members never reach membership: they are quarantined like unlisted ones
        bad_members = [m for m in fam["members"] if SC.member_errors(m)]
        for _ in bad_members:
            audit.append({"event": "MEMBER_QUARANTINED", "family": fid, "evidence_id": None, "detail": "malformed member"})
        fam = dict(fam, members=[m for m in fam["members"] if not SC.member_errors(m)])
        m_out, m_detail, members, quarantined, ingress = PM.family_membership(fam, trust, tau_eval)
        for eid, why in quarantined:
            audit.append({"event": "MEMBER_QUARANTINED", "family": fid, "evidence_id": eid, "detail": why})
        if m_out == "UNVERIFIED":
            fam_out[fid] = {"eligible": False, "status": "MEMBERSHIP_UNVERIFIED", "detail": m_detail}
            audit.append({"event": "MEMBERSHIP_UNVERIFIED", "family": fid, "detail": m_detail})
            reason |= R.RC["MEMBERSHIP_UNVERIFIED"]
            continue
        if m_out == "TAMPERED":
            fam_out[fid] = {"eligible": False, "status": "INVALIDATED", "cause": "MEMBERSHIP_TAMPERED", "detail": m_detail}
            audit.append({"event": "MEMBERSHIP_TAMPERED", "family": fid, "detail": m_detail})
            reason |= R.RC["MEMBERSHIP_TAMPERED"] | R.RC["FAMILY_INVALIDATED"]
            continue
        sub = {"family_id": fid, "members": members}
        root, depth, err = S.canonical_representative(sub)
        if err:
            fam_out[fid] = {"eligible": False, "status": "MALFORMED", "error": err}
            audit.append({"event": "LINEAGE_CLASS_MALFORMED", "family": fid, "detail": err})
            reason |= R.RC["LINEAGE_CLASS_MALFORMED"]
            continue
        prov = {m["evidence_id"]: PV.admissibility(m, trust, pol, tau_eval)
                for m in sorted(members, key=lambda x: _utf8(x["evidence_id"]))}
        prov_out = {eid: {"outcome": o, "detail": d} for eid, (o, d, _) in prov.items()}

        # adverse evidence (A-1): classified before, and independently of, positive support
        for m in sorted(members, key=lambda x: _utf8(x["evidence_id"])):
            if m.get("state") != "CONTRADICTORY":
                continue
            outcome = prov[m["evidence_id"]][0]
            if outcome == "TAMPERED":
                raise_disp("LOCK")
                audit.append({"event": "TAMPERED_CONTRADICTION", "family": fid, "evidence_id": m["evidence_id"]})
                continue
            relevant = m.get("conflict_type") in trust.get("conflict_types", []) and m.get("contradicts") == binding
            if not relevant:
                audit.append({"event": "IRRELEVANT_CONTRADICTION", "family": fid, "evidence_id": m["evidence_id"]})
                continue
            level = "DEFEAT" if outcome == "ADMITTED" else "CONTAIN"
            raise_disp(level)
            dg = contradiction_digest(m)
            rec = obligations.setdefault(dg, {"digest": dg, "disposition": level, "conflict_type": m["conflict_type"],
                                              "occurrences": []})
            if PRECEDENCE[level] > PRECEDENCE[rec["disposition"]]:
                rec["disposition"] = level
            rec["occurrences"].append({"family": fid, "evidence_id": m["evidence_id"], "ingress_channel": ingress,
                                       "provenance": outcome})

        tampered = [(eid, d) for eid, (o, d, _) in prov.items() if o == "TAMPERED"]
        if tampered:
            fam_out[fid] = {"eligible": False, "status": "INVALIDATED", "cause": "PROVENANCE_TAMPERED",
                            "representative": root["evidence_id"], "provenance": prov_out}
            audit.append({"event": "PROVENANCE_TAMPERED", "family": fid,
                          "members": [{"evidence_id": e, "detail": d} for e, d in tampered]})
            reason |= R.RC["PROVENANCE_TAMPERED"] | R.RC["FAMILY_INVALIDATED"]
            continue
        optional = [m for m in members if m["evidence_id"] != root["evidence_id"]]
        states = [m.get("state", "VALID") for m in optional]
        root_state = root.get("state", "VALID")
        if "INVALID" in states or root_state == "INVALID":
            fam_out[fid] = {"eligible": False, "status": "INVALIDATED", "representative": root["evidence_id"],
                            "provenance": prov_out}
            audit.append({"event": "FAMILY_INVALIDATED", "family": fid})
            reason |= R.RC["FAMILY_INVALIDATED"]
            continue
        if root_state != "VALID":  # includes a contradicting root: no admissible positive support
            fam_out[fid] = {"eligible": False, "status": "REPRESENTATIVE_" + root_state,
                            "representative": root["evidence_id"], "provenance": prov_out}
            continue
        V_rep = f(root["value_bits"])
        if not 0.0 <= V_rep <= 1.0:
            fam_out[fid] = {"eligible": False, "status": "MALFORMED", "error": "value outside [0, 1]"}
            audit.append({"event": "LINEAGE_CLASS_MALFORMED", "family": fid, "detail": "value outside [0, 1]"})
            reason |= R.RC["LINEAGE_CLASS_MALFORMED"]
            continue
        # contradicting members are excluded from support and do not alter A
        factors = [prof["kappa_degraded"] if s == "DEGRADED" else 1.0
                   for s in states if s not in ("UNAVAILABLE_OR_STALE", "CONTRADICTORY")]
        A = min(factors) if factors else 1.0
        V = V_rep * A
        outcome, _, atoms = prov[root["evidence_id"]]
        mat = frozenset(a for a in atoms if a.split(":", 1)[0] in prof["material"]) if outcome == "ADMITTED" else frozenset()
        unknown = not mat
        if unknown:
            V = V * kappa_u
            mat = frozenset([UNKNOWN_ATOM])
        w = V / len(mat)
        fam_out[fid] = {
            "eligible": True, "status": "ELIGIBLE", "representative": root["evidence_id"],
            "representative_digest": PV.evidence_digest(root), "lineage_depth": dict(sorted(depth.items())),
            "provenance": prov_out, "ingress_channel": ingress,
            "A_bits": hx(A), "V_bits": hx(V), "unknown_lineage": unknown,
            "atoms": sorted(mat, key=_utf8), "per_atom_share_bits": hx(w),
        }
        eligible.append((fid, mat, w))

    best, owner = {}, {}
    for fid, mat, w in eligible:
        for d in mat:
            if d not in best or w > best[d]:
                best[d], owner[d] = w, fid
    M = 0.0
    for d in sorted(best, key=_utf8):
        M = M + best[d]
    C = 1.0 - prc_exp(-(M / prof["omega"]))
    if not eligible:
        reason |= R.RC["NO_ELIGIBLE_EVIDENCE"]
        M, C = 0.0, 0.0
    chk = R.authorization_check(R.bits_of(C), R.bits_of(T))
    chk_m = R.authorization_check(R.bits_of(C), R.bits_of(T_manual))
    if chk["safe_state_lock"]:
        raise_disp("LOCK")

    if disposition == "LOCK":
        decision = "DENY"
        reason |= R.RC["SAFE_STATE_LOCK"]
    elif disposition == "DEFEAT":
        decision = "DENY"
        reason |= R.RC["DEFEATER_PRESENT"]
    elif disposition == "CONTAIN":
        reason |= R.RC["ADVERSE_CONTAINMENT"]
        decision = "MANUAL_REQUIRED" if chk_m["result"] else "DENY"
        if not chk_m["result"]:
            reason |= R.RC["THRESHOLD_NOT_MET"]
    else:
        decision = "ALLOW" if chk["result"] else "DENY"
        if not chk["result"]:
            reason |= R.RC["THRESHOLD_NOT_MET"]
    if reason & R.RC["CONSEQUENCE_CLASS_UNDEFINED"]:
        decision = "DENY"
    obl = [obligations[k] for k in sorted(obligations)]
    containment = [o["digest"] for o in obl if o["disposition"] == "CONTAIN"]
    return {
        "stage": "PROFILE",
        "scoring_profile": PROFILE_ID,
        "profile_fingerprint": prof["fingerprint"],
        "trust_state_hash": R.H(trust),
        "action": action,
        "tau_eval": tau_eval,
        "families": fam_out,
        "eligible_families": [e[0] for e in eligible],
        "atom_credit_bits": {d: hx(best[d]) for d in sorted(best, key=_utf8)},
        "atom_owner": {d: owner[d] for d in sorted(owner, key=_utf8)},
        "adverse_disposition": disposition,
        "adverse_obligations": obl,
        "containment_digest": R.H(containment) if containment else R.NULL_CONTAINMENT_DIGEST,
        "kappa_unknown_effective_bits": hx(kappa_u),
        "gamma_bits": hx(g) if g is not None else None,
        "M_bits": hx(M),
        "C_bits": f"{chk['c_effective_bits']:016x}",
        "T_bits": hx(T),
        "T_manual_bits": hx(T_manual),
        "safe_state_lock": disposition == "LOCK",
        "decision": decision,
        "reason_bitfield": reason,
        "reason_codes": R.reason_names(reason),
        "audit_events": audit,
    }


### Scoring-profile cases (jaccard) — `tools/scoring_cases.py`

In [12]:
%%writefile tools/scoring_cases.py
"""
Scoring-profile vectors for `jaccard-v0.1.0-alpha` (EXPERIMENTAL).

Imported by tools/generate_vectors.py. Every case declares its intended decision (and,
where it matters, key intermediates); build() aborts if the reference evaluator disagrees.
"""
from __future__ import annotations

import copy
import math
import os
import random
import struct
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "reference"))
import prc_ref as R          # noqa: E402
import prc_scoring as P      # noqa: E402

hx = P.hx


def bits(x: float) -> str:
    return hx(x)


# ---------------------------------------------------------------------------
# Calibrated profile
# ---------------------------------------------------------------------------
# Calibration target (S-09): at Γ(wire.release) = 8.5 the threshold is ≈ 0.92, and
#   three independent families scoring 0.95 / 0.90 / 0.92  -> C ≈ 0.937  ALLOW
#   the same three sharing one upstream feed (I = 1/3)      -> C ≈ 0.603  DENY
#   two independent families                                -> C ≈ 0.846  DENY
# Ω = 1.0 and T_base = 0.5; β chosen so that (1 - T_base)·exp(-8.5β) ≈ 0.08.
PROFILE = {
    "id": P.PROFILE_ID,
    "status": "experimental",
    **P.COMPONENTS,
    "material_dependency_types": ["API_ENDPOINT", "COLLECTOR", "DATABASE", "RUNTIME", "SOURCE"],
    "omega_bits": bits(1.0),
    "t_base_bits": bits(0.5),
    "beta_bits": bits(0.2156),
    "kappa_unknown_bits": bits(0.0),
    "kappa_degraded_bits": bits(0.5),
    "consequence_gamma_bits": {
        "ledger.annotate": bits(0.1),
        "payment.refund": bits(0.5),
        "wire.release": bits(8.5),
    },
}
PROFILE_NET_MATERIAL = dict(PROFILE, material_dependency_types=sorted(PROFILE["material_dependency_types"] + ["NETWORK_AS"]))

WIRE = {"action": "wire.release", "assertion": "beneficiary-verified", "target": "acct-9921",
        "amount_minor": 990000000, "currency": "USD"}
REFUND = {"action": "payment.refund", "assertion": "refund-eligible", "target": "acct-000123"}
ANNOTATE = {"action": "ledger.annotate", "assertion": "memo-accurate", "target": "acct-000123"}

SHARED_FEED = ["SOURCE:feed-core-primary", "DATABASE:aggregator-master", "COLLECTOR:aggregator-17",
               "RUNTIME:aggregator-rt"]
COMMON_NONMATERIAL = ["NETWORK_AS:13335", "TRUST_ROOT:ca-42", "CLOCK:ntp-cluster-2"]


def ev(eid, value, deps, parent=None, state=None, **extra):
    e = {"evidence_id": eid, "parent_id": parent, "value_bits": bits(value) if isinstance(value, float) else value,
         "dependencies": list(deps)}
    if state:
        e["state"] = state
    e.update(extra)
    return e


def fam(fid, *members):
    return {"family_id": fid, "members": list(members)}


def three_confirmations(correlated: bool, vendor_b_webhook_shared=None):
    """Identical apparent confirmations; only dependency atoms differ between the two variants."""
    if correlated:
        da = SHARED_FEED + ["API_ENDPOINT:vendor-a/verify-v2"]
        db = SHARED_FEED + ["API_ENDPOINT:vendor-b/confirm"]
        dw = SHARED_FEED + ["API_ENDPOINT:internal/webhook-beneficiary"]
    else:
        da = ["SOURCE:vendor-a-registry", "DATABASE:vendor-a-db", "API_ENDPOINT:vendor-a/verify-v2"]
        db = ["SOURCE:vendor-b-bureau", "DATABASE:vendor-b-db", "API_ENDPOINT:vendor-b/confirm"]
        dw = ["SOURCE:internal-kyc", "DATABASE:beneficiary-master", "API_ENDPOINT:internal/webhook-beneficiary"]
    if vendor_b_webhook_shared == "partial":  # only vendors A and B echo the shared feed
        da = SHARED_FEED + ["API_ENDPOINT:vendor-a/verify-v2"]
        db = SHARED_FEED + ["API_ENDPOINT:vendor-b/confirm"]
        dw = ["SOURCE:internal-kyc", "DATABASE:beneficiary-master", "API_ENDPOINT:internal/webhook-beneficiary"]
    return [
        fam("F-vendor-a", ev("ev-vendor-a", 0.95, da + COMMON_NONMATERIAL, attestation="sig:vendor-a:88192")),
        fam("F-vendor-b", ev("ev-vendor-b", 0.90, db + COMMON_NONMATERIAL, attestation="sig:vendor-b:55120")),
        fam("F-webhook", ev("ev-internal-webhook", 0.92, dw + COMMON_NONMATERIAL, attestation="sig:internal:30017")),
    ]


def strip_lineage(families):
    out = copy.deepcopy(families)
    for f_ in out:
        for m in f_["members"]:
            m["dependencies"] = []
    return R.encode(out)


# ---------------------------------------------------------------------------
# Boundary search: find representative values whose C lands exactly on / one ulp below T
# ---------------------------------------------------------------------------

def _single(v_bits_hex, action_req):
    return P.evaluate(action_req, [fam("F-solo", ev("ev-solo", v_bits_hex, ["SOURCE:solo-registry"]))], PROFILE)


def boundary_values(action_req):
    tr = _single(bits(0.5), action_req)
    T = P.f(tr["T_bits"])
    lo, hi = 0.0, 1.0  # C is monotone non-decreasing in V; bisect on the bit pattern
    lob, hib = R.bits_of(lo), R.bits_of(hi)
    while hib - lob > 1:
        mid = (lob + hib) // 2
        C = P.f(_single(f"{mid:016x}", action_req)["C_bits"])
        if C >= T:
            hib = mid
        else:
            lob = mid
    eq = hib
    if P.f(_single(f"{eq:016x}", action_req)["C_bits"]) != T:
        raise SystemExit("boundary search: no V gives C == T exactly")
    below = eq
    target_below = struct.unpack(">d", struct.pack(">Q", R.bits_of(T) - 1))[0]
    while P.f(_single(f"{below:016x}", action_req)["C_bits"]) > target_below:
        below -= 1
    if P.f(_single(f"{below:016x}", action_req)["C_bits"]) != target_below:
        raise SystemExit("boundary search: no V gives C == T - 1ulp exactly")
    return f"{eq:016x}", f"{below:016x}", T


# ---------------------------------------------------------------------------
# Cases
# ---------------------------------------------------------------------------

def cases():
    C = []

    def add(cid, title, request, families, expect_decision, profile=PROFILE, checks=None, finding=None,
            note=None, refs=("2.1", "2.2", "2.3")):
        C.append(dict(id=cid, title=title, request=request, families=families, profile=profile,
                      expect=expect_decision, checks=checks or {}, finding=finding, note=note, refs=list(refs)))

    # 1-6: operator semantics
    add("01_no_eligible_evidence", "No eligible evidence: M = 0, C = 0", WIRE, [], "DENY",
        checks={"M": 0.0, "C": 0.0, "reasons": ["THRESHOLD_NOT_MET", "NO_ELIGIBLE_EVIDENCE"]})
    add("02_single_family", "One family takes the singleton identity I = 1.0 (low-consequence action)", ANNOTATE,
        [fam("F1", ev("ev-1", 0.8, ["SOURCE:registry-1", "DATABASE:db-1"]))], "ALLOW", checks={"I": {"F1": 1.0}})
    add("03_identical_dependencies", "Two families with identical material dependencies: I = 0", REFUND,
        [fam("F1", ev("ev-1", 0.9, ["SOURCE:s", "DATABASE:d"])),
         fam("F2", ev("ev-2", 0.9, ["DATABASE:d", "SOURCE:s", "CLOCK:c-1"]))], "DENY",
        checks={"I": {"F1": 0.0, "F2": 0.0}, "M": 0.0},
        note="CLOCK is not a material type in this profile, so the extra atom on ev-2 is ignored.")
    add("04_disjoint_dependencies", "Two families with disjoint material dependencies: I = 1", REFUND,
        [fam("F1", ev("ev-1", 0.6, ["SOURCE:s1", "DATABASE:d1"])),
         fam("F2", ev("ev-2", 0.5, ["SOURCE:s2", "DATABASE:d2"]))], "ALLOW", checks={"I": {"F1": 1.0, "F2": 1.0}})
    add("05a_partial_overlap_half", "Partial overlap |∩|/|∪| = 2/4: I = 0.5", REFUND,
        [fam("F1", ev("ev-1", 0.9, ["SOURCE:a", "DATABASE:b", "COLLECTOR:c"])),
         fam("F2", ev("ev-2", 0.9, ["DATABASE:b", "COLLECTOR:c", "RUNTIME:d"]))], "ALLOW",
        checks={"I": {"F1": 0.5, "F2": 0.5}})
    add("05b_partial_overlap_third", "Partial overlap 1/3: I = 1 - 1/3 exactly as binary64", REFUND,
        [fam("F1", ev("ev-1", 0.9, ["SOURCE:a", "DATABASE:b"])),
         fam("F2", ev("ev-2", 0.9, ["DATABASE:b", "RUNTIME:c"]))], "ALLOW",
        checks={"I": {"F1": 1.0 - 1 / 3, "F2": 1.0 - 1 / 3}})
    add("06_unknown_lineage_fallback", "Empty material dependency set uses κ_unknown = 0.0 (unknown ≠ independent)",
        REFUND,
        [fam("F1", ev("ev-1", 0.6, ["SOURCE:s1"])), fam("F2", ev("ev-2", 0.6, ["SOURCE:s2"])),
         fam("F3", ev("ev-3", 0.6, ["NETWORK_AS:13335"]))], "DENY",
        checks={"I": {"F1": 0.0, "F2": 0.0, "F3": 0.0}, "M": 0.0},
        finding="With min-pairwise I_k, one family of unknown lineage zeroes I_k for every other family. "
                "Two independent confirmations become worth nothing because a third, unrelated item was added.")

    # 7-8: the headline pair
    add("07_three_independent_pass", "Wire: three independent high-quality confirmations", WIRE,
        three_confirmations(False), "ALLOW", checks={"I": {"F-vendor-a": 1.0, "F-vendor-b": 1.0, "F-webhook": 1.0}},
        note="Non-material atoms (NETWORK_AS, TRUST_ROOT, CLOCK) are shared by all three and ignored.")
    add("08_three_correlated_fail", "Wire: the same confirmations, all echoing one upstream feed", WIRE,
        three_confirmations(True), "DENY",
        checks={"I": {"F-vendor-a": 1.0 - 4 / 6, "F-vendor-b": 1.0 - 4 / 6, "F-webhook": 1.0 - 4 / 6},
                "reasons": ["THRESHOLD_NOT_MET"]})
    add("08b_two_of_three_correlated", "Wire: vendors A and B share the feed, the webhook is independent", WIRE,
        three_confirmations(False, "partial"), "DENY")
    add("08c_two_independent_insufficient", "Wire: only two independent confirmations", WIRE,
        [f_ for f_ in three_confirmations(False) if f_["family_id"] != "F-vendor-b"], "DENY",
        note="Calibration evidence: at Γ = 8.5 this profile needs three independent confirmations.")

    # 9-11: numeric boundaries
    eq_bits, below_bits, T_refund = boundary_values(REFUND)
    add("09_boundary_c_equals_t", "C == T exactly (bit-for-bit) must ALLOW", REFUND,
        [fam("F-solo", ev("ev-solo", eq_bits, ["SOURCE:solo-registry"]))], "ALLOW", checks={"C_equals_T": True},
        note="Representative value found by bisection over bit patterns; see `summary`.")
    add("10_boundary_one_ulp_below", "C one binary64 ulp below T must DENY", REFUND,
        [fam("F-solo", ev("ev-solo", below_bits, ["SOURCE:solo-registry"]))], "DENY",
        checks={"C_ulps_below_T": 1})
    add("11a_nan_value_faults_closed", "NaN representative value -> Safe State Lock", ANNOTATE,
        [fam("F1", ev("ev-1", "7ff8000000000000", ["SOURCE:s1"])), fam("F2", ev("ev-2", 0.9, ["SOURCE:s2"]))], "DENY",
        checks={"lock": True, "C": 0.0})
    add("11b_inf_value_faults_closed", "+Inf representative value -> Safe State Lock (not 'infinitely confident')",
        ANNOTATE, [fam("F1", ev("ev-1", "7ff0000000000000", ["SOURCE:s1"]))], "DENY", checks={"lock": True, "C": 0.0},
        note="Without an explicit guard, exp(-Inf) = 0 would give C = 1.0 and ALLOW.")
    add("11c_undefined_consequence_class", "Action with no consequence class -> T undefined -> fault closed",
        {"action": "treasury.sweep", "assertion": "x"}, three_confirmations(False), "DENY",
        checks={"lock": True, "reasons": ["SAFE_STATE_LOCK", "CONSEQUENCE_CLASS_UNDEFINED"]})

    # 12-14: invariances (the generator proves each against its base case)
    base12 = three_confirmations(True) + [fam("F-memo", ev("ev-memo-root", 0.7, ["SOURCE:memo"]),
                                               ev("ev-memo-derived", 0.7, ["SOURCE:memo", "RUNTIME:m"],
                                                  parent="ev-memo-root", state="DEGRADED"))]
    perm = copy.deepcopy(base12)
    random.Random(12).shuffle(perm)
    for f_ in perm:
        f_["members"].reverse()
        for m in f_["members"]:
            m["dependencies"].reverse()
    add("12_permutation_invariance", "Shuffled families, members and dependency lists: identical output bits", WIRE,
        perm, "DENY", checks={"same_trace_as": base12})
    dup = copy.deepcopy(C[3]["families"])  # case 04
    for f_ in dup:
        for m in f_["members"]:
            m["dependencies"] = m["dependencies"] + m["dependencies"][:1]
    add("13_duplicate_atoms", "Duplicate dependency atoms do not change Jaccard (or digests)", REFUND, dup, "ALLOW",
        checks={"same_trace_as": C[3]["families"]})
    root = ev("ev-root", 0.9, ["SOURCE:reg"])
    variants = []
    for nonce in range(1000):
        d = ev("ev-derived", 0.9, ["SOURCE:reg", "COLLECTOR:c"], parent="ev-root", state="VALID", revision=nonce)
        variants.append((P.canonical_digest(d) > P.canonical_digest(root), d))
    hi_d = next(d for gt, d in variants if gt)
    lo_d = next(d for gt, d in variants if not gt)
    add("14a_digest_not_selection_hi", "Derivative with a digest above the root's does not become representative",
        ANNOTATE, [fam("F1", copy.deepcopy(root), hi_d)], "ALLOW", checks={"representative": {"F1": "ev-root"}})
    add("14b_digest_not_selection_lo", "Same family, derivative digest below the root's: same representative", ANNOTATE,
        [fam("F1", copy.deepcopy(root), lo_d)], "ALLOW", checks={"representative": {"F1": "ev-root"},
                                                                  "same_decision_values_as": "14a_digest_not_selection_hi"})

    # structure & findings
    add("15_malformed_two_roots", "Two parentless roots in one class -> class malformed, excluded, audited", ANNOTATE,
        [fam("F1", ev("ev-a", 0.9, ["SOURCE:a"]), ev("ev-b", 0.9, ["SOURCE:b"])),
         fam("F2", ev("ev-c", 0.8, ["SOURCE:c"]))], "ALLOW",
        checks={"reasons": ["LINEAGE_CLASS_MALFORMED"], "eligible": ["F2"]},
        note="Exclusion (not global deny) is chosen so a malformed class can lower C but cannot block on its own (S-02).")
    add("16_malformed_cycle", "Parent-link cycle inside a class -> malformed", ANNOTATE,
        [fam("F1", ev("ev-r", 0.9, ["SOURCE:r"]), ev("ev-x", 0.9, ["SOURCE:r"], parent="ev-y"),
             ev("ev-y", 0.9, ["SOURCE:r"], parent="ev-x"))], "DENY",
        checks={"reasons": ["THRESHOLD_NOT_MET", "NO_ELIGIBLE_EVIDENCE", "LINEAGE_CLASS_MALFORMED"]})
    add("17_non_monotonic_mass", "Adding a fourth confirmation (a mirror of vendor A) flips ALLOW to DENY", WIRE,
        three_confirmations(False) + [fam("F-vendor-a-mirror", ev("ev-vendor-a-mirror", 0.90,
                                          ["SOURCE:vendor-a-registry", "DATABASE:vendor-a-db",
                                           "API_ENDPOINT:vendor-a/verify-v2"]))], "DENY",
        checks={"I": {"F-vendor-a": 0.0, "F-vendor-a-mirror": 0.0}},
        finding="Min-pairwise I_k makes M non-monotone: a redundant confirmation zeroes the independence of the "
                "family it duplicates, so M drops from 2.77 to 1.82. Redundant evidence should add ~0, not subtract.")
    add("18_material_type_sensitivity", "Case 07 with NETWORK_AS declared material: the shared AS flips the decision",
        WIRE, three_confirmations(False), "DENY", profile=PROFILE_NET_MATERIAL,
        checks={"I": {"F-vendor-a": 1.0 - 1 / 7}},
        finding="Which dependency types are material is decisive. Nominating NETWORK_AS makes a shared CDN "
                "look like shared provenance; that choice needs justification, not a default.")
    add("19_degraded_member", "Case 07 with a DEGRADED derivative in vendor A's family (κ = 0.5)", WIRE,
        [fam("F-vendor-a", *(three_confirmations(False)[0]["members"] +
                             [ev("ev-vendor-a-cache", 0.95, ["SOURCE:vendor-a-registry", "RUNTIME:cdn-edge"],
                                 parent="ev-vendor-a", state="DEGRADED")]))] + three_confirmations(False)[1:],
        "DENY")
    add("20_value_out_of_range", "Representative value 1.2 is outside [0, 1] -> class malformed", ANNOTATE,
        [fam("F1", ev("ev-1", 1.2, ["SOURCE:a"])), fam("F2", ev("ev-2", 0.8, ["SOURCE:b"]))], "ALLOW",
        checks={"reasons": ["LINEAGE_CLASS_MALFORMED"], "eligible": ["F2"]})
    return C


# ---------------------------------------------------------------------------
# prc-exp-v1 primitive vectors
# ---------------------------------------------------------------------------
EXP_INPUTS = [0.0, -0.0, 1.0, -1.0, 0.5, -0.5, 1e-300, -1e-300, math.log(2), -math.log(2),
              -0.3695, -0.18475, -2.77, -1.8326, -0.1078, -0.02156,
              -27.046689406673504,  # a value where V8 Math.exp and glibc exp disagree by 1 ulp
              -40.0, -100.0, -700.0, -708.5, -709.0, -744.0, -745.0, -745.1, -1000.0,
              700.0, 709.0, 709.5, math.inf, -math.inf, math.nan]


def exp_vectors():
    out = []
    for x in EXP_INPUTS:
        y = P.prc_exp(x)
        lib = math.exp(x) if x < 709.78 else math.inf
        out.append({"input_bits": hx(x), "input_repr": repr(x), "expected_bits": hx(y), "expected_repr": repr(y),
                    "libm_exp_bits_informative": hx(lib) if not math.isnan(x) else hx(x)})
    return out


# ---------------------------------------------------------------------------
# Build
# ---------------------------------------------------------------------------

def _num(b):
    return P.f(b)


def build():
    all_cases = cases()
    by_id = {}
    out = []
    for c in all_cases:
        tr = P.evaluate(c["request"], c["families"], c["profile"])
        if tr["decision"] != c["expect"]:
            raise SystemExit(f"scoring {c['id']}: evaluator says {tr['decision']} ({tr['reason_codes']}), "
                             f"case declares {c['expect']}")
        ck = c["checks"]
        fams = tr["families"]
        if "M" in ck and _num(tr["M_bits"]) != ck["M"]:
            raise SystemExit(f"scoring {c['id']}: M = {_num(tr['M_bits'])}")
        if "C" in ck and _num(tr["C_bits"]) != ck["C"]:
            raise SystemExit(f"scoring {c['id']}: C = {_num(tr['C_bits'])}")
        for k, v in ck.get("I", {}).items():
            if _num(fams[k]["I_bits"]) != v:
                raise SystemExit(f"scoring {c['id']}: I[{k}] = {_num(fams[k]['I_bits'])} != {v}")
        if "reasons" in ck and sorted(tr["reason_codes"]) != sorted(ck["reasons"]):
            raise SystemExit(f"scoring {c['id']}: reasons {tr['reason_codes']}")
        if "lock" in ck and tr["safe_state_lock"] != ck["lock"]:
            raise SystemExit(f"scoring {c['id']}: lock")
        if "eligible" in ck and tr["eligible_families"] != ck["eligible"]:
            raise SystemExit(f"scoring {c['id']}: eligible {tr['eligible_families']}")
        for k, v in ck.get("representative", {}).items():
            if fams[k]["representative"] != v:
                raise SystemExit(f"scoring {c['id']}: representative")
        if ck.get("C_equals_T") and tr["C_bits"] != tr["T_bits"]:
            raise SystemExit(f"scoring {c['id']}: C != T")
        if "C_ulps_below_T" in ck and int(tr["T_bits"], 16) - int(tr["C_bits"], 16) != ck["C_ulps_below_T"]:
            raise SystemExit(f"scoring {c['id']}: ulp distance")
        if "same_trace_as" in ck:
            base = P.evaluate(c["request"], ck["same_trace_as"], c["profile"])
            if R.encode(base) != R.encode(tr):
                raise SystemExit(f"scoring {c['id']}: trace differs from its base case")
        if "same_decision_values_as" in ck:
            o = by_id[ck["same_decision_values_as"]]
            if (o["M_bits"], o["C_bits"], o["T_bits"], o["decision"]) != (tr["M_bits"], tr["C_bits"], tr["T_bits"], tr["decision"]):
                raise SystemExit(f"scoring {c['id']}: values differ from {ck['same_decision_values_as']}")
        by_id[c["id"]] = tr
        v = {
            "id": c["id"], "title": c["title"], "spec_refs": c["refs"],
            "profile": "calibrated" if c["profile"] is PROFILE else "calibrated+NETWORK_AS-material",
            "input": {"request": c["request"], "families": c["families"]},
            "expected_trace": tr,
            "summary": {
                "decision": tr["decision"],
                "M": repr(_num(tr["M_bits"])), "C": repr(_num(tr["C_bits"])), "T": repr(_num(tr["T_bits"])),
                "I": {k: repr(_num(x["I_bits"])) for k, x in fams.items() if "I_bits" in x},
            },
        }
        if c["note"]:
            v["note"] = c["note"]
        if c["finding"]:
            v["finding"] = c["finding"]
        out.append(v)

    # The headline claim: same apparent confirmations, only lineage differs, decision flips.
    a, b = cases()[7], cases()[8]
    assert a["id"].startswith("07") and b["id"].startswith("08")
    if strip_lineage(a["families"]) != strip_lineage(b["families"]) or a["request"] != b["request"]:
        raise SystemExit("headline pair differs in something other than dependency atoms")
    headline = {
        "claim": "With identical apparent confirmations and identical authorization context, changing only the "
                 "evidence-lineage graph produces a reproducible ALLOW -> DENY transition at a consequence-"
                 "calibrated boundary.",
        "allow_case": "07_three_independent_pass", "deny_case": "08_three_correlated_fail",
        "inputs_identical_except_dependencies": True,
        "stripped_inputs_sha256": R.H_bytes(strip_lineage(a["families"])),
        "allow": out[7]["summary"], "deny": out[8]["summary"],
    }

    # Informative calibration table (not normative).
    calib = {"T_by_action": {}, "C_for_n_independent_families_at_0.9": {}}
    for act in PROFILE["consequence_gamma_bits"]:
        tr = P.evaluate({"action": act}, [], PROFILE)
        calib["T_by_action"][act] = repr(_num(tr["T_bits"]))
    for n in range(1, 7):
        fs = [fam(f"F{i}", ev(f"ev-{i}", 0.9, [f"SOURCE:s{i}"])) for i in range(n)]
        tr = P.evaluate(WIRE, fs, PROFILE)
        calib["C_for_n_independent_families_at_0.9"][str(n)] = f"{repr(_num(tr['C_bits']))} -> {tr['decision']} (wire)"

    return {
        "vector_set": "scoring-profile",
        "scoring_profile": P.PROFILE_ID,
        "status": "EXPERIMENTAL — reference candidate, replaceable; not part of the frozen control architecture",
        "spec_version": R.SPEC_VERSION,
        "assumptions": "docs/SCORING_PROFILE_jaccard-v0.1.0-alpha.md",
        "profiles": {
            "calibrated": {"declaration": PROFILE, "fingerprint": R.H(PROFILE)},
            "calibrated+NETWORK_AS-material": {"declaration": PROFILE_NET_MATERIAL, "fingerprint": R.H(PROFILE_NET_MATERIAL)},
        },
        "prc_exp_v1": {
            "algorithm": "k = floor(x*INV_LN2 + 0.5); r = (x - k*LN2_HI) - k*LN2_LO; p = Horner(c13..c0, r); "
                         "return p * 2^k (two-step scaling outside [-1022, 1023]); "
                         "x > 709 -> +Inf, x < -745 -> 0, NaN -> NaN. No FMA contraction.",
            "constants_bits": {"INV_LN2": hx(P.EXP_INV_LN2), "LN2_HI": hx(P.EXP_LN2_HI), "LN2_LO": hx(P.EXP_LN2_LO),
                               "C0..C13": P.EXP_COEFF_BITS},
            "vectors": exp_vectors(),
        },
        "headline": headline,
        "calibration_informative": calib,
        "cases": out,
    }


### Coverage cases, property witnesses, known exploit — `tools/coverage_cases.py`

In [13]:
%%writefile tools/coverage_cases.py
"""
Vectors for scoring profile `coverage-v0.2.0-alpha` (EXPERIMENTAL) and the engine guard (E-1).

Imported by tools/generate_vectors.py. Each case declares its intended decision and the
invariant it demonstrates; build() aborts if the reference engine disagrees.
Randomised property witnesses use a SHA-256 counter PRNG so they regenerate byte-for-byte.
"""
from __future__ import annotations

import copy
import hashlib
import os
import struct
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "reference"))
import prc_ref as R            # noqa: E402
import prc_scoring as S        # noqa: E402
import prc_coverage as COV     # noqa: E402
import prc_engine as ENG       # noqa: E402
import scoring_cases as J      # noqa: E402  (shared evidence builders; jaccard vectors stay untouched)

bits, ev, fam = J.bits, J.ev, J.fam
WIRE, REFUND, ANNOTATE = J.WIRE, J.REFUND, J.ANNOTATE

PROFILE = {
    "id": COV.PROFILE_ID,
    "status": "experimental",
    **COV.COMPONENTS,
    "material_dependency_types": J.PROFILE["material_dependency_types"],
    "omega_bits": bits(1.0),
    "t_base_bits": bits(0.5),
    "beta_bits": bits(0.2156),
    "kappa_unknown_bits": bits(0.5),
    "kappa_degraded_bits": bits(0.5),
    "unknown_gate_t_bits": bits(0.92),
    "consequence_gamma_bits": J.PROFILE["consequence_gamma_bits"],
}


class DRng:
    """Deterministic PRNG: SHA-256 over (seed, counter). Independent of the Python version."""

    def __init__(self, seed: str):
        self.seed, self.n = seed, 0

    def u64(self) -> int:
        self.n += 1
        return int.from_bytes(hashlib.sha256(f"{self.seed}|{self.n}".encode()).digest()[:8], "big")

    def below(self, k: int) -> int:
        return self.u64() % k

    def unit(self) -> float:  # multiple of 2^-10 in [0, 1] — exact, readable values
        return self.below(1025) / 1024.0

    def pick(self, seq):
        return seq[self.below(len(seq))]


ATOM_POOL = [f"{t}:{v}" for t in ("SOURCE", "DATABASE", "COLLECTOR", "RUNTIME", "API_ENDPOINT")
             for v in ("p", "q", "r")] + ["NETWORK_AS:13335", "TRUST_ROOT:ca-42"]


def random_family(rng: DRng, fid: str) -> dict:
    n = rng.below(5)  # 0 atoms -> unknown lineage
    deps = sorted({rng.pick(ATOM_POOL) for _ in range(n)})
    members = [ev(f"ev-{fid}", rng.unit(), deps)]
    if rng.below(3) == 0:
        members.append(ev(f"ev-{fid}-d", rng.unit(), deps[:1], parent=f"ev-{fid}",
                          state=rng.pick(["VALID", "DEGRADED", "UNAVAILABLE_OR_STALE"])))
    return fam(fid, *members)


def rename_family(f_: dict, new_fid: str) -> dict:
    """Exact structural duplicate under fresh ids (parent links remapped)."""
    ids = {m["evidence_id"]: f"{m['evidence_id']}@{new_fid}" for m in f_["members"]}
    members = []
    for m in f_["members"]:
        n = copy.deepcopy(m)
        n["evidence_id"] = ids[m["evidence_id"]]
        if n.get("parent_id") in ids:
            n["parent_id"] = ids[n["parent_id"]]
        members.append(n)
    return {"family_id": new_fid, "members": members}


def M_of(tr) -> float:
    return S.f(tr["M_bits"])


# ---------------------------------------------------------------------------
# Cases
# ---------------------------------------------------------------------------

def cases():
    C = []

    def add(cid, title, request, families, expect, checks=None, note=None, profile=PROFILE, **extra):
        C.append(dict(id=cid, title=title, request=request, families=families, expect=expect,
                      checks=checks or {}, note=note, profile=profile, **extra))

    ind, cor, part = J.three_confirmations(False), J.three_confirmations(True), J.three_confirmations(False, "partial")
    add("c01_no_eligible_evidence", "No eligible evidence: M = 0, C = 0", WIRE, [], "DENY", {"M": 0.0})
    add("c02_single_family", "One family, low-consequence action", ANNOTATE,
        [fam("F1", ev("ev-1", 0.8, ["SOURCE:registry-1", "DATABASE:db-1"]))], "ALLOW", {"M": 0.8})
    add("c03_three_independent_pass", "Wire: three independent confirmations", WIRE, ind, "ALLOW")
    add("c04_three_correlated_fail", "Wire: the same confirmations sharing one upstream feed", WIRE, cor, "DENY")
    add("c05_two_of_three_correlated", "Wire: vendors A and B share the feed", WIRE, part, "DENY")
    add("c06_two_independent_insufficient", "Wire: only two independent confirmations", WIRE,
        [x for x in ind if x["family_id"] != "F-vendor-b"], "DENY")
    mirror = ind + [fam("F-vendor-a-mirror", ev("ev-vendor-a-mirror", 0.90,
                                                  ["SOURCE:vendor-a-registry", "DATABASE:vendor-a-db",
                                                   "API_ENDPOINT:vendor-a/verify-v2"]))]
    add("c07_mirror_added_neutral", "Fix for F-2: a mirror of vendor A adds nothing and cannot flip ALLOW",
        WIRE, mirror, "ALLOW", {"same_M_as": "c03_three_independent_pass"})
    bridge = ind + [fam("F-bridge", ev("ev-bridge", 0.10, ["SOURCE:vendor-a-registry", "SOURCE:vendor-b-bureau",
                                                            "SOURCE:internal-kyc"]))]
    add("c08_bridge_added_neutral", "A bridging item touching all three sources cannot merge or dilute them",
        WIRE, bridge, "ALLOW", {"same_M_as": "c03_three_independent_pass"})
    add("c09_unknown_at_high_consequence", "Fix for F-3: unattributed item on a wire (T ≥ gate, κ = 0) is neutral",
        WIRE, cor + [fam("F-unattributed", ev("ev-unattributed", 1.0, ["NETWORK_AS:13335"]))], "DENY",
        {"same_M_as": "c04_three_correlated_fail", "kappa_effective": 0.0})
    base10 = [fam("F1", ev("ev-1", 0.6, ["SOURCE:s1"]))]
    add("c10a_refund_known_only", "Refund: one attributed family alone", REFUND, base10, "DENY", {"M": 0.6})
    add("c10b_unknown_at_low_consequence", "Refund (T < gate): unattributed item contributes V·κ once", REFUND,
        base10 + [fam("F-u1", ev("ev-u1", 0.9, []))], "ALLOW", {"M": 0.6 + 0.45, "kappa_effective": 0.5})
    spam = base10 + [fam(f"F-u{i}", ev(f"ev-u{i}", 0.9, [])) for i in range(1, 21)]
    add("c11_unknown_spam_single_sink", "Twenty unattributed items share one UNKNOWN sink: same M as one", REFUND,
        spam, "ALLOW", {"same_M_as": "c10b_unknown_at_low_consequence"})
    dup = base10 + [fam("F1-dup", ev("ev-1-dup", 0.6, ["SOURCE:s1"]))]
    add("c12_duplicate_idempotent", "Exact duplicate (same atoms, same value) leaves M bit-identical", REFUND,
        dup, "DENY", {"same_M_as": "c10a_refund_known_only"})
    base13 = cor + [fam("F-memo", ev("ev-memo-root", 0.7, ["SOURCE:memo"]),
                        ev("ev-memo-derived", 0.7, ["SOURCE:memo", "RUNTIME:m"], parent="ev-memo-root", state="DEGRADED"))]
    perm = copy.deepcopy(base13)
    rng = DRng("c13")
    for i in range(len(perm) - 1, 0, -1):
        j = rng.below(i + 1)
        perm[i], perm[j] = perm[j], perm[i]
    for f_ in perm:
        f_["members"].reverse()
        for m in f_["members"]:
            m["dependencies"].reverse()
    add("c13_permutation_invariance", "Shuffled families, members and atom lists: identical output bits", WIRE, perm,
        "DENY", {"same_trace_as": base13})
    add("c14_degraded_member", "DEGRADED optional member halves vendor A's value", WIRE,
        [fam("F-vendor-a", *(ind[0]["members"] + [ev("ev-vendor-a-cache", 0.95, ["SOURCE:vendor-a-registry"],
                                                     parent="ev-vendor-a", state="DEGRADED")]))] + ind[1:], "DENY")
    add("c15_malformed_class_excluded", "Two roots in one class: excluded and audited, others still count", ANNOTATE,
        [fam("F1", ev("ev-a", 0.9, ["SOURCE:a"]), ev("ev-b", 0.9, ["SOURCE:b"])),
         fam("F2", ev("ev-c", 0.8, ["SOURCE:c"]))], "ALLOW", {"reasons": ["LINEAGE_CLASS_MALFORMED"]})
    add("c16_undefined_consequence", "Action without a consequence class faults closed", {"action": "treasury.sweep"},
        ind, "DENY", {"reasons": ["SAFE_STATE_LOCK", "CONSEQUENCE_CLASS_UNDEFINED"]})
    eqb, belowb = boundary_values()
    add("c17_boundary_c_equals_t", "C == T exactly must ALLOW", REFUND,
        [fam("F-solo", ev("ev-solo", eqb, ["SOURCE:solo"]))], "ALLOW", {"C_equals_T": True})
    add("c18_boundary_one_ulp_below", "C one ulp below T must DENY", REFUND,
        [fam("F-solo", ev("ev-solo", belowb, ["SOURCE:solo"]))], "DENY", {"C_ulps_below_T": 1})

    # engine guard (E-1): profile code is never reached
    add("e01_nan_representative", "Engine guard: NaN representative value", ANNOTATE,
        [fam("F1", ev("ev-1", "7ff8000000000000", ["SOURCE:a"]))], "DENY", {"stage": "ENGINE_GUARD"})
    add("e02_inf_optional_member", "Engine guard: +Inf on an optional (non-representative) member", ANNOTATE,
        [fam("F1", ev("ev-1", 0.9, ["SOURCE:a"]), ev("ev-1d", "7ff0000000000000", ["SOURCE:a"], parent="ev-1"))],
        "DENY", {"stage": "ENGINE_GUARD"})
    add("e03_nan_profile_parameter", "Engine guard: NaN in a profile parameter (Ω)", ANNOTATE, ind, "DENY",
        {"stage": "ENGINE_GUARD"}, profile=dict(PROFILE, omega_bits="7ff8000000000000"))
    add("e04_neg_inf_gamma", "Engine guard: −Inf consequence Γ", WIRE, ind, "DENY", {"stage": "ENGINE_GUARD"},
        profile=dict(PROFILE, consequence_gamma_bits=dict(PROFILE["consequence_gamma_bits"],
                                                          **{"wire.release": "fff0000000000000"})))

    # KNOWN OPEN EXPLOIT: self-declared lineage padding
    pad = [fam(f"F-pad-{i}", ev(f"ev-pad-{i}", 0.95, J.SHARED_FEED + ["API_ENDPOINT:vendor-a/verify-v2"] +
                                [f"COLLECTOR:fabricated-{i}-{k:03d}" for k in range(200)])) for i in (1, 2)]
    add("x01_atom_padding_exploit", "Two self-declared duplicates padded with fabricated atoms flip DENY to ALLOW",
        WIRE, cor + pad, "ALLOW",
        known_exploit={"security_expectation": {"decision": "DENY", "same_M_as": "c04_three_correlated_fail"},
                       "open_until": "verified-provenance boundary (proposed rule P-1) is specified and implemented",
                       "why": "Each padded item re-declares vendor A's lineage and adds 200 unique fabricated "
                              "COLLECTOR atoms, collecting ~V of mass it did not earn."})
    return C


def boundary_values():
    def one(vb):
        return ENG.evaluate(REFUND, [fam("F-solo", ev("ev-solo", vb, ["SOURCE:solo"]))], PROFILE)
    T = S.f(one(bits(0.5))["T_bits"])
    lob, hib = R.bits_of(0.0), R.bits_of(1.0)
    while hib - lob > 1:
        mid = (lob + hib) // 2
        if S.f(one(f"{mid:016x}")["C_bits"]) >= T:
            hib = mid
        else:
            lob = mid
    if S.f(one(f"{hib:016x}")["C_bits"]) != T:
        raise SystemExit("coverage boundary: no V gives C == T")
    target = struct.unpack(">d", struct.pack(">Q", R.bits_of(T) - 1))[0]
    b = hib
    while S.f(one(f"{b:016x}")["C_bits"]) > target:
        b -= 1
    if S.f(one(f"{b:016x}")["C_bits"]) != target:
        raise SystemExit("coverage boundary: no V gives C == T - 1ulp")
    return f"{hib:016x}", f"{b:016x}"


# ---------------------------------------------------------------------------
# Property witnesses (deterministic random pairs, re-checked by the JS cross-check)
# ---------------------------------------------------------------------------

def witnesses(n=24):
    rng = DRng("coverage-witness")
    out = []
    kinds = ["monotone", "idempotent", "bridge"]
    for i in range(n):
        kind = kinds[i % 3]
        base = [random_family(rng, f"W{i}-{k}") for k in range(1 + rng.below(5))]
        req = rng.pick([WIRE, REFUND, ANNOTATE])
        if kind == "monotone":
            added = random_family(rng, f"W{i}-new")
        elif kind == "idempotent":
            added = rename_family(rng.pick(base), f"W{i}-dup")
        else:
            atoms = sorted({a for fb in base for a in fb["members"][0]["dependencies"]})
            pick = sorted({rng.pick(atoms) for _ in range(3)}) if atoms else []
            added = fam(f"W{i}-bridge", ev(f"ev-W{i}-bridge", rng.unit(), pick))
        t0 = ENG.evaluate(req, base, PROFILE)
        t1 = ENG.evaluate(req, base + [added], PROFILE)
        m0, m1 = M_of(t0), M_of(t1)
        ok = m1 >= m0 if kind != "idempotent" else t1["M_bits"] == t0["M_bits"]
        # duplicates of an unknown-lineage family or of a degraded one may still add nothing; the check holds
        if not ok:
            raise SystemExit(f"property witness {i} ({kind}) violated: {m0} -> {m1}")
        out.append({"kind": kind, "request": req, "base": base, "added": added,
                    "M_base_bits": t0["M_bits"], "M_after_bits": t1["M_bits"],
                    "decision_base": t0["decision"], "decision_after": t1["decision"]})
    return out


# ---------------------------------------------------------------------------
# Build
# ---------------------------------------------------------------------------

def build():
    out, by_id = [], {}
    xfail = []
    for c in cases():
        tr = ENG.evaluate(c["request"], c["families"], c["profile"])
        if tr["decision"] != c["expect"]:
            raise SystemExit(f"coverage {c['id']}: engine says {tr['decision']} {tr['reason_codes']}, declared {c['expect']}")
        ck = c["checks"]
        if "M" in ck and S.f(tr["M_bits"]) != ck["M"]:
            raise SystemExit(f"coverage {c['id']}: M = {S.f(tr['M_bits'])} != {ck['M']}")
        if "same_M_as" in ck and tr["M_bits"] != by_id[ck["same_M_as"]]["M_bits"]:
            raise SystemExit(f"coverage {c['id']}: M differs from {ck['same_M_as']}")
        if "same_trace_as" in ck and R.encode(ENG.evaluate(c["request"], ck["same_trace_as"], c["profile"])) != R.encode(tr):
            raise SystemExit(f"coverage {c['id']}: trace differs from its base")
        if "kappa_effective" in ck and S.f(tr["kappa_unknown_effective_bits"]) != ck["kappa_effective"]:
            raise SystemExit(f"coverage {c['id']}: kappa")
        if "reasons" in ck and sorted(tr["reason_codes"]) != sorted(ck["reasons"]):
            raise SystemExit(f"coverage {c['id']}: reasons {tr['reason_codes']}")
        if "stage" in ck and tr["stage"] != ck["stage"]:
            raise SystemExit(f"coverage {c['id']}: stage {tr['stage']}")
        if ck.get("C_equals_T") and tr["C_bits"] != tr["T_bits"]:
            raise SystemExit(f"coverage {c['id']}: C != T")
        if "C_ulps_below_T" in ck and int(tr["T_bits"], 16) - int(tr["C_bits"], 16) != ck["C_ulps_below_T"]:
            raise SystemExit(f"coverage {c['id']}: ulp")
        by_id[c["id"]] = tr
        v = {"id": c["id"], "title": c["title"],
             "profile": "calibrated" if c["profile"] is PROFILE else "variant",
             "input": {"request": c["request"], "families": c["families"]},
             "expected_trace": tr,
             "summary": {"decision": tr["decision"], "stage": tr["stage"],
                         "M": repr(S.f(tr["M_bits"])), "C": repr(S.f(tr["C_bits"])),
                         "T": repr(S.f(tr["T_bits"])) if tr["T_bits"] else None}}
        if c["profile"] is not PROFILE:
            v["profile_declaration"] = c["profile"]
        if c.get("note"):
            v["note"] = c["note"]
        if c.get("known_exploit"):
            ke = c["known_exploit"]
            sec = ke["security_expectation"]
            meets = tr["decision"] == sec["decision"] and tr["M_bits"] == by_id[sec["same_M_as"]]["M_bits"]
            if meets:
                raise SystemExit(f"{c['id']}: exploit no longer reproduces — promote it to a normal case")
            v["known_exploit"] = dict(ke, status="REPRODUCES in this superseded profile; closed in coverage-v0.2.1-alpha (x01_self_declared_padding)")
            xfail.append(c["id"])
        out.append(v)

    return {
        "vector_set": "scoring-profile",
        "scoring_profile": COV.PROFILE_ID,
        "status": "SUPERSEDED by coverage-v0.2.1-alpha (kept frozen as evidence; x01 reproduces here by design)",
        "spec_version": R.SPEC_VERSION,
        "assumptions": "docs/SCORING_PROFILE_coverage-v0.2.0-alpha.md",
        "engine": "reference/prc_engine.py — non-finite guard (E-1) runs before profile dispatch",
        "profiles": {"calibrated": {"declaration": PROFILE, "fingerprint": R.H(PROFILE)}},
        "known_exploits_open": xfail,
        "property_witnesses": witnesses(),
        "cases": out,
    }


### v0.2.1 cases: exploit family, defeaters, witnesses — `tools/coverage_v021_cases.py`

In [14]:
%%writefile tools/coverage_v021_cases.py
"""
Vectors for scoring profile `coverage-v0.2.1-alpha` (EXPERIMENTAL): P-1 verified provenance,
authenticated-contradiction defeaters, and the atom-padding exploit family as MUST-DENY cases.

Imported by tools/generate_vectors.py. Every case declares its intended decision and the
invariant it demonstrates; build() aborts if the reference engine disagrees.
"""
from __future__ import annotations

import copy
import os
import struct
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "reference"))
import prc_ref as R              # noqa: E402
import prc_scoring as S          # noqa: E402
import prc_provenance as PV      # noqa: E402
import prc_coverage_v021 as C21  # noqa: E402
import prc_engine as ENG         # noqa: E402
import scoring_cases as J        # noqa: E402
import coverage_cases as CC      # noqa: E402

bits = J.bits
WIRE_R, REFUND_R, ANNOTATE_R = J.WIRE, J.REFUND, J.ANNOTATE
BASE_MS = 1767225600000
BASE_US = BASE_MS * 1000
DAY = 86_400_000_000
EPOCH = "epoch-2026-01"


def req(r):
    return dict(r, tau_eval=BASE_US)


WIRE, REFUND, ANNOTATE = req(WIRE_R), req(REFUND_R), req(ANNOTATE_R)

# ---------------------------------------------------------------------------
# Keys and trust state (test keys only — seeds are public)
# ---------------------------------------------------------------------------
K_AUD = R.TestKey.derive("provenance", "auditor-1")            # independent lineage auditor, all types
K_AUD_SRC = R.TestKey.derive("provenance", "auditor-src-only")  # authorised for SOURCE atoms only
K_VENDOR_A = R.TestKey.derive("provenance", "vendor-a-self")    # vendor A attesting its own evidence
K_UNREG = R.TestKey.derive("provenance", "not-registered")      # valid key, not in the registry
K_DECISION = R.TestKey.derive("decision", "k1")                 # a decision-domain key (misuse)
AUD, AUD_SRC, VENDOR_A = "org:lineage-auditor-1", "org:source-registry", "org:vendor-a"


def prov_entry(key, attester, types):
    return {"domain": "provenance", "public_key": key.public_hex, "epochs": [EPOCH],
            "attester_id": attester, "authorized_atom_types": sorted(types)}


TRUST = {
    "registry": {
        K_AUD.key_id: prov_entry(K_AUD, AUD, S.DEPENDENCY_TYPES),
        K_AUD_SRC.key_id: prov_entry(K_AUD_SRC, AUD_SRC, ["SOURCE"]),
        K_VENDOR_A.key_id: prov_entry(K_VENDOR_A, VENDOR_A, S.DEPENDENCY_TYPES),
        K_DECISION.key_id: {"domain": "decision", "public_key": K_DECISION.public_hex, "epochs": [EPOCH]},
    },
    "revoked_attestation_ids": [],
}

SHARED = J.SHARED_FEED
NONMAT = J.COMMON_NONMATERIAL
VA = ["SOURCE:vendor-a-registry", "DATABASE:vendor-a-db", "API_ENDPOINT:vendor-a/verify-v2"]
VB = ["SOURCE:vendor-b-bureau", "DATABASE:vendor-b-db", "API_ENDPOINT:vendor-b/confirm"]
VW = ["SOURCE:internal-kyc", "DATABASE:beneficiary-master", "API_ENDPOINT:internal/webhook-beneficiary"]
ALIASES = {"SOURCE:fcp-1": "SOURCE:feed-core-primary", "DATABASE:agg-master-db": "DATABASE:aggregator-master",
           "COLLECTOR:agg17": "COLLECTOR:aggregator-17", "RUNTIME:agg-rt": "RUNTIME:aggregator-rt"}
EXTRA = ["SOURCE:s1", "SOURCE:s2", "SOURCE:solo", "SOURCE:memo", "RUNTIME:m", "COLLECTOR:c-2", "COLLECTOR:c-3",
         "SOURCE:a", "SOURCE:b", "SOURCE:c", "SOURCE:reg"]
REGISTERED = sorted(set(SHARED + NONMAT + VA + VB + VW + EXTRA + CC.ATOM_POOL + [
    "API_ENDPOINT:vendor-a/verify-v2", "API_ENDPOINT:vendor-b/confirm", "API_ENDPOINT:internal/webhook-beneficiary"]),
    key=lambda s: s.encode("utf-8"))

PROFILE = {
    "id": C21.PROFILE_ID,
    "status": "experimental",
    **C21.COMPONENTS,
    "material_dependency_types": J.PROFILE["material_dependency_types"],
    "omega_bits": bits(1.0), "t_base_bits": bits(0.5), "beta_bits": bits(0.2156),
    "kappa_unknown_bits": bits(0.5), "kappa_degraded_bits": bits(0.5), "unknown_gate_t_bits": bits(0.92),
    "consequence_gamma_bits": J.PROFILE["consequence_gamma_bits"],
    "atom_registry": {"mode": "closed", "atoms": REGISTERED, "aliases": ALIASES},
    "max_atoms_per_type": {"API_ENDPOINT": 1, "CLOCK": 1, "COLLECTOR": 2, "DATABASE": 2, "NETWORK_AS": 1,
                           "RUNTIME": 2, "SOURCE": 2, "TRUST_ROOT": 1},
    "attester_independence_required": True,
}
# A naive registry: the alias names are registered as atoms in their own right (no naming authority).
PROFILE_NO_ALIASES = dict(PROFILE, atom_registry={"mode": "closed", "aliases": {},
                                                  "atoms": sorted(set(REGISTERED) | set(ALIASES),
                                                                  key=lambda s: s.encode("utf-8"))})


def attest(member, deps, key=K_AUD, attester=AUD, label=None, **kw):
    m = dict(member)
    m["provenance"] = PV.make_attestation(
        m, key, attester, deps, epoch=kw.pop("epoch", EPOCH),
        tau_issued=kw.pop("tau_issued", BASE_US - DAY), tau_expiry=kw.pop("tau_expiry", BASE_US + 30 * DAY),
        unix_ms=BASE_MS - 86_400_000,
        label=label or f"{m['evidence_id']}|{key.label}|{R.H(sorted(set(deps)))[:12]}", **kw)
    return m


def ev(eid, value, producer, parent=None, state=None, self_declared=None, **extra):
    e = {"evidence_id": eid, "parent_id": parent, "producer": producer,
         "value_bits": bits(value) if isinstance(value, float) else value}
    if state:
        e["state"] = state
    if self_declared is not None:
        e["dependencies"] = list(self_declared)
    e.update(extra)
    return e


fam = J.fam


def three(correlated: bool, partial=False, aliased_b=False):
    da = SHARED + ["API_ENDPOINT:vendor-a/verify-v2"] if (correlated or partial) else VA
    db = SHARED + ["API_ENDPOINT:vendor-b/confirm"] if (correlated or partial) else VB
    if aliased_b:
        db = [{v: k for k, v in ALIASES.items()}.get(a, a) for a in SHARED] + ["API_ENDPOINT:vendor-b/confirm"]
    dw = SHARED + ["API_ENDPOINT:internal/webhook-beneficiary"] if correlated else VW
    return [
        fam("F-vendor-a", attest(ev("ev-vendor-a", 0.95, VENDOR_A, attestation="sig:vendor-a:88192"), da + NONMAT)),
        fam("F-vendor-b", attest(ev("ev-vendor-b", 0.90, "org:vendor-b", attestation="sig:vendor-b:55120"), db + NONMAT)),
        fam("F-webhook", attest(ev("ev-internal-webhook", 0.92, "org:internal-kyc-service",
                                   attestation="sig:internal:30017"), dw + NONMAT)),
    ]


def reattest_rename(f_, new_fid):
    """Structural duplicate under fresh ids, each member re-attested by the same attester."""
    ids = {m["evidence_id"]: f"{m['evidence_id']}@{new_fid}" for m in f_["members"]}
    out = []
    for m in f_["members"]:
        n = {k: copy.deepcopy(v) for k, v in m.items() if k != "provenance"}
        n["evidence_id"] = ids[m["evidence_id"]]
        if n.get("parent_id") in ids:
            n["parent_id"] = ids[n["parent_id"]]
        if "provenance" in m:
            pl = m["provenance"]["payload"]
            key = {K_AUD.key_id: K_AUD, K_AUD_SRC.key_id: K_AUD_SRC, K_VENDOR_A.key_id: K_VENDOR_A}[pl["provenance_key_id"]]
            n = attest(n, pl["dependencies"], key=key, attester=pl["attester_id"])
        out.append(n)
    return {"family_id": new_fid, "members": out}


# ---------------------------------------------------------------------------
# Cases
# ---------------------------------------------------------------------------

def cases():
    C = []

    def add(cid, title, request, families, expect, checks=None, note=None, profile=PROFILE, trust=TRUST, **extra):
        C.append(dict(id=cid, title=title, request=request, families=families, expect=expect,
                      checks=checks or {}, note=note, profile=profile, trust=trust, **extra))

    ind, cor = three(False), three(True)
    # --- headline, now over attested lineage ---------------------------------------------------
    add("c03_three_independent_pass", "Wire: three independent confirmations, lineage attested by an auditor",
        WIRE, ind, "ALLOW")
    add("c04_three_correlated_fail", "Wire: same confirmations, attested lineage shows one shared feed", WIRE,
        cor, "DENY")
    add("c05_two_of_three_correlated", "Wire: vendors A and B share the feed (attested)", WIRE, three(False, partial=True), "DENY")
    add("c06_two_independent_insufficient", "Wire: two independent attested confirmations", WIRE,
        [x for x in ind if x["family_id"] != "F-vendor-b"], "DENY")
    add("c07_mirror_added_neutral", "Attested mirror of vendor A adds nothing", WIRE,
        ind + [fam("F-mirror", attest(ev("ev-vendor-a-mirror", 0.90, "org:vendor-a-mirror"), VA))], "ALLOW",
        {"same_M_as": "c03_three_independent_pass"})
    add("c08_bridge_added_neutral", "Attested bridging item cannot merge or dilute", WIRE,
        ind + [fam("F-bridge", attest(ev("ev-bridge", 0.10, "org:bridge"),
                                       ["SOURCE:vendor-a-registry", "SOURCE:vendor-b-bureau"]))], "ALLOW",
        {"same_M_as": "c03_three_independent_pass"})
    dup = ind + [reattest_rename(ind[0], "F-vendor-a-dup")]
    add("c09_reattested_duplicate_idempotent", "Re-attested structural duplicate leaves M bit-identical", WIRE,
        dup, "ALLOW", {"same_M_as": "c03_three_independent_pass"})
    base10 = [fam("F1", attest(ev("ev-1", 0.6, "org:p1"), ["SOURCE:s1"]))]
    add("c10a_refund_attested_only", "Refund: one attested family", REFUND, base10, "DENY", {"M": 0.6})
    add("c10b_unattested_low_consequence", "Refund (T < gate): an unattested item contributes V·κ once", REFUND,
        base10 + [fam("F-u", ev("ev-u", 0.9, "org:u"))], "ALLOW", {"M": 1.05})
    add("c11_unattested_spam_single_sink", "Twenty unattested items share one UNKNOWN sink", REFUND,
        base10 + [fam(f"F-u{i}", ev(f"ev-u{i}", 0.9, f"org:u{i}")) for i in range(20)], "ALLOW",
        {"same_M_as": "c10b_unattested_low_consequence"})
    perm_base = cor + [fam("F-memo", attest(ev("ev-memo-root", 0.7, "org:memo"), ["SOURCE:memo"]),
                           attest(ev("ev-memo-derived", 0.7, "org:memo", parent="ev-memo-root", state="DEGRADED"),
                                  ["RUNTIME:m", "SOURCE:memo"]))]
    perm = copy.deepcopy(perm_base)
    rng = CC.DRng("v021-perm")
    for i in range(len(perm) - 1, 0, -1):
        j = rng.below(i + 1)
        perm[i], perm[j] = perm[j], perm[i]
    for f_ in perm:
        f_["members"].reverse()
    add("c12_permutation_invariance", "Shuffled families and members: identical output bits", WIRE, perm, "DENY",
        {"same_trace_as": perm_base})
    eqb, belowb = boundary_values()
    add("c13_boundary_c_equals_t", "C == T exactly must ALLOW", REFUND,
        [fam("F-solo", attest(ev("ev-solo", eqb, "org:solo"), ["SOURCE:solo"]))], "ALLOW", {"C_equals_T": True})
    add("c14_boundary_one_ulp_below", "C one ulp below T must DENY", REFUND,
        [fam("F-solo", attest(ev("ev-solo", belowb, "org:solo"), ["SOURCE:solo"]))], "DENY", {"C_ulps_below_T": 1})
    add("e01_engine_guard_nan", "Engine guard still runs first (NaN value)", WIRE,
        [fam("F1", ev("ev-1", "7ff8000000000000", "org:x"))], "DENY", {"stage": "ENGINE_GUARD"})

    # --- the atom-padding family: all MUST DENY / be neutral ---------------------------------
    pads = [fam(f"F-pad-{i}", ev(f"ev-pad-{i}", 0.95, f"org:pad-{i}",
                                 self_declared=SHARED + ["API_ENDPOINT:vendor-a/verify-v2"] +
                                 [f"COLLECTOR:fabricated-{i}-{k:03d}" for k in range(200)])) for i in (1, 2)]
    add("x01_self_declared_padding", "x01 (was the open exploit): self-declared padded lineage is ignored", WIRE,
        cor + pads, "DENY", {"same_M_as": "c04_three_correlated_fail", "outcome": ("F-pad-1", "ev-pad-1", "UNATTESTED")},
        note="Under coverage-v0.2.0-alpha this input produced ALLOW (C = 0.958). P-1 maps it to UNKNOWN:*; κ = 0 on a wire.")
    fab = [f"COLLECTOR:fabricated-{k:03d}" for k in range(200)]
    add("x02_signed_padding_unregistered", "Auditor-signed attestation containing fabricated atoms", WIRE,
        cor + [fam("F-pad", attest(ev("ev-pad", 0.95, "org:pad"), SHARED + fab))], "DENY",
        {"same_M_as": "c04_three_correlated_fail", "outcome": ("F-pad", "ev-pad", "INADMISSIBLE")},
        note="The closed atom registry rejects atoms no naming authority has registered.")
    add("x03_signed_padding_cap", "Auditor-signed attestation with registered atoms above the per-type cap", WIRE,
        cor + [fam("F-pad", attest(ev("ev-pad", 0.95, "org:pad"),
                                    SHARED + ["COLLECTOR:c-2", "COLLECTOR:c-3", "API_ENDPOINT:vendor-a/verify-v2"]))],
        "DENY", {"same_M_as": "c04_three_correlated_fail", "outcome": ("F-pad", "ev-pad", "INADMISSIBLE")})
    add("x04_self_attested", "Vendor A attests its own evidence's lineage (attester = producer)", WIRE,
        [fam("F-vendor-a", attest(ev("ev-vendor-a", 0.95, VENDOR_A), VA, key=K_VENDOR_A, attester=VENDOR_A))] + ind[1:],
        "DENY", {"outcome": ("F-vendor-a", "ev-vendor-a", "INADMISSIBLE")},
        note="Signed self-declaration is accountable but not verification; with κ = 0 vendor A contributes nothing.")
    lifted = attest(ev("ev-vendor-a", 0.95, VENDOR_A, attestation="sig:vendor-a:88192"), VA + NONMAT)["provenance"]
    padded = ev("ev-pad", 0.95, "org:pad")
    padded["provenance"] = copy.deepcopy(lifted)
    add("x05_attestation_lifted", "Vendor A's valid attestation copied onto another evidence item", WIRE,
        cor + [fam("F-pad", padded)], "DENY",
        {"same_M_as": "c04_three_correlated_fail", "outcome": ("F-pad", "ev-pad", "TAMPERED"),
         "reasons_include": ["PROVENANCE_TAMPERED", "FAMILY_INVALIDATED"]})
    add("x06_decision_key_misuse", "Attestation signed with a decision-domain key", WIRE,
        cor + [fam("F-pad", attest(ev("ev-pad", 0.95, "org:pad"), VA, key=K_DECISION, attester=AUD))], "DENY",
        {"same_M_as": "c04_three_correlated_fail", "outcome": ("F-pad", "ev-pad", "TAMPERED")})
    add("x07_alias_resolves_correlation", "Vendor B names the shared feed by aliases; the alias table resolves them",
        WIRE, three(False, partial=True, aliased_b=True), "DENY", {"same_M_as": "c05_two_of_three_correlated"})
    add("x08_alias_table_absent", "Same input with a registry that has no naming authority: correlation hidden",
        WIRE, three(False, partial=True, aliased_b=True), "ALLOW", profile=PROFILE_NO_ALIASES,
        note="Why canonical atom identity matters: without the alias table the shared feed looks like two sources "
             "and the wire is ALLOWED. The vector is kept to pin the risk, not as acceptable behaviour.")
    add("x09_expired_attestation", "Attestation expired before tau_eval", WIRE,
        [fam("F-vendor-a", attest(ev("ev-vendor-a", 0.95, VENDOR_A), VA, tau_expiry=BASE_US - 1))] + ind[1:],
        "DENY", {"outcome": ("F-vendor-a", "ev-vendor-a", "INADMISSIBLE")})
    rev = attest(ev("ev-vendor-a", 0.95, VENDOR_A), VA)
    add("x10_revoked_attestation", "Attestation listed as revoked in the trust state", WIRE,
        [fam("F-vendor-a", rev)] + ind[1:], "DENY", {"outcome": ("F-vendor-a", "ev-vendor-a", "INADMISSIBLE")},
        trust=dict(TRUST, revoked_attestation_ids=[rev["provenance"]["payload"]["attestation_id"]]))
    add("x11_unregistered_attester", "Attestation from a key that is not in the registry", WIRE,
        [fam("F-vendor-a", attest(ev("ev-vendor-a", 0.95, VENDOR_A), VA, key=K_UNREG, attester="org:unknown"))] + ind[1:],
        "DENY", {"outcome": ("F-vendor-a", "ev-vendor-a", "INADMISSIBLE")})
    add("x12_attester_type_not_authorised", "SOURCE-only attester vouches for DATABASE and API atoms", WIRE,
        [fam("F-vendor-a", attest(ev("ev-vendor-a", 0.95, VENDOR_A), VA, key=K_AUD_SRC, attester=AUD_SRC))] + ind[1:],
        "DENY", {"outcome": ("F-vendor-a", "ev-vendor-a", "INADMISSIBLE")})
    add("x13_attester_id_forged", "Payload claims the auditor's identity but is signed by vendor A's key", WIRE,
        [fam("F-vendor-a", attest(ev("ev-vendor-a", 0.95, VENDOR_A), VA, key=K_VENDOR_A, attester=AUD))] + ind[1:],
        "DENY", {"outcome": ("F-vendor-a", "ev-vendor-a", "TAMPERED")})
    add("x14_value_changed_after_attestation", "Score raised after the attestation was issued", WIRE,
        cor[:2] + [fam("F-webhook", dict(cor[2]["members"][0], value_bits=bits(1.0)))], "DENY",
        {"outcome": ("F-webhook", "ev-internal-webhook", "TAMPERED")},
        note="evidence_digest covers value_bits, so the attested score cannot be edited.")

    # --- defeater channel ----------------------------------------------------------------------
    four = ind + [fam("F-registry", attest(ev("ev-registry", 0.9, "org:gov-registry"), ["SOURCE:reg"]))]
    contra = ev("ev-webhook-contra", 0.9, "org:internal-kyc-service", parent="ev-internal-webhook",
                state="CONTRADICTORY", finding="beneficiary name mismatch")
    web_with = lambda m: fam("F-webhook", ind[2]["members"][0], m)
    add("d00_four_independent_pass", "Wire: four independent attested confirmations (baseline)", WIRE, four, "ALLOW")
    add("d01_unverified_contradiction", "Unattested contradiction only zeroes its own family", WIRE,
        [f_ for f_ in four if f_["family_id"] != "F-webhook"] + [web_with(contra)], "ALLOW",
        {"defeaters": 0},
        note="§2.2 behaviour: A(F-webhook) = 0, the other three still carry C above T. Audited as UNVERIFIED_CONTRADICTION.")
    add("d02_authenticated_defeater_vetoes", "Auditor-attested contradiction vetoes an otherwise passing wire", WIRE,
        [f_ for f_ in four if f_["family_id"] != "F-webhook"] + [web_with(attest(contra, ["SOURCE:internal-kyc"]))],
        "DENY", {"defeaters": 1, "reasons_include": ["DEFEATER_PRESENT"]},
        note="Categorical veto, not a negative mass term: M and C are reported unchanged from d01.")
    bad_sig = attest(contra, ["SOURCE:internal-kyc"])
    bad_sig["provenance"]["signature"] = "00" * 64
    add("d03_tampered_defeater", "A contradiction with a forged attestation is tampering, not a veto", WIRE,
        [f_ for f_ in four if f_["family_id"] != "F-webhook"] + [web_with(bad_sig)], "ALLOW",
        {"defeaters": 0, "reasons_include": ["PROVENANCE_TAMPERED"]},
        note="The webhook family is invalidated; the remaining three families still pass.")
    return C


def boundary_values():
    def one(vb):
        return ENG.evaluate(REFUND, [fam("F-solo", attest(ev("ev-solo", vb, "org:solo"), ["SOURCE:solo"]))], PROFILE, TRUST)
    T = S.f(one(bits(0.5))["T_bits"])
    lob, hib = R.bits_of(0.0), R.bits_of(1.0)
    while hib - lob > 1:
        mid = (lob + hib) // 2
        if S.f(one(f"{mid:016x}")["C_bits"]) >= T:
            hib = mid
        else:
            lob = mid
    target = struct.unpack(">d", struct.pack(">Q", R.bits_of(T) - 1))[0]
    b = hib
    while S.f(one(f"{b:016x}")["C_bits"]) > target:
        b -= 1
    if S.f(one(f"{hib:016x}")["C_bits"]) != T or S.f(one(f"{b:016x}")["C_bits"]) != target:
        raise SystemExit("v0.2.1 boundary search failed")
    return f"{hib:016x}", f"{b:016x}"


# ---------------------------------------------------------------------------
# Randomised generators shared with tools/property_tests.py
# ---------------------------------------------------------------------------
POOL = CC.ATOM_POOL


def random_attested_family(rng, fid, attested_p=0.8):
    n = 1 + rng.below(4)
    deps = sorted({rng.pick(POOL) for _ in range(n)}, key=lambda s: s.encode("utf-8"))
    root = ev(f"ev-{fid}", rng.unit(), f"org:{fid}")
    members = [attest(root, deps) if rng.below(10) < attested_p * 10 else root]
    if rng.below(3) == 0:
        d = ev(f"ev-{fid}-d", rng.unit(), f"org:{fid}", parent=f"ev-{fid}",
               state=rng.pick(["VALID", "DEGRADED", "UNAVAILABLE_OR_STALE"]))
        members.append(attest(d, deps[:1]))
    return fam(fid, *members)


def random_defeater_family(rng, fid):
    root = attest(ev(f"ev-{fid}", rng.unit(), f"org:{fid}"), [rng.pick(POOL)])
    contra = attest(ev(f"ev-{fid}-x", rng.unit(), f"org:{fid}", parent=f"ev-{fid}", state="CONTRADICTORY"),
                    [rng.pick(POOL)])
    return fam(fid, root, contra)


def witnesses(n=18):
    rng = CC.DRng("v021-witness")
    out = []
    kinds = ["monotone", "idempotent", "defeater"]
    for i in range(n):
        kind = kinds[i % 3]
        base = [random_attested_family(rng, f"W{i}-{k}") for k in range(1 + rng.below(4))]
        rq = rng.pick([WIRE, REFUND, ANNOTATE])
        added = {"monotone": lambda: random_attested_family(rng, f"W{i}-new"),
                 "idempotent": lambda: reattest_rename(rng.pick(base), f"W{i}-dup"),
                 "defeater": lambda: random_defeater_family(rng, f"W{i}-def")}[kind]()
        t0 = ENG.evaluate(rq, base, PROFILE, TRUST)
        t1 = ENG.evaluate(rq, base + [added], PROFILE, TRUST)
        ok = {"monotone": S.f(t1["M_bits"]) >= S.f(t0["M_bits"]) and not (t0["decision"] == "ALLOW" and t1["decision"] == "DENY"),
              "idempotent": t1["M_bits"] == t0["M_bits"],
              "defeater": t1["decision"] == "DENY"}[kind]
        if not ok:
            raise SystemExit(f"v0.2.1 witness {i} ({kind}) violated")
        out.append({"kind": kind, "request": rq, "base": base, "added": added,
                    "M_base_bits": t0["M_bits"], "M_after_bits": t1["M_bits"],
                    "decision_base": t0["decision"], "decision_after": t1["decision"]})
    return out


# ---------------------------------------------------------------------------
# Build
# ---------------------------------------------------------------------------

def build():
    out, by_id = [], {}
    for c in cases():
        tr = ENG.evaluate(c["request"], c["families"], c["profile"], c["trust"])
        cid, ck = c["id"], c["checks"]
        if tr["decision"] != c["expect"]:
            raise SystemExit(f"v0.2.1 {cid}: engine says {tr['decision']} {tr['reason_codes']}, declared {c['expect']}")
        if "M" in ck and S.f(tr["M_bits"]) != ck["M"]:
            raise SystemExit(f"v0.2.1 {cid}: M = {S.f(tr['M_bits'])}")
        if "same_M_as" in ck and tr["M_bits"] != by_id[ck["same_M_as"]]["M_bits"]:
            raise SystemExit(f"v0.2.1 {cid}: M {S.f(tr['M_bits'])} != {ck['same_M_as']} {S.f(by_id[ck['same_M_as']]['M_bits'])}")
        if "same_trace_as" in ck and R.encode(ENG.evaluate(c["request"], ck["same_trace_as"], c["profile"], c["trust"])) != R.encode(tr):
            raise SystemExit(f"v0.2.1 {cid}: trace differs from its base")
        if "outcome" in ck:
            fid, eid, want = ck["outcome"]
            got = tr["families"][fid]["provenance"][eid]["outcome"]
            if got != want:
                raise SystemExit(f"v0.2.1 {cid}: provenance outcome {got} != {want}")
        if "defeaters" in ck and len(tr["defeaters"]) != ck["defeaters"]:
            raise SystemExit(f"v0.2.1 {cid}: defeaters {tr['defeaters']}")
        for rsn in ck.get("reasons_include", []):
            if rsn not in tr["reason_codes"]:
                raise SystemExit(f"v0.2.1 {cid}: missing reason {rsn}")
        if "stage" in ck and tr["stage"] != ck["stage"]:
            raise SystemExit(f"v0.2.1 {cid}: stage")
        if ck.get("C_equals_T") and tr["C_bits"] != tr["T_bits"]:
            raise SystemExit(f"v0.2.1 {cid}: C != T")
        if "C_ulps_below_T" in ck and int(tr["T_bits"], 16) - int(tr["C_bits"], 16) != ck["C_ulps_below_T"]:
            raise SystemExit(f"v0.2.1 {cid}: ulp")
        by_id[cid] = tr
        v = {"id": cid, "title": c["title"],
             "profile": "calibrated" if c["profile"] is PROFILE else "no-alias-registry",
             "trust": "default" if c["trust"] is TRUST else "variant",
             "input": {"request": c["request"], "families": c["families"]},
             "expected_trace": tr,
             "summary": {"decision": tr["decision"], "stage": tr["stage"], "M": repr(S.f(tr["M_bits"])),
                         "C": repr(S.f(tr["C_bits"])), "reasons": tr["reason_codes"]}}
        if c["trust"] is not TRUST:
            v["trust_state"] = c["trust"]
        if c.get("note"):
            v["note"] = c["note"]
        out.append(v)
    # the d02 veto must leave M and C exactly as in d01 (categorical, not arithmetic)
    if (by_id["d01_unverified_contradiction"]["M_bits"], by_id["d01_unverified_contradiction"]["C_bits"]) != \
            (by_id["d02_authenticated_defeater_vetoes"]["M_bits"], by_id["d02_authenticated_defeater_vetoes"]["C_bits"]):
        raise SystemExit("defeater changed M/C — it must be a categorical veto only")
    return {
        "vector_set": "scoring-profile",
        "scoring_profile": C21.PROFILE_ID,
        "status": "SUPERSEDED by coverage-v0.3.0-alpha (kept as evidence; contradictions still zero A here)",
        "spec_version": R.SPEC_VERSION,
        "assumptions": "docs/SCORING_PROFILE_coverage-v0.2.1-alpha.md",
        "WARNING": "Test keys are derived from public seeds. Never use them outside conformance testing.",
        "profiles": {"calibrated": {"declaration": PROFILE, "fingerprint": R.H(PROFILE)},
                     "no-alias-registry": {"declaration": PROFILE_NO_ALIASES, "fingerprint": R.H(PROFILE_NO_ALIASES)}},
        "trust_state": TRUST,
        "provenance_keys": [{"label": k.label, "seed": k.seed.hex(), "public_key": k.public_hex, "key_id": k.key_id,
                             "domain": k.domain} for k in (K_AUD, K_AUD_SRC, K_VENDOR_A, K_UNREG)],
        "known_exploits_open": [],
        "property_witnesses": witnesses(),
        "cases": out,
    }


### The committed trust state and scoring profile (0.3.0) — `tools/trust_fixture.py`

In [15]:
%%writefile tools/trust_fixture.py
"""
The single evaluation trust context used by the 0.3.0-alpha vectors (TEST KEYS ONLY — public seeds).

    TRUST_STATE            canonical object; its hash is bound into d.trust_state_hash and p.trust_state_hash
    SCORING_PROFILE_V030   coverage-v0.3.0-alpha arithmetic; its hash is d.scoring_profile_hash

Ownership of rules (each rule has exactly one committing object):
    control profile   ManualOnly, Δτ_max, reusable authority classes          -> d.control_profile_hash
    scoring profile   Ω, T_base, T_base_manual, β, κ, gate, Γ, material types -> d.scoring_profile_hash
    trust state       key registry (all six domains, owners, epochs, scopes, authorised atom types),
                      revocations, atom registry + aliases, per-type caps, attester independence,
                      conflict types, clock authorities                        -> d.trust_state_hash
"""
from __future__ import annotations

import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "reference"))
import prc_ref as R              # noqa: E402
import prc_scoring as S          # noqa: E402
import prc_coverage_v030 as C30  # noqa: E402
import coverage_v021_cases as V21  # noqa: E402  (atom registry and aliases carried forward)
import scoring_cases as J        # noqa: E402

EPOCH = "epoch-2026-01"
bits = J.bits

# --- keys (same seeds as earlier vector sets, plus the membership domain) -----------------
K_PROFILE = R.TestKey.derive("profile", "k1")
K_APPROVAL = R.TestKey.derive("approval", "k1")
K_DECISION = R.TestKey.derive("decision", "k1")
K_RECEIPT = R.TestKey.derive("receipt", "k1")
K_AUD, K_AUD_SRC, K_VENDOR_A, K_UNREG = V21.K_AUD, V21.K_AUD_SRC, V21.K_VENDOR_A, V21.K_UNREG
K_MEMB = R.TestKey.derive("membership", "ingest-gateway-1")        # scope F-*
K_MEMB_PARTNER = R.TestKey.derive("membership", "partner-gateway")  # scope F-partner-* only
K_MEMB_UNREG = R.TestKey.derive("membership", "not-registered")
AUD, AUD_SRC, VENDOR_A = V21.AUD, V21.AUD_SRC, V21.VENDOR_A
GATEWAY, PARTNER = "org:ingest-gateway", "org:partner-gateway"


def entry(key, owner, **extra):
    e = {"domain": key.domain, "public_key": key.public_hex, "epochs": [EPOCH], "owner": owner}
    e.update(extra)
    return e


CLOCK_IDEAL = "clk:test-ideal"      # zero uncertainty — keeps the pre-0.3.0 boundary vectors exact
CLOCK_PTP = "clk:exec-ptp"          # 1 ms
CLOCK_NTP = "clk:ntp-pool"          # 500 ms

TRUST_STATE = {
    "trust_state_id": "ts-2026-01",
    "schema_ver": R.SPEC_VERSION,
    "key_registry": {
        K_PROFILE.key_id: entry(K_PROFILE, "org:policy-office"),
        K_APPROVAL.key_id: entry(K_APPROVAL, "org:treasury-approvals", authority_classes=["STANDING", "SUPERVISOR"]),
        K_DECISION.key_id: entry(K_DECISION, "org:pdp"),
        K_RECEIPT.key_id: entry(K_RECEIPT, "org:executor"),
        K_AUD.key_id: entry(K_AUD, AUD, authorized_atom_types=sorted(S.DEPENDENCY_TYPES)),
        K_AUD_SRC.key_id: entry(K_AUD_SRC, AUD_SRC, authorized_atom_types=["SOURCE"]),
        K_VENDOR_A.key_id: entry(K_VENDOR_A, VENDOR_A, authorized_atom_types=sorted(S.DEPENDENCY_TYPES)),
        K_MEMB.key_id: entry(K_MEMB, GATEWAY, membership_scopes=["F-*"]),
        K_MEMB_PARTNER.key_id: entry(K_MEMB_PARTNER, PARTNER, membership_scopes=["F-partner-*"]),
    },
    "revoked_attestation_ids": [],
    "revoked_manifest_ids": [],
    "atom_registry": {"mode": "closed", "atoms": V21.REGISTERED, "aliases": V21.ALIASES},
    "max_atoms_per_type": V21.PROFILE["max_atoms_per_type"],
    "attester_independence_required": True,
    "conflict_types": ["AMOUNT_MISMATCH", "BENEFICIARY_NAME_MISMATCH", "SANCTIONS_HIT", "SOURCE_RETRACTION"],
    "clock_authorities": {
        CLOCK_IDEAL: {"max_uncertainty_us": 0},
        CLOCK_PTP: {"max_uncertainty_us": 1_000},
        CLOCK_NTP: {"max_uncertainty_us": 500_000},
    },
}
TRUST_STATE_HASH = R.H(TRUST_STATE)

SCORING_PROFILE_V030 = {
    "id": C30.PROFILE_ID,
    "status": "experimental",
    **C30.COMPONENTS,
    "material_dependency_types": J.PROFILE["material_dependency_types"],
    "omega_bits": bits(1.0), "t_base_bits": bits(0.5), "t_base_manual_bits": bits(0.5), "beta_bits": bits(0.2156),
    "kappa_unknown_bits": bits(0.5), "kappa_degraded_bits": bits(0.5), "unknown_gate_t_bits": bits(0.92),
    "consequence_gamma_bits": J.PROFILE["consequence_gamma_bits"],
}
SCORING_PROFILE_HASH = R.H(SCORING_PROFILE_V030)


### v0.3.0 cases: containment, membership injection, carry-overs — `tools/coverage_v030_cases.py`

In [16]:
%%writefile tools/coverage_v030_cases.py
"""
Vectors for scoring profile `coverage-v0.3.0-alpha` (EXPERIMENTAL):
  A-1 adverse disposition (CLEAR / CONTAIN / DEFEAT / LOCK) with the MANUAL_REQUIRED outcome,
  M-1 family-membership admissibility via signed family manifests, and the P-1 carry-overs.

Every case declares its intended decision (and key invariants); build() aborts on disagreement.
"""
from __future__ import annotations

import copy
import os
import struct
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "reference"))
import prc_ref as R              # noqa: E402
import prc_scoring as S          # noqa: E402
import prc_provenance as PV      # noqa: E402
import prc_membership as PM      # noqa: E402
import prc_engine as ENG         # noqa: E402
import coverage_cases as CC      # noqa: E402
import coverage_v021_cases as V  # noqa: E402
import trust_fixture as TF       # noqa: E402

bits = V.bits
BASE_MS, BASE_US, DAY = V.BASE_MS, V.BASE_US, V.DAY
TS, PROFILE = TF.TRUST_STATE, TF.SCORING_PROFILE_V030


def req(action, assertion, target, snap):
    return {"action": action, "assertion": assertion, "target": target, "evidence_snapshot_id": snap, "tau_eval": BASE_US}


WIRE = req("wire.release", "beneficiary-verified", "acct-9921", "snap-wire-001")
REFUND = req("payment.refund", "refund-eligible", "acct-000123", "snap-refund-001")
ANNOTATE = req("ledger.annotate", "memo-accurate", "acct-000123", "snap-annotate-001")
BIND_WIRE = {k: WIRE[k] for k in ("action", "assertion", "target", "evidence_snapshot_id")}
ev, attest = V.ev, V.attest


def manifest_for(family, key=TF.K_MEMB, authority=TF.GATEWAY, **kw):
    return PM.make_manifest(family, key, authority, epoch=TF.EPOCH,
                            tau_issued=kw.pop("tau_issued", BASE_US - DAY), tau_expiry=kw.pop("tau_expiry", BASE_US + 30 * DAY),
                            unix_ms=BASE_MS - 86_400_000, **kw)


def fam(fid, *members, manifest=True, **mkw):
    f = {"family_id": fid, "members": list(members)}
    if manifest:
        f["manifest"] = manifest_for(f, **mkw)
    return f


def inject(f, *members):
    """Append members AFTER the manifest was issued (they are not listed)."""
    g = copy.deepcopy(f)
    g["members"] += list(members)
    return g


def contra(eid, parent, producer="org:internal-kyc-service", conflict="BENEFICIARY_NAME_MISMATCH",
           claim="beneficiary name on file differs from payment instruction", binding=None, **extra):
    return ev(eid, 0.9, producer, parent=parent, state="CONTRADICTORY", conflict_type=conflict,
              contradicts=binding or BIND_WIRE, claim=claim, **extra)


def three(correlated=False):
    out = []
    for f in V.three(correlated):
        out.append(fam(f["family_id"], *f["members"]))
    return out


def four():
    return three() + [fam("F-registry", attest(ev("ev-registry", 0.9, "org:gov-registry"), ["SOURCE:reg"]))]


def with_member(families, fid, member, listed=True):
    out = []
    for f in families:
        if f["family_id"] != fid:
            out.append(f)
            continue
        base = {"family_id": fid, "members": f["members"] + [member]}
        out.append(fam(fid, *base["members"]) if listed else inject(f, member))
    return out


# ---------------------------------------------------------------------------
# Cases
# ---------------------------------------------------------------------------

def cases():
    C = []

    def add(cid, title, request, families, expect, checks=None, note=None, trust=TS, **extra):
        C.append(dict(id=cid, title=title, request=request, families=families, expect=expect,
                      checks=checks or {}, note=note, trust=trust, **extra))

    add("c01_three_independent_pass", "Wire: three independent attested confirmations with manifests", WIRE, three(), "ALLOW")
    add("c02_three_correlated_fail", "Wire: same confirmations, one shared upstream feed", WIRE, three(True), "DENY")
    add("c03_two_independent_insufficient", "Wire: two independent confirmations", WIRE,
        [f for f in three() if f["family_id"] != "F-vendor-b"], "DENY")

    # --- A-1 adverse disposition -----------------------------------------------------------------
    base = four()
    add("a00_four_independent_pass", "Wire: four independent confirmations (baseline)", WIRE, base, "ALLOW",
        {"disposition": "CLEAR"})
    add("a01_admitted_contradiction_defeats", "Auditor-attested relevant contradiction -> DEFEAT", WIRE,
        with_member(base, "F-webhook", attest(contra("ev-contra", "ev-internal-webhook"), ["SOURCE:internal-kyc"])),
        "DENY", {"disposition": "DEFEAT", "same_MC_as": "a00_four_independent_pass"})
    add("a02_unattested_contradiction_contains", "Unattested relevant contradiction -> CONTAIN -> MANUAL_REQUIRED",
        WIRE, with_member(base, "F-webhook", contra("ev-contra", "ev-internal-webhook")), "MANUAL_REQUIRED",
        {"disposition": "CONTAIN", "same_MC_as": "a00_four_independent_pass", "obligations": 1},
        note="The contradiction changes neither M, C nor T; it removes eligibility for automatic authorization.")
    add("a03_inadmissible_contradiction_contains", "Self-attested (inadmissible) contradiction -> CONTAIN", WIRE,
        with_member(base, "F-webhook", attest(contra("ev-contra", "ev-internal-webhook", producer=TF.VENDOR_A),
                                              ["SOURCE:vendor-a-registry"], key=TF.K_VENDOR_A, attester=TF.VENDOR_A)),
        "MANUAL_REQUIRED", {"disposition": "CONTAIN", "same_MC_as": "a00_four_independent_pass"})
    bad = attest(contra("ev-contra", "ev-internal-webhook"), ["SOURCE:internal-kyc"])
    bad["provenance"]["signature"] = "00" * 64
    add("a04_tampered_contradiction_locks", "Listed contradiction with a forged attestation -> LOCK", WIRE,
        with_member(base, "F-webhook", bad), "DENY", {"disposition": "LOCK", "reasons_include": ["SAFE_STATE_LOCK"]})
    two = [f for f in three() if f["family_id"] != "F-vendor-b"]
    add("a05_contain_below_manual_threshold", "CONTAIN but C < T_manual -> DENY", WIRE,
        with_member(two, "F-webhook", contra("ev-contra", "ev-internal-webhook")), "DENY",
        {"disposition": "CONTAIN", "reasons_include": ["ADVERSE_CONTAINMENT", "THRESHOLD_NOT_MET"]})
    dup2 = with_member(with_member(base, "F-webhook", contra("ev-contra", "ev-internal-webhook")),
                       "F-vendor-b", contra("ev-contra-b", "ev-vendor-b", producer="org:vendor-b"))
    add("a06_same_claim_two_families_one_obligation", "The same adverse claim in two families is one obligation",
        WIRE, dup2, "MANUAL_REQUIRED", {"obligations": 1, "occurrences": 2})
    many = base[:2] + [fam("F-webhook", base[2]["members"][0],
                           *[contra(f"ev-contra-{i:02d}", "ev-internal-webhook") for i in range(20)])] + base[3:]
    add("a07_twenty_identical_claims_one_obligation", "Twenty identical submissions create one review obligation",
        WIRE, many, "MANUAL_REQUIRED", {"obligations": 1, "occurrences": 20,
                                        "same_containment_as": "a02_unattested_contradiction_contains"})
    add("a08_contradiction_other_target_irrelevant", "Contradiction bound to a different target has no effect", WIRE,
        with_member(base, "F-webhook", contra("ev-contra", "ev-internal-webhook", binding=dict(BIND_WIRE, target="acct-1"))),
        "ALLOW", {"disposition": "CLEAR", "same_MC_as": "a00_four_independent_pass"})
    add("a09_unrecognised_conflict_type_irrelevant", "Unrecognised conflict_type has no effect", WIRE,
        with_member(base, "F-webhook", contra("ev-contra", "ev-internal-webhook", conflict="VIBES")), "ALLOW",
        {"disposition": "CLEAR"})
    root_contra = fam("F-tip", contra("ev-tip", None, producer="org:tipline"))
    add("a10_contradicting_root", "A family whose root is an unattested contradiction: no support, CONTAIN", WIRE,
        base + [root_contra], "MANUAL_REQUIRED", {"disposition": "CONTAIN", "same_MC_as": "a00_four_independent_pass"})
    both = with_member(with_member(base, "F-webhook", contra("ev-contra", "ev-internal-webhook")),
                       "F-registry", attest(contra("ev-contra-r", "ev-registry", producer="org:gov-registry",
                                                   conflict="SANCTIONS_HIT", claim="beneficiary on sanctions list"),
                                            ["SOURCE:reg"]))
    add("a11_defeat_outranks_contain", "DEFEAT and CONTAIN together: DEFEAT wins", WIRE, both, "DENY",
        {"disposition": "DEFEAT", "obligations": 2})
    add("a12_contradiction_no_longer_zeroes_family", "Three independent + unattested contradiction: M unchanged (was zeroed in 0.2.x)",
        WIRE, with_member(three(), "F-webhook", contra("ev-contra", "ev-internal-webhook")), "MANUAL_REQUIRED",
        {"same_MC_as": "c01_three_independent_pass"},
        note="coverage-v0.2.1-alpha set A(F-webhook) = 0 for this input (M 2.77 -> 1.85, DENY).")

    # --- M-1 family membership -------------------------------------------------------------------
    ind = three()
    forged = ev("ev-forged", 1.0, "org:attacker", parent="ev-vendor-a", state="INVALID")
    add("m01_injected_invalid_member_quarantined", "Injected INVALID member cannot invalidate vendor A's family", WIRE,
        [inject(ind[0], forged)] + ind[1:], "ALLOW", {"same_MC_as": "c01_three_independent_pass", "quarantined": 1},
        note="In 0.2.x any INVALID member invalidated its family: a one-item denial of service.")
    add("m02_injected_second_root_quarantined", "Injected second root cannot make the family malformed", WIRE,
        [inject(ind[0], ev("ev-rogue-root", 0.99, "org:attacker"))] + ind[1:], "ALLOW",
        {"same_MC_as": "c01_three_independent_pass"})
    add("m03_injected_unattested_contradiction", "Injected unattested contradiction: no containment", WIRE,
        [inject(ind[0], contra("ev-inj", "ev-vendor-a", producer="org:attacker"))] + ind[1:], "ALLOW",
        {"disposition": "CLEAR"})
    add("m04_injected_attested_contradiction", "Injected contradiction with a valid auditor attestation: still quarantined",
        WIRE, [inject(ind[0], attest(contra("ev-inj", "ev-vendor-a", producer="org:attacker"), ["SOURCE:reg"]))] + ind[1:],
        "ALLOW", {"disposition": "CLEAR"},
        note="Adverse evidence must enter through a membership authority; a valid attestation alone does not place it in a family.")
    badinj = attest(contra("ev-inj", "ev-vendor-a", producer="org:attacker"), ["SOURCE:reg"])
    badinj["provenance"]["signature"] = "11" * 64
    add("m05_injected_tampered_contradiction", "Injected forged contradiction cannot trigger LOCK", WIRE,
        [inject(ind[0], badinj)] + ind[1:], "ALLOW", {"disposition": "CLEAR"})
    impostor = ev("ev-vendor-a", 0.10, "org:attacker")
    add("m06_same_id_impostor", "Impostor reusing a listed evidence_id is quarantined; the listed item is used", WIRE,
        [{"family_id": ind[0]["family_id"], "members": [impostor] + ind[0]["members"], "manifest": ind[0]["manifest"]}]
        + ind[1:], "ALLOW", {"same_MC_as": "c01_three_independent_pass"})
    add("m07_no_manifest", "Family without a manifest is excluded", WIRE,
        [fam(ind[0]["family_id"], *ind[0]["members"], manifest=False)] + ind[1:], "DENY",
        {"family_status": ("F-vendor-a", "MEMBERSHIP_UNVERIFIED")})
    add("m08_manifest_out_of_scope", "Partner gateway's scope (F-partner-*) does not cover F-vendor-a", WIRE,
        [fam(ind[0]["family_id"], *ind[0]["members"], key=TF.K_MEMB_PARTNER, authority=TF.PARTNER)] + ind[1:], "DENY",
        {"family_status": ("F-vendor-a", "MEMBERSHIP_UNVERIFIED")})
    add("m09_manifest_unregistered_key", "Manifest from a key not in the trust state", WIRE,
        [fam(ind[0]["family_id"], *ind[0]["members"], key=TF.K_MEMB_UNREG, authority="org:unknown")] + ind[1:], "DENY",
        {"family_status": ("F-vendor-a", "MEMBERSHIP_UNVERIFIED")})
    add("m10_manifest_wrong_key_domain", "Manifest signed with a provenance-domain key", WIRE,
        [fam(ind[0]["family_id"], *ind[0]["members"], key=TF.K_AUD, authority=TF.AUD)] + ind[1:], "DENY",
        {"family_status": ("F-vendor-a", "INVALIDATED"), "reasons_include": ["MEMBERSHIP_TAMPERED"]})
    replay = dict(ind[0], family_id="F-vendor-a-copy")
    add("m11_manifest_replayed_onto_other_family", "Vendor A's manifest attached to a different family id", WIRE,
        [replay] + ind[1:], "DENY", {"family_status": ("F-vendor-a-copy", "INVALIDATED")})
    altered = copy.deepcopy(ind[0])
    altered["members"][0]["value_bits"] = bits(1.0)
    add("m12_listed_member_altered", "Listed member edited after the manifest was issued", WIRE,
        [altered] + ind[1:], "DENY", {"family_status": ("F-vendor-a", "INVALIDATED"), "reasons_include": ["MEMBERSHIP_TAMPERED"]})
    listed_c = with_member(base, "F-webhook", contra("ev-contra", "ev-internal-webhook"))
    withheld = [dict(f, members=[m for m in f["members"] if m["evidence_id"] != "ev-contra"]) if f["family_id"] == "F-webhook" else f
                for f in listed_c]
    add("m13_listed_contradiction_withheld", "Submitter drops a listed contradiction to avoid containment", WIRE,
        withheld, "ALLOW", {"family_status": ("F-webhook", "MEMBERSHIP_UNVERIFIED"), "disposition": "CLEAR"},
        note="The family is excluded rather than silently cleared; four families minus one still pass here, "
             "so withholding costs the submitter that family's support and gains nothing.")
    add("m14_manifest_expired", "Expired manifest", WIRE,
        [fam(ind[0]["family_id"], *ind[0]["members"], tau_expiry=BASE_US - 1)] + ind[1:], "DENY",
        {"family_status": ("F-vendor-a", "MEMBERSHIP_UNVERIFIED")})
    add("m15_manifest_revoked", "Revoked manifest", WIRE, ind, "DENY",
        {"family_status": ("F-vendor-a", "MEMBERSHIP_UNVERIFIED")},
        trust=dict(TS, revoked_manifest_ids=[ind[0]["manifest"]["payload"]["manifest_id"]]))

    # --- P-1 carried over under M-1 ------------------------------------------------------------------
    cor = three(True)
    pad = fam("F-pad", ev("ev-pad", 0.95, "org:pad", self_declared=V.SHARED + [f"COLLECTOR:fab-{k:03d}" for k in range(200)]))
    add("x01_self_declared_padding", "Listed but self-declared padded lineage is ignored", WIRE, cor + [pad], "DENY",
        {"same_MC_as": "c02_three_correlated_fail"})
    lifted = ev("ev-lift", 0.95, "org:pad")
    lifted["provenance"] = copy.deepcopy(ind[0]["members"][0]["provenance"])
    add("x05_attestation_lifted", "Valid attestation copied onto other listed evidence", WIRE, cor + [fam("F-lift", lifted)],
        "DENY", {"same_MC_as": "c02_three_correlated_fail", "reasons_include": ["PROVENANCE_TAMPERED"]})
    alias = [fam(f["family_id"], *f["members"]) for f in V.three(False, partial=True, aliased_b=True)]
    add("x07_alias_resolves_correlation", "Aliased shared feed is resolved through the trust state's alias table",
        WIRE, alias, "DENY")

    # --- boundaries and the total-function boundary -------------------------------------------------
    eqb, belowb = boundary_values()
    add("b01_c_equals_t", "C == T exactly must ALLOW", REFUND, [solo(eqb)], "ALLOW", {"C_equals_T": True})
    add("b02_one_ulp_below", "C one ulp below T must DENY", REFUND, [solo(belowb)], "DENY", {"C_ulps_below_T": 1})
    add("e01_engine_guard_nan", "Engine guard still runs first (NaN value)", WIRE,
        [fam("F1", ev("ev-1", "7ff8000000000000", "org:x"))], "DENY", {"stage": "ENGINE_GUARD"})
    add("f01_malformed_family_is_structured", "A family without family_id is excluded and audited, not an exception",
        WIRE, ind + [{"members": []}], "ALLOW", {"stage": "PROFILE", "reasons_include": ["LINEAGE_CLASS_MALFORMED"]})
    add("f02_malformed_member_quarantined", "A member with junk value_bits is quarantined, not an exception", WIRE,
        [inject(ind[0], {"evidence_id": "ev-junk", "parent_id": "ev-vendor-a", "producer": "org:x", "value_bits": [1, 2]})]
        + ind[1:], "ALLOW", {"same_MC_as": "c01_three_independent_pass"})
    return C


def solo(vb):
    return fam("F-solo", attest(ev("ev-solo", vb, "org:solo"), ["SOURCE:solo"]))


def boundary_values():
    def one(vb):
        return ENG.evaluate(REFUND, [solo(vb)], PROFILE, TS)
    T = S.f(one(bits(0.5))["T_bits"])
    lob, hib = R.bits_of(0.0), R.bits_of(1.0)
    while hib - lob > 1:
        mid = (lob + hib) // 2
        if S.f(one(f"{mid:016x}")["C_bits"]) >= T:
            hib = mid
        else:
            lob = mid
    target = struct.unpack(">d", struct.pack(">Q", R.bits_of(T) - 1))[0]
    b = hib
    while S.f(one(f"{b:016x}")["C_bits"]) > target:
        b -= 1
    if S.f(one(f"{hib:016x}")["C_bits"]) != T or S.f(one(f"{b:016x}")["C_bits"]) != target:
        raise SystemExit("v0.3.0 boundary search failed")
    return f"{hib:016x}", f"{b:016x}"


# ---------------------------------------------------------------------------
# Random generators (shared with property tests)
# ---------------------------------------------------------------------------

def random_family(rng, fid, attested_p=8):
    f = V.random_attested_family(rng, fid)
    return fam(fid, *f["members"])


def random_contradiction(rng, parent, eid, attested):
    c = contra(eid, parent, producer=f"org:{eid}", conflict=rng.pick(TS["conflict_types"]),
               claim=f"claim-{rng.below(4)}")
    return attest(c, [rng.pick(V.POOL)]) if attested else c


def witnesses(n=18):
    rng = CC.DRng("v030-witness")
    kinds = ["monotone", "contain", "inject"]
    out = []
    for i in range(n):
        kind = kinds[i % 3]
        basef = [random_family(rng, f"F-W{i}-{k}") for k in range(2 + rng.below(4))]
        rq = rng.pick([ANNOTATE, ANNOTATE, REFUND, WIRE])
        if kind == "monotone":
            after = basef + [random_family(rng, f"F-W{i}-new")]
        elif kind == "contain":
            tgt = basef[0]
            root_id = tgt["members"][0]["evidence_id"]
            b = dict(BIND_WIRE, **{k: rq[k] for k in ("action", "assertion", "target", "evidence_snapshot_id")})
            c = contra(f"ev-W{i}-c", root_id, producer=f"org:W{i}c", binding=b)
            after = [fam(tgt["family_id"], *tgt["members"], c)] + basef[1:]
        else:
            tgt = basef[0]
            after = [inject(tgt, ev(f"ev-W{i}-inj", 1.0, "org:attacker", parent=tgt["members"][0]["evidence_id"],
                                    state=rng.pick(["INVALID", "CONTRADICTORY"])))] + basef[1:]
        t0 = ENG.evaluate(rq, basef, PROFILE, TS)
        t1 = ENG.evaluate(rq, after, PROFILE, TS)
        ok = {"monotone": S.f(t1["M_bits"]) >= S.f(t0["M_bits"]) and not (t0["decision"] == "ALLOW" and t1["decision"] == "DENY"),
              "contain": (t1["M_bits"], t1["C_bits"]) == (t0["M_bits"], t0["C_bits"]) and t1["decision"] != "ALLOW",
              "inject": (t1["M_bits"], t1["C_bits"], t1["decision"]) == (t0["M_bits"], t0["C_bits"], t0["decision"])}[kind]
        if not ok:
            raise SystemExit(f"v0.3.0 witness {i} ({kind}) violated: {t0['decision']} -> {t1['decision']}")
        out.append({"kind": kind, "request": rq, "base": basef, "after": after,
                    "M_base_bits": t0["M_bits"], "M_after_bits": t1["M_bits"],
                    "C_base_bits": t0["C_bits"], "C_after_bits": t1["C_bits"],
                    "decision_base": t0["decision"], "decision_after": t1["decision"]})
    return out


# ---------------------------------------------------------------------------
# Build
# ---------------------------------------------------------------------------

def build():
    out, by_id = [], {}
    for c in cases():
        tr = ENG.evaluate(c["request"], c["families"], PROFILE, c["trust"])
        cid, ck = c["id"], c["checks"]
        fail = lambda m: SystemExit(f"v0.3.0 {cid}: {m}")  # noqa: E731
        if tr["decision"] != c["expect"]:
            raise fail(f"engine says {tr['decision']} {tr['reason_codes']}, declared {c['expect']}")
        if "stage" in ck and tr["stage"] != ck["stage"]:
            raise fail(f"stage {tr['stage']}")
        if "disposition" in ck and tr.get("adverse_disposition") != ck["disposition"]:
            raise fail(f"disposition {tr.get('adverse_disposition')}")
        if "same_MC_as" in ck:
            o = by_id[ck["same_MC_as"]]
            if (tr["M_bits"], tr["C_bits"]) != (o["M_bits"], o["C_bits"]):
                raise fail(f"M/C differ from {ck['same_MC_as']}: {S.f(tr['M_bits'])} vs {S.f(o['M_bits'])}")
        if "same_containment_as" in ck and tr["containment_digest"] != by_id[ck["same_containment_as"]]["containment_digest"]:
            raise fail("containment digest differs")
        if "obligations" in ck and len(tr["adverse_obligations"]) != ck["obligations"]:
            raise fail(f"obligations {len(tr['adverse_obligations'])}")
        if "occurrences" in ck and sum(len(o["occurrences"]) for o in tr["adverse_obligations"]) != ck["occurrences"]:
            raise fail("occurrences")
        if "quarantined" in ck and sum(e["event"] == "MEMBER_QUARANTINED" for e in tr["audit_events"]) != ck["quarantined"]:
            raise fail("quarantine count")
        if "family_status" in ck:
            fid, st = ck["family_status"]
            if tr["families"][fid]["status"] != st:
                raise fail(f"family {fid} status {tr['families'][fid]['status']}")
        for rsn in ck.get("reasons_include", []):
            if rsn not in tr["reason_codes"]:
                raise fail(f"missing reason {rsn}")
        if ck.get("C_equals_T") and tr["C_bits"] != tr["T_bits"]:
            raise fail("C != T")
        if "C_ulps_below_T" in ck and int(tr["T_bits"], 16) - int(tr["C_bits"], 16) != ck["C_ulps_below_T"]:
            raise fail("ulp")
        by_id[cid] = tr
        v = {"id": cid, "title": c["title"], "input": {"request": c["request"], "families": c["families"]},
             "expected_trace": tr,
             "summary": {"decision": tr["decision"], "disposition": tr.get("adverse_disposition"), "stage": tr["stage"],
                         "M": repr(S.f(tr["M_bits"])), "C": repr(S.f(tr["C_bits"])), "reasons": tr["reason_codes"]}}
        if c["trust"] is not TS:
            v["trust_state"] = c["trust"]
        if c.get("note"):
            v["note"] = c["note"]
        out.append(v)
    return {
        "vector_set": "scoring-profile",
        "scoring_profile": PROFILE["id"],
        "status": "EXPERIMENTAL — reference candidate; supersedes coverage-v0.2.1-alpha",
        "spec_version": R.SPEC_VERSION,
        "assumptions": "docs/SCORING_PROFILE_coverage-v0.3.0-alpha.md",
        "WARNING": "Test keys are derived from public seeds. Never use them outside conformance testing.",
        "scoring_profile_declaration": PROFILE, "scoring_profile_hash": TF.SCORING_PROFILE_HASH,
        "trust_state": TS, "trust_state_hash": TF.TRUST_STATE_HASH,
        "exploit_statement": "No known open exploit UNDER the declared trust state, atom registry and alias authority, "
                             "family-membership authorities, clock authorities and calibration. Each of those is an "
                             "assumption, not a result (see spec §5 and docs/CONFORMANCE_ASSUMPTIONS.md Q-1..Q-6).",
        "property_witnesses": witnesses(),
        "cases": out,
    }


### Randomised property tests — `tools/property_tests.py`

In [17]:
%%writefile tools/property_tests.py
#!/usr/bin/env python3
"""
Randomised property tests for the scoring profiles.

For coverage-v0.2.0-alpha these MUST all hold (exit status 1 otherwise):
  P1 monotone      M(E ∪ {e}) ≥ M(E) for any added eligible family e
  P2 idempotent    M(E ∪ {dup}) == M(E) bit-for-bit, dup = structural copy of a member of E under new ids
  P3 order-free    permuting families, members and atom lists leaves the whole trace bit-identical
  P4 bridge-safe   adding a family whose atoms are drawn from several existing families never lowers M
  P5 decision      none of the additions above turns ALLOW into DENY

The same generators are run against jaccard-v0.1.0-alpha for comparison; its violations are
reported, not failed on (they are the reason coverage exists).

    python tools/property_tests.py [--trials N] [--seed S]
"""
from __future__ import annotations

import argparse
import copy
import os
import random
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.join(HERE, "..", "reference"))
sys.path.insert(0, HERE)
import prc_ref as R            # noqa: E402
import prc_scoring as S        # noqa: E402
import prc_engine as ENG       # noqa: E402
import scoring_cases as J      # noqa: E402
import coverage_cases as CC    # noqa: E402

ATOMS = CC.ATOM_POOL
REQS = [J.WIRE, J.REFUND, J.ANNOTATE]


def rfam(rng: random.Random, fid: str) -> dict:
    deps = sorted({rng.choice(ATOMS) for _ in range(rng.randrange(5))})
    members = [J.ev(f"ev-{fid}", rng.randrange(1025) / 1024.0, deps)]
    if rng.random() < 0.3:
        members.append(J.ev(f"ev-{fid}-d", rng.randrange(1025) / 1024.0, deps[:1], parent=f"ev-{fid}",
                            state=rng.choice(["VALID", "DEGRADED", "UNAVAILABLE_OR_STALE"])))
    return J.fam(fid, *members)


def permute(rng, fams):
    p = copy.deepcopy(fams)
    rng.shuffle(p)
    for f_ in p:
        rng.shuffle(f_["members"])
        for m in f_["members"]:
            rng.shuffle(m["dependencies"])
    return p


def run(profile, trials, seed):
    rng = random.Random(seed)
    viol = {"P1": 0, "P2": 0, "P3": 0, "P4": 0, "P5": 0}
    example = {}
    for t in range(trials):
        req = rng.choice(REQS)
        base = [rfam(rng, f"B{t}-{k}") for k in range(1 + rng.randrange(6))]
        t0 = ENG.evaluate(req, base, profile)
        m0 = S.f(t0["M_bits"])
        adds = {
            "P1": rfam(rng, f"N{t}"),
            "P2": CC.rename_family(rng.choice(base), f"D{t}"),
        }
        pool = sorted({a for f_ in base for a in f_["members"][0]["dependencies"]})
        adds["P4"] = J.fam(f"X{t}", J.ev(f"ev-X{t}", rng.randrange(1025) / 1024.0,
                                         sorted({rng.choice(pool) for _ in range(3)}) if pool else []))
        for prop, fam_ in adds.items():
            t1 = ENG.evaluate(req, base + [fam_], profile)
            m1 = S.f(t1["M_bits"])
            bad = (t1["M_bits"] != t0["M_bits"]) if prop == "P2" else (m1 < m0)
            if bad:
                viol[prop] += 1
                example.setdefault(prop, (m0, m1))
            if t0["decision"] == "ALLOW" and t1["decision"] == "DENY":
                viol["P5"] += 1
                example.setdefault("P5", (m0, m1))
        if R.encode(ENG.evaluate(req, permute(rng, base), profile)) != R.encode(t0):
            viol["P3"] += 1
    return viol, example


def run_v021(trials, seed):
    """coverage-v0.2.1-alpha over attested evidence. Adds:
       P6 defeater      adding an admitted CONTRADICTORY member always yields DENY
       P7 unattested    on a wire (κ_eff = 0) an unattested family leaves M bit-identical
       P8 tampered      a family whose attestation is forged/lifted leaves M bit-identical"""
    import coverage_v021_cases as V
    rng = CC.DRng(f"props-v021-{seed}")
    prof, trust = V.PROFILE, V.TRUST
    viol = {k: 0 for k in ("P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8")}
    for t in range(trials):
        rq = rng.pick([V.WIRE, V.REFUND, V.ANNOTATE])
        base = [V.random_attested_family(rng, f"B{t}-{k}") for k in range(1 + rng.below(5))]
        t0 = ENG.evaluate(rq, base, prof, trust)
        m0 = t0["M_bits"]

        def after(extra, r=rq, b=base):
            return ENG.evaluate(r, b + [extra], prof, trust)
        t1 = after(V.random_attested_family(rng, f"N{t}"))
        viol["P1"] += S.f(t1["M_bits"]) < S.f(m0)
        viol["P5"] += t0["decision"] == "ALLOW" and t1["decision"] == "DENY"
        viol["P2"] += after(V.reattest_rename(rng.pick(base), f"D{t}"))["M_bits"] != m0
        pool = sorted({a for f_ in base for m in f_["members"] if "provenance" in m
                       for a in m["provenance"]["payload"]["dependencies"]}, key=lambda s: s.encode())
        bridge = V.fam(f"X{t}", V.attest(V.ev(f"ev-X{t}", rng.unit(), f"org:X{t}"),
                                         sorted({rng.pick(pool) for _ in range(2)}) if pool else ["SOURCE:p"]))
        tb = after(bridge)
        viol["P4"] += S.f(tb["M_bits"]) < S.f(m0)
        viol["P5"] += t0["decision"] == "ALLOW" and tb["decision"] == "DENY"
        viol["P6"] += after(V.random_defeater_family(rng, f"Y{t}"))["decision"] != "DENY"
        tw0 = ENG.evaluate(V.WIRE, base, prof, trust)
        tw1 = ENG.evaluate(V.WIRE, base + [V.fam(f"U{t}", V.ev(f"ev-U{t}", rng.unit(), f"org:U{t}"))], prof, trust)
        viol["P7"] += tw1["M_bits"] != tw0["M_bits"]
        src = next((m for f_ in base for m in f_["members"] if "provenance" in m), None)
        if src is not None:
            forged = V.ev(f"ev-Z{t}", 1.0, f"org:Z{t}")
            forged["provenance"] = copy.deepcopy(src["provenance"])  # lifted onto new evidence
            viol["P8"] += after(V.fam(f"Z{t}", forged))["M_bits"] != m0
        shuffled = copy.deepcopy(base)
        for i in range(len(shuffled) - 1, 0, -1):
            j = rng.below(i + 1)
            shuffled[i], shuffled[j] = shuffled[j], shuffled[i]
        for f_ in shuffled:
            f_["members"].reverse()
        viol["P3"] += R.encode(ENG.evaluate(rq, shuffled, prof, trust)) != R.encode(t0)
    return viol


def run_v030(trials, seed):
    """coverage-v0.3.0-alpha over attested evidence with signed family manifests. Adds:
       P9  injection    a member NOT listed in the manifest (INVALID, CONTRADICTORY, attested or forged
                        contradiction, rogue root, same-id impostor) changes no M, C, disposition or decision
       P10 containment  a listed unattested relevant contradiction leaves M and C bit-identical and
                        prevents ALLOW (ALLOW becomes MANUAL_REQUIRED)"""
    import coverage_v030_cases as W
    rng = CC.DRng(f"props-v030-{seed}")
    prof, ts = W.PROFILE, W.TS
    viol = {k: 0 for k in ("P1", "P2", "P3", "P4", "P5", "P6", "P9", "P10")}
    for t in range(trials):
        rq = rng.pick([W.ANNOTATE, W.ANNOTATE, W.REFUND, W.WIRE])
        base = [W.random_family(rng, f"F-B{t}-{k}") for k in range(2 + rng.below(4))]
        t0 = ENG.evaluate(rq, base, prof, ts)
        key0 = (t0["M_bits"], t0["C_bits"], t0.get("adverse_disposition"), t0["decision"])
        ev_ = lambda fams: ENG.evaluate(rq, fams, prof, ts)  # noqa: E731
        t1 = ev_(base + [W.random_family(rng, f"F-N{t}")])
        viol["P1"] += S.f(t1["M_bits"]) < S.f(t0["M_bits"])
        viol["P5"] += t0["decision"] == "ALLOW" and t1["decision"] == "DENY"
        src = rng.pick(base)
        dup = W.fam(f"F-D{t}", *__import__("coverage_v021_cases").reattest_rename(src, f"F-D{t}")["members"])
        viol["P2"] += ev_(base + [dup])["M_bits"] != t0["M_bits"]
        pool = sorted({a for f_ in base for m in f_["members"] if "provenance" in m
                       for a in m["provenance"]["payload"]["dependencies"]}, key=lambda x: x.encode())
        br = W.fam(f"F-X{t}", W.attest(W.ev(f"ev-X{t}", rng.unit(), f"org:X{t}"),
                                       sorted({rng.pick(pool) for _ in range(2)}) if pool else ["SOURCE:p"]))
        tb = ev_(base + [br])
        viol["P4"] += S.f(tb["M_bits"]) < S.f(t0["M_bits"])
        viol["P5"] += t0["decision"] == "ALLOW" and tb["decision"] == "DENY"
        binding = {k: rq[k] for k in ("action", "assertion", "target", "evidence_snapshot_id")}
        tgt_i = rng.below(len(base))
        tgt = base[tgt_i]
        root_id = tgt["members"][0]["evidence_id"]
        # P6: a listed, auditor-attested relevant contradiction always defeats
        c_att = W.attest(W.contra(f"ev-Y{t}", root_id, producer=f"org:Y{t}", binding=binding), [rng.pick(V_POOL())])
        fams6 = base[:tgt_i] + [W.fam(tgt["family_id"], *tgt["members"], c_att)] + base[tgt_i + 1:]
        viol["P6"] += ev_(fams6)["decision"] != "DENY"
        # P10: a listed unattested relevant contradiction contains
        c_un = W.contra(f"ev-Z{t}", root_id, producer=f"org:Z{t}", binding=binding)
        fams10 = base[:tgt_i] + [W.fam(tgt["family_id"], *tgt["members"], c_un)] + base[tgt_i + 1:]
        t10 = ev_(fams10)
        viol["P10"] += ((t10["M_bits"], t10["C_bits"]) != (t0["M_bits"], t0["C_bits"])
                        or t10["decision"] == "ALLOW"
                        or (t0["decision"] == "ALLOW" and t10["decision"] != "MANUAL_REQUIRED"))
        # P9: injection of an unlisted member of every adversarial kind
        kind = rng.below(6)
        if kind == 0:
            m = W.ev(f"ev-I{t}", 1.0, "org:attacker", parent=root_id, state="INVALID")
        elif kind == 1:
            m = c_un
        elif kind == 2:
            m = c_att
        elif kind == 3:
            m = copy.deepcopy(c_att)
            m["provenance"]["signature"] = "22" * 64
        elif kind == 4:
            m = W.ev(f"ev-R{t}", 0.99, "org:attacker")
        else:
            m = W.ev(root_id, 0.01, "org:attacker")  # same-id impostor
        fams9 = base[:tgt_i] + [W.inject(tgt, m)] + base[tgt_i + 1:]
        t9 = ev_(fams9)
        viol["P9"] += (t9["M_bits"], t9["C_bits"], t9.get("adverse_disposition"), t9["decision"]) != key0
        sh = copy.deepcopy(base)
        for i in range(len(sh) - 1, 0, -1):
            j = rng.below(i + 1)
            sh[i], sh[j] = sh[j], sh[i]
        for f_ in sh:
            f_["members"].reverse()
        viol["P3"] += R.encode(ev_(sh)) != R.encode(t0)
    return viol


def V_POOL():
    import coverage_v021_cases as V
    return V.POOL


def fuzz_verifiers(trials, seed):
    """P11 totality. Mutates signed decisions/approvals from the vector suite and malformed scoring
    inputs; counts exceptions escaping the *inner* verifiers (must be 0 for DecisionBindingValid) and
    faults that the scoring boundary converted into DENY (reported)."""
    import json
    import random as _r
    import generate_vectors as G
    import coverage_v030_cases as W
    rng = _r.Random(seed)
    doc = json.load(open(os.path.join(HERE, "..", "tests", "vectors", "auth_expiry_skew.json"), encoding="utf-8"))
    ts = doc["trust_state"]
    vs = [v for v in doc["vectors"] if v["expected_output"]["DecisionBindingValid"]]
    junk = [None, True, -1, 2 ** 63 - 1, "", "x", [], {}, [1, "a"], {"k": None}, 0.5, "0" * 64, "zz"]
    raised = flipped_allowed = resigned = 0
    for _ in range(trials):
        v = rng.choice(vs)
        i = v["input"]
        ctx = dict(i["context"], trust_state=ts)
        # (a) random byte flip without re-signing: must never be valid
        b = bytearray(bytes.fromhex(i["decision_bytes"]))
        k = rng.randrange(len(b))
        b[k] ^= 1 << rng.randrange(8)
        try:
            ok, _ = R._decision_binding_valid(ctx, b.hex(), i["decision_signature"], i["approval_bytes"], i["approval_signature"])
            flipped_allowed += ok
        except Exception:  # noqa: BLE001
            raised += 1
        # (b) replace a random field with junk and RE-SIGN validly: must never raise
        d = copy.deepcopy(i["decision_payload"])
        fld = rng.choice(sorted(d))
        d[fld] = copy.deepcopy(rng.choice(junk))
        if rng.random() < 0.2:
            d.pop(rng.choice(sorted(d)))
        try:
            db = R.encode(d)
        except Exception:  # noqa: BLE001  (unencodable junk, e.g. float is fine; skip nothing)
            continue
        sig = G.K_DECISION.sign_digest(R.H_bytes(db))
        p_hex, p_sig = i["approval_bytes"], i["approval_signature"]
        if p_hex and rng.random() < 0.5:
            p = copy.deepcopy(i["approval_payload"])
            p[rng.choice(sorted(p))] = copy.deepcopy(rng.choice(junk))
            pb = R.encode(p)
            p_hex, p_sig = pb.hex(), G.K_APPROVAL.sign_digest(R.H_bytes(pb))
        try:
            R._decision_binding_valid(ctx, db.hex(), sig, p_hex, p_sig)
            resigned += 1
        except Exception:  # noqa: BLE001
            raised += 1
    # scoring inputs
    faults = 0
    for t in range(trials // 4):
        fams = [W.random_family(CC.DRng(f"fz{seed}-{t}-{k}"), f"F-Z{t}-{k}") for k in range(2)]
        f_ = rng.choice(fams)
        target = rng.choice([f_, rng.choice(f_["members"]), f_.get("manifest", {}).get("payload", {})])
        if isinstance(target, dict) and target:
            target[rng.choice(sorted(target))] = copy.deepcopy(rng.choice(junk))
        tr = ENG.evaluate(W.WIRE, fams, W.PROFILE, W.TS)   # public boundary: must return, never raise
        faults += tr["stage"] == "VERIFIER_FAULT"
    return {"decision_verifier_exceptions": raised, "bitflips_accepted": flipped_allowed,
            "resigned_mutations_evaluated": resigned, "scoring_faults_converted_to_deny": faults}


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--trials", type=int, default=1500)
    ap.add_argument("--seed", type=int, default=20260925)
    a = ap.parse_args(argv)
    cov, cex = run(CC.PROFILE, a.trials, a.seed)
    jac, jex = run(J.PROFILE, a.trials, a.seed)
    print(f"property tests, {a.trials} random trials per property")
    print(f"  {'':<14}{'P1 mono':>9}{'P2 idem':>9}{'P3 order':>10}{'P4 bridge':>11}{'P5 decision':>13}")
    for name, v in (("coverage-v0.2", cov), ("jaccard-v0.1", jac)):
        print(f"  {name:<14}{v['P1']:>9}{v['P2']:>9}{v['P3']:>10}{v['P4']:>11}{v['P5']:>13}")
    if jex:
        k = next(iter(jex))
        print(f"  jaccard counterexample ({k}): M {jex[k][0]!r} -> {jex[k][1]!r}")
    v21 = run_v021(max(1, a.trials // 3), a.seed)
    print(f"  coverage-v0.2.1-alpha over attested evidence, {max(1, a.trials // 3)} trials: " +
          ", ".join(f"{k}={v}" for k, v in v21.items()))
    print("    (P6 defeater always DENY, P7 unattested neutral on a wire, P8 forged/lifted attestation neutral)")
    n30 = max(1, a.trials // 3)
    v30 = run_v030(n30, a.seed)
    print(f"  coverage-v0.3.0-alpha with membership manifests, {n30} trials: " + ", ".join(f"{k}={v}" for k, v in v30.items()))
    print("    (P9 unlisted injection changes nothing, P10 containment keeps M/C and forbids ALLOW)")
    fz = fuzz_verifiers(max(1, a.trials), a.seed)
    print(f"  P11 totality fuzz ({max(1, a.trials)} mutations): " + ", ".join(f"{k}={v}" for k, v in fz.items()))
    failed = (any(cov.values()) or any(v21.values()) or any(v30.values())
              or fz["decision_verifier_exceptions"] or fz["bitflips_accepted"] or fz["scoring_faults_converted_to_deny"])
    print("coverage profiles:", "FAIL" if failed else "all properties hold")
    return 1 if failed else 0


if __name__ == "__main__":
    sys.exit(main())


### Transport round trip (Protobuf is transport only) — `tools/transport_roundtrip.py`

In [18]:
%%writefile tools/transport_roundtrip.py
#!/usr/bin/env python3
"""
Transport round trip: canonical object -> Protobuf -> bytes -> Protobuf -> canonical object
-> PRC-LP-0 -> SHA-256 -> Ed25519 verify, for every signed object in the vector suite.

Also demonstrates why Protobuf bytes must never be hashed:
  * two different, valid wire encodings of the same message (field splitting + merge)
  * an unknown field that a lenient parser keeps and a strict verifier must reject

Requires: pip install grpcio-tools protobuf     (skips with exit 0 if they are missing and
PRC_REQUIRE_TRANSPORT is not set)

    python tools/transport_roundtrip.py
"""
from __future__ import annotations

import glob
import hashlib
import importlib
import json
import os
import sys
import tempfile

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.abspath(os.path.join(HERE, ".."))
sys.path.insert(0, os.path.join(ROOT, "reference"))
import prc_ref as R  # noqa: E402

PROTO = os.path.join(ROOT, "schema", "protobuf", "provenance_razor_v1alpha3.proto")
FBS = os.path.join(ROOT, "schema", "flatbuffers", "provenance_razor_v1alpha3.fbs")
FB_NS = "Provenance.Razor.V1Alpha3"


class TransportError(ValueError):
    pass


def load_pb2():
    from grpc_tools import protoc
    out = tempfile.mkdtemp(prefix="prc_pb2_")
    rc = protoc.main(["protoc", f"-I{os.path.dirname(PROTO)}", f"--python_out={out}", PROTO])
    if rc != 0:
        raise SystemExit("protoc failed to compile the transport schema")
    sys.path.insert(0, out)
    return importlib.import_module("provenance_razor_v1alpha3_pb2")


# field kinds: s=string, i=int64, b=bool, v=Value, d=32-byte digest, e:<Enum>=enum, oi=optional int64, atoms
APPROVAL = [("domain", "s"), ("schema_ver", "s"), ("approval_id", "s"), ("approval_key_id", "s"),
            ("approval_trust_epoch", "s"), ("authority_class", "s"), ("authority_scope", "s"), ("x", "v"),
            ("a", "v"), ("target", "s"), ("state_ver", "i"), ("control_profile_hash", "d"), ("tau_issued", "i"),
            ("tau_expiry", "i"), ("decision", "e:DecisionOutcome"), ("actor", "v"), ("req_id", "s"),
            ("capability_id", "s"), ("scoring_profile_hash", "d"), ("trust_state_hash", "d"),
            ("adverse_disposition", "e:AdverseDisposition"), ("containment_digest", "d")]
DECISION = [("domain", "s"), ("schema_ver", "s"), ("profile_version", "s"), ("lineage_schema_version", "s"),
            ("capability_id", "s"), ("req_id", "s"), ("evidence_snapshot_id", "s"), ("actor", "v"),
            ("auth_mode", "e:AuthorizationMode"), ("x", "v"), ("a", "v"), ("target", "s"), ("state_ver", "i"),
            ("control_profile_hash", "d"), ("evidence_hash", "d"), ("lineage_hash", "d"), ("approval_hash", "d"),
            ("decision_key_id", "s"), ("decision_trust_epoch", "s"), ("tau_eval", "i"), ("tau_expiry", "i"),
            ("decision", "e:DecisionOutcome"), ("reason_codes", "i"), ("scoring_profile_hash", "d"),
            ("trust_state_hash", "d"), ("adverse_disposition", "e:AdverseDisposition"), ("containment_digest", "d"),
            ("eval_clock_id", "s")]
RECEIPT = [("domain", "s"), ("schema_ver", "s"), ("receipt_id", "s"), ("capability_id", "s"), ("req_id", "s"),
           ("decision_digest", "d"), ("x", "v"), ("target", "s"), ("control_profile_hash", "d"),
           ("execution_status", "e:ExecutionStatus"), ("failure_phase", "e:FailurePhase"), ("reason_code", "i"),
           ("reconciliation_required", "b"), ("pre_state_digest", "d"), ("post_state_digest", "d"),
           ("tau_dispatch", "i"), ("tau_finalized", "i"), ("receipt_key_id", "s"), ("receipt_trust_epoch", "s"),
           ("expected_state_ver", "i"), ("observed_pre_state_ver", "i"), ("observed_post_state_ver", "oi")]
ATTESTATION = [("domain", "s"), ("schema_ver", "s"), ("attestation_id", "s"), ("provenance_key_id", "s"),
               ("provenance_trust_epoch", "s"), ("attester_id", "s"), ("evidence_digest", "d"),
               ("dependencies", "atoms"), ("tau_issued", "i"), ("tau_expiry", "i")]
MANIFEST = [("domain", "s"), ("schema_ver", "s"), ("manifest_id", "s"), ("membership_key_id", "s"),
            ("membership_trust_epoch", "s"), ("authority_id", "s"), ("family_id", "s"), ("ingress_channel", "s"),
            ("members", "members"), ("tau_issued", "i"), ("tau_expiry", "i")]
ENUM_PREFIX = {"AdverseDisposition": "ADVERSE_DISPOSITION_", "DecisionOutcome": "DECISION_OUTCOME_", "AuthorizationMode": "AUTHORIZATION_MODE_",
               "ExecutionStatus": "EXECUTION_STATUS_", "FailurePhase": "FAILURE_PHASE_"}


class Codec:
    def __init__(self, pb2):
        self.pb = pb2
        from google.protobuf import unknown_fields
        self._unknown = unknown_fields.UnknownFieldSet

    # ---- canonical Value <-> proto Value (lossless) ----
    def to_value(self, v):
        m = self.pb.Value()
        if v is None:
            m.null_value = True
        elif isinstance(v, bool):
            m.bool_value = v
        elif isinstance(v, int):
            m.int_value = v
        elif isinstance(v, R.F64):
            m.f64_bits = v.bits
        elif isinstance(v, str):
            m.string_value = v
        elif isinstance(v, bytes):
            m.bytes_value = v
        elif isinstance(v, list):
            m.list_value.SetInParent()
            for it in v:
                m.list_value.items.append(self.to_value(it))
        elif isinstance(v, dict):
            m.map_value.SetInParent()
            for k in sorted(v):
                e = m.map_value.entries.add()
                e.key = k
                e.value.CopyFrom(self.to_value(v[k]))
        else:
            raise TypeError(type(v))
        return m

    def from_value(self, m):
        self._strict(m)
        k = m.WhichOneof("kind")
        if k is None:
            raise TransportError("Value has no kind")
        if k == "null_value":
            if not m.null_value:
                raise TransportError("null_value must be true")
            return None
        if k == "bool_value":
            return m.bool_value
        if k == "int_value":
            return m.int_value
        if k == "f64_bits":
            return R.F64(m.f64_bits)
        if k == "string_value":
            return m.string_value
        if k == "bytes_value":
            return bytes(m.bytes_value)
        if k == "list_value":
            return [self.from_value(i) for i in m.list_value.items]
        out = {}
        for e in m.map_value.entries:
            if e.key in out:
                raise TransportError(f"duplicate map key {e.key!r}")
            if not e.HasField("value"):
                raise TransportError("map entry without value")
            out[e.key] = self.from_value(e.value)
        return out

    def _strict(self, msg):
        if len(self._unknown(msg)):
            raise TransportError(f"unknown fields present in {msg.DESCRIPTOR.name}")

    # ---- canonical object <-> proto message ----
    def to_proto(self, obj, spec, cls):
        m = cls()
        if set(obj) != {k for k, _ in spec}:
            raise TransportError(f"canonical object keys do not match {cls.DESCRIPTOR.name}")
        for key, kind in spec:
            v = obj[key]
            if kind in ("s", "i", "b"):
                setattr(m, key, v)
            elif kind == "v":
                getattr(m, key).CopyFrom(self.to_value(v))
            elif kind == "d":
                raw = bytes.fromhex(v)
                if len(raw) != 32:
                    raise TransportError(f"{key} is not 32 bytes")
                setattr(m, key, raw)
            elif kind.startswith("e:"):
                enum = getattr(self.pb, kind[2:])
                setattr(m, key, enum.Value(ENUM_PREFIX[kind[2:]] + v))
            elif kind == "oi":
                if v is not None:
                    setattr(m, key, v)
            elif kind == "atoms":
                for a in v:
                    t, _, i = a.partition(":")
                    at = m.dependencies.add()
                    at.dependency_type, at.dependency_id = t, i
            elif kind == "members":
                for mm in v:
                    e = m.members.add()
                    e.evidence_id, e.evidence_digest = mm["evidence_id"], bytes.fromhex(mm["evidence_digest"])
                    if mm["parent_id"] is not None:
                        e.parent_id = mm["parent_id"]
        return m

    def from_proto(self, m, spec):
        self._strict(m)
        obj = {}
        for key, kind in spec:
            if kind == "s":
                if getattr(m, key) == "":
                    raise TransportError(f"required string {key} missing")
                obj[key] = getattr(m, key)
            elif kind in ("i", "b"):
                obj[key] = getattr(m, key)
            elif kind == "v":
                if not m.HasField(key):
                    raise TransportError(f"required value {key} missing")
                obj[key] = self.from_value(getattr(m, key))
            elif kind == "d":
                raw = getattr(m, key)
                if len(raw) != 32:
                    raise TransportError(f"{key} must be exactly 32 bytes")
                obj[key] = raw.hex()
            elif kind.startswith("e:"):
                name = getattr(self.pb, kind[2:]).Name(getattr(m, key))
                if name.endswith("_UNSPECIFIED"):
                    raise TransportError(f"{key} unspecified")
                obj[key] = name[len(ENUM_PREFIX[kind[2:]]):]
            elif kind == "oi":
                obj[key] = getattr(m, key) if m.HasField(key) else None
            elif kind == "atoms":
                atoms = []
                for at in m.dependencies:
                    self._strict(at)
                    atoms.append(f"{at.dependency_type}:{at.dependency_id}")
                obj[key] = atoms  # canonical-order rule is enforced by admissibility, not silently fixed here
            elif kind == "members":
                out = []
                for e in m.members:
                    self._strict(e)
                    if len(e.evidence_digest) != 32 or e.evidence_id == "":
                        raise TransportError("manifest member malformed")
                    out.append({"evidence_id": e.evidence_id, "evidence_digest": e.evidence_digest.hex(),
                                "parent_id": e.parent_id if e.HasField("parent_id") else None})
                obj[key] = out
        return obj


def collect(tests):
    """Every signed object in the suite: (label, kind, canonical payload, public key | None, signature | None)."""
    items = []
    g = json.load(open(os.path.join(tests, "fixtures", "golden_vectors.json"), encoding="utf-8"))
    keys = {k["key_id"]: k["public_key"] for k in g["keys"]}
    for o in g["objects"]:
        if o["artifact_type"] in ("approval", "decision", "receipt"):
            items.append((f"golden:{o['id']}", o["artifact_type"], o["payload"], keys[o["signing_key_id"]],
                          o["expected_signature"]))
    auth = json.load(open(os.path.join(tests, "vectors", "auth_expiry_skew.json"), encoding="utf-8"))
    for v in auth["vectors"]:
        i = v["input"]
        if v["test_id"].startswith("auth_schema_"):  # deliberately schema-invalid: not representable
            continue
        if v["test_id"] != "auth_expiry_013":  # 013 is deliberately non-canonical bytes
            items.append((f"auth:{v['test_id']}:d", "decision", i["decision_payload"], None, None))
        if i["approval_payload"]:
            items.append((f"auth:{v['test_id']}:p", "approval", i["approval_payload"], None, None))
    import yaml
    for f in sorted(glob.glob(os.path.join(tests, "fixtures", "cas_reconciliation", "*.yaml"))):
        e = yaml.safe_load(open(f, encoding="utf-8"))["expected"]
        items.append((f"cas:{os.path.basename(f)}", "receipt", e["receipt_payload"], None, None))
    v30 = json.load(open(os.path.join(tests, "vectors", "scoring_coverage_v0.3.0-alpha.json"), encoding="utf-8"))
    reg30 = v30["trust_state"]["key_registry"]
    seen_m = set()
    for c in v30["cases"]:
        for f_ in c["input"]["families"]:
            env = f_.get("manifest") if isinstance(f_, dict) else None
            if env and env["payload"]["manifest_id"] not in seen_m:
                seen_m.add(env["payload"]["manifest_id"])
                pk = reg30.get(env["payload"]["membership_key_id"], {}).get("public_key")
                items.append((f"v030:{c['id']}:{f_['family_id']}", "manifest", env["payload"], pk, env["signature"]))
            for m in (f_.get("members", []) if isinstance(f_, dict) else []):
                pe = m.get("provenance") if isinstance(m, dict) else None
                if pe and pe["payload"]["attestation_id"] not in seen_m:
                    seen_m.add(pe["payload"]["attestation_id"])
                    pk = reg30.get(pe["payload"]["provenance_key_id"], {}).get("public_key")
                    items.append((f"v030:{c['id']}:{m['evidence_id']}", "attestation", pe["payload"], pk, pe["signature"]))
    v21 = json.load(open(os.path.join(tests, "vectors", "scoring_coverage_v0.2.1-alpha.json"), encoding="utf-8"))
    reg = v21["trust_state"]["registry"]
    seen = set()
    for c in v21["cases"]:
        for f_ in c["input"]["families"]:
            for m in f_["members"]:
                env = m.get("provenance")
                if env and env["payload"]["attestation_id"] not in seen:
                    seen.add(env["payload"]["attestation_id"])
                    pk = reg.get(env["payload"]["provenance_key_id"], {}).get("public_key")
                    items.append((f"v021:{c['id']}:{m['evidence_id']}", "attestation", env["payload"], pk,
                                  env["signature"]))
    return items


# ---------------------------------------------------------------------------
# FlatBuffers: canonical -> flatc JSON -> binary -> flatc JSON -> canonical (uses the flatc CLI)
# ---------------------------------------------------------------------------
FB_ROOT = {"approval": "SignedApproval", "decision": "SignedDecision", "receipt": "SignedReceipt",
           "attestation": "SignedProvenanceAttestation", "manifest": "SignedFamilyManifest"}


def fb_value(v):
    if v is None:
        return {"kind_type": "NullValue", "kind": {}}
    if isinstance(v, bool):
        return {"kind_type": "BoolValue", "kind": {"value": v}}
    if isinstance(v, int):
        return {"kind_type": "IntValue", "kind": {"value": v}}
    if isinstance(v, R.F64):
        return {"kind_type": "F64Bits", "kind": {"bits": v.bits}}
    if isinstance(v, str):
        return {"kind_type": "StringValue", "kind": {"value": v}}
    if isinstance(v, bytes):
        return {"kind_type": "BytesValue", "kind": {"value": list(v)}}
    if isinstance(v, list):
        return {"kind_type": "ValueList", "kind": {"items": [fb_value(i) for i in v]}}
    return {"kind_type": "ValueMap", "kind": {"entries": [{"key": k, "value": fb_value(v[k])} for k in sorted(v)]}}


def fb_unvalue(j):
    t, k = j.get("kind_type"), j.get("kind", {})
    if t == "NullValue":
        return None
    if t == "BoolValue":
        return bool(k.get("value", False))
    if t == "IntValue":
        return int(k.get("value", 0))
    if t == "F64Bits":
        return R.F64(int(k.get("bits", 0)))
    if t == "StringValue":
        return k["value"]
    if t == "BytesValue":
        return bytes(k["value"])
    if t == "ValueList":
        return [fb_unvalue(i) for i in k["items"]]
    if t == "ValueMap":
        out = {}
        for e in k["entries"]:
            if e["key"] in out:
                raise TransportError("duplicate map key")
            out[e["key"]] = fb_unvalue(e["value"])
        return out
    raise TransportError(f"unknown Value kind {t!r}")


def to_fb(obj, spec):
    out = {}
    for key, kind in spec:
        v = obj[key]
        if kind in ("s", "i", "b"):
            out[key] = v
        elif kind == "v":
            out[key] = fb_value(v)
        elif kind == "d":
            out[key] = list(bytes.fromhex(v))
        elif kind.startswith("e:"):
            out[key] = v
        elif kind == "oi":
            if v is not None:
                out[key] = {"value": v}
        elif kind == "atoms":
            out[key] = [{"dependency_type": a.partition(":")[0], "dependency_id": a.partition(":")[2]} for a in v]
        elif kind == "members":
            out[key] = [dict({"evidence_id": m["evidence_id"], "evidence_digest": list(bytes.fromhex(m["evidence_digest"]))},
                             **({"parent_id": m["parent_id"]} if m["parent_id"] is not None else {})) for m in v]
    return out


def from_fb(j, spec):
    allowed = {k for k, _ in spec}
    if set(j) - allowed:
        raise TransportError(f"unexpected FlatBuffers fields {sorted(set(j) - allowed)}")
    obj = {}
    for key, kind in spec:
        if kind == "s":
            if not j.get(key):
                raise TransportError(f"required string {key} missing")
            obj[key] = j[key]
        elif kind == "i":
            obj[key] = int(j.get(key, 0))
        elif kind == "b":
            obj[key] = bool(j.get(key, False))
        elif kind == "v":
            obj[key] = fb_unvalue(j[key])
        elif kind == "d":
            if len(j.get(key, [])) != 32:
                raise TransportError(f"{key} must be exactly 32 bytes")
            obj[key] = bytes(j[key]).hex()
        elif kind.startswith("e:"):
            if j.get(key, "UNSPECIFIED") == "UNSPECIFIED":
                raise TransportError(f"{key} unspecified")
            obj[key] = j[key]
        elif kind == "oi":
            obj[key] = int(j[key].get("value", 0)) if key in j else None
        elif kind == "atoms":
            obj[key] = [f"{a['dependency_type']}:{a['dependency_id']}" for a in j.get(key, [])]
        elif kind == "members":
            obj[key] = [{"evidence_id": m["evidence_id"], "evidence_digest": bytes(m["evidence_digest"]).hex(),
                         "parent_id": m.get("parent_id")} for m in j.get(key, [])]
    return obj


def flatc_path():
    import shutil
    return os.environ.get("FLATC") or shutil.which("flatc")


def flatbuffers_roundtrip(items, specs):
    import subprocess
    flatc = flatc_path()
    if not flatc:
        return None
    work = tempfile.mkdtemp(prefix="prc_fb_")
    fails, count, sigs = [], 0, 0
    by_kind = {}
    for n, (label, kind, payload, pub, sig) in enumerate(items):
        by_kind.setdefault(kind, []).append((n, label, payload, pub, sig))
    for kind, group in by_kind.items():
        spec = specs[kind]
        root = f"{FB_NS}.{FB_ROOT[kind]}"
        ins, outs = os.path.join(work, kind, "in"), os.path.join(work, kind, "bin")
        back = os.path.join(work, kind, "back")
        for d in (ins, outs, back):
            os.makedirs(d, exist_ok=True)
        for n, label, payload, pub, sig in group:
            doc = {"payload": to_fb(payload, spec), "signature": list(bytes.fromhex(sig)) if sig else []}
            json.dump(doc, open(os.path.join(ins, f"o{n}.json"), "w"))
        files = sorted(glob.glob(os.path.join(ins, "*.json")))
        r = subprocess.run([flatc, "-b", "--root-type", root, "-o", outs, FBS] + files, capture_output=True, text=True)
        if r.returncode != 0:
            return {"error": r.stderr.strip()[:300]}
        bins = sorted(glob.glob(os.path.join(outs, "*.bin")))
        r = subprocess.run([flatc, "-t", "--strict-json", "--defaults-json", "--raw-binary", "--root-type", root,
                            "-o", back, FBS, "--"] + bins, capture_output=True, text=True)
        if r.returncode != 0:
            return {"error": r.stderr.strip()[:300]}
        for n, label, payload, pub, sig in group:
            try:
                j = json.load(open(os.path.join(back, f"o{n}.json")))
                canon = from_fb(j["payload"], spec)
            except (TransportError, KeyError, ValueError) as ex:
                fails.append(f"{label}: {ex}")
                continue
            if R.H(canon) != R.H(payload):
                fails.append(f"{label}: canonical hash changed across FlatBuffers transport")
            if pub and sig:
                sigs += 1
                if R.ed25519_verify(pub, R.H(payload), sig) != R.ed25519_verify(pub, R.H(canon), bytes(j.get("signature", [])).hex()):
                    fails.append(f"{label}: signature verdict changed")
            count += 1
    # FlatBuffers bytes are not canonical either: same message, different field order in the source JSON
    first = by_kind["decision"][0]
    a = to_fb(first[2], specs["decision"])
    b = dict(reversed(list(a.items())))
    for name, doc in (("a", a), ("b", b)):
        json.dump({"payload": doc, "signature": []}, open(os.path.join(work, f"demo_{name}.json"), "w"))
    subprocess.run([flatc, "-b", "--root-type", f"{FB_NS}.SignedDecision", "-o", work, FBS,
                    os.path.join(work, "demo_a.json"), os.path.join(work, "demo_b.json")], capture_output=True)
    ba, bb = open(os.path.join(work, "demo_a.bin"), "rb").read(), open(os.path.join(work, "demo_b.bin"), "rb").read()
    unknown = dict(a, not_in_schema=1)
    json.dump({"payload": unknown, "signature": []}, open(os.path.join(work, "demo_u.json"), "w"))
    ru = subprocess.run([flatc, "-b", "--root-type", f"{FB_NS}.SignedDecision", "-o", work, FBS,
                         os.path.join(work, "demo_u.json")], capture_output=True, text=True)
    return {"count": count, "sigs": sigs, "fails": fails,
            "demo": {"field_order_changes_bytes": ba != bb, "flatc_rejects_unknown_field": ru.returncode != 0}}


def main():
    try:
        pb = load_pb2()
    except ImportError:
        if os.environ.get("PRC_REQUIRE_TRANSPORT"):
            raise
        print("transport round trip: SKIPPED (pip install grpcio-tools protobuf)")
        return 0
    codec = Codec(pb)
    spec = {"approval": (APPROVAL, pb.ExternalApproval, pb.SignedApproval),
            "decision": (DECISION, pb.DecisionCapability, pb.SignedDecision),
            "receipt": (RECEIPT, pb.ExecutionReceipt, pb.SignedReceipt),
            "attestation": (ATTESTATION, pb.ProvenanceAttestation, pb.SignedProvenanceAttestation),
            "manifest": (MANIFEST, pb.FamilyManifest, pb.SignedFamilyManifest)}
    fails, counts, sig_checked = [], {}, 0
    for label, kind, payload, pub, sig in collect(os.path.join(ROOT, "tests")):
        fields, cls, signed_cls = spec[kind]
        try:
            wrapper = signed_cls()
            wrapper.payload.CopyFrom(codec.to_proto(payload, fields, cls))
            wrapper.signature = bytes.fromhex(sig) if sig else b""
            wire = wrapper.SerializeToString()
            back = signed_cls.FromString(wire)
            codec._strict(back)
            canon = codec.from_proto(back.payload, fields)
        except TransportError as ex:
            fails.append(f"{label}: {ex}")
            continue
        if R.H(canon) != R.H(payload):
            fails.append(f"{label}: canonical hash changed across transport")
        if pub and sig:
            before = R.ed25519_verify(pub, R.H(payload), sig)
            after = R.ed25519_verify(pub, R.H(canon), back.signature.hex())
            sig_checked += 1
            if before != after:
                fails.append(f"{label}: signature verdict changed across transport")
        counts[kind] = counts.get(kind, 0) + 1

    # --- why Protobuf bytes are not canonical --------------------------------------------------
    g = json.load(open(os.path.join(ROOT, "tests", "fixtures", "golden_vectors.json"), encoding="utf-8"))
    d = next(o for o in g["objects"] if o["id"] == "decision_automated")
    full = pb.SignedDecision()
    full.payload.CopyFrom(codec.to_proto(d["payload"], DECISION, pb.DecisionCapability))
    full.signature = bytes.fromhex(d["expected_signature"])
    wire_a = full.SerializeToString()
    first, second = pb.SignedDecision(), pb.SignedDecision()
    names = [k for k, _ in DECISION]
    for i, n in enumerate(names):
        tgt = first if i % 2 else second
        fld = getattr(full.payload, n)
        if hasattr(fld, "CopyFrom"):
            getattr(tgt.payload, n).CopyFrom(fld)
        else:
            setattr(tgt.payload, n, fld)
    first.signature = full.signature
    wire_b = second.SerializeToString() + first.SerializeToString()  # protobuf merges concatenated messages
    parsed_b = pb.SignedDecision.FromString(wire_b)
    same_msg = parsed_b == full
    canon_b = R.H(codec.from_proto(parsed_b.payload, DECISION))
    demo1 = {"same_parsed_message": same_msg,
             "wire_sha256_differs": hashlib.sha256(wire_a).hexdigest() != hashlib.sha256(wire_b).hexdigest(),
             "canonical_hash_identical": canon_b == d["expected_hash"]}
    tampered = wire_a + bytes([0x98, 0x06, 0x01])  # field 99, varint 1: unknown to the schema
    lenient = pb.SignedDecision.FromString(tampered)
    try:
        codec._strict(lenient)
        strict_rejects = False
    except TransportError:
        strict_rejects = True
    demo2 = {"lenient_parser_accepts": lenient.payload == full.payload, "strict_verifier_rejects": strict_rejects}

    print(f"transport round trip: {sum(counts.values())} signed objects "
          f"({', '.join(f'{k}={v}' for k, v in sorted(counts.items()))}), {sig_checked} signatures re-verified, "
          f"{len(fails)} failures")
    print(f"  non-canonical wire demo: {demo1}")
    print(f"  unknown-field demo:      {demo2}")
    for f in fails:
        print("  FAIL", f)
    ok = not fails and all(demo1.values()) and all(demo2.values())
    fb = flatbuffers_roundtrip(collect(os.path.join(ROOT, "tests")), {k: v[0] for k, v in spec.items()})
    if fb is None:
        print("  FlatBuffers: SKIPPED (set FLATC or put flatc on PATH)")
    elif "error" in fb:
        print("  FlatBuffers: flatc error:", fb["error"])
        ok = False
    else:
        print(f"  FlatBuffers round trip: {fb['count']} signed objects, {fb['sigs']} signatures re-verified, "
              f"{len(fb['fails'])} failures; demo {fb['demo']}")
        for f in fb["fails"]:
            print("  FAIL (fb)", f)
        ok = ok and not fb["fails"] and all(fb["demo"].values())
    return 0 if ok else 1


if __name__ == "__main__":
    sys.exit(main())


### Vector generator — `tools/generate_vectors.py`

In [19]:
%%writefile tools/generate_vectors.py
#!/usr/bin/env python3
"""
Deterministic conformance-vector generator for Provenance Razor v0.3.0-alpha.

    python tools/generate_vectors.py            # writes into ./tests
    python tools/generate_vectors.py --out DIR  # writes elsewhere (used by the determinism check)

Every case declares its *intended* outcome (from the spec text). The generator asks the
reference model for the outcome and refuses to write anything if the two disagree, so a
bug in the reference model cannot silently become a "golden" value.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import struct
import sys
from datetime import datetime, timezone

import yaml

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "reference"))
try:
    import prc_ref as R  # noqa: E402
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
except ModuleNotFoundError:
    raise SystemExit(
        "prc_ref not found. This script needs reference/prc_ref.py next to tools/ "
        "(repo layout: reference/prc_ref.py, tools/%s). In the notebook, run the "
        "'Reference model' %%%%writefile cell first, or unzip the whole repo." % os.path.basename(__file__))

import trust_fixture as TF  # noqa: E402  (single committed trust context, 0.3.0)

BASE_MS = 1767225600000            # 2026-01-01T00:00:00Z — fixed, never the wall clock
BASE_US = BASE_MS * 1000
S = 1_000_000                      # one second in microseconds
EPOCH = "epoch-2026-01"


def iso(us: int) -> str:
    return datetime.fromtimestamp(us / S, tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")


# ---------------------------------------------------------------------------
# Fixed test keys and shared material
# ---------------------------------------------------------------------------
K_PROFILE = R.TestKey.derive("profile", "k1")
K_APPROVAL = R.TestKey.derive("approval", "k1")
K_DECISION = R.TestKey.derive("decision", "k1")
K_RECEIPT = R.TestKey.derive("receipt", "k1")
KEYS = [K_PROFILE, K_APPROVAL, K_DECISION, K_RECEIPT]

REGISTRY = {
    K_PROFILE.key_id: R.registry_entry(K_PROFILE, [EPOCH]),
    K_APPROVAL.key_id: R.registry_entry(K_APPROVAL, [EPOCH], ["STANDING", "SUPERVISOR"]),
    K_DECISION.key_id: R.registry_entry(K_DECISION, [EPOCH]),
    K_RECEIPT.key_id: R.registry_entry(K_RECEIPT, [EPOCH]),
}

PROFILE = {
    "profile_version": "1.0.0",
    "manual_only_actions": ["wire.release"],
    "max_eval_skew_us": {"*": 300 * S, "payment.refund": 120 * S},
    "reusable_authority_classes": ["STANDING"],
}
PROFILE_HASH = R.H(PROFILE)

EVIDENCE = [
    {"evidence_id": R.uuid7(BASE_MS - 5000, "ev-1"), "source": "ledger", "value_bits": "3fe999999999999a"},
    {"evidence_id": R.uuid7(BASE_MS - 4000, "ev-2"), "source": "kyc", "value_bits": "3fe8000000000000"},
]
EVIDENCE_HASH = R.H(EVIDENCE)
LINEAGE = {"nodes": [e["evidence_id"] for e in EVIDENCE], "edges": []}
LINEAGE_HASH = R.H(LINEAGE)

X_REFUND = {"action": "payment.refund", "params": {"amount_minor": 12500, "currency": "USD"}}
A_REFUND = {"assertion": "refund-eligible", "subject": "order-8841"}
X_WIRE = {"action": "wire.release", "params": {"amount_minor": 9900000, "currency": "USD"}}
A_WIRE = {"assertion": "beneficiary-verified", "subject": "wire-5521"}
TARGET = "acct-000123"
ACTOR = {"id": "svc-ledger-bot", "type": "service"}
STATE_VER = 7


def decision(tag: str, **o) -> dict:
    d = {
        "domain": R.DOMAIN_DECISION,
        "schema_ver": R.SPEC_VERSION,
        "profile_version": "1.0.0",
        "lineage_schema_version": "1.0.0",
        "capability_id": R.uuid7(BASE_MS, "cap|" + tag),
        "req_id": R.uuid7(BASE_MS, "req|" + tag),
        "evidence_snapshot_id": R.uuid7(BASE_MS, "snap|" + tag),
        "actor": ACTOR,
        "auth_mode": "AUTOMATED",
        "x": X_REFUND,
        "a": A_REFUND,
        "target": TARGET,
        "state_ver": STATE_VER,
        "control_profile_hash": PROFILE_HASH,
        "scoring_profile_hash": TF.SCORING_PROFILE_HASH,
        "trust_state_hash": TF.TRUST_STATE_HASH,
        "evidence_hash": EVIDENCE_HASH,
        "lineage_hash": LINEAGE_HASH,
        "approval_hash": R.NULL_APPROVAL_DIGEST,
        "adverse_disposition": "CLEAR",
        "containment_digest": R.NULL_CONTAINMENT_DIGEST,
        "eval_clock_id": TF.CLOCK_IDEAL,
        "decision_key_id": K_DECISION.key_id,
        "decision_trust_epoch": EPOCH,
        "tau_eval": BASE_US,
        "tau_expiry": BASE_US + 600 * S,
        "decision": "ALLOW",
        "reason_codes": 0,
    }
    d.update(o)
    return d


def approval(tag: str, **o) -> dict:
    p = {
        "domain": R.DOMAIN_APPROVAL,
        "schema_ver": R.SPEC_VERSION,
        "approval_id": R.uuid7(BASE_MS - 60000, "appr|" + tag),
        "approval_key_id": K_APPROVAL.key_id,
        "approval_trust_epoch": EPOCH,
        "authority_class": "SUPERVISOR",
        "authority_scope": "wire.release:acct-*",
        "x": X_WIRE,
        "a": A_WIRE,
        "target": TARGET,
        "actor": ACTOR,
        "req_id": R.uuid7(BASE_MS, "req|" + tag),
        "capability_id": R.uuid7(BASE_MS, "cap|" + tag),
        "state_ver": STATE_VER,
        "control_profile_hash": PROFILE_HASH,
        "scoring_profile_hash": TF.SCORING_PROFILE_HASH,
        "trust_state_hash": TF.TRUST_STATE_HASH,
        "adverse_disposition": "CLEAR",
        "containment_digest": R.NULL_CONTAINMENT_DIGEST,
        "tau_issued": BASE_US - 60 * S,
        "tau_expiry": BASE_US + 900 * S,
        "decision": "ALLOW",
    }
    p.update(o)
    return p


def context(tau_exec=BASE_US + 60 * S, clock=None, uncertainty_us=0, **o) -> dict:
    c = {
        "x": X_REFUND,
        "a": A_REFUND,
        "current_target": TARGET,
        "current_state_ver": STATE_VER,
        "active_control_profile_hash": PROFILE_HASH,
        "active_scoring_profile_hash": TF.SCORING_PROFILE_HASH,
        "trust_state_hash": TF.TRUST_STATE_HASH,
        "evidence_hash": EVIDENCE_HASH,
        "lineage_hash": LINEAGE_HASH,
        "adverse_disposition": "CLEAR",
        "containment_digest": R.NULL_CONTAINMENT_DIGEST,
        "exec_clock": {"clock_id": clock or TF.CLOCK_IDEAL, "tau_exec": tau_exec, "uncertainty_us": uncertainty_us},
        "profile": PROFILE,
        "consumed_capabilities": [],
        "consumed_approvals": [],
    }
    c.update(o)
    return c


def wire_ctx(**o) -> dict:
    return context(x=X_WIRE, a=A_WIRE, **o)


def encode_map_unsorted(d: dict, order: list) -> bytes:
    """Deliberately non-canonical map encoding (keys in the given order)."""
    body = b"".join(R.encode(k) + R.encode(d[k]) for k in order)
    return bytes([R.T_MAP]) + struct.pack(">Q", len(body)) + body


# ---------------------------------------------------------------------------
# Authorization & expiry vectors (§3.1–3.5)
# ---------------------------------------------------------------------------

def auth_cases():
    """(id, scenario, spec_refs, d, sign_key, p, p_sign_key, ctx, expected_reason_names, extras)"""
    C = []

    def add(tid, scenario, refs, d, ctx, expect, key=K_DECISION, p=None, pkey=K_APPROVAL,
            d_bytes=None, gap=None, note=None, trust=None):
        C.append(dict(tid=tid, scenario=scenario, refs=refs, d=d, key=key, p=p, pkey=pkey, ctx=ctx,
                      expect=expect, d_bytes=d_bytes, gap=gap, note=note, trust=trust))

    # --- freshness & skew -----------------------------------------------------------------
    add("auth_expiry_001", "Valid AUTOMATED decision", ["3.5"], decision("001"), context(), [])
    add("auth_expiry_002", "Expired decision (exec after tau_expiry)", ["3.5"],
        decision("002", tau_eval=BASE_US - 3600 * S, tau_expiry=BASE_US - 1800 * S),
        context(tau_exec=BASE_US), ["EXPIRED_CAPABILITY", "EVAL_SKEW_EXCEEDED"],
        note="Exec is 1h after eval, so the 120s skew limit is also exceeded; both bits must be set.")
    add("auth_expiry_003", "Exec exactly at tau_expiry is valid (<=)", ["3.5"],
        decision("003", tau_expiry=BASE_US + 100 * S), context(tau_exec=BASE_US + 100 * S), [])
    add("auth_expiry_004", "Exec one microsecond after tau_expiry", ["3.5"],
        decision("004", tau_expiry=BASE_US + 100 * S), context(tau_exec=BASE_US + 100 * S + 1),
        ["EXPIRED_CAPABILITY"])
    add("auth_expiry_005", "Exec before tau_eval (decision from the future)", ["3.5"],
        decision("005"), context(tau_exec=BASE_US - 1), ["NOT_YET_VALID"])
    add("auth_expiry_006", "Unexpired but evaluation skew exceeded", ["3.5"],
        decision("006", tau_expiry=BASE_US + 3600 * S), context(tau_exec=BASE_US + 121 * S),
        ["EVAL_SKEW_EXCEEDED"])
    add("auth_expiry_007", "Skew exactly at per-action limit is valid", ["3.5"],
        decision("007"), context(tau_exec=BASE_US + 120 * S), [])
    add("auth_expiry_008", "Default ('*') skew limit applies to actions without an override", ["3.5"],
        decision("008", x=X_WIRE, a=A_WIRE, auth_mode="MANUAL",
                 approval_hash=R.H(approval("008"))),
        wire_ctx(tau_exec=BASE_US + 300 * S), [], p=approval("008"),
        note="Exec 300s after eval: exceeds payment.refund's 120s but equals the default 300s.")

    # --- key domains & signatures ---------------------------------------------------------
    add("auth_expiry_009", "Decision signed with a receipt-domain key", ["1", "3.5"],
        decision("009", decision_key_id=K_RECEIPT.key_id), context(), ["SIGNATURE_INVALID", "DECISION_KEY_UNAUTHORIZED"],
        key=K_RECEIPT, note="0.3.0: keys are resolved only inside the required domain, so a receipt key is not "
                            "found and the signature cannot verify — the same classification approvals get (031).")
    add("auth_expiry_010", "Unknown decision key id", ["3.5"],
        decision("010", decision_key_id="kid:decision:0000000000000000"), context(),
        ["SIGNATURE_INVALID", "DECISION_KEY_UNAUTHORIZED"])
    add("auth_expiry_011", "Decision key used outside its trust epoch", ["3.5"],
        decision("011", decision_trust_epoch="epoch-2025-12"), context(), ["DECISION_KEY_UNAUTHORIZED"])
    tampered = decision("012")
    add("auth_expiry_012", "Payload altered after signing", ["3.5"],
        dict(tampered, x={"action": "payment.refund", "params": {"amount_minor": 99999999, "currency": "USD"}}),
        context(x={"action": "payment.refund", "params": {"amount_minor": 99999999, "currency": "USD"}}),
        ["SIGNATURE_INVALID"], note="signature is over the original payload (see sign_original)")
    C[-1]["sign_original"] = tampered
    d013 = decision("013")
    order = list(d013.keys())  # insertion order, not canonical order
    add("auth_expiry_013", "Non-canonical encoding (unsorted map keys) with a valid signature", ["1", "3.5"],
        d013, context(), ["DECODE_INVALID"], d_bytes=encode_map_unsorted(d013, order))
    add("auth_expiry_014", "Wrong domain string in decision payload", ["3.5"],
        decision("014", domain=R.DOMAIN_RECEIPT), context(), ["DOMAIN_MISMATCH"])

    # --- bindings ---------------------------------------------------------------------------
    add("auth_expiry_015", "State version advanced since evaluation", ["3.5"],
        decision("015"), context(current_state_ver=STATE_VER + 1), ["STATE_VERSION_MISMATCH"])
    add("auth_expiry_016", "Different target entity", ["3.5"],
        decision("016"), context(current_target="acct-000999"), ["TARGET_MISMATCH"])
    add("auth_expiry_017", "Active profile changed", ["3.5"],
        decision("017"), context(active_control_profile_hash=R.H({"profile_version": "1.0.1"})), ["PROFILE_HASH_MISMATCH"])
    add("auth_expiry_018", "Evidence and lineage changed", ["3.5"],
        decision("018"), context(evidence_hash=R.H([]), lineage_hash=R.H({})),
        ["EVIDENCE_HASH_MISMATCH", "LINEAGE_HASH_MISMATCH"])
    add("auth_expiry_019", "Decision value is not ALLOW", ["3.5"],
        decision("019", decision="DENY"), context(), ["DECISION_NOT_ALLOW"])
    d020 = decision("020")
    add("auth_expiry_020", "Capability already consumed (replay)", ["3.5", "4.1"],
        d020, context(consumed_capabilities=[d020["capability_id"]]), ["CAPABILITY_UNAVAILABLE"])
    add("auth_expiry_021", "Multiple failures accumulate in the bitfield", ["3.5"],
        decision("021", tau_expiry=BASE_US + 30 * S), context(current_state_ver=STATE_VER + 2),
        ["STATE_VERSION_MISMATCH", "EXPIRED_CAPABILITY"])

    # --- authorization mode (bidirectional, §3.4) -------------------------------------------
    add("auth_expiry_022", "AUTOMATED on a manual-only action", ["3.4"],
        decision("022", x=X_WIRE, a=A_WIRE), wire_ctx(), ["AUTH_MODE_INVALID"])
    add("auth_expiry_023", "AUTOMATED with a non-null approval_hash", ["3.2", "3.4"],
        decision("023", approval_hash=R.H("something-else")), context(), ["AUTH_MODE_INVALID"])
    p024 = approval("024", x=X_REFUND, a=A_REFUND, authority_scope="payment.refund:*")
    add("auth_expiry_024", "MANUAL on an action that is not manual-only", ["3.4"],
        decision("024", auth_mode="MANUAL", approval_hash=R.H(p024)), context(), ["AUTH_MODE_INVALID"],
        p=p024, note="Approval itself is valid; the mode is rejected because x is not in ManualOnly.")
    add("auth_expiry_025", "MANUAL without an approval envelope", ["3.4"],
        decision("025", x=X_WIRE, a=A_WIRE, auth_mode="MANUAL", approval_hash=R.H(approval("025"))),
        wire_ctx(), ["AUTH_MODE_INVALID"])

    # --- manual approvals (§3.1) ------------------------------------------------------------
    def manual(tag, p, ctx, expect, pkey=K_APPROVAL, scenario="", note=None, gap=None, dover=None):
        d = decision(tag, x=X_WIRE, a=A_WIRE, auth_mode="MANUAL", approval_hash=R.H(p), **(dover or {}))
        add(f"auth_expiry_{tag}", scenario, ["3.1", "3.4", "3.5"], d, ctx, expect, p=p, pkey=pkey,
            note=note, gap=gap)

    manual("026", approval("026"), wire_ctx(), [], scenario="Valid MANUAL decision with approval")
    manual("027", approval("027", tau_expiry=BASE_US + 30 * S), wire_ctx(),
           ["APPROVAL_EXPIRED", "AUTH_MODE_INVALID"], scenario="Approval expired before execution")
    p028 = approval("028")
    manual("028", p028, wire_ctx(consumed_approvals=[p028["approval_id"]]),
           ["APPROVAL_CONSUMED", "AUTH_MODE_INVALID"], scenario="Single-use approval already consumed")
    p029 = approval("029", authority_class="STANDING")
    manual("029", p029, wire_ctx(consumed_approvals=[p029["approval_id"]]), [],
           scenario="Reusable (STANDING) approval already used is still valid")
    manual("030", approval("030", authority_scope="wire.release:acct-9*"), wire_ctx(),
           ["APPROVAL_SCOPE_DENIED", "AUTH_MODE_INVALID"], scenario="Approval scope does not cover target")
    manual("031", approval("031", approval_key_id=K_DECISION.key_id), wire_ctx(),
           ["APPROVAL_SIGNATURE_INVALID", "APPROVAL_KEY_UNAUTHORIZED", "AUTH_MODE_INVALID"],
           pkey=K_DECISION, scenario="Approval signed with a decision-domain key")
    manual("032", approval("032", authority_class="EXECUTIVE"), wire_ctx(),
           ["APPROVAL_KEY_UNAUTHORIZED", "AUTH_MODE_INVALID"],
           scenario="Approval key not authorized for the claimed authority class")
    manual("033", approval("033", state_ver=STATE_VER - 1), wire_ctx(),
           ["APPROVAL_BINDING_MISMATCH", "AUTH_MODE_INVALID"],
           scenario="Approval bound to an older state version")
    manual("034", approval("034", domain=R.DOMAIN_DECISION), wire_ctx(),
           ["APPROVAL_DOMAIN_MISMATCH", "AUTH_MODE_INVALID"], scenario="Approval carries the decision domain string")
    # approval_hash in d points to a different approval than the one presented
    p035 = approval("035")
    d035 = decision("035", x=X_WIRE, a=A_WIRE, auth_mode="MANUAL", approval_hash=R.H(approval("035-other")))
    add("auth_expiry_035", "approval_hash does not match presented approval", ["3.3", "3.4"],
        d035, wire_ctx(), ["APPROVAL_HASH_MISMATCH", "AUTH_MODE_INVALID"], p=p035)

    # --- 0.2.0 approval bindings: former spec gaps G-01..G-03, now must-reject ------------------
    manual("036", approval("036", target="acct-000777"), wire_ctx(),
           ["APPROVAL_SUBJECT_MISMATCH", "AUTH_MODE_INVALID"],
           scenario="Approval names a different target than the decision (closes G-01)",
           note="0.1.0 accepted this: an approval for acct-000777 authorised a wire on acct-000123.")
    manual("037", approval("037", tau_issued=BASE_US + 3600 * S), wire_ctx(),
           ["APPROVAL_NOT_YET_VALID", "AUTH_MODE_INVALID"],
           scenario="Approval issued after evaluation (closes G-02)",
           note="0.2.0 requires p.tau_issued <= d.tau_eval <= tau_exec <= p.tau_expiry.")
    manual("038", approval("038"), wire_ctx(), ["APPROVAL_SUBJECT_MISMATCH", "AUTH_MODE_INVALID"],
           dover={"actor": {"id": "someone-else", "type": "user"}},
           scenario="Decision actor differs from the approved actor (closes G-03)")
    manual("039", approval("039", req_id=R.uuid7(BASE_MS, "req|some-other-request")), wire_ctx(),
           ["APPROVAL_SUBJECT_MISMATCH", "AUTH_MODE_INVALID"],
           scenario="Approval for a different request id is replayed onto this decision (closes G-03)")
    manual("040", approval("040", capability_id=R.uuid7(BASE_MS, "cap|some-other-capability")), wire_ctx(),
           ["APPROVAL_SUBJECT_MISMATCH", "AUTH_MODE_INVALID"],
           scenario="Approval bound to a different capability id (closes G-03)",
           note="capability_id is pre-allocated at request time so the approver can bind to it.")
    manual("041", approval("041", tau_issued=BASE_US), wire_ctx(), [],
           scenario="Approval issued exactly at d.tau_eval is valid (<=)")

    # --- 0.3.0 CanonicalObjectValid: validly signed but malformed objects (V-1: never raise) --------
    def mut(tag, fn, **o):
        d = decision(tag, **o)
        fn(d)
        return d
    schema = [
        ("s01", "Extra field in a signed decision", lambda d: d.update(debug_override="allow-all")),
        ("s02", "Required field tau_eval missing", lambda d: d.pop("tau_eval")),
        ("s03", "tau_eval given as a string", lambda d: d.update(tau_eval="2026-01-01T00:00:00Z")),
        ("s04", "capability_id is a UUIDv4, not UUIDv7", lambda d: d.update(capability_id="0b7b2e8c-2f5c-4b9a-9e4d-5a2f1c3d4e5f")),
        ("s05", "evidence_hash is 31 bytes", lambda d: d.update(evidence_hash=d["evidence_hash"][:62])),
        ("s06", "Unsupported schema_ver", lambda d: d.update(schema_ver="9.9.9")),
        ("s07", "Unknown auth_mode string", lambda d: d.update(auth_mode="SEMI_AUTOMATED")),
        ("s08", "Negative state_ver", lambda d: d.update(state_ver=-1)),
        ("s09", "reason_codes with reserved bit 63 set (negative int64)", lambda d: d.update(reason_codes=-(2 ** 63))),
        ("s10", "actor.type outside {service, user}", lambda d: d.update(actor={"id": "svc-ledger-bot", "type": "robot"})),
        ("s11", "Boolean where an integer is required", lambda d: d.update(state_ver=True)),
        ("s12", "Uppercase hex digest", lambda d: d.update(lineage_hash=d["lineage_hash"].upper())),
    ]
    for sid, title, fn in schema:
        add(f"auth_schema_{sid}", title, ["1.5", "3.5"], mut(sid, fn), context(), ["OBJECT_SCHEMA_INVALID"],
            note="Before 0.3.0 some of these were accepted and some raised an exception inside the verifier.")
    p_extra = approval("s13")
    p_extra["note_to_self"] = "approved over the phone"
    add("auth_schema_s13", "Approval with an extra field", ["1.5", "3.1"],
        decision("s13", x=X_WIRE, a=A_WIRE, auth_mode="MANUAL", approval_hash=R.H(p_extra)), wire_ctx(),
        ["APPROVAL_SCHEMA_INVALID", "AUTH_MODE_INVALID"], p=p_extra)
    p_missing = approval("s14")
    p_missing.pop("capability_id")
    add("auth_schema_s14", "Approval missing capability_id", ["1.5", "3.1"],
        decision("s14", x=X_WIRE, a=A_WIRE, auth_mode="MANUAL", approval_hash=R.H(p_missing)), wire_ctx(),
        ["APPROVAL_SCHEMA_INVALID", "AUTH_MODE_INVALID"], p=p_missing)

    # --- 0.3.0 trust-context binding --------------------------------------------------------------
    other_ts = dict(TF.TRUST_STATE, revoked_attestation_ids=["00000000-0000-7000-8000-000000000000"])
    add("auth_trust_t01", "Decision evaluated under a different trust state", ["3.3", "3.5"],
        decision("t01", trust_state_hash=R.H(other_ts)), context(), ["TRUST_STATE_MISMATCH"])
    add("auth_trust_t02", "Decision evaluated under a different scoring profile", ["3.3", "3.5"],
        decision("t02", scoring_profile_hash=R.H({"id": "coverage-v0.2.1-alpha"})), context(), ["SCORING_PROFILE_MISMATCH"])
    add("auth_trust_t03", "Verifier handed a trust state that does not match the hash it was told to enforce",
        ["3.5"], decision("t03"), context(), ["TRUST_STATE_MISMATCH"], trust=other_ts)

    # --- 0.3.0 adverse disposition: containment requires bound manual authorization ----------------
    CD = R.H(["c0ffee" * 10 + "abcd"])  # a containment digest (hash of the obligation list)
    cctx = dict(adverse_disposition="CONTAIN", containment_digest=CD)
    add("auth_contain_a01", "CONTAIN evidence with an AUTOMATED decision", ["2.5", "3.4"],
        decision("a01", adverse_disposition="CONTAIN", containment_digest=CD), context(**cctx),
        ["AUTH_MODE_INVALID", "ADVERSE_CONTAINMENT"])
    p_ack = approval("a02", x=X_REFUND, a=A_REFUND, authority_scope="payment.refund:*",
                     adverse_disposition="CONTAIN", containment_digest=CD)
    add("auth_contain_a02", "CONTAIN evidence with MANUAL approval that acknowledges the exact claims", ["2.5", "3.4"],
        decision("a02", adverse_disposition="CONTAIN", containment_digest=CD, auth_mode="MANUAL",
                 approval_hash=R.H(p_ack)), context(**cctx), [], p=p_ack,
        note="payment.refund is not ManualOnly; containment is what makes MANUAL the required mode.")
    p_clear = approval("a03", x=X_REFUND, a=A_REFUND, authority_scope="payment.refund:*")
    add("auth_contain_a03", "Approval acknowledges CLEAR but the decision carries CONTAIN", ["2.5", "3.1"],
        decision("a03", adverse_disposition="CONTAIN", containment_digest=CD, auth_mode="MANUAL",
                 approval_hash=R.H(p_clear)), context(**cctx), ["ADVERSE_DISPOSITION_MISMATCH", "AUTH_MODE_INVALID"],
        p=p_clear)
    p_other = approval("a04", x=X_REFUND, a=A_REFUND, authority_scope="payment.refund:*",
                       adverse_disposition="CONTAIN", containment_digest=R.H(["another-claim"]))
    add("auth_contain_a04", "Approval acknowledged a different set of adverse claims", ["2.5", "3.1"],
        decision("a04", adverse_disposition="CONTAIN", containment_digest=CD, auth_mode="MANUAL",
                 approval_hash=R.H(p_other)), context(**cctx), ["ADVERSE_DISPOSITION_MISMATCH", "AUTH_MODE_INVALID"],
        p=p_other)
    add("auth_contain_a05", "Decision says CLEAR but current evidence is CONTAIN", ["2.5", "3.5"],
        decision("a05"), context(**cctx), ["ADVERSE_DISPOSITION_MISMATCH"])
    add("auth_contain_a06", "CONTAIN with the null containment digest is internally inconsistent", ["2.5", "3.3"],
        decision("a06", adverse_disposition="CONTAIN"), context(adverse_disposition="CONTAIN"),
        ["ADVERSE_DISPOSITION_MISMATCH", "AUTH_MODE_INVALID", "ADVERSE_CONTAINMENT"])

    # --- 0.3.0 C-1 clock authority (closes G-10) --------------------------------------------------
    add("auth_clock_k01", "Executor clock is not a registered clock authority", ["3.5"],
        decision("k01"), context(clock="clk:laptop"), ["CLOCK_UNTRUSTED"])
    add("auth_clock_k02", "Executor reports uncertainty above its authority's maximum", ["3.5"],
        decision("k02"), context(clock=TF.CLOCK_NTP, uncertainty_us=600_000), ["CLOCK_UNTRUSTED"])
    add("auth_clock_k03", "Evaluator clock in the decision is not registered", ["3.5"],
        decision("k03", eval_clock_id="clk:rogue"), context(), ["CLOCK_UNTRUSTED"])
    add("auth_clock_k04", "200 ms before expiry with a 1 ms clock: valid", ["3.5"],
        decision("k04", tau_expiry=BASE_US + 60 * S + 200_000), context(clock=TF.CLOCK_PTP, uncertainty_us=1_000), [])
    add("auth_clock_k05", "Same timing with a 500 ms clock: expiry cannot be excluded", ["3.5"],
        decision("k05", tau_expiry=BASE_US + 60 * S + 200_000), context(clock=TF.CLOCK_NTP, uncertainty_us=500_000),
        ["EXPIRED_CAPABILITY"])
    add("auth_clock_k06", "Exec 100 ms after eval with a 500 ms clock: ordering cannot be proven", ["3.5"],
        decision("k06"), context(tau_exec=BASE_US + 100_000, clock=TF.CLOCK_NTP, uncertainty_us=500_000),
        ["NOT_YET_VALID"])
    add("auth_clock_k07", "Skew limit tightened by clock uncertainty", ["3.5"],
        decision("k07", tau_expiry=BASE_US + 3600 * S), context(tau_exec=BASE_US + 119_600_000, clock=TF.CLOCK_NTP,
                                                              uncertainty_us=500_000),
        ["EVAL_SKEW_EXCEEDED"], note="119.6 s + 0.5 s > 120 s for payment.refund.")
    return C


def build_auth_vectors():
    out = []
    for c in auth_cases():
        d, key, ctx = c["d"], c["key"], c["ctx"]
        if c.get("sign_original"):
            sig = key.sign_digest(R.H(c["sign_original"]))
            d_bytes = R.encode(d)
        elif c["d_bytes"] is not None:
            d_bytes = c["d_bytes"]
            sig = key.sign_digest(R.H_bytes(d_bytes))
        else:
            d_bytes = R.encode(d)
            sig = key.sign_digest(R.H_bytes(d_bytes))
        p_hex = p_sig = None
        if c["p"] is not None:
            pb = R.encode(c["p"])
            p_hex, p_sig = pb.hex(), c["pkey"].sign_digest(R.H_bytes(pb))
        full_ctx = dict(ctx, trust_state=c["trust"] or TF.TRUST_STATE)
        valid, bits = R.decision_binding_valid(full_ctx, d_bytes.hex(), sig, p_hex, p_sig)
        want = sum(R.RC[n] for n in c["expect"])
        if bits != want:
            raise SystemExit(f"{c['tid']}: reference model gives {R.reason_names(bits)}, "
                             f"case declares {c['expect']}")
        v = {
            "test_id": c["tid"],
            "scenario": c["scenario"],
            "spec_refs": c["refs"],
            "input": {
                "decision_payload": d,
                "decision_bytes": d_bytes.hex(),
                "decision_hash": R.H_bytes(d_bytes),
                "decision_signature": sig,
                "approval_payload": c["p"],
                "approval_bytes": p_hex,
                "approval_signature": p_sig,
                "context": ctx,
            },
            "expected_output": {
                "DecisionBindingValid": valid,
                "reason_bitfield": bits,
                "reason_codes": R.reason_names(bits),
            },
            "timestamps_iso": {k: (iso(v) if isinstance(v, int) and not isinstance(v, bool) and v >= 0 else None)
                               for k, v in (("tau_eval", d.get("tau_eval")), ("tau_expiry", d.get("tau_expiry")),
                                            ("tau_exec", ctx["exec_clock"]["tau_exec"]))},
        }
        if c["trust"] is not None:
            v["trust_state"] = c["trust"]
        if c["note"]:
            v["note"] = c["note"]
        if c["gap"]:
            v["spec_gap"] = c["gap"]
        out.append(v)
    return {
        "vector_set": "authorization-expiry-skew",
        "spec_version": R.SPEC_VERSION,
        "encoding": R.ENCODING_ID,
        "time_unit": "microseconds since Unix epoch, UTC",
        "trust_state": TF.TRUST_STATE,
        "trust_state_hash": TF.TRUST_STATE_HASH,
        "reason_code_registry": "tests/reason_codes.json",
        "vectors": out,
    }


# ---------------------------------------------------------------------------
# Golden binary vectors (§1, §3, §4.3)
# ---------------------------------------------------------------------------

def tj(v):
    """Python value -> typed JSON (lossless, language-neutral)."""
    if v is None:
        return {"type": "null"}
    if isinstance(v, bool):
        return {"type": "bool", "value": v}
    if isinstance(v, int):
        return {"type": "int", "value": str(v)}
    if isinstance(v, R.F64):
        return {"type": "f64", "bits": f"{v.bits:016x}"}
    if isinstance(v, str):
        return {"type": "str", "value": v}
    if isinstance(v, bytes):
        return {"type": "bytes", "hex": v.hex()}
    if isinstance(v, list):
        return {"type": "list", "items": [tj(i) for i in v]}
    if isinstance(v, dict):
        return {"type": "map", "entries": [[k, tj(v[k])] for k in v]}
    raise TypeError(type(v))


def build_golden():
    prims = [
        ("null", None), ("false", False), ("true", True),
        ("int_zero", 0), ("int_neg_one", -1), ("int64_max", R.INT64_MAX), ("int64_min", R.INT64_MIN),
        ("f64_pos_zero", R.F64(0x0000000000000000)), ("f64_neg_zero", R.F64(0x8000000000000000)),
        ("f64_min_subnormal", R.F64(0x0000000000000001)), ("f64_quiet_nan", R.F64(0x7FF8000000000000)),
        ("str_empty", ""), ("str_ascii", "provenance"), ("str_unicode", "\u00e9\U0001F642\u4e2d"),
        ("str_nfc_vs_nfd", "é"),
        ("bytes_empty", b""), ("bytes_sha256_len", bytes(range(32))),
        ("list_empty", []), ("list_mixed", [1, "a", None, True]),
        ("map_empty", {}),
        ("map_key_order", {"b": 1, "a": 2, "B": 3, "aa": 4, "é": 5, "": 6}),
        ("map_nested", {"z": {"y": [1, {"x": None}]}, "a": []}),
    ]
    primitives = []
    for pid, v in prims:
        b = R.encode(v)
        assert R.encode(R.decode(b)) == b
        primitives.append({"id": pid, "value": tj(v), "expected_bytes": b.hex(), "expected_hash": R.H_bytes(b)})

    good = R.encode({"a": 1, "b": 2})
    rejects = [
        ("unsorted_keys", encode_map_unsorted({"a": 1, "b": 2}, ["b", "a"]), "map keys not ascending"),
        ("duplicate_keys", bytes([R.T_MAP]) + struct.pack(">Q", 2 * len(R.encode("a") + R.encode(1)))
         + (R.encode("a") + R.encode(1)) * 2, "duplicate map key"),
        ("trailing_bytes", good + b"\x00", "trailing bytes after top-level value"),
        ("truncated", good[:-1], "length exceeds buffer"),
        ("bool_with_length", bytes([R.T_TRUE]) + struct.pack(">Q", 1) + b"\x01", "bool must have length 0"),
        ("int_short", bytes([R.T_INT]) + struct.pack(">Q", 4) + b"\x00\x00\x00\x01", "int must be 8 bytes"),
        ("invalid_utf8", bytes([R.T_STR]) + struct.pack(">Q", 2) + b"\xc3\x28", "invalid UTF-8"),
        ("non_string_key", bytes([R.T_MAP]) + struct.pack(">Q", len(R.encode(1) + R.encode(2)))
         + R.encode(1) + R.encode(2), "map key must be a string"),
        ("unknown_tag", bytes([0x7F]) + struct.pack(">Q", 0), "unknown tag"),
    ]
    reject_out = []
    for rid, b, why in rejects:
        try:
            R.decode(b)
            raise SystemExit(f"reject vector {rid} was accepted by the reference decoder")
        except R.CanonicalDecodeError:
            pass
        reject_out.append({"id": rid, "bytes": b.hex(), "must_reject": True, "reason": why})

    p = approval("golden-manual")
    d_auto = decision("golden-auto")
    d_man = decision("golden-manual", x=X_WIRE, a=A_WIRE, auth_mode="MANUAL", approval_hash=R.H(p))
    pre = {"version": STATE_VER, "data": {"balance_minor": 50000}}
    post = R.apply_mutation(pre, {"balance_minor": 37500})
    r = R.build_receipt(d=d_auto, d_hash=R.H(d_auto), receipt_id=R.uuid7(BASE_MS + 61000, "rcpt|golden"),
                        status="COMMITTED", failure_phase="NONE", reason=0, recon_required=False,
                        pre_state=pre, post_state=post, tau_dispatch=BASE_US + 60 * S,
                        tau_finalized=BASE_US + 61 * S, key=K_RECEIPT, epoch=EPOCH)
    objects = []
    for oid, typ, payload, key in [
        ("profile_default", "profile", PROFILE, K_PROFILE),
        ("approval_supervisor", "approval", p, K_APPROVAL),
        ("decision_automated", "decision", d_auto, K_DECISION),
        ("decision_manual", "decision", d_man, K_DECISION),
        ("receipt_committed", "receipt", r, K_RECEIPT),
    ]:
        b = R.encode(payload)
        h = R.H_bytes(b)
        objects.append({"id": oid, "artifact_type": typ, "payload": payload, "expected_bytes": b.hex(),
                        "expected_hash": h, "signing_key_id": key.key_id, "expected_signature": key.sign_digest(h)})

    return {
        "vector_set": "golden-binary",
        "spec_version": R.SPEC_VERSION,
        "encoding": R.ENCODING_ID,
        "encoding_definition": "docs/CONFORMANCE_ASSUMPTIONS.md#a-01",
        "hash": "SHA-256 over the PRC-LP-0 encoding (single layer, A-02)",
        "signature": "Ed25519 (RFC 8032) over the raw 32-byte digest H(z)",
        "WARNING": "Test keys are derived from public seeds. Never use them outside conformance testing.",
        "keys": [{"domain": k.domain, "label": k.label, "seed": k.seed.hex(), "public_key": k.public_hex,
                  "key_id": k.key_id} for k in KEYS],
        "constants": {
            "NULL_APPROVAL_DIGEST": {"preimage": R.NO_APPROVAL_LABEL,
                                     "preimage_bytes": R.encode(R.NO_APPROVAL_LABEL).hex(),
                                     "value": R.NULL_APPROVAL_DIGEST},
            "NULL_STATE_DIGEST": {"preimage": R.NO_STATE_LABEL,
                                  "preimage_bytes": R.encode(R.NO_STATE_LABEL).hex(),
                                  "value": R.NULL_STATE_DIGEST},
            "UUIDv7_example": {"unix_ms": BASE_MS, "label": "example", "value": R.uuid7(BASE_MS, "example")},
        },
        "primitives": primitives,
        "must_reject": reject_out,
        "objects": objects,
    }


# ---------------------------------------------------------------------------
# Numerical vectors (§2.3)
# ---------------------------------------------------------------------------
OP_AUTH_CHECK, OP_KAHAN_SUM = 1, 2
b = R.bits_of
NUM_CASES = [
    # (id, op, input bits, expected result(bool)|None, expected output bits, expected lock, note)
    ("above_threshold", OP_AUTH_CHECK, [b(0.75), b(0.7)], True, b(0.75), False, ""),
    ("equal_threshold", OP_AUTH_CHECK, [b(0.7), b(0.7)], True, b(0.7), False, ">= is inclusive"),
    ("one_ulp_below", OP_AUTH_CHECK, [b(0.7) - 1, b(0.7)], False, b(0.7) - 1, False, "no epsilon tolerance"),
    ("sum_0.1_0.2_vs_0.3", OP_AUTH_CHECK, [b(0.1 + 0.2), b(0.3)], True, b(0.1 + 0.2), False,
     "0.1+0.2 = 0.30000000000000004 > 0.3"),
    ("0.3_vs_sum_0.1_0.2", OP_AUTH_CHECK, [b(0.3), b(0.1 + 0.2)], False, b(0.3), False, ""),
    ("neg_zero_normalised", OP_AUTH_CHECK, [0x8000000000000000, b(0.0)], True, 0, False, "-0.0 -> +0.0"),
    ("threshold_neg_zero", OP_AUTH_CHECK, [b(0.0), 0x8000000000000000], True, 0, False, ""),
    ("min_subnormal_preserved", OP_AUTH_CHECK, [0x1, b(0.0)], True, 0x1, False, "no flush-to-zero"),
    ("subnormal_order", OP_AUTH_CHECK, [0x1, 0x2], False, 0x1, False, "5e-324 < 1e-323"),
    ("max_subnormal_vs_min_normal", OP_AUTH_CHECK, [0x000FFFFFFFFFFFFF, 0x0010000000000000], False,
     0x000FFFFFFFFFFFFF, False, ""),
    ("quiet_nan", OP_AUTH_CHECK, [0x7FF8000000000000, b(0.5)], False, 0, True, "C forced to 0.0, State 2"),
    ("signaling_nan", OP_AUTH_CHECK, [0x7FF0000000000001, b(0.0)], False, 0, True,
     "lock denies even though 0.0 >= 0.0 (A-05)"),
    ("negative_nan", OP_AUTH_CHECK, [0xFFF8000000000000, b(0.5)], False, 0, True, ""),
    ("pos_inf", OP_AUTH_CHECK, [0x7FF0000000000000, b(0.5)], False, 0, True, "Inf is an exception, not 'very high'"),
    ("neg_inf", OP_AUTH_CHECK, [0xFFF0000000000000, b(0.5)], False, 0, True, ""),
    ("nan_threshold", OP_AUTH_CHECK, [b(0.9), 0x7FF8000000000000], False, 0, True, "A-04: exceptional T also locks"),
    ("kahan_small_terms", OP_KAHAN_SUM, [b(1.0)] + [b(1e-16)] * 10, None, b(1.000000000000001), False,
     "naive left-to-right sum returns exactly 1.0"),
    ("kahan_tenths", OP_KAHAN_SUM, [b(0.1)] * 10, None, b(1.0), False, "naive sum gives 0.9999999999999999"),
    ("kahan_neg_zeros", OP_KAHAN_SUM, [0x8000000000000000] * 2, None, 0, False, "-0.0 result normalised"),
    ("kahan_subnormals", OP_KAHAN_SUM, [0x1] * 3, None, 0x3, False, "subnormals preserved"),
    ("kahan_empty", OP_KAHAN_SUM, [], None, 0, False, ""),
    ("kahan_overflow", OP_KAHAN_SUM, [b(1e308), b(1e308)], None, 0, True, "intermediate Inf -> lock"),
    ("kahan_nan_input", OP_KAHAN_SUM, [b(1.0), 0x7FF8000000000000], None, 0, True, ""),
]


def build_numeric():
    recs, manifest = [], []
    for cid, op, ins, want_res, want_bits, want_lock, note in NUM_CASES:
        if op == OP_AUTH_CHECK:
            got = R.authorization_check(*ins)
            ok = (got["result"], got["c_effective_bits"], got["safe_state_lock"]) == (want_res, want_bits, want_lock)
            out_bits, res = got["c_effective_bits"], got["result"]
        else:
            got = R.kahan_sum(ins)
            ok = (got["result_bits"], got["safe_state_lock"]) == (want_bits, want_lock)
            out_bits, res = got["result_bits"], False
        if not ok:
            raise SystemExit(f"numeric case {cid}: reference gives {got}")
        flags = (1 if res else 0) | (2 if want_lock else 0)
        recs.append(struct.pack("<BBHI", op, flags, 0, len(ins)) + b"".join(struct.pack("<Q", x) for x in ins)
                    + struct.pack("<Q", out_bits))
        manifest.append({
            "id": cid, "op": {1: "AUTH_CHECK", 2: "KAHAN_SUM"}[op],
            "inputs_bits": [f"{x:016x}" for x in ins],
            "inputs_repr": [repr(R.f64_from_bits(x)) for x in ins],
            "expected": ({"result": res} if op == OP_AUTH_CHECK else {}) | {
                "output_bits": f"{out_bits:016x}", "output_repr": repr(R.f64_from_bits(out_bits)),
                "safe_state_lock": want_lock},
            "normative": op == OP_AUTH_CHECK,
            "note": note,
        })
    blob = b"PRNV" + struct.pack("<HHI", 1, 0, len(recs)) + b"".join(recs)
    doc = {
        "vector_set": "numerical-ieee754",
        "spec_version": R.SPEC_VERSION,
        "binary_file": "kahan_cordic_ieee754.bin",
        "binary_sha256": hashlib.sha256(blob).hexdigest(),
        "binary_layout": {
            "header": "magic 'PRNV' | u16 version=1 | u16 reserved=0 | u32 record_count  (little-endian)",
            "record": "u8 op (1=AUTH_CHECK, 2=KAHAN_SUM) | u8 flags (bit0 result, bit1 safe_state_lock) | "
                      "u16 reserved=0 | u32 n | n x u64 input bit patterns | u64 expected output bits",
            "AUTH_CHECK": "inputs [C, T]; output = effective C after §2.3 handling",
            "KAHAN_SUM": "informative harness op (not in spec text); output = compensated sum",
        },
        "cordic": "No CORDIC vectors: the specification does not define a CORDIC operation (see A-06).",
        "cases": manifest,
    }
    return blob, doc


# ---------------------------------------------------------------------------
# Lineage fixtures (§2.1–2.2)
# ---------------------------------------------------------------------------

def rep(i, v):
    return {"id": i, "value": v}


LINEAGE_CASES = [
    ("01_singleton_valid", "Single class, valid optional member", {
        "kappa_degraded": 0.5,
        "classes": [{"id": "F1", "representative": rep("ev-a", 0.8),
                     "optional_members": [{"id": "ev-a2", "state": "VALID"}]}]},
     {"eligible": ["F1"], "V": {"F1": 0.8}, "I": {"F1": 1.0}}),
    ("02_degraded", "DEGRADED member applies kappa", {
        "kappa_degraded": 0.5,
        "classes": [{"id": "F1", "representative": rep("ev-a", 0.9),
                     "optional_members": [{"id": "ev-a2", "state": "DEGRADED"}]}]},
     {"eligible": ["F1"], "V": {"F1": 0.45}, "I": {"F1": 1.0}}),
    ("03_contradictory", "CONTRADICTORY member zeroes A; class stays eligible", {
        "kappa_degraded": 0.5,
        "classes": [{"id": "F1", "representative": rep("ev-a", 0.9),
                     "optional_members": [{"id": "ev-b", "state": "CONTRADICTORY"}]}]},
     {"eligible": ["F1"], "V": {"F1": 0.0}, "I": {"F1": 1.0}}),
    ("04_unavailable_ignored", "UNAVAILABLE_OR_STALE leaves A unaffected", {
        "kappa_degraded": 0.5,
        "classes": [{"id": "F1", "representative": rep("ev-a", 0.7),
                     "optional_members": [{"id": "ev-a2", "state": "UNAVAILABLE_OR_STALE"}]}]},
     {"eligible": ["F1"], "V": {"F1": 0.7}, "I": {"F1": 1.0}}),
    ("05_invalid_family", "INVALID member invalidates the family and emits an audit event", {
        "kappa_degraded": 0.5,
        "classes": [{"id": "F1", "representative": rep("ev-a", 0.95),
                     "optional_members": [{"id": "ev-a2", "state": "INVALID"}]}]},
     {"eligible": [], "V": {}, "I": {}, "C": 0.0, "audit": ["F1"]}),
    ("06_empty_evidence", "No evidence at all: M = C = 0.0", {"kappa_degraded": 0.5, "classes": []},
     {"eligible": [], "V": {}, "I": {}, "C": 0.0, "audit": []}),
    ("07_pairwise_min", "Three classes: I_k is the minimum pairwise conditional independence", {
        "kappa_degraded": 0.5,
        "classes": [{"id": "F1", "representative": rep("ev-a", 0.8)},
                    {"id": "F2", "representative": rep("ev-b", 0.6)},
                    {"id": "F3", "representative": rep("ev-c", 0.5)}],
        "pairwise_independence": {"F1|F2": 0.9, "F1|F3": 0.4, "F2|F1": 0.7, "F2|F3": 1.0,
                                  "F3|F1": 0.3, "F3|F2": 0.6}},
     {"eligible": ["F1", "F2", "F3"], "V": {"F1": 0.8, "F2": 0.6, "F3": 0.5}, "I": {"F1": 0.4, "F2": 0.7, "F3": 0.3}}),
    ("08_invalidation_shrinks_pairs", "An invalidated class is excluded from pairwise reduction", {
        "kappa_degraded": 0.5,
        "classes": [{"id": "F1", "representative": rep("ev-a", 0.8)},
                    {"id": "F2", "representative": rep("ev-b", 0.6)},
                    {"id": "F3", "representative": rep("ev-c", 0.5),
                     "optional_members": [{"id": "ev-c2", "state": "INVALID"}]}],
        "pairwise_independence": {"F1|F2": 0.9, "F1|F3": 0.1, "F2|F1": 0.7, "F2|F3": 0.1,
                                  "F3|F1": 0.1, "F3|F2": 0.1}},
     {"eligible": ["F1", "F2"], "V": {"F1": 0.8, "F2": 0.6}, "I": {"F1": 0.9, "F2": 0.7}, "audit": ["F3"]}),
    ("09_invalidation_to_singleton", "Invalidation leaves one class, which takes the singleton identity I = 1.0", {
        "kappa_degraded": 0.5,
        "classes": [{"id": "F1", "representative": rep("ev-a", 0.8)},
                    {"id": "F2", "representative": rep("ev-b", 0.6),
                     "optional_members": [{"id": "ev-b2", "state": "INVALID"}]}],
        "pairwise_independence": {"F1|F2": 0.2, "F2|F1": 0.2}},
     {"eligible": ["F1"], "V": {"F1": 0.8}, "I": {"F1": 1.0}, "audit": ["F2"]}),
    ("10_mixed_states_min", "Several optional members: A takes the most adverse factor", {
        "kappa_degraded": 0.25,
        "classes": [{"id": "F1", "representative": rep("ev-a", 0.8),
                     "optional_members": [{"id": "m1", "state": "VALID"}, {"id": "m2", "state": "DEGRADED"},
                                          {"id": "m3", "state": "UNAVAILABLE_OR_STALE"}]}]},
     {"eligible": ["F1"], "V": {"F1": 0.2}, "I": {"F1": 1.0}}),
]


def build_lineage():
    files = {}
    for fid, title, case, want in LINEAGE_CASES:
        got = R.evaluate_lineage(case)
        assert got["eligible"] == want["eligible"], (fid, got)
        for k, v in want["V"].items():
            assert got["classes"][k]["V"] == v, (fid, k, got["classes"][k]["V"], v)
        for k, v in want["I"].items():
            assert got["classes"][k]["I"] == v, (fid, k)
        if "C" in want:
            assert got["C"] == want["C"], fid
        assert [e["class"] for e in got["audit_events"]] == want.get("audit", []), fid
        exp = {"eligible_classes": got["eligible"], "classes": {}, "audit_events": got["audit_events"],
               "M": got["M"], "C": got["C"], "reason_bitfield": got["reason_code"],
               "reason_codes": R.reason_names(got["reason_code"])}
        for k, cv in got["classes"].items():
            e = {"eligible": cv["eligible"], "invalidated": cv["invalidated"]}
            if cv["eligible"]:
                e.update({"A": cv["A"], "V": cv["V"], "V_bits": f"{R.bits_of(cv['V']):016x}", "I": cv["I"]})
            exp["classes"][k] = e
        files[f"{fid}.yaml"] = {
            "name": title, "spec_refs": ["2.1", "2.2"], "spec_version": R.SPEC_VERSION,
            "input": case, "expected": exp,
        }
    return files


# ---------------------------------------------------------------------------
# CAS / reconciliation fixtures (§4)
# ---------------------------------------------------------------------------
CAS_CASES = [
    ("01_local_commit", "local", "Local CAS succeeds", {"version": 7}, [], None,
     ("COMMITTED", "NONE", [], False)),
    ("02_local_cas_mismatch", "local", "Local CAS precondition failure (state advanced)", {"version": 8}, [], None,
     ("REJECTED", "PRECONDITION", ["CAS_PRECONDITION_FAILED"], False)),
    ("03_local_capability_consumed", "local", "Local replay of a consumed capability", {"version": 7}, ["SELF"], None,
     ("REJECTED", "PRECONDITION", ["CAPABILITY_UNAVAILABLE"], False)),
    ("04_dist_ack", "distributed", "Target acknowledges", {"version": 7}, [], {"dispatch": "DELIVERED"},
     ("COMMITTED", "NONE", [], False)),
    ("05_dist_timeout_applied", "distributed", "Timeout, reconciliation finds mutation applied", {"version": 7}, [],
     {"dispatch": "TIMEOUT", "reconcile": "APPLIED"}, ("COMMITTED", "NONE", ["DISPATCH_TIMEOUT"], False)),
    ("06_dist_timeout_not_applied", "distributed", "Timeout, reconciliation finds nothing applied", {"version": 7}, [],
     {"dispatch": "TIMEOUT", "reconcile": "NOT_APPLIED"}, ("ROLLED_BACK", "DISPATCH", ["DISPATCH_TIMEOUT"], False)),
    ("07_dist_unresolved", "distributed", "Timeout, reconciliation cannot determine outcome", {"version": 7}, [],
     {"dispatch": "TIMEOUT", "reconcile": "UNKNOWN"},
     ("UNRESOLVED", "RECONCILIATION", ["DISPATCH_TIMEOUT", "RECONCILIATION_UNRESOLVED"], True)),
    ("08_dist_target_reject", "distributed", "Target rejects the request", {"version": 7}, [], {"dispatch": "REJECT"},
     ("REJECTED", "TARGET", ["TARGET_REJECTED"], False)),
    ("09_dist_target_cas_fail", "distributed", "Target reports State-Precondition mismatch", {"version": 9}, [],
     {"dispatch": "DELIVERED"}, ("REJECTED", "PRECONDITION", ["CAS_PRECONDITION_FAILED"], False)),
    ("10_dist_capability_reserved", "distributed", "Capability already reserved by an earlier attempt",
     {"version": 7}, ["SELF"], {"dispatch": "DELIVERED"}, ("REJECTED", "PREPARE", ["CAPABILITY_UNAVAILABLE"], False)),
]


def build_cas():
    files = {}
    for i, (fid, kind, title, st, consumed_spec, target, want) in enumerate(CAS_CASES):
        d = decision(f"cas-{fid}")
        d_hash = R.H(d)
        consumed = [d["capability_id"] if c == "SELF" else c for c in consumed_spec]
        pre = {"version": st["version"], "data": {"balance_minor": 50000}}
        mutation = {"balance_minor": 37500}
        if kind == "local":
            res = R.execute_local(pre, consumed, d, mutation)
        else:
            res = R.execute_distributed(pre, consumed, d, mutation, target)
        want_status, want_phase, want_reasons, want_recon = want
        got = (res["status"], res["phase"], res["reason"], res["recon"])
        if got != (want_status, want_phase, sum(R.RC[n] for n in want_reasons), want_recon):
            raise SystemExit(f"CAS case {fid}: reference gives {got}")
        t_dispatch = BASE_US + 60 * S
        t_final = t_dispatch + (30 * S if "timeout" in fid or "unresolved" in fid else 1 * S)
        rcpt = R.build_receipt(d=d, d_hash=d_hash, receipt_id=R.uuid7(BASE_MS + 61000, "rcpt|" + fid),
                               status=res["status"], failure_phase=res["phase"], reason=res["reason"],
                               recon_required=res["recon"], pre_state=pre,
                               post_state=res["post_state"],
                               tau_dispatch=t_dispatch, tau_finalized=t_final, key=K_RECEIPT, epoch=EPOCH)
        rb = R.encode(rcpt)
        rh = R.H_bytes(rb)
        files[f"{fid}.yaml"] = {
            "name": title, "spec_refs": ["4.1"] if kind == "local" else ["4.2"], "spec_version": R.SPEC_VERSION,
            "boundary": kind,
            "input": {
                "initial_state": pre, "consumed_or_reserved_capabilities": consumed, "mutation": mutation,
                "decision_payload": d, "decision_hash": d_hash, "target_behaviour": target,
            },
            "expected": {
                "execution_status": res["status"], "failure_phase": res["phase"],
                "reason_bitfield": res["reason"], "reason_codes": R.reason_names(res["reason"]),
                "reconciliation_required": res["recon"],
                "final_state": res["post_state"],
                "capability_consumed": d["capability_id"] in res["consumed"],
                "phase_trace": res["trace"],
                "receipt_payload": rcpt, "receipt_bytes": rb.hex(), "receipt_hash": rh,
                "receipt_signature": K_RECEIPT.sign_digest(rh), "receipt_key_id": K_RECEIPT.key_id,
            },
        }
    return files


# ---------------------------------------------------------------------------
# Writing
# ---------------------------------------------------------------------------

def dump_json(path, obj):
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, sort_keys=False)
        f.write("\n")


def dump_yaml(path, obj):
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write("# GENERATED by tools/generate_vectors.py — do not edit by hand.\n")
        yaml.safe_dump(obj, f, sort_keys=False, allow_unicode=True, width=120)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "tests"))
    args = ap.parse_args(argv)
    out = os.path.abspath(args.out)
    for sub in ("fixtures/lineage_graphs", "fixtures/cas_reconciliation", "vectors"):
        os.makedirs(os.path.join(out, sub), exist_ok=True)

    written = {}

    def rel(p):
        return os.path.relpath(p, out).replace(os.sep, "/")

    p = os.path.join(out, "reason_codes.json")
    dump_json(p, {"spec_version": R.SPEC_VERSION, "status": "provisional (A-03)",
                  "encoding": "non-negative int64 bitfield; bit 63 is reserved and must be zero (a mask with it "
                              "set is negative in int64 and fails CanonicalObjectValid)",
                  "bits": [{"bit": i, "name": n, "mask": str(1 << i)} for i, n in enumerate(R.REASON_BITS)],
                  "reserved": [{"bit": 63, "reason": "sign bit of int64"}]})
    written[rel(p)] = p
    p = os.path.join(out, "fixtures", "golden_vectors.json")
    dump_json(p, build_golden()); written[rel(p)] = p
    p = os.path.join(out, "vectors", "auth_expiry_skew.json")
    dump_json(p, build_auth_vectors()); written[rel(p)] = p
    blob, ndoc = build_numeric()
    p = os.path.join(out, "vectors", "kahan_cordic_ieee754.bin")
    with open(p, "wb") as f:
        f.write(blob)
    written[rel(p)] = p
    p = os.path.join(out, "vectors", "kahan_cordic_ieee754.json")
    dump_json(p, ndoc); written[rel(p)] = p
    for name, doc in build_lineage().items():
        p = os.path.join(out, "fixtures", "lineage_graphs", name)
        dump_yaml(p, doc); written[rel(p)] = p
    import scoring_cases  # EXPERIMENTAL scoring profile jaccard-v0.1.0-alpha
    p = os.path.join(out, "vectors", "scoring_jaccard_v0.1.0-alpha.json")
    dump_json(p, scoring_cases.build()); written[rel(p)] = p
    import coverage_cases  # EXPERIMENTAL scoring profile coverage-v0.2.0-alpha + engine guard
    p = os.path.join(out, "vectors", "scoring_coverage_v0.2.0-alpha.json")
    dump_json(p, coverage_cases.build()); written[rel(p)] = p
    import coverage_v021_cases  # EXPERIMENTAL coverage-v0.2.1-alpha: P-1 provenance + defeater channel
    p = os.path.join(out, "vectors", "scoring_coverage_v0.2.1-alpha.json")
    dump_json(p, coverage_v021_cases.build()); written[rel(p)] = p
    import coverage_v030_cases  # EXPERIMENTAL coverage-v0.3.0-alpha: A-1 containment + M-1 membership
    p = os.path.join(out, "vectors", "scoring_coverage_v0.3.0-alpha.json")
    dump_json(p, coverage_v030_cases.build()); written[rel(p)] = p
    for name, doc in build_cas().items():
        p = os.path.join(out, "fixtures", "cas_reconciliation", name)
        dump_yaml(p, doc); written[rel(p)] = p

    manifest = {"spec_version": R.SPEC_VERSION, "generator": "tools/generate_vectors.py", "files": {}}
    for r in sorted(written):
        with open(written[r], "rb") as f:
            manifest["files"][r] = hashlib.sha256(f.read()).hexdigest()
    dump_json(os.path.join(out, "MANIFEST.json"), manifest)
    print(f"wrote {len(written)} artifact files + MANIFEST.json to {out}")


if __name__ == "__main__":
    main()


### Python checker — `tools/check_vectors.py`

In [20]:
%%writefile tools/check_vectors.py
#!/usr/bin/env python3
"""
Checks the committed conformance vectors.

  1. MANIFEST.json hashes match the files on disk (vectors have not drifted).
  2. Every expected value is recomputed from the vector's *inputs* by the reference model.
  3. Internal consistency (payload <-> bytes <-> hash <-> signature <-> public key <-> seed).
  4. Determinism: regenerating into a temp directory reproduces every file byte-for-byte.

Exit status is non-zero on any failure.
"""
from __future__ import annotations

import filecmp
import hashlib
import json
import os
import struct
import sys
import tempfile

import yaml

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.abspath(os.path.join(HERE, ".."))
sys.path.insert(0, HERE)
sys.path.insert(0, os.path.join(ROOT, "reference"))
try:
    import prc_ref as R  # noqa: E402
except ModuleNotFoundError:
    raise SystemExit(
        "prc_ref not found. This script needs reference/prc_ref.py next to tools/ "
        "(repo layout: reference/prc_ref.py, tools/%s). In the notebook, run the "
        "'Reference model' %%%%writefile cell first, or unzip the whole repo." % os.path.basename(__file__))

FAIL: list = []
COUNT = {"checks": 0}


def check(cond, msg):
    COUNT["checks"] += 1
    if not cond:
        FAIL.append(msg)


def from_tj(t):
    k = t["type"]
    if k == "null":
        return None
    if k == "bool":
        return t["value"]
    if k == "int":
        return int(t["value"])
    if k == "f64":
        return R.F64(int(t["bits"], 16))
    if k == "str":
        return t["value"]
    if k == "bytes":
        return bytes.fromhex(t["hex"])
    if k == "list":
        return [from_tj(i) for i in t["items"]]
    if k == "map":
        return {kk: from_tj(v) for kk, v in t["entries"]}
    raise ValueError(k)


def check_manifest(tests):
    m = json.load(open(os.path.join(tests, "MANIFEST.json")))
    for rel, h in m["files"].items():
        with open(os.path.join(tests, rel), "rb") as f:
            check(hashlib.sha256(f.read()).hexdigest() == h, f"manifest: {rel} hash drift")


def check_golden(tests):
    g = json.load(open(os.path.join(tests, "fixtures", "golden_vectors.json"), encoding="utf-8"))
    keys = {}
    for k in g["keys"]:
        tk = R.TestKey.derive(k["domain"], k["label"])
        check(tk.seed.hex() == k["seed"] and tk.public_hex == k["public_key"] and tk.key_id == k["key_id"],
              f"golden: key {k['key_id']} derivation")
        keys[k["key_id"]] = k
    for name in ("NULL_APPROVAL_DIGEST", "NULL_STATE_DIGEST"):
        c = g["constants"][name]
        check(R.encode(c["preimage"]).hex() == c["preimage_bytes"], f"golden: {name} preimage")
        check(R.H(c["preimage"]) == c["value"] == getattr(R, name), f"golden: {name} value")
    u = g["constants"]["UUIDv7_example"]
    check(R.uuid7(u["unix_ms"], u["label"]) == u["value"], "golden: uuid7")
    for p in g["primitives"]:
        b = R.encode(from_tj(p["value"]))
        check(b.hex() == p["expected_bytes"], f"golden: primitive {p['id']} bytes")
        check(R.H_bytes(b) == p["expected_hash"], f"golden: primitive {p['id']} hash")
    for r in g["must_reject"]:
        try:
            R.decode(bytes.fromhex(r["bytes"]))
            check(False, f"golden: reject {r['id']} was accepted")
        except R.CanonicalDecodeError:
            check(True, "")
    for o in g["objects"]:
        b = R.encode(o["payload"])
        check(b.hex() == o["expected_bytes"], f"golden: object {o['id']} bytes")
        check(R.H_bytes(b) == o["expected_hash"], f"golden: object {o['id']} hash")
        k = keys[o["signing_key_id"]]
        check(k["domain"] == o["artifact_type"], f"golden: object {o['id']} signed outside its key domain")
        check(R.ed25519_verify(k["public_key"], o["expected_hash"], o["expected_signature"]),
              f"golden: object {o['id']} signature")
    return len(g["primitives"]) + len(g["must_reject"]) + len(g["objects"])


def check_auth(tests):
    doc = json.load(open(os.path.join(tests, "vectors", "auth_expiry_skew.json"), encoding="utf-8"))
    check(R.H(doc["trust_state"]) == doc["trust_state_hash"], "auth: trust_state_hash")
    for v in doc["vectors"]:
        i, e = v["input"], v["expected_output"]
        ctx = dict(i["context"], trust_state=v.get("trust_state", doc["trust_state"]))
        valid, bits = R.decision_binding_valid(ctx, i["decision_bytes"], i["decision_signature"],
                                               i["approval_bytes"], i["approval_signature"])
        tid = v["test_id"]
        check(valid == e["DecisionBindingValid"], f"auth: {tid} validity")
        check(bits == e["reason_bitfield"], f"auth: {tid} bitfield {R.reason_names(bits)} != {e['reason_codes']}")
        check(R.reason_names(e["reason_bitfield"]) == e["reason_codes"], f"auth: {tid} names/bits disagree")
        check(R.H_bytes(bytes.fromhex(i["decision_bytes"])) == i["decision_hash"], f"auth: {tid} hash")
        canonical = R.encode(i["decision_payload"]).hex() == i["decision_bytes"]
        check(canonical == ("DECODE_INVALID" not in e["reason_codes"]), f"auth: {tid} payload/bytes consistency")
        if i["approval_bytes"]:
            check(R.encode(i["approval_payload"]).hex() == i["approval_bytes"], f"auth: {tid} approval bytes")
        import prc_schema as SC
        schema_ok = not SC.validate(i["decision_payload"], "decision")
        check(schema_ok == ("OBJECT_SCHEMA_INVALID" not in e["reason_codes"] and "DECODE_INVALID" not in e["reason_codes"])
              or "DECODE_INVALID" in e["reason_codes"], f"auth: {tid} schema verdict consistency")
    return len(doc["vectors"])


def check_numeric(tests):
    vdir = os.path.join(tests, "vectors")
    blob = open(os.path.join(vdir, "kahan_cordic_ieee754.bin"), "rb").read()
    doc = json.load(open(os.path.join(vdir, "kahan_cordic_ieee754.json")))
    check(hashlib.sha256(blob).hexdigest() == doc["binary_sha256"], "numeric: bin sha256")
    check(blob[:4] == b"PRNV", "numeric: magic")
    ver, _res, n = struct.unpack("<HHI", blob[4:12])
    check(ver == 1 and n == len(doc["cases"]), "numeric: header")
    pos = 12
    for case in doc["cases"]:
        op, flags, _r, cnt = struct.unpack("<BBHI", blob[pos:pos + 8]); pos += 8
        ins = list(struct.unpack(f"<{cnt}Q", blob[pos:pos + 8 * cnt])); pos += 8 * cnt
        (out,) = struct.unpack("<Q", blob[pos:pos + 8]); pos += 8
        cid = case["id"]
        check([f"{x:016x}" for x in ins] == case["inputs_bits"], f"numeric: {cid} inputs bin/json")
        check(f"{out:016x}" == case["expected"]["output_bits"], f"numeric: {cid} output bin/json")
        check(bool(flags & 2) == case["expected"]["safe_state_lock"], f"numeric: {cid} lock bin/json")
        if op == 1:
            g = R.authorization_check(*ins)
            check(g["result"] == bool(flags & 1) == case["expected"]["result"], f"numeric: {cid} result")
            check(g["c_effective_bits"] == out and g["safe_state_lock"] == bool(flags & 2), f"numeric: {cid} recompute")
        else:
            g = R.kahan_sum(ins)
            check(g["result_bits"] == out and g["safe_state_lock"] == bool(flags & 2), f"numeric: {cid} recompute")
    check(pos == len(blob), "numeric: trailing bytes in bin")
    return len(doc["cases"])


def check_lineage(tests):
    d = os.path.join(tests, "fixtures", "lineage_graphs")
    n = 0
    for f in sorted(os.listdir(d)):
        doc = yaml.safe_load(open(os.path.join(d, f), encoding="utf-8"))
        got, exp = R.evaluate_lineage(doc["input"]), doc["expected"]
        check(got["eligible"] == exp["eligible_classes"], f"lineage: {f} eligible")
        check(got["M"] == exp["M"] and got["C"] == exp["C"], f"lineage: {f} M/C")
        check(got["audit_events"] == exp["audit_events"], f"lineage: {f} audit")
        check(got["reason_code"] == exp["reason_bitfield"], f"lineage: {f} reason")
        for k, e in exp["classes"].items():
            c = got["classes"][k]
            check(c["eligible"] == e["eligible"] and c["invalidated"] == e["invalidated"], f"lineage: {f} {k} flags")
            if e["eligible"]:
                check(f"{R.bits_of(c['V']):016x}" == e["V_bits"] and c["A"] == e["A"] and c["I"] == e["I"],
                      f"lineage: {f} {k} values")
        n += 1
    return n


def check_cas(tests):
    d = os.path.join(tests, "fixtures", "cas_reconciliation")
    keys = {k.key_id: k for k in [R.TestKey.derive("receipt", "k1")]}
    n = 0
    for f in sorted(os.listdir(d)):
        doc = yaml.safe_load(open(os.path.join(d, f), encoding="utf-8"))
        i, e = doc["input"], doc["expected"]
        dd = i["decision_payload"]
        check(R.H(dd) == i["decision_hash"], f"cas: {f} decision hash")
        if doc["boundary"] == "local":
            res = R.execute_local(i["initial_state"], i["consumed_or_reserved_capabilities"], dd, i["mutation"])
        else:
            res = R.execute_distributed(i["initial_state"], i["consumed_or_reserved_capabilities"], dd,
                                        i["mutation"], i["target_behaviour"])
        check((res["status"], res["phase"], res["reason"], res["recon"]) ==
              (e["execution_status"], e["failure_phase"], e["reason_bitfield"], e["reconciliation_required"]),
              f"cas: {f} outcome")
        check(res["post_state"] == e["final_state"], f"cas: {f} final state")
        check((dd["capability_id"] in res["consumed"]) == e["capability_consumed"], f"cas: {f} capability")
        check(res["trace"] == e["phase_trace"], f"cas: {f} trace")
        r = e["receipt_payload"]
        check(r["execution_status"] == e["execution_status"] and r["decision_digest"] == i["decision_hash"],
              f"cas: {f} receipt fields")
        known = e["final_state"] is not None
        check(r["post_state_digest"] == (R.H(e["final_state"]) if known else R.NULL_STATE_DIGEST),
              f"cas: {f} post_state_digest")
        check(r["pre_state_digest"] == R.H(i["initial_state"]), f"cas: {f} pre_state_digest")
        # R-1: no receipt may contradict itself (expected vs observed state)
        check(r["observed_pre_state_ver"] == i["initial_state"]["version"], f"cas: {f} observed_pre_state_ver")
        check(r["expected_state_ver"] == dd["state_ver"], f"cas: {f} expected_state_ver")
        check(r["observed_post_state_ver"] == (e["final_state"]["version"] if known else None),
              f"cas: {f} observed_post_state_ver")
        check(known or e["execution_status"] in ("UNRESOLVED", "INDETERMINATE"), f"cas: {f} unknown post only if unresolved")
        rb = R.encode(r)
        check(rb.hex() == e["receipt_bytes"] and R.H_bytes(rb) == e["receipt_hash"], f"cas: {f} receipt bytes/hash")
        k = keys[e["receipt_key_id"]]
        check(R.ed25519_verify(k.public_hex, e["receipt_hash"], e["receipt_signature"]), f"cas: {f} receipt sig")
        n += 1
    return n


def check_scoring(tests):
    import prc_scoring as P
    doc = json.load(open(os.path.join(tests, "vectors", "scoring_jaccard_v0.1.0-alpha.json"), encoding="utf-8"))
    profiles = {k: v["declaration"] for k, v in doc["profiles"].items()}
    for k, v in doc["profiles"].items():
        check(R.H(v["declaration"]) == v["fingerprint"], f"scoring: profile {k} fingerprint")
    for x in doc["prc_exp_v1"]["vectors"]:
        check(P.hx(P.prc_exp(P.f(x["input_bits"]))) == x["expected_bits"], f"scoring: prc_exp({x['input_repr']})")
    by_id = {}
    for c in doc["cases"]:
        tr = P.evaluate(c["input"]["request"], c["input"]["families"], profiles[c["profile"]])
        check(R.encode(tr) == R.encode(c["expected_trace"]), f"scoring: {c['id']} trace")
        check(tr["profile_fingerprint"] == doc["profiles"][c["profile"]]["fingerprint"], f"scoring: {c['id']} fp")
        by_id[c["id"]] = c
    h = doc["headline"]
    a, b = by_id[h["allow_case"]], by_id[h["deny_case"]]
    check(a["expected_trace"]["decision"] == "ALLOW" and b["expected_trace"]["decision"] == "DENY", "scoring: headline flip")
    def strip(fams):
        import copy
        f2 = copy.deepcopy(fams)
        for f_ in f2:
            for m in f_["members"]:
                m["dependencies"] = []
        return R.encode(f2)
    check(strip(a["input"]["families"]) == strip(b["input"]["families"]) and
          a["input"]["request"] == b["input"]["request"], "scoring: headline inputs differ beyond lineage")
    check(a["expected_trace"]["T_bits"] == b["expected_trace"]["T_bits"], "scoring: headline thresholds differ")
    return len(doc["cases"]) + len(doc["prc_exp_v1"]["vectors"])


def check_coverage(tests):
    import prc_engine as ENG
    import prc_scoring as P
    doc = json.load(open(os.path.join(tests, "vectors", "scoring_coverage_v0.2.0-alpha.json"), encoding="utf-8"))
    base = doc["profiles"]["calibrated"]
    check(R.H(base["declaration"]) == base["fingerprint"], "coverage: profile fingerprint")
    by_id = {}
    for c in doc["cases"]:
        prof = c.get("profile_declaration", base["declaration"])
        try:
            tr = ENG.evaluate(c["input"]["request"], c["input"]["families"], prof)
        except Exception as ex:  # a regression may raise instead of failing closed
            check(False, f"coverage: {c['id']} raised {type(ex).__name__}: {ex}")
            by_id[c["id"]] = c
            continue
        check(R.encode(tr) == R.encode(c["expected_trace"]), f"coverage: {c['id']} trace")
        by_id[c["id"]] = c
    xfail = 0
    for c in doc["cases"]:
        ke = c.get("known_exploit")
        if ke:
            sec = ke["security_expectation"]
            meets = (c["expected_trace"]["decision"] == sec["decision"] and
                     c["expected_trace"]["M_bits"] == by_id[sec["same_M_as"]]["expected_trace"]["M_bits"])
            check(not meets, f"coverage: {c['id']} exploit no longer reproduces (XPASS) — promote it")
            xfail += 1
    for i, w in enumerate(doc["property_witnesses"]):
        prof = base["declaration"]
        t0 = ENG.evaluate(w["request"], w["base"], prof)
        t1 = ENG.evaluate(w["request"], w["base"] + [w["added"]], prof)
        check(t0["M_bits"] == w["M_base_bits"] and t1["M_bits"] == w["M_after_bits"], f"coverage: witness {i} bits")
        ok = (w["M_after_bits"] == w["M_base_bits"]) if w["kind"] == "idempotent" else \
            P.f(w["M_after_bits"]) >= P.f(w["M_base_bits"])
        check(ok and not (w["decision_base"] == "ALLOW" and w["decision_after"] == "DENY"), f"coverage: witness {i} property")
    import property_tests
    import contextlib, io
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        rc = property_tests.main(["--trials", "400"])
    check(rc == 0, "coverage: randomised property tests failed:\n" + buf.getvalue())
    COUNT["xfail"] = xfail
    return len(doc["cases"]) + len(doc["property_witnesses"])


def check_coverage_v021(tests):
    import prc_engine as ENG
    import prc_scoring as P
    doc = json.load(open(os.path.join(tests, "vectors", "scoring_coverage_v0.2.1-alpha.json"), encoding="utf-8"))
    profs = {k: v["declaration"] for k, v in doc["profiles"].items()}
    for k, v in doc["profiles"].items():
        check(R.H(v["declaration"]) == v["fingerprint"], f"v0.2.1: profile {k} fingerprint")
    for k in doc["provenance_keys"]:
        tk = R.TestKey.derive(k["domain"], k["label"])
        check(tk.public_hex == k["public_key"] and tk.key_id == k["key_id"], f"v0.2.1: key {k['label']}")
    check(doc["known_exploits_open"] == [], "v0.2.1: no known exploit may remain open in the current profile")
    for c in doc["cases"]:
        trust = c.get("trust_state", doc["trust_state"])
        try:
            tr = ENG.evaluate(c["input"]["request"], c["input"]["families"], profs[c["profile"]], trust)
        except Exception as ex:
            check(False, f"v0.2.1: {c['id']} raised {type(ex).__name__}: {ex}")
            continue
        check(R.encode(tr) == R.encode(c["expected_trace"]), f"v0.2.1: {c['id']} trace")
    by = {c["id"]: c["expected_trace"] for c in doc["cases"]}
    check(by["x01_self_declared_padding"]["decision"] == "DENY" and
          by["x01_self_declared_padding"]["M_bits"] == by["c04_three_correlated_fail"]["M_bits"], "v0.2.1: x01 must deny")
    check(by["c03_three_independent_pass"]["decision"] == "ALLOW" and by["c04_three_correlated_fail"]["decision"] == "DENY",
          "v0.2.1: lineage flip")
    d1, d2 = by["d01_unverified_contradiction"], by["d02_authenticated_defeater_vetoes"]
    check((d1["M_bits"], d1["C_bits"]) == (d2["M_bits"], d2["C_bits"]) and d1["decision"] == "ALLOW" and
          d2["decision"] == "DENY", "v0.2.1: defeater must be a categorical veto")
    for i, w in enumerate(doc["property_witnesses"]):
        t0 = ENG.evaluate(w["request"], w["base"], profs["calibrated"], doc["trust_state"])
        t1 = ENG.evaluate(w["request"], w["base"] + [w["added"]], profs["calibrated"], doc["trust_state"])
        check(t0["M_bits"] == w["M_base_bits"] and t1["M_bits"] == w["M_after_bits"], f"v0.2.1: witness {i} bits")
        ok = {"monotone": P.f(w["M_after_bits"]) >= P.f(w["M_base_bits"]) and
                          not (w["decision_base"] == "ALLOW" and w["decision_after"] == "DENY"),
              "idempotent": w["M_after_bits"] == w["M_base_bits"],
              "defeater": w["decision_after"] == "DENY"}[w["kind"]]
        check(ok, f"v0.2.1: witness {i} ({w['kind']}) property")
    return len(doc["cases"]) + len(doc["property_witnesses"])


def check_transport():
    """Optional: runs only when grpcio-tools/protobuf are installed (or PRC_REQUIRE_TRANSPORT=1)."""
    import contextlib, io
    if os.environ.get("PRC_SKIP_TRANSPORT"):
        COUNT["transport"] = "transport round trip: skipped (PRC_SKIP_TRANSPORT)"
        return
    import transport_roundtrip as TR
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        rc = TR.main()
    out = buf.getvalue().strip()
    check(rc == 0, "transport round trip failed:\n" + out)
    COUNT["transport"] = out.splitlines()[0] if out else "no output"


def check_coverage_v030(tests):
    import prc_engine as ENG
    doc = json.load(open(os.path.join(tests, "vectors", "scoring_coverage_v0.3.0-alpha.json"), encoding="utf-8"))
    check(R.H(doc["scoring_profile_declaration"]) == doc["scoring_profile_hash"], "v0.3.0: scoring profile hash")
    check(R.H(doc["trust_state"]) == doc["trust_state_hash"], "v0.3.0: trust state hash")
    prof = doc["scoring_profile_declaration"]
    by = {}
    for c in doc["cases"]:
        tr = ENG.evaluate(c["input"]["request"], c["input"]["families"], prof, c.get("trust_state", doc["trust_state"]))
        check(R.encode(tr) == R.encode(c["expected_trace"]), f"v0.3.0: {c['id']} trace")
        by[c["id"]] = c["expected_trace"]
    base = by["a00_four_independent_pass"]
    for cid, want in (("a01_admitted_contradiction_defeats", "DENY"), ("a02_unattested_contradiction_contains", "MANUAL_REQUIRED")):
        check((by[cid]["M_bits"], by[cid]["C_bits"]) == (base["M_bits"], base["C_bits"]) and by[cid]["decision"] == want,
              f"v0.3.0: {cid} must leave M/C untouched and yield {want}")
    for cid in [k for k in by if k.startswith("m0") and k < "m07"]:
        check(by[cid]["decision"] == "ALLOW", f"v0.3.0: injection {cid} must be neutral")
    for i, w in enumerate(doc["property_witnesses"]):
        t0 = ENG.evaluate(w["request"], w["base"], prof, doc["trust_state"])
        t1 = ENG.evaluate(w["request"], w["after"], prof, doc["trust_state"])
        check((t0["M_bits"], t1["M_bits"], t0["C_bits"], t1["C_bits"]) ==
              (w["M_base_bits"], w["M_after_bits"], w["C_base_bits"], w["C_after_bits"]), f"v0.3.0: witness {i} bits")
    return len(doc["cases"]) + len(doc["property_witnesses"])


def check_determinism(tests):
    import generate_vectors as G
    with tempfile.TemporaryDirectory() as tmp:
        try:
            G.main(["--out", tmp])
        except SystemExit as e:  # generator refuses when the model disagrees with a declared intent
            check(False, f"determinism: generator refused to run: {e}")
            return
        for root, _dirs, files in os.walk(tmp):
            for fn in files:
                a = os.path.join(root, fn)
                rel = os.path.relpath(a, tmp)
                b = os.path.join(tests, rel)
                check(os.path.exists(b) and filecmp.cmp(a, b, shallow=False), f"determinism: {rel} differs")


def main():
    tests = os.path.join(ROOT, "tests")
    check_manifest(tests)
    counts = {
        "golden": check_golden(tests),
        "auth_expiry": check_auth(tests),
        "numeric": check_numeric(tests),
        "lineage": check_lineage(tests),
        "cas": check_cas(tests),
        "scoring": check_scoring(tests),
        "coverage_v0.2.0": check_coverage(tests),
        "coverage_v0.2.1": check_coverage_v021(tests),
        "coverage_v0.3.0": check_coverage_v030(tests),
    }
    check_transport()
    check_determinism(tests)
    print("vectors:", ", ".join(f"{k}={v}" for k, v in counts.items()))
    print(COUNT.get("transport", ""))
    print(f"checks run: {COUNT['checks']}, failures: {len(FAIL)}")
    print("current profile coverage-v0.3.0-alpha: no known open exploit UNDER the declared trust state, atom "
          "registry/aliases, membership authorities, clock authorities and calibration")
    print(f"superseded coverage-v0.2.0-alpha still reproduces its atom-padding exploit by design: {COUNT.get('xfail', 0)}")
    for m in FAIL:
        print("  FAIL", m)
    sys.exit(1 if FAIL else 0)


if __name__ == "__main__":
    main()


### Independent JavaScript cross-check — `tools/xcheck.mjs`

In [21]:
%%writefile tools/xcheck.mjs
// Independent cross-check of the conformance vectors in JavaScript (Node >= 18, no dependencies).
//
// This file shares no code with reference/prc_ref.py. It re-implements, from the spec and
// docs/CONFORMANCE_ASSUMPTIONS.md only: the PRC-LP-0 encoder and strict decoder, SHA-256 hashing,
// Ed25519 key derivation/sign/verify, DecisionBindingValid (§3.1–3.5), and the §2.3 numeric ops.
// If both implementations agree on every vector, the vectors are much less likely to encode a
// bug in either one.
//
//   node tools/xcheck.mjs
import { readFileSync } from "node:fs";
import { createHash, createPrivateKey, createPublicKey, sign, verify } from "node:crypto";
import { dirname, join } from "node:path";
import { fileURLToPath } from "node:url";

const ROOT = join(dirname(fileURLToPath(import.meta.url)), "..");
const T = (p) => join(ROOT, "tests", p);
const fails = [];
let checks = 0;
const ok = (c, m) => { checks++; if (!c) fails.push(m); };

// ---------------- PRC-LP-0 ----------------
const TAG = { NULL: 0x00, FALSE: 0x01, TRUE: 0x02, INT: 0x10, F64: 0x11, STR: 0x20, BYTES: 0x21, LIST: 0x30, MAP: 0x31 };
const u64be = (n) => { const b = Buffer.alloc(8); b.writeBigUInt64BE(BigInt(n)); return b; };
const frame = (tag, payload) => Buffer.concat([Buffer.from([tag]), u64be(payload.length), payload]);
const cmpBytes = (a, b) => Buffer.compare(Buffer.from(a, "utf8"), Buffer.from(b, "utf8"));

// typed-JSON value (golden primitives)
function encTJ(t) {
  switch (t.type) {
    case "null": return frame(TAG.NULL, Buffer.alloc(0));
    case "bool": return frame(t.value ? TAG.TRUE : TAG.FALSE, Buffer.alloc(0));
    case "int": { const b = Buffer.alloc(8); b.writeBigInt64BE(BigInt(t.value)); return frame(TAG.INT, b); }
    case "f64": return frame(TAG.F64, Buffer.from(t.bits, "hex"));
    case "str": return frame(TAG.STR, Buffer.from(t.value, "utf8"));
    case "bytes": return frame(TAG.BYTES, Buffer.from(t.hex, "hex"));
    case "list": return frame(TAG.LIST, Buffer.concat(t.items.map(encTJ)));
    case "map": {
      const es = [...t.entries].sort((x, y) => cmpBytes(x[0], y[0]));
      return frame(TAG.MAP, Buffer.concat(es.flatMap(([k, v]) => [encTJ({ type: "str", value: k }), encTJ(v)])));
    }
  }
  throw new Error("bad type " + t.type);
}
// plain JSON payloads: numbers must be integers (payloads contain no floats)
function enc(v) {
  if (v === null) return encTJ({ type: "null" });
  if (typeof v === "boolean") return encTJ({ type: "bool", value: v });
  if (typeof v === "number") {
    if (!Number.isSafeInteger(v)) throw new Error("non-integer number in payload: " + v);
    return encTJ({ type: "int", value: String(v) });
  }
  if (typeof v === "string") return encTJ({ type: "str", value: v });
  if (Array.isArray(v)) return frame(TAG.LIST, Buffer.concat(v.map(enc)));
  const keys = Object.keys(v).sort(cmpBytes);
  return frame(TAG.MAP, Buffer.concat(keys.flatMap((k) => [enc(k), enc(v[k])])));
}
function decodeStrict(buf) {
  function at(b, pos, end) {
    if (pos + 9 > end) throw new Error("truncated");
    const tag = b[pos];
    const len = Number(b.readBigUInt64BE(pos + 1));
    const s = pos + 9, e = s + len;
    if (e > end) throw new Error("overrun");
    const body = b.subarray(s, e);
    switch (tag) {
      case TAG.NULL: case TAG.FALSE: case TAG.TRUE:
        if (len) throw new Error("len"); return [tag === TAG.NULL ? null : tag === TAG.TRUE, e];
      case TAG.INT: if (len !== 8) throw new Error("int len"); return [body.readBigInt64BE(0), e];
      case TAG.F64: if (len !== 8) throw new Error("f64 len"); return [{ f64: body.toString("hex") }, e];
      case TAG.STR: return [new TextDecoder("utf-8", { fatal: true }).decode(body), e];
      case TAG.BYTES: return [Buffer.from(body), e];
      case TAG.LIST: { const out = []; let p = s; while (p < e) { const [x, q] = at(b, p, e); out.push(x); p = q; } return [out, e]; }
      case TAG.MAP: {
        const out = {}; let p = s, prev = null;
        while (p < e) {
          const [k, q] = at(b, p, e);
          if (typeof k !== "string") throw new Error("key type");
          if (prev !== null && cmpBytes(k, prev) <= 0) throw new Error("key order");
          prev = k; if (q >= e) throw new Error("dangling key");
          const [v, r] = at(b, q, e); out[k] = v; p = r;
        }
        return [out, e];
      }
    }
    throw new Error("tag");
  }
  const [v, end] = at(buf, 0, buf.length);
  if (end !== buf.length) throw new Error("trailing");
  return v;
}
const sha = (b) => createHash("sha256").update(b).digest("hex");
const H = (v) => sha(enc(v));
// decoded values carry BigInt ints; convert back to JSON-ish for comparisons
const plain = (v) => typeof v === "bigint" ? Number(v) : Array.isArray(v) ? v.map(plain)
  : v && typeof v === "object" && !Buffer.isBuffer(v) && !("f64" in v) ? Object.fromEntries(Object.entries(v).map(([k, x]) => [k, plain(x)])) : v;
const same = (a, b) => { try { return enc(a).equals(enc(b)); } catch { return false; } };

// ---------------- Ed25519 ----------------
const SPKI = Buffer.from("302a300506032b6570032100", "hex");
const PKCS8 = Buffer.from("302e020100300506032b657004220420", "hex");
const pubKey = (hex) => createPublicKey({ key: Buffer.concat([SPKI, Buffer.from(hex, "hex")]), format: "der", type: "spki" });
const edVerify = (pubHex, digestHex, sigHex) => {
  try { return verify(null, Buffer.from(digestHex, "hex"), pubKey(pubHex), Buffer.from(sigHex, "hex")); } catch { return false; }
};

// ---------------- golden ----------------
const g = JSON.parse(readFileSync(T("fixtures/golden_vectors.json"), "utf8"));
const keyById = {};
for (const k of g.keys) {
  const seed = createHash("sha256").update(`PRC-TEST-KEY|${k.domain}|${k.label}`).digest();
  ok(seed.toString("hex") === k.seed, `golden key seed ${k.key_id}`);
  const priv = createPrivateKey({ key: Buffer.concat([PKCS8, seed]), format: "der", type: "pkcs8" });
  const pub = createPublicKey(priv).export({ format: "der", type: "spki" }).subarray(12).toString("hex");
  ok(pub === k.public_key, `golden key pub ${k.key_id}`);
  ok(`kid:${k.domain}:${sha(Buffer.from(pub, "hex")).slice(0, 16)}` === k.key_id, `golden key id ${k.key_id}`);
  keyById[k.key_id] = { ...k, priv };
}
for (const [name, c] of Object.entries(g.constants)) {
  if (!name.startsWith("NULL_")) continue;
  ok(enc(c.preimage).toString("hex") === c.preimage_bytes && H(c.preimage) === c.value, `golden ${name}`);
}
for (const p of g.primitives) {
  const b = encTJ(p.value);
  ok(b.toString("hex") === p.expected_bytes && sha(b) === p.expected_hash, `golden primitive ${p.id}`);
}
for (const r of g.must_reject) {
  let rejected = false;
  try { decodeStrict(Buffer.from(r.bytes, "hex")); } catch { rejected = true; }
  ok(rejected, `golden reject ${r.id}`);
}
for (const o of g.objects) {
  const b = enc(o.payload), h = sha(b);
  ok(b.toString("hex") === o.expected_bytes && h === o.expected_hash, `golden object ${o.id} bytes/hash`);
  const k = keyById[o.signing_key_id];
  const sig = sign(null, Buffer.from(h, "hex"), k.priv).toString("hex");
  ok(sig === o.expected_signature, `golden object ${o.id} deterministic signature`);
  ok(edVerify(k.public_key, h, o.expected_signature), `golden object ${o.id} verify`);
}
const NULL_APPROVAL = g.constants.NULL_APPROVAL_DIGEST.value;

// ---------------- DecisionBindingValid (independent) ----------------
const auth = JSON.parse(readFileSync(T("vectors/auth_expiry_skew.json"), "utf8"));
const REASONS = JSON.parse(readFileSync(T("reason_codes.json"), "utf8")).bits.map((b) => b.name);
const glob = (s, pat) => new RegExp("^" + pat.replace(/[.+^${}()|[\]\\]/g, "\\$&").replace(/\*/g, ".*").replace(/\?/g, ".") + "$").test(s);
const keyAuth = (reg, kid, epoch, domain, cls) => {
  const e = reg[kid];
  if (!e || e.domain !== domain || !e.epochs.includes(epoch)) return false;
  return cls === undefined || (e.authority_classes || []).includes(cls);
};
// ---- CanonicalObjectValid (independent re-implementation of spec 0.3.0 §1.5) ----
const MAXU = 2n ** 63n - 1n;
const isInt = (v) => typeof v === "bigint" || Number.isSafeInteger(v);
const K = {
  str: (v) => typeof v === "string" && v !== "",
  uint: (v) => (typeof v === "bigint" && v >= 0n && v <= MAXU) || (Number.isSafeInteger(v) && v >= 0),
  hex32: (v) => typeof v === "string" && /^[0-9a-f]{64}$/.test(v),
  uuid7: (v) => typeof v === "string" && /^[0-9a-f]{8}-[0-9a-f]{4}-7[0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$/.test(v),
  semver: (v) => typeof v === "string" && /^(0|[1-9]\d*)\.(0|[1-9]\d*)\.(0|[1-9]\d*)(-[0-9A-Za-z.-]+)?$/.test(v),
  keyid: (v) => typeof v === "string" && /^kid:[a-z]+:[0-9a-f]{16}$/.test(v),
  schema_ver: (v) => v === "0.3.0-alpha",
  bool: (v) => typeof v === "boolean",
  map: (v) => v !== null && typeof v === "object" && !Array.isArray(v) && !Buffer.isBuffer(v) && !("f64" in v),
  actor: (v) => K.map(v) && Object.keys(v).sort().join() === "id,type" && K.str(v.id) && ["service", "user"].includes(v.type),
  action: (v) => K.map(v) && K.str(v.action),
  atoms: (v) => Array.isArray(v) && v.every((a) => typeof a === "string" && /^[A-Z_]+:\S+$/.test(a)),
  manifest_members: (v) => Array.isArray(v) && v.every((m) => K.map(m) && Object.keys(m).sort().join() === "evidence_digest,evidence_id,parent_id"
    && K.str(m.evidence_id) && K.hex32(m.evidence_digest) && (m.parent_id === null || K.str(m.parent_id))),
};
const KS = K;
const kind = (spec) => spec.startsWith("enum:") ? (v) => spec.slice(5).split("|").includes(v)
  : spec.startsWith("opt_") ? (v) => v === null || K[spec.slice(4)](v) : K[spec];
const SCH = {
  decision: "domain:str schema_ver:schema_ver profile_version:semver lineage_schema_version:semver capability_id:uuid7 req_id:uuid7 evidence_snapshot_id:uuid7 actor:actor auth_mode:enum:AUTOMATED|MANUAL x:action a:map target:str state_ver:uint control_profile_hash:hex32 scoring_profile_hash:hex32 trust_state_hash:hex32 evidence_hash:hex32 lineage_hash:hex32 approval_hash:hex32 adverse_disposition:enum:CLEAR|CONTAIN containment_digest:hex32 decision_key_id:keyid decision_trust_epoch:str eval_clock_id:str tau_eval:uint tau_expiry:uint decision:enum:ALLOW|DENY reason_codes:uint",
  approval: "domain:str schema_ver:schema_ver approval_id:uuid7 approval_key_id:keyid approval_trust_epoch:str authority_class:str authority_scope:str x:action a:map target:str actor:actor req_id:uuid7 capability_id:uuid7 state_ver:uint control_profile_hash:hex32 scoring_profile_hash:hex32 trust_state_hash:hex32 adverse_disposition:enum:CLEAR|CONTAIN containment_digest:hex32 tau_issued:uint tau_expiry:uint decision:enum:ALLOW|DENY",
  attestation: "domain:str schema_ver:schema_ver attestation_id:uuid7 provenance_key_id:keyid provenance_trust_epoch:str attester_id:str evidence_digest:hex32 dependencies:atoms tau_issued:uint tau_expiry:uint",
  manifest: "domain:str schema_ver:schema_ver manifest_id:uuid7 membership_key_id:keyid membership_trust_epoch:str authority_id:str family_id:str ingress_channel:str members:manifest_members tau_issued:uint tau_expiry:uint",
};
const SCHEMA = Object.fromEntries(Object.entries(SCH).map(([n, txt]) => [n, Object.fromEntries(txt.split(" ").map((t) => { const i = t.indexOf(":"); return [t.slice(0, i), kind(t.slice(i + 1))]; }))]));
function schemaValid(z, name) {
  if (!K.map(z)) return false;
  const spec = SCHEMA[name];
  const keys = Object.keys(z);
  if (keys.length !== Object.keys(spec).length || keys.some((k) => !(k in spec))) return false;
  return Object.entries(spec).every(([k, f]) => k in z && f(z[k]));
}
const envelopeOk = (env) => K.map(env) && Object.keys(env).sort().join() === "payload,signature" && typeof env.signature === "string" && /^[0-9a-f]{128}$/.test(env.signature);
const domainKey = (reg, kid, domain) => { const e = typeof kid === "string" ? reg[kid] : undefined; return e && e.domain === domain ? e : null; };
const verifyIn = (reg, kid, domain, digest, sig) => { const e = domainKey(reg, kid, domain); return !!e && typeof sig === "string" && edVerify(e.public_key, digest, sig); };
const NULL_CONTAINMENT = sha(enc("PROVENANCE-RAZOR-NO-CONTAINMENT-V0.3.0-Alpha"));
function clock(ts, evalId, ex) {
  const cl = ts.clock_authorities || {};
  const ev = cl[evalId], xa = ex && cl[ex.clock_id];
  const u = ex && Number.isSafeInteger(ex.uncertainty_us) && ex.uncertainty_us >= 0 ? ex.uncertainty_us : 0;
  const trusted = !!ev && !!xa && Number.isSafeInteger(ex.uncertainty_us) && ex.uncertainty_us >= 0 && ex.uncertainty_us <= xa.max_uncertainty_us;
  return [trusted, (ev ? ev.max_uncertainty_us : 0) + u];
}
function approvalCheck(ctx, ts, pHex, pSig, d, U) {
  const R = new Set(), reg = ts.key_registry;
  let raw, p;
  try { raw = Buffer.from(pHex, "hex"); const dec = decodeStrict(raw); if (!schemaValid(dec, "approval")) return ["APPROVAL_SCHEMA_INVALID"]; p = plain(dec); }
  catch { return ["APPROVAL_SCHEMA_INVALID"]; }
  if (!verifyIn(reg, p.approval_key_id, "approval", sha(raw), pSig)) R.add("APPROVAL_SIGNATURE_INVALID");
  if (p.domain !== "PROVENANCE-RAZOR-APPROVAL-V0.3.0-Alpha") R.add("APPROVAL_DOMAIN_MISMATCH");
  if (!keyAuth(reg, p.approval_key_id, p.approval_trust_epoch, "approval", p.authority_class)) R.add("APPROVAL_KEY_UNAUTHORIZED");
  if (!glob(`${ctx.x.action}:${p.target}`, p.authority_scope)) R.add("APPROVAL_SCOPE_DENIED");
  if (!same(p.x, ctx.x) || !same(p.a, ctx.a) || p.state_ver !== ctx.current_state_ver || p.control_profile_hash !== ctx.active_control_profile_hash
      || p.scoring_profile_hash !== d.scoring_profile_hash || p.trust_state_hash !== d.trust_state_hash) R.add("APPROVAL_BINDING_MISMATCH");
  if (p.target !== ctx.current_target || p.target !== d.target || !same(p.actor, d.actor) || p.req_id !== d.req_id || p.capability_id !== d.capability_id)
    R.add("APPROVAL_SUBJECT_MISMATCH");
  if (p.adverse_disposition !== d.adverse_disposition || p.containment_digest !== d.containment_digest) R.add("ADVERSE_DISPOSITION_MISMATCH");
  if (!(p.tau_issued <= d.tau_eval)) R.add("APPROVAL_NOT_YET_VALID");
  if (!(ctx.exec_clock.tau_exec + U <= p.tau_expiry)) R.add("APPROVAL_EXPIRED");
  const reusable = (ctx.profile.reusable_authority_classes || []).includes(p.authority_class);
  if ((ctx.consumed_approvals || []).includes(p.approval_id) && !reusable) R.add("APPROVAL_CONSUMED");
  return [...R];
}
function dbv(ctx, ts, dHex, dSig, pHex, pSig) {
  let d, raw;
  try { raw = Buffer.from(dHex, "hex"); const dec = decodeStrict(raw); if (!schemaValid(dec, "decision")) return ["OBJECT_SCHEMA_INVALID"]; d = plain(dec); }
  catch { return ["DECODE_INVALID"]; }
  const R = new Set(), reg = ts.key_registry;
  if (!verifyIn(reg, d.decision_key_id, "decision", sha(raw), dSig)) R.add("SIGNATURE_INVALID");
  if (d.domain !== "PROVENANCE-RAZOR-DECISION-V0.3.0-Alpha") R.add("DOMAIN_MISMATCH");
  if (!same(d.x, ctx.x)) R.add("ACTION_MISMATCH");
  if (!same(d.a, ctx.a)) R.add("ASSERTION_MISMATCH");
  if (d.target !== ctx.current_target) R.add("TARGET_MISMATCH");
  if (d.state_ver !== ctx.current_state_ver) R.add("STATE_VERSION_MISMATCH");
  if (d.control_profile_hash !== ctx.active_control_profile_hash) R.add("PROFILE_HASH_MISMATCH");
  if (d.scoring_profile_hash !== ctx.active_scoring_profile_hash) R.add("SCORING_PROFILE_MISMATCH");
  if (d.trust_state_hash !== ctx.trust_state_hash || d.trust_state_hash !== H(ts)) R.add("TRUST_STATE_MISMATCH");
  if (d.evidence_hash !== ctx.evidence_hash) R.add("EVIDENCE_HASH_MISMATCH");
  if (d.lineage_hash !== ctx.lineage_hash) R.add("LINEAGE_HASH_MISMATCH");
  if (d.adverse_disposition !== ctx.adverse_disposition || d.containment_digest !== ctx.containment_digest) R.add("ADVERSE_DISPOSITION_MISMATCH");
  if ((d.adverse_disposition === "CLEAR") !== (d.containment_digest === NULL_CONTAINMENT)) R.add("ADVERSE_DISPOSITION_MISMATCH");
  if (d.decision !== "ALLOW") R.add("DECISION_NOT_ALLOW");
  const [trusted, U] = clock(ts, d.eval_clock_id, ctx.exec_clock);
  if (!trusted) R.add("CLOCK_UNTRUSTED");
  const tx = ctx.exec_clock.tau_exec;
  if (!(d.tau_eval + U <= tx)) R.add("NOT_YET_VALID");
  if (!(tx + U <= d.tau_expiry)) R.add("EXPIRED_CAPABILITY");
  const skew = ctx.profile.max_eval_skew_us[ctx.x.action] ?? ctx.profile.max_eval_skew_us["*"];
  if (!(tx - d.tau_eval + U <= skew)) R.add("EVAL_SKEW_EXCEEDED");
  if (!keyAuth(reg, d.decision_key_id, d.decision_trust_epoch, "decision")) R.add("DECISION_KEY_UNAUTHORIZED");
  if ((ctx.consumed_capabilities || []).includes(d.capability_id)) R.add("CAPABILITY_UNAVAILABLE");
  const manualOnly = (ctx.profile.manual_only_actions || []).includes(ctx.x.action);
  const contained = d.adverse_disposition === "CONTAIN";
  let modeOk = false;
  if (d.auth_mode === "AUTOMATED") { modeOk = !manualOnly && !contained && d.approval_hash === NULL_APPROVAL; if (contained) R.add("ADVERSE_CONTAINMENT"); }
  else if (pHex) {
    const pr = approvalCheck(ctx, ts, pHex, pSig, d, U);
    const hashOk = d.approval_hash === sha(Buffer.from(pHex, "hex"));
    if (!hashOk) pr.push("APPROVAL_HASH_MISMATCH");
    pr.forEach((x) => R.add(x));
    modeOk = (manualOnly || contained) && hashOk && pr.length === 0;
  }
  if (!modeOk) R.add("AUTH_MODE_INVALID");
  return REASONS.filter((n) => R.has(n));
}
ok(H(auth.trust_state) === auth.trust_state_hash, "auth trust_state_hash");
for (const v of auth.vectors) {
  const i = v.input;
  const got = dbv(i.context, v.trust_state || auth.trust_state, i.decision_bytes, i.decision_signature, i.approval_bytes, i.approval_signature);
  ok(JSON.stringify(got) === JSON.stringify(v.expected_output.reason_codes), `auth ${v.test_id}: js=${got} vector=${v.expected_output.reason_codes}`);
  ok((got.length === 0) === v.expected_output.DecisionBindingValid, `auth ${v.test_id} validity`);
  const mask = got.reduce((m, n) => m | (1n << BigInt(REASONS.indexOf(n))), 0n);
  ok(mask === BigInt(v.expected_output.reason_bitfield), `auth ${v.test_id} bitfield`);
}

// ---------------- §2.3 numerics (independent) ----------------
const blob = readFileSync(T("vectors/kahan_cordic_ieee754.bin"));
const nman = JSON.parse(readFileSync(T("vectors/kahan_cordic_ieee754.json"), "utf8"));
ok(sha(blob) === nman.binary_sha256, "numeric bin sha256");
const dv = new DataView(blob.buffer, blob.byteOffset, blob.byteLength);
const toF = (bits) => { const d = new DataView(new ArrayBuffer(8)); d.setBigUint64(0, bits, true); return d.getFloat64(0, true); };
const toB = (x) => { const d = new DataView(new ArrayBuffer(8)); d.setFloat64(0, x, true); return d.getBigUint64(0, true); };
const exceptional = (bits) => ((bits >> 52n) & 0x7ffn) === 0x7ffn;
const nz = (x) => (x === 0 ? 0 : x); // -0 -> +0
let pos = 12;
const count = dv.getUint32(8, true);
ok(blob.subarray(0, 4).toString() === "PRNV" && count === nman.cases.length, "numeric header");
for (const c of nman.cases) {
  const op = dv.getUint8(pos), flags = dv.getUint8(pos + 1), n = dv.getUint32(pos + 4, true); pos += 8;
  const ins = []; for (let k = 0; k < n; k++) { ins.push(dv.getBigUint64(pos, true)); pos += 8; }
  const out = dv.getBigUint64(pos, true); pos += 8;
  let res = false, lock = false, bits = 0n;
  if (op === 1) {
    if (exceptional(ins[0]) || exceptional(ins[1])) lock = true;
    else { const C = nz(toF(ins[0])), Tt = nz(toF(ins[1])); res = C >= Tt; bits = toB(C); }
  } else {
    if (ins.some(exceptional)) lock = true;
    else {
      let s = 0, comp = 0;
      for (const b of ins) {
        const y = toF(b) - comp, t = s + y; comp = (t - s) - y; s = t;
        if (!Number.isFinite(s) || !Number.isFinite(comp)) { lock = true; break; }
      }
      if (!lock) bits = toB(nz(s));
    }
  }
  ok(bits === out && lock === Boolean(flags & 2) && (op !== 1 || res === Boolean(flags & 1)), `numeric ${c.id}`);
}

// ---------------- scoring profile jaccard-v0.1.0-alpha (independent) ----------------
{
  const sc = JSON.parse(readFileSync(T("vectors/scoring_jaccard_v0.1.0-alpha.json"), "utf8"));
  const fb = (h) => Buffer.from(h, "hex").readDoubleBE(0);
  const hb = (x) => { const b = Buffer.alloc(8); b.writeDoubleBE(x); return b.toString("hex"); };
  const bitsBig = (h) => BigInt("0x" + h);
  const K = sc.prc_exp_v1.constants_bits;
  const INV_LN2 = fb(K.INV_LN2), LN2_HI = fb(K.LN2_HI), LN2_LO = fb(K.LN2_LO);
  // coefficients recomputed here (1/n!), then checked against the published bits
  const COEF = []; { let fct = 1; for (let n = 0; n < 14; n++) { if (n) fct *= n; COEF.push(1 / fct); } }
  ok(COEF.map(hb).join() === K["C0..C13"].join(), "prc_exp coefficients");
  const pow2 = (k) => { const b = Buffer.alloc(8);
    if (k >= -1022) b.writeBigUInt64BE(BigInt(k + 1023) << 52n); else b.writeBigUInt64BE(1n << BigInt(k + 1074));
    return b.readDoubleBE(0); };
  const pexp = (x) => {
    if (Number.isNaN(x)) return x; if (x > 709) return Infinity; if (x < -745) return 0;
    const k = Math.floor(x * INV_LN2 + 0.5);
    const r = (x - k * LN2_HI) - k * LN2_LO;
    let p = COEF[13]; for (let n = 12; n >= 0; n--) p = p * r + COEF[n];
    if (k < -1022) return (p * pow2(-1022)) * pow2(k + 1022);
    if (k > 1023) return (p * pow2(1023)) * pow2(k - 1023);
    return p * pow2(k);
  };
  let expMismatchLibm = 0;
  for (const v of sc.prc_exp_v1.vectors) {
    ok(hb(pexp(fb(v.input_bits))) === v.expected_bits, `prc_exp ${v.input_repr}`);
    if (hb(Math.exp(fb(v.input_bits))) !== v.libm_exp_bits_informative) expMismatchLibm++;
  }
  const TYPES = ["SOURCE","DATABASE","COLLECTOR","RUNTIME","CLOCK","NETWORK_AS","TRUST_ROOT","API_ENDPOINT"];
  const utf8sort = (a) => [...a].sort(cmpBytes);
  const digest = (e) => { const n = { ...e, dependencies: utf8sort(new Set(e.dependencies || [])) }; return H(n); };
  const exceptional = (x) => !Number.isFinite(x);
  function evaluate(req, families, prof) {
    const omega = fb(prof.omega_bits), tb = fb(prof.t_base_bits), beta = fb(prof.beta_bits);
    const kU = fb(prof.kappa_unknown_bits), kD = fb(prof.kappa_degraded_bits);
    const material = new Set(prof.material_dependency_types);
    const R = new Set(); const out = { families: {} }; const elig = [];
    for (const fam of [...families].sort((a, b) => cmpBytes(a.family_id, b.family_id))) {
      const ids = fam.members.map((m) => m.evidence_id), byId = Object.fromEntries(fam.members.map((m) => [m.evidence_id, m]));
      let bad = new Set(ids).size !== ids.length;
      const roots = fam.members.filter((m) => m.parent_id === null || !(m.parent_id in byId));
      bad = bad || roots.length !== 1;
      const depth = {};
      if (!bad) for (const m of fam.members) {
        let cur = m, d = 0; const seen = new Set();
        while (cur.evidence_id !== roots[0].evidence_id) { if (seen.has(cur.evidence_id)) { bad = true; break; }
          seen.add(cur.evidence_id); cur = byId[cur.parent_id]; d++; }
        depth[m.evidence_id] = d;
      }
      bad = bad || fam.members.some((m) => (m.dependencies || []).some((a) => { const mm = /^([A-Z_]+):(\S+)$/.exec(a); return !mm || !TYPES.includes(mm[1]); }));
      if (bad) { out.families[fam.family_id] = { status: "MALFORMED" }; R.add("LINEAGE_CLASS_MALFORMED"); continue; }
      const root = roots[0], opt = fam.members.filter((m) => m !== root);
      const states = opt.map((m) => m.state || "VALID"), rs = root.state || "VALID";
      if (states.includes("INVALID") || rs === "INVALID") { out.families[fam.family_id] = { status: "INVALIDATED" }; R.add("FAMILY_INVALIDATED"); continue; }
      if (rs !== "VALID") { out.families[fam.family_id] = { status: "REPRESENTATIVE_" + rs }; continue; }
      const fac = states.filter((s) => s !== "UNAVAILABLE_OR_STALE").map((s) => s === "DEGRADED" ? kD : s === "VALID" ? 1 : 0);
      const A = fac.length ? Math.min(...fac) : 1;
      const Vr = fb(root.value_bits);
      if (Number.isFinite(Vr) && !(Vr >= 0 && Vr <= 1)) { out.families[fam.family_id] = { status: "MALFORMED" }; R.add("LINEAGE_CLASS_MALFORMED"); continue; }
      const V = Vr * A;
      const mat = new Set((root.dependencies || []).filter((a) => material.has(a.split(":")[0])));
      out.families[fam.family_id] = { status: "ELIGIBLE", representative: root.evidence_id, representative_digest: digest(root), A_bits: hb(A), V_bits: hb(V), depth };
      elig.push({ rep: root.evidence_id, fid: fam.family_id, V, mat });
    }
    elig.sort((a, b) => cmpBytes(a.rep, b.rep));
    const pair = {}; let unknown = false;
    for (const k of elig) {
      if (elig.length === 1) { k.I = 1; break; }
      let mn = Infinity;
      for (const j of elig) { if (j === k) continue;
        let i; if (!k.mat.size || !j.mat.size) { i = kU; unknown = true; }
        else { let inter = 0; for (const a of k.mat) if (j.mat.has(a)) inter++; const uni = k.mat.size + j.mat.size - inter; i = 1 - inter / uni; }
        pair[`${k.fid}|${j.fid}`] = hb(i); mn = Math.min(mn, i); }
      k.I = mn;
    }
    let M = 0; for (const e of elig) { M = M + e.V * e.I; out.families[e.fid].I_bits = hb(e.I); }
    let C = Number.isFinite(M) ? 1 - pexp(-(M / omega)) : NaN;
    const g = prof.consequence_gamma_bits[req.action];
    let Tt = NaN; if (g !== undefined) Tt = 1 - (1 - tb) * pexp(-(fb(g) * beta)); else R.add("CONSEQUENCE_CLASS_UNDEFINED");
    if (!elig.length) { R.add("NO_ELIGIBLE_EVIDENCE"); M = 0; C = 0; }
    let lock = false, res = false;
    if (exceptional(C) || exceptional(Tt)) { lock = true; C = 0; R.add("SAFE_STATE_LOCK"); }
    else { C = C === 0 ? 0 : C; res = C >= (Tt === 0 ? 0 : Tt); if (!res) R.add("THRESHOLD_NOT_MET"); }
    const allow = res && !R.has("CONSEQUENCE_CLASS_UNDEFINED");
    return { ...out, pair, unknown, M: hb(M), C: hb(C), T: hb(Tt), lock, decision: allow ? "ALLOW" : "DENY",
             reasons: REASONS.filter((n) => R.has(n)), eligible: elig.map((e) => e.fid) };
  }
  const sortObj = (o) => JSON.stringify(Object.fromEntries(Object.entries(o).sort((a, b) => cmpBytes(a[0], b[0]))));
  for (const c of sc.cases) {
    const prof = sc.profiles[c.profile].declaration, e = c.expected_trace;
    ok(H(prof) === sc.profiles[c.profile].fingerprint, `scoring ${c.id} profile fingerprint`);
    const g = evaluate(c.input.request, c.input.families, prof);
    const tag = `scoring ${c.id}`;
    ok(g.decision === e.decision, `${tag} decision js=${g.decision}`);
    ok(JSON.stringify(g.reasons) === JSON.stringify(e.reason_codes), `${tag} reasons js=${g.reasons}`);
    ok(g.M === e.M_bits && g.C === e.C_bits && g.T === e.T_bits, `${tag} M/C/T bits js=${g.M},${g.C},${g.T}`);
    ok(g.lock === e.safe_state_lock && g.unknown === e.unknown_lineage_fallback_used, `${tag} flags`);
    ok(JSON.stringify(g.eligible) === JSON.stringify(e.eligible_families), `${tag} eligible`);
    ok(sortObj(g.pair) === sortObj(e.pairwise_independence_bits), `${tag} pairwise`);
    for (const [fid, ef] of Object.entries(e.families)) {
      const gf = g.families[fid] || {};
      ok(gf.status === ef.status, `${tag} ${fid} status`);
      if (ef.status === "ELIGIBLE")
        ok(gf.representative === ef.representative && gf.representative_digest === ef.representative_digest &&
           gf.A_bits === ef.A_bits && gf.V_bits === ef.V_bits && gf.I_bits === ef.I_bits &&
           sortObj(gf.depth) === sortObj(ef.lineage_depth), `${tag} ${fid} values`);
    }
  }
  const h = sc.headline;
  const byId = Object.fromEntries(sc.cases.map((c) => [c.id, c]));
  const strip = (fs) => enc(fs.map((f_) => ({ ...f_, members: f_.members.map((m) => ({ ...m, dependencies: [] })) })));
  ok(strip(byId[h.allow_case].input.families).equals(strip(byId[h.deny_case].input.families)), "headline inputs identical except lineage");
  // ---------- coverage-v0.2.0-alpha + engine guard E-1 (independent) ----------
  const cv = JSON.parse(readFileSync(T("vectors/scoring_coverage_v0.2.0-alpha.json"), "utf8"));
  const nonfinite = (h) => !Number.isFinite(fb(h));
  function engine(req, families, prof) {
    const bad = [];
    for (const k of Object.keys(prof).sort(cmpBytes)) if (k.endsWith("_bits") && typeof prof[k] === "string" && nonfinite(prof[k])) bad.push(`profile.${k}`);
    for (const a of Object.keys(prof.consequence_gamma_bits || {}).sort(cmpBytes)) if (nonfinite(prof.consequence_gamma_bits[a])) bad.push(`profile.consequence_gamma_bits.${a}`);
    for (const fm of [...families].sort((a, b) => cmpBytes(a.family_id, b.family_id)))
      for (const m of [...fm.members].sort((a, b) => cmpBytes(a.evidence_id, b.evidence_id)))
        if (m.value_bits !== undefined && nonfinite(m.value_bits)) bad.push(`${fm.family_id}/${m.evidence_id}.value_bits`);
    if (bad.length) return { stage: "ENGINE_GUARD", bad, M: hb(0), C: hb(0), T: null, lock: true, decision: "DENY", reasons: ["SAFE_STATE_LOCK", "NONFINITE_INPUT"] };
    return coverage(req, families, prof);
  }
  function coverage(req, families, prof) {
    const omega = fb(prof.omega_bits), tb = fb(prof.t_base_bits), beta = fb(prof.beta_bits);
    const kU = fb(prof.kappa_unknown_bits), kD = fb(prof.kappa_degraded_bits), gate = fb(prof.unknown_gate_t_bits);
    const material = new Set(prof.material_dependency_types);
    const R = new Set(); const fams = {}; const elig = [];
    const g = prof.consequence_gamma_bits[req.action];
    let Tt = NaN; if (g !== undefined) Tt = 1 - (1 - tb) * pexp(-(fb(g) * beta)); else R.add("CONSEQUENCE_CLASS_UNDEFINED");
    const kEff = (Tt === Tt && Tt >= gate) ? 0 : kU;
    for (const fam of [...families].sort((a, b) => cmpBytes(a.family_id, b.family_id))) {
      const byId = Object.fromEntries(fam.members.map((m) => [m.evidence_id, m]));
      let bad = new Set(fam.members.map((m) => m.evidence_id)).size !== fam.members.length;
      const roots = fam.members.filter((m) => m.parent_id === null || !(m.parent_id in byId));
      bad = bad || roots.length !== 1;
      if (!bad) for (const m of fam.members) { let cur = m; const seen = new Set();
        while (cur.evidence_id !== roots[0].evidence_id) { if (seen.has(cur.evidence_id)) { bad = true; break; } seen.add(cur.evidence_id); cur = byId[cur.parent_id]; } }
      bad = bad || fam.members.some((m) => (m.dependencies || []).some((a) => { const mm = /^([A-Z_]+):(\S+)$/.exec(a); return !mm || !TYPES.includes(mm[1]); }));
      if (bad) { fams[fam.family_id] = { status: "MALFORMED" }; R.add("LINEAGE_CLASS_MALFORMED"); continue; }
      const root = roots[0], states = fam.members.filter((m) => m !== root).map((m) => m.state || "VALID"), rs = root.state || "VALID";
      if (states.includes("INVALID") || rs === "INVALID") { fams[fam.family_id] = { status: "INVALIDATED" }; R.add("FAMILY_INVALIDATED"); continue; }
      if (rs !== "VALID") { fams[fam.family_id] = { status: "REPRESENTATIVE_" + rs }; continue; }
      const Vr = fb(root.value_bits);
      if (!(Vr >= 0 && Vr <= 1)) { fams[fam.family_id] = { status: "MALFORMED" }; R.add("LINEAGE_CLASS_MALFORMED"); continue; }
      const fac = states.filter((x) => x !== "UNAVAILABLE_OR_STALE").map((x) => x === "DEGRADED" ? kD : x === "VALID" ? 1 : 0);
      let V = Vr * (fac.length ? Math.min(...fac) : 1);
      let atoms = [...new Set((root.dependencies || []).filter((a) => material.has(a.split(":")[0])))];
      if (!atoms.length) { V = V * kEff; atoms = ["UNKNOWN:*"]; }
      const w = V / atoms.length;
      fams[fam.family_id] = { status: "ELIGIBLE", V_bits: hb(V), share: hb(w), atoms: [...atoms].sort(cmpBytes) };
      elig.push({ fid: fam.family_id, atoms, w });
    }
    const best = new Map();
    for (const e of elig) for (const a of e.atoms) if (!best.has(a) || e.w > best.get(a)) best.set(a, e.w);
    let M = 0; for (const a of [...best.keys()].sort(cmpBytes)) M = M + best.get(a);
    let C = 1 - pexp(-(M / omega));
    if (!elig.length) { R.add("NO_ELIGIBLE_EVIDENCE"); M = 0; C = 0; }
    let lock = false, res = false;
    if (!Number.isFinite(C) || !Number.isFinite(Tt)) { lock = true; C = 0; R.add("SAFE_STATE_LOCK"); }
    else { C = C === 0 ? 0 : C; res = C >= Tt; if (!res) R.add("THRESHOLD_NOT_MET"); }
    return { stage: "PROFILE", fams, eligible: elig.map((e) => e.fid), kEff: hb(kEff), M: hb(M), C: hb(C), T: hb(Tt), lock,
             decision: res && !R.has("CONSEQUENCE_CLASS_UNDEFINED") ? "ALLOW" : "DENY", reasons: REASONS.filter((n) => R.has(n)) };
  }
  const cvBase = cv.profiles.calibrated.declaration;
  ok(H(cvBase) === cv.profiles.calibrated.fingerprint, "coverage profile fingerprint");
  const cvById = {};
  let xfail = 0;
  for (const c of cv.cases) {
    const e = c.expected_trace, tag = `coverage ${c.id}`;
    const g = engine(c.input.request, c.input.families, c.profile_declaration || cvBase);
    cvById[c.id] = e;
    ok(g.stage === e.stage && g.decision === e.decision, `${tag} stage/decision js=${g.stage}/${g.decision}`);
    ok(JSON.stringify(g.reasons) === JSON.stringify(e.reason_codes), `${tag} reasons js=${g.reasons}`);
    ok(g.M === e.M_bits && g.C === e.C_bits && (e.T_bits === null ? g.T === null : g.T === e.T_bits), `${tag} M/C/T bits`);
    if (e.stage === "ENGINE_GUARD") { ok(JSON.stringify(g.bad) === JSON.stringify(e.nonfinite_inputs), `${tag} guard locations`); continue; }
    ok(g.kEff === e.kappa_unknown_effective_bits && JSON.stringify(g.eligible) === JSON.stringify(e.eligible_families), `${tag} kappa/eligible`);
    for (const [fid, ef] of Object.entries(e.families)) {
      const gf = g.fams[fid] || {};
      ok(gf.status === ef.status, `${tag} ${fid} status`);
      if (ef.status === "ELIGIBLE") ok(gf.V_bits === ef.V_bits && gf.share === ef.per_atom_share_bits && JSON.stringify(gf.atoms) === JSON.stringify(ef.atoms), `${tag} ${fid} values`);
    }
  }
  for (const c of cv.cases) if (c.known_exploit) {
    const s = c.known_exploit.security_expectation;
    const meets = c.expected_trace.decision === s.decision && c.expected_trace.M_bits === cvById[s.same_M_as].M_bits;
    ok(!meets, `coverage ${c.id} exploit no longer reproduces`); xfail++;
  }
  cv.property_witnesses.forEach((w, i) => {
    const t0 = engine(w.request, w.base, cvBase), t1 = engine(w.request, [...w.base, w.added], cvBase);
    ok(t0.M === w.M_base_bits && t1.M === w.M_after_bits, `coverage witness ${i} bits`);
    const prop = w.kind === "idempotent" ? t1.M === t0.M : fb(t1.M) >= fb(t0.M);
    ok(prop && !(t0.decision === "ALLOW" && t1.decision === "DENY"), `coverage witness ${i} (${w.kind}) property`);
  });
  console.log(`coverage-v0.2.0 (superseded): ${cv.cases.length} cases, ${cv.property_witnesses.length} witnesses, ${xfail} exploit reproduced by design (closed in v0.2.1)`);
  // ---------- coverage-v0.2.1-alpha: P-1 provenance admissibility + defeater veto (independent) ----------
  const c21 = JSON.parse(readFileSync(T("vectors/scoring_coverage_v0.2.1-alpha.json"), "utf8"));
  const PROV_DOMAIN = "PROVENANCE-RAZOR-PROVENANCE-V0.3.0-Alpha";
  const REQ = ["domain","schema_ver","attestation_id","provenance_key_id","provenance_trust_epoch","attester_id","evidence_digest","dependencies","tau_issued","tau_expiry"];
  const core = (m) => Object.fromEntries(Object.entries(m).filter(([k]) => k !== "provenance" && k !== "dependencies"));
  function admit(m, trust, prof, tauEval) {
    const env = m.provenance;
    if (env === undefined) return ["UNATTESTED", null];
    if (!envelopeOk(env)) return ["TAMPERED", null];
    const p = env.payload;
    if (!schemaValid(p, "attestation")) return ["TAMPERED", null];
    if (p.domain !== PROV_DOMAIN) return ["TAMPERED", null];
    const e = (trust.key_registry || trust.registry)[p.provenance_key_id];
    if (!e) return ["INADMISSIBLE", null];
    if (e.domain !== "provenance") return ["TAMPERED", null];
    if (!edVerify(e.public_key, H(p), env.signature)) return ["TAMPERED", null];
    if (p.attester_id !== (e.owner ?? e.attester_id)) return ["TAMPERED", null];
    if (p.evidence_digest !== H(core(m))) return ["TAMPERED", null];
    if (!e.epochs.includes(p.provenance_trust_epoch)) return ["INADMISSIBLE", null];
    if ((trust.revoked_attestation_ids || []).includes(p.attestation_id)) return ["INADMISSIBLE", null];
    if (!(p.tau_issued <= tauEval && tauEval <= p.tau_expiry)) return ["INADMISSIBLE", null];
    if (prof.attester_independence_required && p.attester_id === m.producer) return ["INADMISSIBLE", null];
    if (JSON.stringify(p.dependencies) !== JSON.stringify(utf8sort(new Set(p.dependencies)))) return ["INADMISSIBLE", null];
    const reg = new Set(prof.atom_registry.atoms), al = prof.atom_registry.aliases || {};
    const res = new Set();
    for (const a of p.dependencies) {
      const mm = typeof a === "string" ? /^([A-Z_]+):(\S+)$/.exec(a) : null;
      if (!mm || !TYPES.includes(mm[1])) return ["INADMISSIBLE", null];
      if (!(e.authorized_atom_types || []).includes(mm[1])) return ["INADMISSIBLE", null];
      const c = al[a] ?? a; if (!reg.has(c)) return ["INADMISSIBLE", null];
      res.add(c);
    }
    const cnt = {}; for (const c of res) { const t = c.split(":")[0]; cnt[t] = (cnt[t] || 0) + 1; }
    for (const [t, n] of Object.entries(cnt)) if (n > (prof.max_atoms_per_type[t] ?? 0)) return ["INADMISSIBLE", null];
    return ["ADMITTED", res];
  }
  function cov21(req, families, prof, trust) {
    const g0 = engine(req, families, prof);  // reuse the E-1 guard only
    if (g0.stage === "ENGINE_GUARD") return g0;
    const omega = fb(prof.omega_bits), tb = fb(prof.t_base_bits), beta = fb(prof.beta_bits);
    const kU = fb(prof.kappa_unknown_bits), kD = fb(prof.kappa_degraded_bits), gate = fb(prof.unknown_gate_t_bits);
    const material = new Set(prof.material_dependency_types);
    const R = new Set(), fams = {}, elig = [], defeaters = [];
    const g = prof.consequence_gamma_bits[req.action];
    let Tt = NaN; if (g !== undefined) Tt = 1 - (1 - tb) * pexp(-(fb(g) * beta)); else R.add("CONSEQUENCE_CLASS_UNDEFINED");
    const kEff = (Tt === Tt && Tt >= gate) ? 0 : kU;
    for (const fam of [...families].sort((a, b) => cmpBytes(a.family_id, b.family_id))) {
      const byId = Object.fromEntries(fam.members.map((m) => [m.evidence_id, m]));
      let bad = new Set(fam.members.map((m) => m.evidence_id)).size !== fam.members.length;
      const roots = fam.members.filter((m) => m.parent_id === null || !(m.parent_id in byId));
      bad = bad || roots.length !== 1;
      if (!bad) for (const m of fam.members) { let cur = m; const seen = new Set();
        while (cur.evidence_id !== roots[0].evidence_id) { if (seen.has(cur.evidence_id)) { bad = true; break; } seen.add(cur.evidence_id); cur = byId[cur.parent_id]; } }
      if (bad) { fams[fam.family_id] = { status: "MALFORMED" }; R.add("LINEAGE_CLASS_MALFORMED"); continue; }
      const prov = Object.fromEntries(fam.members.map((m) => [m.evidence_id, admit(m, trust, prof, req.tau_eval)]));
      const root = roots[0], opt = fam.members.filter((m) => m !== root);
      const outcomes = Object.fromEntries(Object.entries(prov).map(([k, v]) => [k, v[0]]));
      if (Object.values(outcomes).includes("TAMPERED")) { fams[fam.family_id] = { status: "INVALIDATED", outcomes }; R.add("PROVENANCE_TAMPERED"); R.add("FAMILY_INVALIDATED"); continue; }
      const states = opt.map((m) => m.state || "VALID"), rs = root.state || "VALID";
      if (states.includes("INVALID") || rs === "INVALID") { fams[fam.family_id] = { status: "INVALIDATED", outcomes }; R.add("FAMILY_INVALIDATED"); continue; }
      for (const m of opt) if (m.state === "CONTRADICTORY" && prov[m.evidence_id][0] === "ADMITTED") defeaters.push(m.evidence_id);
      if (rs !== "VALID") { fams[fam.family_id] = { status: "REPRESENTATIVE_" + rs, outcomes }; continue; }
      const Vr = fb(root.value_bits);
      if (!(Vr >= 0 && Vr <= 1)) { fams[fam.family_id] = { status: "MALFORMED" }; R.add("LINEAGE_CLASS_MALFORMED"); continue; }
      const fac = states.filter((x) => x !== "UNAVAILABLE_OR_STALE").map((x) => x === "DEGRADED" ? kD : x === "VALID" ? 1 : 0);
      let V = Vr * (fac.length ? Math.min(...fac) : 1);
      const [o, atoms] = prov[root.evidence_id];
      let mat = o === "ADMITTED" ? [...atoms].filter((a) => material.has(a.split(":")[0])) : [];
      if (!mat.length) { V = V * kEff; mat = ["UNKNOWN:*"]; }
      const w = V / mat.length;
      fams[fam.family_id] = { status: "ELIGIBLE", outcomes, V_bits: hb(V), share: hb(w), atoms: utf8sort(mat) };
      elig.push({ fid: fam.family_id, mat, w });
    }
    const best = new Map();
    for (const e of elig) for (const a of e.mat) if (!best.has(a) || e.w > best.get(a)) best.set(a, e.w);
    let M = 0; for (const a of [...best.keys()].sort(cmpBytes)) M = M + best.get(a);
    let C = 1 - pexp(-(M / omega));
    if (!elig.length) { R.add("NO_ELIGIBLE_EVIDENCE"); M = 0; C = 0; }
    let res = false;
    if (!Number.isFinite(C) || !Number.isFinite(Tt)) { C = 0; R.add("SAFE_STATE_LOCK"); }
    else { res = C >= Tt; if (!res) R.add("THRESHOLD_NOT_MET"); }
    if (defeaters.length) R.add("DEFEATER_PRESENT");
    return { stage: "PROFILE", fams, eligible: elig.map((e) => e.fid), defeaters: defeaters.length, M: hb(M), C: hb(C), T: hb(Tt),
             decision: res && !defeaters.length && !R.has("CONSEQUENCE_CLASS_UNDEFINED") ? "ALLOW" : "DENY", reasons: REASONS.filter((n) => R.has(n)) };
  }
  const c21p = Object.fromEntries(Object.entries(c21.profiles).map(([k, v]) => [k, v.declaration]));
  for (const [k, v] of Object.entries(c21.profiles)) ok(H(v.declaration) === v.fingerprint, `v0.2.1 profile ${k} fingerprint`);
  for (const c of c21.cases) {
    const e = c.expected_trace, tag = `v0.2.1 ${c.id}`;
    const g = cov21(c.input.request, c.input.families, c21p[c.profile], c.trust_state || c21.trust_state);
    ok(g.decision === e.decision && g.stage === e.stage, `${tag} decision js=${g.decision}`);
    ok(JSON.stringify(g.reasons) === JSON.stringify(e.reason_codes), `${tag} reasons js=${g.reasons}`);
    ok(g.M === e.M_bits && g.C === e.C_bits, `${tag} M/C bits`);
    if (e.stage !== "PROFILE") continue;
    ok(g.defeaters === e.defeaters.length && JSON.stringify(g.eligible) === JSON.stringify(e.eligible_families), `${tag} defeaters/eligible`);
    for (const [fid, ef] of Object.entries(e.families)) {
      const gf = g.fams[fid] || {};
      ok(gf.status === ef.status, `${tag} ${fid} status js=${gf.status}`);
      if (ef.provenance) for (const [eid, po] of Object.entries(ef.provenance)) ok(gf.outcomes && gf.outcomes[eid] === po.outcome, `${tag} ${eid} provenance js=${gf.outcomes && gf.outcomes[eid]} py=${po.outcome}`);
      if (ef.status === "ELIGIBLE") ok(gf.V_bits === ef.V_bits && gf.share === ef.per_atom_share_bits && JSON.stringify(gf.atoms) === JSON.stringify(ef.atoms), `${tag} ${fid} values`);
    }
  }
  c21.property_witnesses.forEach((w, i) => {
    const t0 = cov21(w.request, w.base, c21p.calibrated, c21.trust_state), t1 = cov21(w.request, [...w.base, w.added], c21p.calibrated, c21.trust_state);
    ok(t0.M === w.M_base_bits && t1.M === w.M_after_bits, `v0.2.1 witness ${i} bits`);
    const prop = { monotone: fb(t1.M) >= fb(t0.M) && !(t0.decision === "ALLOW" && t1.decision === "DENY"), idempotent: t1.M === t0.M, defeater: t1.decision === "DENY" }[w.kind];
    ok(prop, `v0.2.1 witness ${i} (${w.kind})`);
  });
  ok(c21.known_exploits_open.length === 0, "v0.2.1 has no open exploits");
  console.log(`coverage-v0.2.1: ${c21.cases.length} cases, ${c21.property_witnesses.length} witnesses, provenance admissibility re-derived independently`);
  // ---------- coverage-v0.3.0-alpha: M-1 membership manifests + A-1 adverse disposition (independent) ----------
  const c30 = JSON.parse(readFileSync(T("vectors/scoring_coverage_v0.3.0-alpha.json"), "utf8"));
  const MEMB_DOMAIN = "PROVENANCE-RAZOR-MEMBERSHIP-V0.3.0-Alpha";
  const memberOk = (m) => KS.map(m) && KS.str(m.evidence_id) && "parent_id" in m && (m.parent_id === null || KS.str(m.parent_id))
    && KS.str(m.producer) && typeof m.value_bits === "string" && /^[0-9a-f]{16}$/.test(m.value_bits)
    && (!("state" in m) || ["VALID", "UNAVAILABLE_OR_STALE", "DEGRADED", "CONTRADICTORY", "INVALID"].includes(m.state))
    && (!("dependencies" in m) || Array.isArray(m.dependencies));
  function membership(fam, ts, tauEval) {
    const env = fam.manifest;
    if (env === undefined) return ["UNVERIFIED", [], 0, ""];
    if (!envelopeOk(env) || !schemaValid(env.payload, "manifest")) return ["TAMPERED", [], 0, ""];
    const p = env.payload;
    if (p.domain !== MEMB_DOMAIN) return ["TAMPERED", [], 0, ""];
    const e = ts.key_registry[p.membership_key_id];
    if (!e) return ["UNVERIFIED", [], 0, ""];
    if (e.domain !== "membership" || !edVerify(e.public_key, H(p), env.signature) || p.authority_id !== e.owner || p.family_id !== fam.family_id)
      return ["TAMPERED", [], 0, ""];
    const ids = p.members.map((m) => m.evidence_id);
    if (JSON.stringify(ids) !== JSON.stringify(utf8sort(new Set(ids)))) return ["TAMPERED", [], 0, ""];
    if (!e.epochs.includes(p.membership_trust_epoch) || (ts.revoked_manifest_ids || []).includes(p.manifest_id)
        || !(p.tau_issued <= tauEval && tauEval <= p.tau_expiry) || !(e.membership_scopes || []).some((pat) => glob(fam.family_id, pat)))
      return ["UNVERIFIED", [], 0, ""];
    const listed = Object.fromEntries(p.members.map((m) => [m.evidence_id, m]));
    const groups = new Map();
    for (const m of fam.members) groups.set(m.evidence_id, [...(groups.get(m.evidence_id) || []), m]);
    const admitted = []; let quarantined = 0, altered = false;
    for (const [eid, cands] of groups) {
      if (!(eid in listed)) { quarantined += cands.length; continue; }
      const match = cands.filter((m) => H(core(m)) === listed[eid].evidence_digest && m.parent_id === listed[eid].parent_id);
      if (match.length) { admitted.push(match[0]); quarantined += cands.length - 1; } else altered = true;
    }
    if (altered) return ["TAMPERED", [], quarantined, p.ingress_channel];
    if (Object.keys(listed).some((eid) => !groups.has(eid))) return ["UNVERIFIED", [], quarantined, p.ingress_channel];
    return ["VERIFIED", admitted, quarantined, p.ingress_channel];
  }
  const PREC = { CLEAR: 0, CONTAIN: 1, DEFEAT: 2, LOCK: 3 };
  function cov30(req, families, prof, ts) {
    // E-1 guard (independent): any non-finite binary64 input anywhere -> DENY before profile logic
    const hexOk = (h) => typeof h === "string" && /^[0-9a-f]{16}$/.test(h);
    const nonfin = [...Object.entries(prof).filter(([k, v]) => k.endsWith("_bits") && typeof v === "string").map(([, v]) => v),
      ...Object.values(prof.consequence_gamma_bits || {}),
      ...families.filter((f) => KS.map(f) && Array.isArray(f.members)).flatMap((f) => f.members.filter((m) => KS.map(m) && hexOk(m.value_bits)).map((m) => m.value_bits))]
      .some((h) => !Number.isFinite(fb(h)));
    if (nonfin) return { stage: "ENGINE_GUARD", M: hb(0), C: hb(0), decision: "DENY", reasons: ["SAFE_STATE_LOCK", "NONFINITE_INPUT"] };
    const f64 = (h) => fb(h);
    const omega = f64(prof.omega_bits), tb = f64(prof.t_base_bits), tbm = f64(prof.t_base_manual_bits), beta = f64(prof.beta_bits);
    const kU = f64(prof.kappa_unknown_bits), kD = f64(prof.kappa_degraded_bits), gate = f64(prof.unknown_gate_t_bits);
    const material = new Set(prof.material_dependency_types);
    const R = new Set(), fams = {}, elig = []; let disp = "CLEAR", quarantined = 0; const obl = new Map();
    const raise = (l) => { if (PREC[l] > PREC[disp]) disp = l; };
    const g = prof.consequence_gamma_bits[req.action];
    let Tt = NaN, Tm = NaN;
    if (g !== undefined) { Tt = 1 - (1 - tb) * pexp(-(f64(g) * beta)); Tm = 1 - (1 - tbm) * pexp(-(f64(g) * beta)); } else R.add("CONSEQUENCE_CLASS_UNDEFINED");
    const kEff = (Tt === Tt && Tt >= gate) ? 0 : kU;
    const bind = { action: req.action, assertion: req.assertion, target: req.target, evidence_snapshot_id: req.evidence_snapshot_id };
    const good = [];
    for (const f of families) { if (KS.map(f) && KS.str(f.family_id) && Array.isArray(f.members)) good.push(f); else R.add("LINEAGE_CLASS_MALFORMED"); }
    for (const fam0 of good.sort((a, b) => cmpBytes(a.family_id, b.family_id))) {
      const fid = fam0.family_id;
      quarantined += fam0.members.filter((m) => !memberOk(m)).length;
      const fam = { ...fam0, members: fam0.members.filter(memberOk) };
      const [mo, members, q, ingress] = membership(fam, ts, req.tau_eval);
      quarantined += q;
      if (mo === "UNVERIFIED") { fams[fid] = { status: "MEMBERSHIP_UNVERIFIED" }; R.add("MEMBERSHIP_UNVERIFIED"); continue; }
      if (mo === "TAMPERED") { fams[fid] = { status: "INVALIDATED" }; R.add("MEMBERSHIP_TAMPERED"); R.add("FAMILY_INVALIDATED"); continue; }
      const byId = Object.fromEntries(members.map((m) => [m.evidence_id, m]));
      const roots = members.filter((m) => m.parent_id === null || !(m.parent_id in byId));
      let bad = roots.length !== 1;
      if (!bad) for (const m of members) { let cur = m; const seen = new Set();
        while (cur.evidence_id !== roots[0].evidence_id) { if (seen.has(cur.evidence_id)) { bad = true; break; } seen.add(cur.evidence_id); cur = byId[cur.parent_id]; } }
      if (bad) { fams[fid] = { status: "MALFORMED" }; R.add("LINEAGE_CLASS_MALFORMED"); continue; }
      const prov = Object.fromEntries(members.map((m) => [m.evidence_id, admit(m, ts, ts, req.tau_eval)]));
      for (const m of members) {
        if (m.state !== "CONTRADICTORY") continue;
        const o = prov[m.evidence_id][0];
        if (o === "TAMPERED") { raise("LOCK"); continue; }
        if (!(ts.conflict_types || []).includes(m.conflict_type) || !same(m.contradicts, bind)) continue;
        const lvl = o === "ADMITTED" ? "DEFEAT" : "CONTAIN";
        raise(lvl);
        const dg = H({ conflict_type: m.conflict_type ?? null, contradicts: m.contradicts ?? null, claim: m.claim ?? null });
        const rec = obl.get(dg) || { lvl, n: 0 };
        if (PREC[lvl] > PREC[rec.lvl]) rec.lvl = lvl;
        rec.n++; obl.set(dg, rec);
      }
      if (Object.values(prov).some((v) => v[0] === "TAMPERED")) { fams[fid] = { status: "INVALIDATED" }; R.add("PROVENANCE_TAMPERED"); R.add("FAMILY_INVALIDATED"); continue; }
      const root = roots[0], states = members.filter((m) => m !== root).map((m) => m.state || "VALID"), rs = root.state || "VALID";
      if (states.includes("INVALID") || rs === "INVALID") { fams[fid] = { status: "INVALIDATED" }; R.add("FAMILY_INVALIDATED"); continue; }
      if (rs !== "VALID") { fams[fid] = { status: "REPRESENTATIVE_" + rs }; continue; }
      const Vr = f64(root.value_bits);
      if (!(Vr >= 0 && Vr <= 1)) { fams[fid] = { status: "MALFORMED" }; R.add("LINEAGE_CLASS_MALFORMED"); continue; }
      const fac = states.filter((x) => x !== "UNAVAILABLE_OR_STALE" && x !== "CONTRADICTORY").map((x) => x === "DEGRADED" ? kD : 1);
      let V = Vr * (fac.length ? Math.min(...fac) : 1);
      const [o, atoms] = prov[root.evidence_id];
      let mat = o === "ADMITTED" ? [...atoms].filter((a) => material.has(a.split(":")[0])) : [];
      if (!mat.length) { V = V * kEff; mat = ["UNKNOWN:*"]; }
      fams[fid] = { status: "ELIGIBLE", V_bits: hb(V), atoms: utf8sort(mat) };
      elig.push({ fid, mat, w: V / mat.length });
    }
    const best = new Map();
    for (const e of elig) for (const a of e.mat) if (!best.has(a) || e.w > best.get(a)) best.set(a, e.w);
    let M = 0; for (const a of [...best.keys()].sort(cmpBytes)) M = M + best.get(a);
    let C = 1 - pexp(-(M / omega));
    if (!elig.length) { R.add("NO_ELIGIBLE_EVIDENCE"); M = 0; C = 0; }
    const excep = !Number.isFinite(C) || !Number.isFinite(Tt);
    if (excep) { raise("LOCK"); C = 0; }
    let decision;
    if (disp === "LOCK") { decision = "DENY"; R.add("SAFE_STATE_LOCK"); }
    else if (disp === "DEFEAT") { decision = "DENY"; R.add("DEFEATER_PRESENT"); }
    else if (disp === "CONTAIN") { R.add("ADVERSE_CONTAINMENT"); if (C >= Tm) decision = "MANUAL_REQUIRED"; else { decision = "DENY"; R.add("THRESHOLD_NOT_MET"); } }
    else { if (C >= Tt) decision = "ALLOW"; else { decision = "DENY"; R.add("THRESHOLD_NOT_MET"); } }
    if (R.has("CONSEQUENCE_CLASS_UNDEFINED")) decision = "DENY";
    const cont = [...obl.entries()].filter(([, r]) => r.lvl === "CONTAIN").map(([d]) => d).sort();
    return { stage: "PROFILE", fams, disp, M: hb(M), C: hb(C), decision, reasons: REASONS.filter((n) => R.has(n)), quarantined,
             obligations: obl.size, occurrences: [...obl.values()].reduce((a, r) => a + r.n, 0),
             containment: cont.length ? H(cont) : NULL_CONTAINMENT, eligible: elig.map((e) => e.fid) };
  }
  ok(H(c30.scoring_profile_declaration) === c30.scoring_profile_hash && H(c30.trust_state) === c30.trust_state_hash, "v0.3.0 hashes");
  for (const c of c30.cases) {
    const e = c.expected_trace, tag = `v0.3.0 ${c.id}`;
    if (e.stage === "VERIFIER_FAULT") continue;
    const gt = cov30(c.input.request, c.input.families, c30.scoring_profile_declaration, c.trust_state || c30.trust_state);
    ok(gt.decision === e.decision && gt.stage === e.stage, `${tag} decision js=${gt.decision} py=${e.decision}`);
    ok(JSON.stringify(gt.reasons) === JSON.stringify(e.reason_codes), `${tag} reasons js=${gt.reasons} py=${e.reason_codes}`);
    ok(gt.M === e.M_bits && gt.C === e.C_bits, `${tag} M/C bits`);
    if (e.stage !== "PROFILE") continue;
    ok(gt.disp === e.adverse_disposition && gt.containment === e.containment_digest, `${tag} disposition js=${gt.disp}`);
    ok(gt.obligations === e.adverse_obligations.length && gt.occurrences === e.adverse_obligations.reduce((a, o) => a + o.occurrences.length, 0), `${tag} obligations`);
    ok(gt.quarantined === e.audit_events.filter((a) => a.event === "MEMBER_QUARANTINED").length, `${tag} quarantine count js=${gt.quarantined}`);
    ok(JSON.stringify(gt.eligible) === JSON.stringify(e.eligible_families), `${tag} eligible`);
    for (const [fid, ef] of Object.entries(e.families)) {
      const gf = gt.fams[fid] || {};
      ok(gf.status === ef.status, `${tag} ${fid} status js=${gf.status} py=${ef.status}`);
      if (ef.status === "ELIGIBLE") ok(gf.V_bits === ef.V_bits && JSON.stringify(gf.atoms) === JSON.stringify(ef.atoms), `${tag} ${fid} values`);
    }
  }
  c30.property_witnesses.forEach((w, i) => {
    const t0 = cov30(w.request, w.base, c30.scoring_profile_declaration, c30.trust_state);
    const t1 = cov30(w.request, w.after, c30.scoring_profile_declaration, c30.trust_state);
    ok(t0.M === w.M_base_bits && t1.M === w.M_after_bits && t0.C === w.C_base_bits && t1.C === w.C_after_bits, `v0.3.0 witness ${i} bits`);
    ok(t0.decision === w.decision_base && t1.decision === w.decision_after, `v0.3.0 witness ${i} decisions`);
  });
  console.log(`coverage-v0.3.0: ${c30.cases.length} cases, ${c30.property_witnesses.length} witnesses; membership, containment and defeat re-derived independently`);
  console.log(`scoring: prc_exp matches published bits on every input; platform Math.exp differs from the published libm reference on ${expMismatchLibm} of ${sc.prc_exp_v1.vectors.length} inputs`);
}

console.log(`xcheck (JavaScript, independent): ${checks} checks, ${fails.length} failures`);
for (const f of fails) console.log("  FAIL", f);
process.exit(fails.length ? 1 : 0);


### Assumptions and spec gaps — `docs/CONFORMANCE_ASSUMPTIONS.md`

In [22]:
%%writefile docs/CONFORMANCE_ASSUMPTIONS.md
# Conformance assumptions and spec gaps — v0.3.0-alpha

The vectors in `tests/` need exact answers wherever the specification leaves a choice open. This file records each choice (**A-xx**) and each place where the spec as written allows something it probably shouldn't (**G-xx**). All of them are provisional. Each one should be written into the normative text (or replaced), and the vectors regenerated after any change.

Vectors that pin behaviour from a G-item carry a `spec_gap` field. Their expected values follow the spec **as written**, so fixing the spec will flip them. That is intended.

---

## Assumptions

### A-01 — Canonical encoding `PRC-LP-0`
§1 requires "length-prefixed canonical byte encodings" but names no format. §5 mentions Protobuf/FlatBuffers, but neither has a canonical form by default. Until a schema registry exists, the vectors use this tag-length-value encoding:

```
value := tag:u8 || len:u64 big-endian || payload[len]
0x00 null   len=0          0x10 int    len=8, int64 two's complement BE
0x01 false  len=0          0x11 f64    len=8, IEEE 754 bits BE (no normalisation)
0x02 true   len=0          0x20 string UTF-8
                           0x21 bytes  raw
0x30 list   concatenated encoded items
0x31 map    concatenated (string key, value) pairs; keys strictly ascending by UTF-8 bytes
```
- The decoder is strict. It rejects unsorted or duplicate keys, trailing bytes, wrong fixed lengths, invalid UTF-8, non-string keys and unknown tags (`golden_vectors.json → must_reject`).
- Strings are not Unicode-normalised: `"é"` (NFC) and `"é"` (NFD) encode differently. Callers must normalise before encoding if they need them to compare equal.
- Hashes, key ids and UUIDs inside payloads are lowercase hex / canonical text strings, not raw bytes.

### A-02 — One encoding layer
The spec defines `H(z) = SHA256(CanonicalEncode(z))`, but also defines `d`, `p` and `r` as `CanonicalEncode(…)`, and writes the null digests as `H(CanonicalEncode(label))`. Read literally, that encodes twice. The vectors treat `d`, `p` and `r` as the structured objects and encode exactly once:
`NULL_APPROVAL_DIGEST = SHA256(PRC-LP-0("PROVENANCE-RAZOR-NO-APPROVAL-V0.3.0-Alpha"))`.

### A-03 — Reason codes
- `reason_codes` (in `d`) and `reason_code` (in `r`) are a single unsigned 64-bit bitfield, encoded as an int.
- The bit assignments in `tests/reason_codes.json` are provisional.
- Verifiers report every failing conjunct, not just the first (A-18).

### A-04 — Exceptional threshold
§2.3 only says NaN/Inf in *C* forces `C = 0.0`. The vectors treat NaN/Inf in *T* the same way (lock, deny).

### A-05 — Safe State Lock denies
If a Safe State Lock fires, `AuthorizationCheck` returns FALSE even when the forced `C = 0.0 ≥ T` (for example `T = 0.0`). The lock must never produce an ALLOW.

### A-06 — No CORDIC vectors
The spec text defines no CORDIC operation, so none are generated. `KAHAN_SUM` is an **informative** harness operation for comparing runtimes. Only `AUTH_CHECK` records are normative. The file keeps the name given in §5.

### A-07 — `ScopePermits(scope, x, target)`
Case-sensitive glob (`*`, `?`) matched against `"<x.action>:<p.target>"`, e.g. `wire.release:acct-*`.

### A-08 — `A(F̄_k)` with several optional members
Each member's state gives a factor: `VALID` → 1.0, `DEGRADED` → κ, `CONTRADICTORY` → 0.0, `UNAVAILABLE_OR_STALE` → ignored. `A` is the **minimum** factor, or 1.0 if nothing contributes.

### A-09 — `CanonicalRepresentative`
The selection rule isn't defined, so each fixture names its representative explicitly as input.

### A-10 — Eligibility
A class is in `F_eligible` unless one of its members is `INVALID`. That invalidates the whole class and emits a `FAMILY_INVALIDATED` audit event.

### A-11 — Undefined aggregates (in the core lineage fixtures)
*The experimental scoring profile `jaccard-v0.1.0-alpha` supplies candidate definitions; see its own document. The core fixtures stay profile-independent.*
- `I(F̄_k | F̄_j)` isn't defined, so fixtures supply it as an input matrix. `I_k` is the minimum over the other eligible classes, or 1.0 for a singleton.
- `M(x,a)` and `C(x,a)` are only defined for an empty eligible set (both 0.0). Otherwise the fixtures mark them `UNDEFINED_BY_SPEC`.

### A-12 — Local CAS abort still produces a receipt
§4.1 writes the receipt *inside* the transaction, so an abort would also discard the receipt. But §4.3 defines `REJECTED`/`PRECONDITION` receipts. The vectors therefore:
- roll back the transaction, leaving the capability unconsumed and the state unchanged;
- write a `REJECTED` receipt in a separate transaction.

### A-13 — Distributed capabilities are never released
Once reserved at Prepare, a capability counts as consumed whatever the outcome (`REJECTED`, `ROLLED_BACK`, `UNRESOLVED`). This stops replays under the same Idempotency-Key. Retrying needs a new decision.

### A-14 — Timeout that reconciles to COMMITTED
`failure_phase = NONE`, and `reason_code` keeps `DISPATCH_TIMEOUT` so the history is recorded. A timeout that reconciles to not-applied is `ROLLED_BACK` / `DISPATCH`.

### A-15 — Time, identifiers, keys
- All timestamps are int64 microseconds since the Unix epoch, UTC. The `*_iso` fields in the vectors are for reading only.
- UUIDv7 follows RFC 9562. The 74 random bits are `SHA256("PRC-UUID7|" + label)`, so identifiers are deterministic.
- Test keys are Ed25519 with seed `SHA256("PRC-TEST-KEY|<domain>|<label>")`. Key id is `kid:<domain>:<first 16 hex of SHA256(pubkey)>`. **The seeds are public. These keys are for testing only.**

### A-16 — Signatures
`σ = Ed25519.Sign(sk, H(z))` over the raw 32-byte digest. Ed25519 is deterministic, so signatures are stable golden values.

### A-17 — Profile-driven parameters
The profile supplies:
- `ManualOnly` = `profile.manual_only_actions`;
- `Δτ_max(x, a, profile)` = `profile.max_eval_skew_us[x.action]`, falling back to `["*"]`;
- "reusable as classified" = `authority_class ∈ profile.reusable_authority_classes`.

### A-18 — No short-circuit
`DecisionBindingValid` evaluates every conjunct and sets every failing bit. The one exception is `CanonicalDecodeValid`: if decoding fails, only `DECODE_INVALID` is reported.

### A-19 — Key lookup and domain separation
- `Verify_{K_decision^pub}` uses the key named by `decision_key_id` in the trust registry.
- A valid signature from a key in another domain passes the signature conjunct, but `DecisionKeyAuthorized` rejects it (`auth_expiry_009`).
- For approvals, the named key must be an `approval`-domain key for the signature to count (`auth_expiry_031`).

---

## Spec gaps: status after 0.2.0-alpha

The normative text is `spec/PROVENANCE_RAZOR_SPEC.md` (0.3.0-alpha). Eleven of the twelve 0.1.0 gaps are now closed in the spec and pinned by vectors; G-12 is partly addressed.

| ID | Gap (0.1.0) | Status in 0.2.0 | Evidence |
|---|---|---|---|
| G-01 | approval target not bound to `d.target` / `CurrentTarget` | **closed** (§3.1) | `auth_expiry_036` now must-reject |
| G-02 | post-dated approvals accepted | **closed**: `p.τ_issued ≤ d.τ_eval ≤ τ_exec ≤ p.τ_expiry` | `auth_expiry_037` must-reject, `041` boundary |
| G-03 | approval not bound to actor / req_id / capability_id | **closed** (§3.1) | `auth_expiry_038`–`040` must-reject |
| G-04 | double encoding | **closed**: single layer, detached signatures (§1.2–1.3) | golden vectors |
| G-05 | M, C, T, I undefined | **closed by delegation** to a scoring profile (§2.1); reference `coverage-v0.2.1-alpha` | scoring vectors |
| G-06 | receipt inside an aborting transaction | **closed**: abort writes REJECTED receipt separately (§4.1) | cas `02`, `03` |
| G-07 | NaN/Inf in T; lock vs `0 ≥ T` | **closed**: E-1, lock always denies (§2.3) | numeric, `e0x` |
| G-08 | `reason_codes` shape | **closed**: uint64 bitfield (§3.3) | `tests/reason_codes.json` |
| G-09 | capability release after failure | **closed**: never released (§4.2) | cas `06`, `08`, `09` |
| G-10 | evaluator/executor clock disagreement; source of `τ_exec` | **closed in 0.3.0**: C-1 clock authorities with combined uncertainty U (§3.5) | `auth_clock_k01`–`k07` |
| G-11 | "SPECIFICATION FROZEN"; schemas claimed but absent | **closed**: status table rewritten; transport schemas added and marked provisional | `schema/` |
| G-12 | non-VALID representative | **partly**: a non-VALID representative gives no positive support (S-05); INVALID/TAMPERED invalidates the family. A normative sentence is still needed | scoring vectors |

### Open questions

| ID | Question | Status in 0.3.0 |
|---|---|---|
| Q-1 | **Naming authority.** Who maintains the atom registry and alias table, and how is alias equivalence proven? `x07` vs `x08` show this decides whether correlation is visible. | open; the registry is now committed by `trust_state_hash`, so it is at least bound and auditable |
| Q-2 | **Unverified contradictions** could suppress a family's support. | **closed**: A-1 (§2.5). An unattested contradiction contains (MANUAL_REQUIRED) and never changes M or C (`a02`, `a12`) |
| Q-3 | **Family membership** was trusted upstream. | **addressed**: M-1 (§2.6) signed family manifests; injection vectors `m01`–`m06` are neutral. What remains is Q-6 |
| Q-4 | **Attester collusion.** Independence is only `attester ≠ producer`. | open |
| Q-5 | **Calibration.** Ω, T_base, T_base_manual, β and Γ are policy choices without empirical validation. | open |
| Q-6 | **Membership authorities are now a trust root.** The manifest issuer decides what is in a family. Two consequences follow from the current rules. First, authenticated adverse evidence arriving outside a manifest is quarantined (`m04`), so the ingestion path must be able to add contradictions to families. Second, withholding a listed member excludes the whole family (`m13`). That is fail-safe for support, but it does not by itself raise containment. Who may issue manifests, how quickly contradictions can be appended, and whether withholding should also CONTAIN are unspecified. | new |


### Scoring profile definitions, calibration and findings — `docs/SCORING_PROFILE_jaccard-v0.1.0-alpha.md`

In [23]:
%%writefile docs/SCORING_PROFILE_jaccard-v0.1.0-alpha.md
# PRC Scoring Profile `jaccard-v0.1.0-alpha`

> **Superseded for mass computation by `coverage-v0.2.0-alpha`** (docs/SCORING_PROFILE_coverage-v0.2.0-alpha.md), which fixes F-2 and F-3. This profile and its vectors are kept frozen as the evidence that motivated the change.

```yaml
scoring-profile:
  id: jaccard-v0.1.0-alpha
  status: experimental
  normative-role: reference candidate (replaceable)
  canonical-representative: structural-root-v1
  independence-function: material-dependency-jaccard-v1
  mass-function: quality-weighted-independent-mass-v1
  confidence-function: exponential-saturation-v1
  threshold-function: consequence-exponential-v1
  exp-function: prc-exp-v1
  summation-order: representative-evidence-id-utf8-ascending
```

The control architecture should require *a* deterministic scoring profile, identified by id and by fingerprint `H(profile)`. This is the first candidate. Its job is to produce evidence about the model. It is not settled, and nothing here belongs in the frozen core yet.

- Reference: `reference/prc_scoring.py` (pure functions; no signing, CAS or receipts)
- Vectors: `tests/vectors/scoring_jaccard_v0.1.0-alpha.json` (26 scoring cases + 32 `prc-exp-v1` vectors)
- Independent JavaScript implementation: the "scoring" block in `tools/xcheck.mjs`

## Success criterion — met

> With identical apparent confirmations and identical authorization context, changing only the evidence-lineage graph produces a reproducible ALLOW → DENY transition at a consequence-calibrated boundary.

| Case | Lineage | I per family | M | C | T (wire, Γ = 8.5) | Decision |
|---|---|---|---|---|---|---|
| `07_three_independent_pass` | disjoint material dependencies | 1, 1, 1 | 2.77 | 0.93734 | 0.92000 | **ALLOW** |
| `08_three_correlated_fail` | all three share one upstream feed | ⅓, ⅓, ⅓ | 0.92333 | 0.60281 | 0.92000 | **DENY** |
| `08b_two_of_three_correlated` | vendors A and B share the feed | ⅓, ⅓, 1 | 1.53667 | 0.78490 | 0.92000 | DENY |
| `08c_two_independent_insufficient` | two disjoint confirmations | 1, 1 | 1.87 | 0.84588 | 0.92000 | DENY |

The generator and both checkers verify that cases 07 and 08 are **byte-identical once dependency atoms are removed**: same evidence ids, scores, attestations, request and profile. They produce the same bits in Python and in the independent JavaScript implementation.

## Definitions

All arithmetic is binary64, uses only `+ − × ÷` (correctly rounded) and exact power-of-two scaling, and has no FMA contraction.

**Representative (structural-root-v1).** A family must have exactly one *root*: a member whose `parent_id` is null or points outside the family. Every other member must reach the root through in-family parents without a cycle. `F*_k` is that root. Digest order is never used for selection. Zero roots, two or more roots, a cycle, a duplicate id or an unknown atom type make the class **MALFORMED** (S-02).

**Dependency atoms.** Typed strings `TYPE:value`, where `TYPE` is one of `SOURCE, DATABASE, COLLECTOR, RUNTIME, CLOCK, NETWORK_AS, TRUST_ROOT, API_ENDPOINT`. Atoms form a set, so duplicates and order are irrelevant. The compiled profile nominates which types are **material**; only those count. Here the material types are `SOURCE, DATABASE, COLLECTOR, RUNTIME, API_ENDPOINT`. `D_k` is the representative's set of material atoms.

**Independence (material-dependency-jaccard-v1).**

$$I(\bar F_k\mid\bar F_j)=\begin{cases}\kappa_{\text{unknown}} & D_k=\varnothing \lor D_j=\varnothing\\[2pt] 1-\dfrac{|D_k\cap D_j|}{|D_k\cup D_j|} & \text{otherwise}\end{cases}\qquad I_k=\begin{cases}1 & m=1\\ \min_{j\ne k} I(\bar F_k\mid\bar F_j) & m>1\end{cases}$$

The quotient is an int÷int correctly rounded division. The empty-union case is covered by the first branch.

**Class value.** `V(F̄_k) = V(F*_k) · A(F̄_k)`, using the §2.2 state rules (A-08). `V(F*)` must be finite and in [0, 1]; a finite value outside that range makes the class MALFORMED (S-08).

**Mass (quality-weighted-independent-mass-v1).** `M = Σ V(F̄_k) · I_k`, summed left to right with families in ascending UTF-8 order of representative `evidence_id` (S-06). Fixing the order is what makes the result permutation-invariant bit for bit.

**Confidence (exponential-saturation-v1).** `C = 1 − prc_exp(−(M / Ω))`. If M is NaN or ±Inf, C is NaN, which triggers the §2.3 Safe State Lock. Without this guard, `+Inf` would give `exp(−Inf) = 0` and therefore `C = 1` (case `11b`).

**Threshold (consequence-exponential-v1).** `T = 1 − (1 − T_base) · prc_exp(−(Γ(x) · β))`, with `T_base ∈ [0.5, 1)` and `Γ ∈ [0.1, 10]`. An action with no consequence class gets `T = NaN`, which faults closed (`CONSEQUENCE_CLASS_UNDEFINED`).

**Check.** This is the existing §2.3 `AuthorizationCheck(C, T)`.

**prc-exp-v1.**
1. `k = floor(x·INV_LN2 + 0.5)`
2. `r = (x − k·LN2_HI) − k·LN2_LO` (Cody–Waite reduction; `LN2_HI` has 21 trailing zero bits, so `k·LN2_HI` is exact)
3. Horner evaluation of the degree-13 Taylor polynomial with coefficients `1/n!`
4. Multiply by an exact `2^k` built from its bit pattern, in two steps outside [−1022, 1023]
5. Special cases: `x > 709 → +Inf`, `x < −745 → 0`, `NaN → NaN`

It is within 1 ulp of glibc `exp` across 197k samples. It is not correctly rounded, but it gives the same bits on every conforming runtime. All constants are published as bit patterns in the vector file.

### Calibrated parameters

| Parameter | Value |
|---|---|
| Ω | 1.0 |
| T_base | 0.5 |
| β | 0.2156 |
| κ_unknown | 0.0 |
| κ_degraded | 0.5 |
| Γ(ledger.annotate) | 0.1 |
| Γ(payment.refund) | 0.5 |
| Γ(wire.release) | 8.5 |

Resulting thresholds: T(annotate) = 0.5107, T(refund) = 0.5511, T(wire) = 0.9200.

At Γ = 8.5, confidence from *n* independent families each scoring 0.9:

| n | C | Wire decision |
|---|---|---|
| 1 | 0.593 | DENY |
| 2 | 0.835 | DENY |
| 3 | 0.933 | ALLOW |
| 4–6 | ≥ 0.973 | ALLOW |

So this profile's policy is that a $9.9M wire needs three independent confirmations. That was chosen deliberately (S-09). Your worked example's Ω = 2 would have required about six independent confirmations; with Ω = 2 even three fully independent ones reach only 0.750.

C is a calibrated *score*, not a probability. Nothing yet shows that these numbers correspond to real risk. That calibration is the open research question.

## Findings — where the model needs work

**F-1: Platform `exp` breaks bit-exact authorization.**

- V8's `Math.exp` and glibc's `exp` disagree by 1 ulp on about 10% of inputs in [−40, 0] (19,543 of 200,000 sampled).
- Evaluated with platform `exp` on the same inputs, case `10_boundary_one_ulp_below` is **ALLOW in Python/glibc and DENY in Node/V8**.

§2.3 demands bit-exact comparison without tolerance, so any profile that uses transcendental functions must specify the algorithm. This is what `prc-exp-v1` does. The finding belongs in the normative core as a rule: *scoring profiles must use only basic IEEE operations or profile-specified algorithms.*

**F-2: Mass is non-monotone in evidence** (case `17`).

Adding a fourth confirmation that mirrors vendor A's lineage zeroes both A's and the mirror's I_k under the min rule. M drops from 2.77 to 1.82, and the decision flips from ALLOW to DENY. A redundant confirmation should add roughly nothing; it should not subtract. The min-pairwise `I_k` comes from §2.1 itself. Candidate fixes:
- cluster families by lineage and give each cluster one unit of mass; or
- discount sequentially in canonical order, so only later duplicates are reduced.

**F-3: Unknown lineage poisons the whole set** (case `06`).

With κ_unknown = 0, a single family with no declared material dependencies makes `I_k = 0` for *every* family. That is a denial-of-service lever: anyone who can inject one unattributed evidence item can block an action. It follows from F-2's min rule combined with a conservative κ. Excluding unknown-lineage families from the pairwise reduction, and capping their own contribution, would avoid it.

**F-4: The choice of material types decides outcomes** (case `18`).

Declaring `NETWORK_AS` material makes three vendors behind one CDN look like shared provenance. I drops to 6/7, C to 0.9069, and the decision flips to DENY. Which types are material must be a justified, reviewed profile choice.

**F-5: A single family always gets I = 1** (§2.1 singleton identity), even when its lineage is unknown. It is harmless at high Γ, since one family can't pass there, but worth stating.

## Assumptions (S-xx)

- **S-01** Families (equivalence classes) are supplied as input. How raw evidence is partitioned into classes is out of scope for this profile.
- **S-02** A malformed class is excluded and audited (`LINEAGE_CLASS_MALFORMED`) rather than denying the whole evaluation. Exclusion can only lower C, so it is safe, and it stops one malformed item from blocking everything. Your security policy may prefer global deny; that is a profile switch.
- **S-03** D_k is the *representative's* material atoms, following the formula. Derivatives' atoms are ignored for independence.
- **S-04** κ_unknown applies when either set is empty. There is no division by an empty union.
- **S-05** A representative whose state is not VALID gives no positive support. `INVALID` invalidates the family (G-12).
- **S-06** The canonical summation order is representative `evidence_id`, ascending by UTF-8 bytes. Min is order-independent anyway.
- **S-07** No platform transcendental functions (F-1).
- **S-08** Quality scores are in [0, 1]. NaN and ±Inf fault closed; other out-of-range values make the class malformed.
- **S-09** The calibration above is a policy choice made to demonstrate the lineage flip, not a validated risk model.

## On the comparison with OPA and Cedar

OPA and Cedar are general-purpose policy engines. An application can feed either one a lineage graph, or a precomputed score, and enforce the same decision. So the distinction isn't that they *can't* do this. What the Provenance Razor adds is a **normative, deterministic preprocessing and execution-control model**. It specifies how to derive action-specific confidence from evidence lineage, compares it with a consequence-calibrated threshold, and binds the result (via `evidence_hash`, `lineage_hash` and the profile fingerprint) into a consumable execution capability.

The value lies in the specification being exact, reproducible across runtimes and auditable. The findings above show those properties aren't automatic, which is exactly why they need specifying.

## Suggested status block for the spec

```
[CONTROL ARCHITECTURE STABLE]    subject to G-01..G-03 (approval target/time/actor binding) being closed
[SCORING PROFILE EXPERIMENTAL]   jaccard-v0.1.0-alpha — F-2 and F-3 open
```


### Coverage profile, properties and proposed rules E-1 / N-1 / P-1 — `docs/SCORING_PROFILE_coverage-v0.2.0-alpha.md`

In [24]:
%%writefile docs/SCORING_PROFILE_coverage-v0.2.0-alpha.md
# PRC Scoring Profile `coverage-v0.2.0-alpha`

```yaml
scoring-profile:
  id: coverage-v0.2.0-alpha
  status: experimental
  normative-role: reference candidate (replaceable); supersedes jaccard-v0.1.0-alpha's mass rule
  canonical-representative: structural-root-v1
  mass-function: material-dependency-coverage-v1
  unknown-lineage: single-sink-consequence-gated-v1
  confidence-function: exponential-saturation-v1
  threshold-function: consequence-exponential-v1
  exp-function: prc-exp-v1
  summation-order: dependency-atom-utf8-ascending
  dependency-source: self-declared-UNVERIFIED     # see P-1 and known exploit x01
```

- Reference: `reference/prc_coverage.py`
- Engine guard: `reference/prc_engine.py`
- Vectors: `tests/vectors/scoring_coverage_v0.2.0-alpha.json` (24 cases, 24 property witnesses)
- Randomised property tests: `tools/property_tests.py`
- Independent JavaScript implementation: `tools/xcheck.mjs`

The representative, state machine, confidence, threshold and `prc-exp-v1` are unchanged from `jaccard-v0.1.0-alpha`; see that document. The calibration (Ω = 1, T_base = 0.5, β = 0.2156, κ_degraded = 0.5, same Γ table) is also carried over.

## Mass

For each eligible family k with value `V_k = V(F*_k) · A(F̄_k)` and material atom set `D_k`:

$$w_k=\frac{V_k}{|D_k|},\qquad \operatorname{best}(d)=\max_{k:\,d\in D_k} w_k,\qquad M=\sum_{d\,\in\,\bigcup_k D_k}\operatorname{best}(d)$$

- The sum runs in ascending UTF-8 order of `d`, left to right.
- Ties keep the first family in canonical order as the atom's recorded owner. This affects only the trace, not M.

**Unknown lineage (single-sink-consequence-gated-v1).** A family with no material atoms is given `D_k = {UNKNOWN:*}` and `V_k · κ_eff`, where:
- `κ_eff = 0` if `T(x,a) ≥ unknown_gate_t` (0.92 in this profile);
- otherwise `κ_eff = κ_unknown` (0.5).

All unattributed families share the one sink atom. Together they contribute at most `max V_k · κ_eff`, however many there are.

## Properties

M is a weighted coverage function. Adding a family either leaves each `best(d)` unchanged, raises it, or adds a new atom with a non-negative credit. Round-to-nearest addition is monotone (`a ≤ a′ ⇒ fl(a+b) ≤ fl(a′+b)`), so by induction over the fixed-order sum the properties hold **bit for bit**, not just in exact arithmetic:

| Property | Statement | coverage-v0.2 violations | jaccard-v0.1 violations |
|---|---|---|---|
| P1 monotone | adding an eligible family never lowers M | 0 / 1500 | 287 / 1500 |
| P2 idempotent | a structural duplicate leaves M bit-identical | 0 / 1500 | 720 / 1500 |
| P3 order-free | permuting families, members or atom lists changes no output bit | 0 / 1500 | 0 / 1500 |
| P4 bridge-safe | a family drawn from several others' atoms never lowers M | 0 / 1500 | 463 / 1500 |
| P5 decision | none of the above turns ALLOW into DENY | 0 | 245 |

The counts come from `python tools/property_tests.py` (seed 20260925). The checker reruns 400 trials, and 24 deterministic witnesses are recomputed in both Python and JavaScript.

A family that is malformed, invalidated or has a non-VALID representative is excluded and cannot change anyone else's credit. `κ_eff` depends only on the action's threshold, never on the evidence, so adding evidence cannot switch the gate.

## Results

| Case | M | C | Decision | Notes |
|---|---|---|---|---|
| c03 three independent (wire) | 2.77 | 0.93734 | **ALLOW** | |
| c04 same three, shared feed | 1.314 | 0.73126 | **DENY** | the lineage flip is preserved |
| c05 two of three shared | 2.05 | 0.87127 | DENY | |
| c06 two independent | 1.87 | 0.84588 | DENY | |
| c07 + mirror of vendor A | 2.77 | 0.93734 | ALLOW | M bit-identical to c03 (F-2 fixed) |
| c08 + bridging item | 2.77 | 0.93734 | ALLOW | M bit-identical to c03 |
| c09 + unattributed item (wire, κ_eff = 0) | 1.314 | 0.73126 | DENY | M bit-identical to c04 (F-3 fixed) |
| c10a refund, one known family | 0.6 | 0.45119 | DENY | |
| c10b + one unattributed (κ_eff = 0.5) | 1.05 | 0.65006 | ALLOW | low-consequence policy lets unattributed evidence help, once |
| c11 + twenty unattributed | 1.05 | 0.65006 | ALLOW | M bit-identical to c10b: the single sink holds |
| c12 exact duplicate | 0.6 | 0.45119 | DENY | M bit-identical to c10a |
| c17 / c18 boundary | | C = T / T − 1 ulp | ALLOW / DENY | |
| e01–e04 NaN/Inf anywhere | 0 | 0 | DENY | stopped at `ENGINE_GUARD`; profile code never runs |
| **x01 atom padding** | 3.168 | 0.95790 | **ALLOW** | **known exploit, open (expected failure)** |

## Proposed normative rules

**E-1: Engine input guard.** Before dispatching to any scoring profile, the engine MUST check that every binary64 input is finite. That covers every `*_bits` profile parameter, every consequence Γ, and every evidence member's `value_bits`, including optional members. If any is NaN or ±Inf, the engine MUST return DENY with `SAFE_STATE_LOCK | NONFINITE_INPUT` without invoking the profile.

§2.3's rule that NaN/Inf forces C to 0.0 remains as a second line of defence for intermediate values. The guard is kept apart from profile validation deliberately: vectors e03 and e04 show that without it, a NaN parameter surfaces as a misleading "omega must be > 0" error instead of a clean fail-closed decision.

**N-1: Arithmetic.** Profiles MUST use binary64 with every intermediate rounded separately. Fused multiply-add, `-ffast-math`, reassociation and x87 extended precision are forbidden (`-ffp-contract=off`, strict FP modes). Transcendental functions MUST be profile-specified algorithms over basic operations (`prc-exp-v1`), never the platform library. JavaScript cannot express FMA portably, and FMA rounds `a·b+c` once instead of twice, so allowing it would split runtimes.

**P-1: Verified provenance.** This is the proposed rule that would close x01.
- Dependency atoms MUST NOT be taken from unauthenticated evidence bodies.
- An atom is admissible only if it is either:
  (a) carried in a provenance attestation signed by a key in a dedicated `K_provenance` domain (a fifth key domain under §1), bound to the evidence's canonical digest, and issued by an attester the profile authorises for that atom type; or
  (b) computed by the verifier itself from topology it observed.
- Anything else maps to `UNKNOWN:*`.
- Profiles SHOULD also bound atoms per type per family (for example one `SOURCE`). A signed attestation makes padding attributable, but not impossible.

Under P-1, x01's padded items would collapse to the unknown sink and contribute 0 on a wire. The vector records that as its security expectation: DENY, with M equal to c04. The checker treats the case as an expected failure, and fails the build if the exploit ever stops reproducing without the case being promoted.

## Still open

- **x01 / P-1:** needs the provenance attestation schema, the attester authorisation model and test vectors. It is the prerequisite for any lineage-based profile being sound.
- **Contradiction.** Coverage is monotone even for families with `CONTRADICTORY` members: contradiction zeroes only that family's own value, exactly as §2.2 states. Evidence that *refutes* the assertion therefore can't lower confidence built on other evidence. If refutation should act as a veto, it needs a separate channel (a defeater set checked before mass), not a negative mass term. That channel would break P1 by design.
- **Atom granularity** is still a modelling choice. Splitting one real dependency into several atoms changes shares (`V/|D|`). Verified provenance fixes who declares atoms, not how finely.
- **Calibration** is carried over and is still a policy choice, not a validated risk model.


### coverage-v0.2.1-alpha: P-1, D-1 and exploit-family results — `docs/SCORING_PROFILE_coverage-v0.2.1-alpha.md`

In [25]:
%%writefile docs/SCORING_PROFILE_coverage-v0.2.1-alpha.md
# PRC Scoring Profile `coverage-v0.2.1-alpha`

```yaml
scoring-profile:
  id: coverage-v0.2.1-alpha
  status: experimental
  normative-role: reference candidate; supersedes coverage-v0.2.0-alpha
  canonical-representative: structural-root-v1
  mass-function: material-dependency-coverage-v1      # unchanged from v0.2.0
  unknown-lineage: single-sink-consequence-gated-v1   # unchanged
  confidence-function: exponential-saturation-v1
  threshold-function: consequence-exponential-v1
  exp-function: prc-exp-v1
  summation-order: dependency-atom-utf8-ascending
  dependency-source: attested-P-1                     # was self-declared-UNVERIFIED
  defeater-policy: authenticated-contradiction-veto-v1 # new
```

- Reference: `reference/prc_provenance.py`, `reference/prc_coverage_v021.py`
- Vectors: `tests/vectors/scoring_coverage_v0.2.1-alpha.json` (32 cases, 18 witnesses)
- Normative text: `spec/PROVENANCE_RAZOR_SPEC.md` §2.4 (P-1) and §2.5 (D-1)

The coverage mass rule and the calibration are unchanged from v0.2.0. Only two things change: where the atoms come from, and how contradiction acts.

## Profile additions

| Field | Value in the calibrated profile | Purpose |
|---|---|---|
| `atom_registry.mode` | `closed` | only registered atoms are admissible (the naming authority) |
| `atom_registry.atoms` | sorted, unique list | canonical atom identities |
| `atom_registry.aliases` | e.g. `SOURCE:fcp-1 → SOURCE:feed-core-primary` | different names for one dependency resolve to one identity |
| `max_atoms_per_type` | API_ENDPOINT 1, CLOCK 1, COLLECTOR 2, DATABASE 2, NETWORK_AS 1, RUNTIME 2, SOURCE 2, TRUST_ROOT 1 | bounds padding even by an honest-but-careless attester |
| `attester_independence_required` | `true` | the attester must not be the evidence producer |

The trust state (key registry and revocations) is an input to evaluation, not part of the profile. Each provenance key is registered with its owner's `attester_id`, its authorised epochs, and the atom types it may vouch for.

## Results

| Case | Provenance outcome | Decision | Notes |
|---|---|---|---|
| c03 three independent (attested) | ADMITTED | **ALLOW** (C = 0.93734) | the lineage flip is preserved over attested lineage |
| c04 same three, shared feed (attested) | ADMITTED | **DENY** (C = 0.73126) | |
| c07 / c08 / c09 mirror, bridge, re-attested duplicate | ADMITTED | ALLOW | M bit-identical to c03 |
| c10b / c11 unattested at refund | UNATTESTED | ALLOW | contributes V·κ once; twenty contribute the same as one |
| **x01 self-declared padding** (the v0.2.0 exploit) | UNATTESTED | **DENY** | M bit-identical to c04; was ALLOW (C = 0.958) in v0.2.0 |
| x02 auditor-signed fabricated atoms | INADMISSIBLE: unregistered atom | DENY | M = c04 |
| x03 registered atoms over the cap | INADMISSIBLE: 3 COLLECTOR > 2 | DENY | M = c04 |
| x04 vendor attests its own evidence | INADMISSIBLE: self-attested | DENY | |
| x05 valid attestation copied onto other evidence | **TAMPERED**: digest mismatch | DENY | family invalidated, audited |
| x06 attestation signed with a decision key | **TAMPERED**: key domain | DENY | |
| x07 shared feed named by aliases | ADMITTED (resolved) | DENY | M = c05: correlation detected |
| x08 same input, registry without aliases | ADMITTED | **ALLOW** | pins the naming-authority risk (Q-1) |
| x09 / x10 / x11 / x12 expired, revoked, unknown key, type not authorised | INADMISSIBLE | DENY | |
| x13 attester_id forged | **TAMPERED** | DENY | |
| x14 score edited after attestation | **TAMPERED**: digest mismatch | DENY | the digest covers `value_bits` |
| d00 four independent | ADMITTED | ALLOW (C = 0.97452) | baseline |
| d01 unattested contradiction | UNATTESTED | ALLOW | zeroes only its family; audited as UNVERIFIED_CONTRADICTION |
| **d02 auditor-attested contradiction** | ADMITTED defeater | **DENY** | M and C bit-identical to d01: a categorical veto |
| d03 contradiction with a forged signature | TAMPERED | ALLOW | family invalidated; the other three still pass |

## Properties

`tools/property_tests.py` runs 500 trials over randomly generated attested evidence. The results:

| | Property | Violations |
|---|---|---|
| P1 | monotone positive mass | 0 |
| P2 | re-attested duplicate idempotent | 0 |
| P3 | order-free | 0 |
| P4 | bridge-safe | 0 |
| P5 | none of P1–P4 turns ALLOW into DENY | 0 |
| P6 | an admitted defeater always yields DENY | 0 |
| P7 | an unattested family is neutral on a wire | 0 |
| P8 | a lifted or forged attestation is neutral | 0 |

P5 now excludes defeaters by design: a veto is the one addition that may turn ALLOW into DENY.

## Open questions

These are summarised in `docs/CONFORMANCE_ASSUMPTIONS.md` as Q-1 to Q-5:

- Q-1 who governs the atom registry;
- Q-2 whether unattested contradictions should have any effect;
- Q-3 admissibility of family membership;
- Q-4 attester collusion;
- Q-5 calibration.


### coverage-v0.3.0-alpha: containment, membership, properties — `docs/SCORING_PROFILE_coverage-v0.3.0-alpha.md`

In [26]:
%%writefile docs/SCORING_PROFILE_coverage-v0.3.0-alpha.md
# PRC Scoring Profile `coverage-v0.3.0-alpha`

```yaml
scoring-profile:
  id: coverage-v0.3.0-alpha
  status: experimental
  normative-role: reference candidate; supersedes coverage-v0.2.1-alpha
  canonical-representative: structural-root-v1
  mass-function: material-dependency-coverage-v1       # unchanged since v0.2.0
  unknown-lineage: single-sink-consequence-gated-v1    # unchanged
  confidence-function: exponential-saturation-v1
  threshold-function: consequence-exponential-v1       # also used for T_manual (t_base_manual)
  exp-function: prc-exp-v1
  summation-order: dependency-atom-utf8-ascending
  dependency-source: attested-P-1
  membership: signed-family-manifest-M-1               # new
  adverse-disposition: contain-defeat-lock-A-1         # new; replaces D-1
```

- Reference: `reference/prc_coverage_v030.py`, `reference/prc_membership.py`, `reference/prc_schema.py`
- Vectors: `tests/vectors/scoring_coverage_v0.3.0-alpha.json` (39 cases, 18 witnesses)
- Normative text: spec §2.5 (A-1), §2.6 (M-1), §2.7 (trust context)
- Trust context: `tools/trust_fixture.py`

The mass function and the calibration are unchanged. In this version the profile holds only arithmetic. The atom registry, aliases, caps, independence rule, conflict types and clock authorities moved to the committed trust state.

## What changed and what it closes

| Case | Result | What it shows |
|---|---|---|
| a00 four independent | ALLOW (C = 0.97452) | baseline |
| a01 + auditor-attested contradiction | **DENY** (DEFEAT) | M and C bit-identical to a00 |
| a02 + unattested contradiction | **MANUAL_REQUIRED** (CONTAIN) | M and C bit-identical to a00; automatic ALLOW prohibited |
| a03 + self-attested contradiction | MANUAL_REQUIRED | INADMISSIBLE contains exactly like UNATTESTED |
| a04 + listed contradiction with a forged attestation | **DENY** (LOCK) | Safe State Lock |
| a05 CONTAIN with C below T_manual | DENY | containment never lowers the bar |
| a06 / a07 same claim in two families / twenty times | one obligation | review obligations deduplicate |
| a08 / a09 wrong target / unknown conflict type | ALLOW | irrelevant contradictions are audited only |
| a10 root is itself an unattested contradiction | MANUAL_REQUIRED | no support from that family; containment |
| a11 DEFEAT and CONTAIN together | DENY (DEFEAT) | precedence |
| a12 three independent + unattested contradiction | MANUAL_REQUIRED, M = 2.77 | coverage-v0.2.1 zeroed this family (M = 1.85, DENY) |
| m01–m06 injected INVALID member, second root, unattested / attested / forged contradiction, same-id impostor | ALLOW, M bit-identical | injection into another source's family is neutral (quarantined) |
| m07–m15 no manifest, out of scope, unregistered key, wrong key domain, replayed, altered, withheld, expired, revoked | family excluded or invalidated | membership admissibility |
| x01 / x05 / x07 padding, lifted attestation, aliases | DENY | P-1 still holds under M-1 |
| b01 / b02 boundary | ALLOW / DENY | C = T and one ulp below |
| f01 / f02 malformed family or member | excluded, audited | V-1: structured, never an exception |

## Properties

`tools/property_tests.py` runs 200 trials for the v0.3.0 properties and 600 mutations for P11. The results:

| | Property | Violations |
|---|---|---|
| P1 | monotone positive mass | 0 |
| P2 | re-attested, re-manifested duplicate idempotent | 0 |
| P3 | order-free | 0 |
| P4 | bridge-safe | 0 |
| P5 | none of P1–P4 turns ALLOW into DENY | 0 |
| P6 | a listed, admitted, relevant contradiction always DENYs | 0 |
| P9 | an unlisted member of any adversarial kind changes nothing (M, C, disposition, decision) | 0 |
| P10 | a listed unattested relevant contradiction keeps M and C bit-identical, never yields ALLOW, and turns ALLOW into MANUAL_REQUIRED | 0 |
| P11 | totality under 600 random mutations: exceptions escaping the decision verifier; byte-flipped decisions accepted; scoring faults | 0; 0; 0 |

## Open questions

These are listed in `docs/CONFORMANCE_ASSUMPTIONS.md`:

- Q-1 naming authority;
- Q-4 attester collusion;
- Q-5 calibration;
- Q-6 membership authorities as a trust root: authenticated adverse evidence outside a manifest is quarantined (m04), and withholding excludes a family but does not raise containment (m13).


### Normative specification 0.3.0-alpha (same text as the first cell) — `spec/PROVENANCE_RAZOR_SPEC.md`

In [27]:
%%writefile spec/PROVENANCE_RAZOR_SPEC.md
---
title: Provenance Razor Control Specification
version: 0.3.0-alpha
supersedes: 0.2.0-alpha
status: alpha
confidence-classification: provisional
artifact-class: research-specification
license: MIT
owner: Kevin R. Barrow

evidence-basis:
  - mathematical-formalization
  - executable-semantics
  - conformance-vectors
  - cross-runtime-verification
  - randomized-property-and-totality-testing

maturity:
  control-architecture: stable-draft
  canonical-encoding: provisional        # PRC-LP-0
  transport-schemas: provisional         # Protobuf and FlatBuffers, both compiled and round-trip tested
  scoring-profile: experimental          # coverage-v0.3.0-alpha
  reference-runtime: python
  conformance-vectors: generated-provisional
  cross-runtime-verification: python-and-javascript
  dag-decomposition: pending
  independent-verification: pending

version-policy:
  major: incompatible normative change
  minor: normative extension
  patch: clarification or correction
  pre-1.0: incompatible normative changes increment the minor version (SemVer 2.0.0 item 4)
---

## Provenance Razor Control Specification
#### v0.3.0 Alpha

The Provenance Razor Control Specification is an evolving research and engineering framework for deterministic, lineage-aware execution control. Everything in this document is provisional. Version status reflects the quantity and quality of supporting evidence, and nothing here is frozen.

Consistent with the Provenance Razor itself, the specification treats its own assertions as hypotheses. Its objective is not certainty. It is a deterministic framework for judging whether an assertion has enough evidence, verified provenance, verified membership, observation and reconciliation to justify confidence proportional to the consequences of error. Each result carries only the authority its provenance warrants.

**Changes** are summarised in §6. Every normative change is backed by conformance vectors in `tests/`.

---

### 1. Key Domains, Canonical Encoding, Signatures and Validity

#### 1.1 Key domains

$$\text{Domain}(K) \in \{ K_{\text{profile}}, K_{\text{approval}}, K_{\text{decision}}, K_{\text{receipt}}, K_{\text{provenance}}, K_{\text{membership}} \}$$

1. $K_{\text{profile}}$ signs control profiles, scoring profiles and trust-state objects.
2. $K_{\text{approval}}$ signs manual approvals ($p$).
3. $K_{\text{decision}}$ signs decision capabilities ($d$).
4. $K_{\text{receipt}}$ signs receipts ($r$).
5. $K_{\text{provenance}}$ signs provenance attestations ($\pi$, §2.4).
6. $K_{\text{membership}}$ signs family manifests ($\mu$, §2.6). New in 0.3.0.

**Domain-scoped resolution.** $\text{Verify}_{K_{\text{dom}}}$ means three separate steps:
1. resolve the key identifier *only* among keys registered in domain `dom`;
2. verify the signature;
3. check epoch and policy authorisation.

A key registered in another domain is treated as not found. The same rule applies to every object type.

#### 1.2 Canonical encoding and hashing

$$\mathcal{H}(z) \equiv \text{SHA256}\left( \text{PRC-LP-0}(z) \right)$$

- $z$ is the structured canonical object, encoded exactly once.
- PRC-LP-0 is the canonical, length-prefixed tag-length-value encoding (A-01). Decoders MUST reject every non-canonical byte string.
- Integers are int64, and every integer this specification defines is non-negative. Reason-code masks therefore use bits 0–62; **bit 63 is reserved.**
- Timestamps are int64 microseconds since the Unix epoch, UTC.

#### 1.3 Detached signatures

$$\sigma_z = \text{Sign}_{K_{\text{domain}(z)}}\left( \mathcal{H}(z) \right)$$

A signature is never a field of the object it signs.

#### 1.4 Transport encodings

Protobuf and FlatBuffers (`schema/`) are transport and storage formats only. Their bytes MUST NOT be hashed, signed or compared. A verifier MUST:

1. parse the transport message;
2. reject unknown fields;
3. validate presence, lengths and enumerations;
4. convert to the canonical object;
5. apply §1.5;
6. compute $\mathcal{H}$ over PRC-LP-0;
7. verify the detached signature.

#### 1.5 Canonical object validity and total verification

**CanonicalObjectValid(z, schema)** runs after decoding and before any signature or semantic predicate. It requires:

- exactly the schema's fields, with none missing and none extra;
- every field of its declared type, with a boolean never accepted as an integer;
- non-negative integers within int64;
- a supported `schema_ver`;
- 32-byte lowercase-hex digests;
- canonical lowercase UUIDv7 identifiers;
- key identifiers of the form `kid:<domain>:<16 hex>`;
- enumeration values from the defined set.

Schemas are defined for $d$, $p$, $r$, $\pi$ and $\mu$. Evidence members have an open schema: `evidence_id`, `parent_id`, `producer`, `value_bits` (16 hex digits) and optional `state` are required to be well formed, and claim fields may be added.

**V-1 (total verification).** For every input $z$:

$$\text{Verify}(z) \in \{\text{ALLOW}, \text{MANUAL\_REQUIRED}, \text{DENY}\}$$

Verify MUST NOT raise or terminate abnormally. A malformed object yields a structured rejection: `DECODE_INVALID`, `OBJECT_SCHEMA_INVALID` or `APPROVAL_SCHEMA_INVALID`. Structurally invalid families or members are excluded and audited. Any residual internal fault is converted at the verifier boundary into DENY with `SAFE_STATE_LOCK | VERIFIER_FAULT`.

---

### 2. Evidence, Provenance, Membership and Deterministic Math

#### 2.1 Equivalence classes and the scoring profile

Evidence is partitioned into lineage families $\bar{F}_1, \dots, \bar{F}_m$. The members of a family are exactly those admitted by §2.6.

- $F_k^*$ is the family's unique structural root among its admitted members.
- A family with zero or several roots, a cycle, or a duplicate identifier is malformed. It is excluded and audited.
- The family value is:

$$V(\bar{F}_k) = V(F_k^*) \cdot A(\bar{F}_k), \qquad V(F_k^*) \in [0,1]$$

$M(x,a)$, $C(x,a)$, $T(x,a)$ and $T_{\text{manual}}(x,a)$ are defined by the active **scoring profile**. The scoring profile is bound by $d.\text{scoring\_profile\_hash}$ (§2.7). A scoring profile MUST:
- be deterministic;
- obey §2.3;
- take dependency atoms only from §2.4 and family members only from §2.6.

Positive mass SHOULD be monotone. The reference candidate is `coverage-v0.3.0-alpha`.

$$F_{\text{eligible}}(x,a) = \emptyset \implies M(x,a) = 0.0 \implies C(x,a) = 0.0$$

#### 2.2 Optional member states

$$\text{State}(F_i) \in \{\text{VALID}, \text{UNAVAILABLE\_OR\_STALE}, \text{DEGRADED}, \text{CONTRADICTORY}, \text{INVALID}\}$$

* **UNAVAILABLE_OR_STALE:** no positive support. $A$ is unaffected.
* **DEGRADED:** $A(\bar{F}_k) = \kappa_{\text{degraded}}$.
* **CONTRADICTORY:** no positive support, and **no effect on $A$.** Its effect on authorization is governed only by §2.5.
* **INVALID:** the family is invalidated and a security audit event is emitted.

$A(\bar{F}_k)$ is the minimum factor over contributing members, or 1.0 if none contribute. A TAMPERED provenance attestation (§2.4) makes its member INVALID.

#### 2.3 Numerical comparison

$$\text{AuthorizationCheck}(C, T) \iff (C(x,a) \ge T(x,a)) = \text{TRUE}$$

- **E-1:** every binary64 input MUST be finite before any profile logic runs. Otherwise the result is DENY with Safe State Lock.
- **N-1:** binary64 with every intermediate rounded separately; no fused multiply-add, reassociation or extended precision. Transcendental functions MUST be profile-specified algorithms, such as `prc-exp-v1`.
- No epsilon tolerances. `-0.0` normalises to `+0.0`. Subnormals are preserved. A NaN or ±Inf forces $C = 0.0$ and **State 2 (Safe State Lock)**, which always denies.

#### 2.4 Verified provenance (P-1)

Self-declared dependencies MUST NOT be used. An atom `TYPE:id` is admissible only if it arrives in a provenance attestation that satisfies CanonicalObjectValid and:

$$\text{DependencyAdmissible}(\pi, e) \iff \begin{cases} \text{Verify}_{K_{\text{provenance}}}(\mathcal{H}(\pi), \sigma_\pi) \land \pi.\text{domain} = \text{"PROVENANCE-RAZOR-PROVENANCE-V0.3.0-Alpha"} \\ \land\; \pi.\text{attester\_id} = \text{Owner}(\pi.\text{provenance\_key\_id}) \land \pi.\text{evidence\_digest} = \mathcal{H}(\text{EvidenceCore}(e)) \\ \land\; \text{EpochAuthorized} \land \neg\text{Revoked}(\pi) \land \pi.\tau_{\text{issued}} \le \tau_{\text{eval}} \le \pi.\tau_{\text{expiry}} \land \pi.\text{attester\_id} \ne e.\text{producer} \\ \land\; \forall t: \text{AttesterAuthorized}(\text{type}(t)) \land \text{Registered}(\text{Resolve}(t)) \land \forall \theta: |\text{atoms of type } \theta| \le \text{Cap}(\theta) \end{cases}$$

- The atom registry, the alias table, the caps and the independence rule belong to the **trust state** (§2.7).
- Outcomes:
  - **ADMITTED:** the resolved atoms are used.
  - **UNATTESTED** or **INADMISSIBLE:** the lineage is unknown.
  - **TAMPERED:** the member is INVALID.

#### 2.5 Adverse disposition (A-1)

A contradicting member is **relevant** only if it meets both conditions:
- its `conflict_type` is recognised by the trust state;
- its `contradicts` field equals the request's $\langle$action, assertion, target, evidence_snapshot_id$\rangle$.

Irrelevant contradictions are audited and have no effect. For each relevant contradicting member $e$ admitted by §2.6:

$$\text{ContradictionDisposition}(e)= \begin{cases} \text{DEFEAT}, & \text{Provenance}(e)=\text{ADMITTED}\\ \text{CONTAIN}, & \text{Provenance}(e)\in\{\text{UNATTESTED},\text{INADMISSIBLE}\}\\ \text{LOCK}, & \text{Provenance}(e)=\text{TAMPERED} \end{cases}$$

$$\text{AdverseDisposition}(x,a) = \max_{\prec}\{\text{ContradictionDisposition}(e)\} \cup \{\text{CLEAR}\}, \qquad \text{CLEAR} \prec \text{CONTAIN} \prec \text{DEFEAT} \prec \text{LOCK}$$

A numeric Safe State Lock also yields LOCK. Contradictions never change $A$, $V$, $M$, $C$ or $T$. They affect only which authorization modes are eligible:

$$\text{AutomatedAllow}(x,a) \iff C \ge T \land \text{AdverseDisposition} = \text{CLEAR}$$

$$\text{ManualAllow}(x,a) \iff C \ge T_{\text{manual}} \land \text{AdverseDisposition} \in \{\text{CLEAR}, \text{CONTAIN}\} \land \text{ExternalApprovalValid}(p, d)$$

| Disposition | Outcome |
|---|---|
| LOCK | DENY, with Safe State Lock |
| DEFEAT | DENY. An ordinary manual approval cannot override it; any future override needs a separately named, higher-authority exception object. |
| CONTAIN | MANUAL_REQUIRED if $C \ge T_{\text{manual}}$, otherwise DENY |
| CLEAR | ALLOW if $C \ge T$, otherwise DENY |

**Containment obligations.**
- Relevant adverse claims collapse under the canonical contradiction digest $\mathcal{H}(\langle\text{conflict\_type}, \text{contradicts}, \text{claim}\rangle)$. Repeated submissions of the same claim create one obligation.
- Each occurrence is audited with its ingress channel.
- `containment_digest` is $\mathcal{H}$ of the sorted CONTAIN digests, or NULL_CONTAINMENT_DIGEST when the disposition is CLEAR.

$$\text{NULL\_CONTAINMENT\_DIGEST} = \mathcal{H}\left(\text{"PROVENANCE-RAZOR-NO-CONTAINMENT-V0.3.0-Alpha"}\right)$$

#### 2.6 Family membership (M-1)

Membership is asserted by a **family manifest** $\mu$ signed in $K_{\text{membership}}$ by a membership authority, typically the ingestion gateway. It contains:
- domain, schema_ver, manifest_id, membership_key_id, membership_trust_epoch;
- authority_id and family_id;
- ingress_channel;
- members, each as $\langle$evidence_id, $\mathcal{H}(\text{EvidenceCore}(e))$, parent_id$\rangle$, sorted and unique;
- $\tau_{\text{issued}}$ and $\tau_{\text{expiry}}$.

Evaluating a family's manifest gives one of three outcomes:

- **TAMPERED** — the family is invalidated and a security event emitted. Causes:
  - a malformed envelope or payload, or a wrong domain string;
  - a key from another domain, or a bad signature;
  - an authority_id that is not the key's owner, or a family_id that doesn't match (a replayed manifest);
  - a non-canonical member list;
  - a listed member altered after the manifest was issued.
- **UNVERIFIED** — the family is excluded from support, and its contradictions have no effect. Causes:
  - no manifest, or a manifest signed by a key that isn't registered;
  - an unauthorised epoch, a revoked manifest, or a manifest that isn't fresh;
  - the authority's `membership_scopes` don't cover the family_id;
  - a listed member withheld from the submission.
- **VERIFIED** — members are classified individually:
  - **LISTED:** the digest and parent match the manifest.
  - **QUARANTINED:** the member is not listed, or it reuses a listed evidence_id without matching the manifest. It is excluded entirely and audited only: it can neither invalidate, attenuate, contain nor defeat the family it names.

Structurally invalid members (§1.5) are quarantined before membership is evaluated.

#### 2.7 Trust context and binding

Each rule is committed by exactly one object:

| Object | Commits | Bound as |
|---|---|---|
| Control profile | ManualOnly, $\Delta\tau_{\max}$, reusable authority classes | `control_profile_hash` |
| Scoring profile | Ω, $T_{\text{base}}$, $T_{\text{base,manual}}$, β, κ values, gate, Γ, material types, component ids | `scoring_profile_hash` |
| Trust state | key registry for all six domains (owners, epochs, authority classes, authorised atom types, membership scopes), revocations of attestations and manifests, atom registry and aliases, per-type caps, attester independence, conflict types, clock authorities | `trust_state_hash` |

The verifier MUST confirm all three:
- $d.\text{trust\_state\_hash} = \mathcal{H}(\text{TrustState}_{\text{active}})$;
- $d.\text{scoring\_profile\_hash} = \mathcal{H}(\text{ScoringProfile}_{\text{active}})$;
- $d.\text{control\_profile\_hash} = \mathcal{H}(\text{ControlProfile}_{\text{active}})$.

Two evaluators that use different registries, alias authorities, revocation sets or attester authorisations therefore produce different decision bindings.

---

### 3. Canonical Objects and Bidirectional Binding

#### 3.1 External manual approval ($p$)

$p$ carries:
- domain `"PROVENANCE-RAZOR-APPROVAL-V0.3.0-Alpha"`, schema_ver, approval_id, approval_key_id, approval_trust_epoch;
- authority_class and authority_scope;
- $x$, $a$, target, actor, req_id, capability_id, state_ver;
- control_profile_hash, scoring_profile_hash, trust_state_hash;
- adverse_disposition, containment_digest;
- $\tau_{\text{issued}}$, $\tau_{\text{expiry}}$, and decision = ALLOW.

$$\text{ExternalApprovalValid}(x,a,d,p,\sigma_p) \iff \begin{cases} \text{CanonicalObjectValid}(p, \text{approval}) \land \text{Verify}_{K_{\text{approval}}}(\mathcal{H}(p), \sigma_p) \land \text{domain ok} \\ \land\; \text{ApprovalKeyAuthorized} \land \text{ScopePermits}(p.\text{authority\_scope}, x, p.\text{target}) \\ \land\; p.x = x \land p.a = a \land p.\text{state\_ver} = \text{CurrentStateVersion} \land p.\text{control\_profile\_hash} = \text{Active} \\ \land\; p.\text{scoring\_profile\_hash} = d.\text{scoring\_profile\_hash} \land p.\text{trust\_state\_hash} = d.\text{trust\_state\_hash} \\ \land\; p.\text{target} = d.\text{target} = \text{CurrentTarget} \land p.\text{actor} = d.\text{actor} \land p.\text{req\_id} = d.\text{req\_id} \land p.\text{capability\_id} = d.\text{capability\_id} \\ \land\; p.\text{adverse\_disposition} = d.\text{adverse\_disposition} \land p.\text{containment\_digest} = d.\text{containment\_digest} \\ \land\; p.\tau_{\text{issued}} \le d.\tau_{\text{eval}} \land \tau_{\text{exec}} + U \le p.\tau_{\text{expiry}} \land \text{ApprovalUnconsumedOrReusableAsClassified}(p) \end{cases}$$

The approver acknowledges exactly the adverse claims the decision carries. An approval of a clean request cannot be reused once containment appears, and an approval of one set of claims cannot cover another.

#### 3.2 Null digests

$$\text{NULL\_APPROVAL\_DIGEST} = \mathcal{H}\left( \text{"PROVENANCE-RAZOR-NO-APPROVAL-V0.3.0-Alpha"} \right)$$

#### 3.3 Decision capability ($d$)

$d$ carries:
- domain `"PROVENANCE-RAZOR-DECISION-V0.3.0-Alpha"`, schema_ver, profile_version, lineage_schema_version;
- capability_id, req_id, evidence_snapshot_id, actor, auth_mode ∈ {AUTOMATED, MANUAL};
- $x$, $a$, target, state_ver;
- **control_profile_hash, scoring_profile_hash, trust_state_hash**, evidence_hash, lineage_hash, approval_hash;
- **adverse_disposition ∈ {CLEAR, CONTAIN}, containment_digest**;
- decision_key_id, decision_trust_epoch, **eval_clock_id**;
- $\tau_{\text{eval}}$, $\tau_{\text{expiry}}$, decision = ALLOW, and reason_codes (a non-negative bitfield).

A decision is never issued for DEFEAT or LOCK.

#### 3.4 Bidirectional authorization mode validation

$$\text{AuthorizationModeValid} \iff \begin{cases} d.\text{auth\_mode} = \text{AUTOMATED} \land x \notin \text{ManualOnly} \land d.\text{adverse\_disposition} = \text{CLEAR} \land d.\text{approval\_hash} = \text{NULL\_APPROVAL\_DIGEST} \\ \lor\; d.\text{auth\_mode} = \text{MANUAL} \land (x \in \text{ManualOnly} \lor d.\text{adverse\_disposition} = \text{CONTAIN}) \land d.\text{approval\_hash} = \mathcal{H}(p) \land \text{ExternalApprovalValid} \end{cases}$$

Consistency: $d.\text{adverse\_disposition} = \text{CLEAR} \iff d.\text{containment\_digest} = \text{NULL\_CONTAINMENT\_DIGEST}$.

#### 3.5 Decision verification predicate and clock authority (C-1)

The evaluator's clock is $d.\text{eval\_clock\_id}$. The executor's clock is $\langle\text{clock\_id}, \tau_{\text{exec}}, u_{\text{exec}}\rangle$. Both MUST be registered clock authorities in the trust state, and $u_{\text{exec}}$ MUST NOT exceed the executor authority's maximum; otherwise the result is `CLOCK_UNTRUSTED`. Every time comparison uses the combined worst-case uncertainty:

$$U = u_{\text{eval}}^{\max} + u_{\text{exec}}$$

$$\text{DecisionBindingValid} \iff \begin{cases} \text{CanonicalDecodeValid}(d) \land \text{CanonicalObjectValid}(d, \text{decision}) \land \text{Verify}_{K_{\text{decision}}}(\mathcal{H}(d), \sigma_d) \land \text{domain ok} \\ \land\; d.x = x \land d.a = a \land d.\text{target} = \text{CurrentTarget} \land d.\text{state\_ver} = \text{CurrentStateVersion} \\ \land\; d.\text{control\_profile\_hash}, d.\text{scoring\_profile\_hash}, d.\text{trust\_state\_hash} \text{ match the active objects (§2.7)} \\ \land\; d.\text{evidence\_hash} = \mathcal{H}(F_{\text{eligible}}) \land d.\text{lineage\_hash} = \mathcal{H}(D_{x,a}) \\ \land\; d.\text{adverse\_disposition}, d.\text{containment\_digest} \text{ match the current evaluation} \land d.\text{decision} = \text{ALLOW} \\ \land\; \text{ClocksTrusted} \land d.\tau_{\text{eval}} + U \le \tau_{\text{exec}} \land \tau_{\text{exec}} + U \le d.\tau_{\text{expiry}} \land (\tau_{\text{exec}} - d.\tau_{\text{eval}}) + U \le \Delta\tau_{\max} \\ \land\; \text{DecisionKeyAuthorized} \land \text{CapabilityAvailable} \land \text{AuthorizationModeValid} \end{cases}$$

Verifiers report every failing reason bit. On a decoding or schema failure, only that single bit is reported.

---

### 4. Transaction Boundaries, State Preconditions and Signed Receipts

#### 4.1 Local targets (compare-and-swap)

The transaction proceeds in this order:

1. BEGIN, then lock the target.
2. Verify CapabilityAvailable.
3. Verify CurrentStateVersion = $d$.state_ver.
4. MarkCapabilityConsumed.
5. ApplyMutation.
6. AdvanceStateVersion.
7. WriteReceipt.
8. COMMIT.

If a verification fails, the transaction rolls back and a REJECTED/PRECONDITION receipt is written in a separate transaction.

#### 4.2 Distributed targets

This section is unchanged from 0.2.0. The target enforces the state precondition. Outcomes are COMMITTED, REJECTED (PRECONDITION, TARGET or PREPARE), ROLLED_BACK (DISPATCH) or UNRESOLVED (RECONCILIATION, with reconciliation_required). A reserved capability is never released.

#### 4.3 Execution receipt ($r$)

$r$ carries:
- domain `"PROVENANCE-RAZOR-RECEIPT-V0.3.0-Alpha"`, schema_ver, receipt_id;
- capability_id, req_id, decision_digest = $\mathcal{H}(d)$, $x$, target, **control_profile_hash**;
- expected_state_ver = $d$.state_ver, observed_pre_state_ver, observed_post_state_ver (null if unknown);
- execution_status, failure_phase, reason_code, reconciliation_required;
- pre_state_digest, post_state_digest ($\mathcal{H}(\text{State}_{\text{post}})$ if known, else NULL_STATE_DIGEST);
- $\tau_{\text{dispatch}}$, $\tau_{\text{finalized}}$, receipt_key_id, receipt_trust_epoch.

The scoring profile and trust state are committed through `decision_digest`.

$$\text{NULL\_STATE\_DIGEST} = \mathcal{H}\left( \text{"PROVENANCE-RAZOR-NO-STATE-V0.3.0-Alpha"} \right)$$

---

### 5. Conformance Status

| Deliverable | Status | Location / harness |
|---|---|---|
| Normative control specification | **ALPHA**; control architecture **STABLE DRAFT** | this document |
| Canonical encoding PRC-LP-0 | **PROVISIONAL** | A-01 |
| Transport schemas | **PROVISIONAL**: Protobuf and FlatBuffers compiled, round-trip tested, strict | `schema/`, `tools/transport_roundtrip.py` |
| Scoring profile `coverage-v0.3.0-alpha` | **EXPERIMENTAL** | `tests/vectors/scoring_coverage_v0.3.0-alpha.json` |
| Superseded profiles | `jaccard-v0.1.0-alpha`, `coverage-v0.2.0-alpha` (exploit reproduces by design), `coverage-v0.2.1-alpha` | `tests/vectors/` |
| Golden, numerical, lineage, authorization, transaction vectors | **GENERATED, PROVISIONAL** | `tests/` |
| Randomised property and totality tests | P1–P6, P9–P11 hold | `tools/property_tests.py` |
| Cross-runtime conformance | **PYTHON + JAVASCRIPT**; Go / Rust / C++ pending | `tools/check_vectors.py`, `tools/xcheck.mjs` |
| Independent external verification | **PENDING** | — |

**Exploit statement.** For `coverage-v0.3.0-alpha` there is no known open exploit **under the declared assumptions**:
- the trust state (key owners, authorisations, revocations);
- the atom registry and alias authority;
- the family-membership authorities;
- the clock authorities;
- the calibration.

Each of these is an assumption, not a result. The open questions are listed in `docs/CONFORMANCE_ASSUMPTIONS.md` (Q-1 to Q-6).

---

### 6. Changes

#### 6.1 From 0.2.0-alpha to 0.3.0-alpha

| Change | Section | Closes |
|---|---|---|
| CanonicalObjectValid for $d$, $p$, $r$, $\pi$, $\mu$ and evidence members; V-1 total verification | 1.5 | validly signed malformed decisions were accepted or raised exceptions |
| Domain-scoped key resolution for every object type | 1.1 | asymmetric cross-domain diagnostics |
| Reason masks non-negative; bit 63 reserved | 1.2, 3.3 | signed/unsigned mismatch |
| Contradictions no longer change $A$; A-1 disposition CLEAR / CONTAIN / DEFEAT / LOCK; MANUAL_REQUIRED outcome | 2.2, 2.5 | Q-2 (unverified contradictions suppressing support) |
| Containment obligations deduplicated; approval must acknowledge `containment_digest` | 2.5, 3.1 | review spam; stale approvals |
| $K_{\text{membership}}$ and signed family manifests; quarantine of unlisted members | 1.1, 2.6 | Q-3 (injection into another source's family) |
| Trust state as a committed object; `control_profile_hash`, `scoring_profile_hash`, `trust_state_hash` on $d$ and $p$ | 2.7, 3 | scoring profile and registry were not bound |
| C-1 clock authorities and combined uncertainty $U$ | 3.5 | G-10 |
| Receipt `profile_hash` renamed `control_profile_hash` | 4.3 | naming ambiguity |
| FlatBuffers compiled and round-trip tested | 1.4 | transport completeness |
| Exploit statement qualified by its assumptions | 5 | overclaim |
| Domain strings move to `V0.3.0-Alpha` | all | cross-version replay |

#### 6.2 From 0.1.0-alpha to 0.2.0-alpha (summary)

- `K_provenance` and P-1 verified provenance.
- D-1 authenticated defeaters.
- Approvals bound to target, actor, req_id and capability_id (G-01 to G-03).
- Expected vs observed receipt state.
- Single encoding layer and detached signatures (G-04).
- E-1 and N-1.
- Scoring delegated to profiles (G-05).
- Transport schemas.
- The "SPECIFICATION FROZEN" status removed.


### Protobuf transport schema (v1alpha3) — `schema/protobuf/provenance_razor_v1alpha3.proto`

In [28]:
%%writefile schema/protobuf/provenance_razor_v1alpha3.proto
// Provenance Razor Control Specification 0.3.0-alpha — TRANSPORT SCHEMA
//
// STATUS: provisional, transport and storage only.
//
// These messages define structure, types and field presence for moving objects between
// systems. They do NOT define the bytes that are hashed or signed. Protobuf serialization is
// not canonical: field order, merged/repeated fields, default-value omission and unknown
// fields all allow several valid encodings of one message.
//
// Canonical representation (normative): PRC-LP-0 over the canonical object
//     H(z) = SHA-256(PRC-LP-0(z))          sigma = Ed25519(sk, H(z))
//
// Verifier flow:
//     receive protobuf -> parse -> REJECT unknown fields -> validate presence/lengths/enums
//     -> convert to the canonical object -> PRC-LP-0 -> SHA-256 -> verify detached signature
// Never hash or verify protobuf bytes. Never re-serialize protobuf and compare.
//
// Signatures are detached: every signed object travels as Signed*{payload, signature}; the
// signature is never a field of the payload it signs.
//
// Changes in v1alpha3 (spec 0.3.0-alpha):
//   * profile_hash renamed control_profile_hash (same field numbers: wire-compatible rename)
//   * scoring_profile_hash, trust_state_hash, adverse_disposition, containment_digest on p and d;
//     eval_clock_id on d
//   * FamilyManifest (M-1) and SignedFamilyManifest
//   * reason masks are non-negative: bit 63 is reserved
//
// Changes from the v1alpha1 draft (in v1alpha2):
//   * signature fields moved out of payloads into Signed* wrappers
//   * x, a and actor carried as lossless canonical Values (not "canonical strings")
//   * approval bound to actor, req_id, capability_id (spec 0.3.0 §3.1; closes G-01..G-03)
//   * receipt separates expected from observed state versions (spec 0.3.0 §4.3, R-1)
//   * ProvenanceAttestation and the K_provenance domain (spec 0.3.0 §2.4, P-1)
//   * integers are int64 to match PRC-LP-0's int64; timestamps are int64 microseconds UTC
//   * new fields take new numbers; nothing is renumbered
syntax = "proto3";

package provenance.razor.v1alpha3;

option go_package = "provenance/razor/v1alpha3;razorv1alpha3";
option java_package = "com.provenance.razor.v1alpha3";

// ---------------------------------------------------------------------------
// Lossless canonical value (mirrors the PRC-LP-0 type system exactly)
// ---------------------------------------------------------------------------
message Value {
  oneof kind {
    bool null_value = 1;          // presence (with value true) means PRC-LP-0 null
    bool bool_value = 2;
    sint64 int_value = 3;         // PRC-LP-0 int (int64)
    fixed64 f64_bits = 4;         // PRC-LP-0 f64: raw IEEE 754 bit pattern, never a decimal
    string string_value = 5;      // valid UTF-8, not normalised
    bytes bytes_value = 6;
    ValueList list_value = 7;
    ValueMap map_value = 8;
  }
}
message ValueList { repeated Value items = 1; }
message ValueMap { repeated MapEntry entries = 1; }   // keys must be unique; order is irrelevant
message MapEntry { string key = 1; Value value = 2; }

enum AuthorizationMode {
  AUTHORIZATION_MODE_UNSPECIFIED = 0;   // always rejected
  AUTHORIZATION_MODE_AUTOMATED = 1;
  AUTHORIZATION_MODE_MANUAL = 2;
}
enum DecisionOutcome {
  DECISION_OUTCOME_UNSPECIFIED = 0;     // always rejected
  DECISION_OUTCOME_ALLOW = 1;
  DECISION_OUTCOME_DENY = 2;            // representable, never valid in d or p
}
enum AdverseDisposition {
  ADVERSE_DISPOSITION_UNSPECIFIED = 0;  // always rejected
  ADVERSE_DISPOSITION_CLEAR = 1;
  ADVERSE_DISPOSITION_CONTAIN = 2;      // decision must be MANUAL with an approval acknowledging containment_digest
}
enum ExecutionStatus {
  EXECUTION_STATUS_UNSPECIFIED = 0;
  EXECUTION_STATUS_COMMITTED = 1;
  EXECUTION_STATUS_ROLLED_BACK = 2;
  EXECUTION_STATUS_REJECTED = 3;
  EXECUTION_STATUS_INDETERMINATE = 4;
  EXECUTION_STATUS_UNRESOLVED = 5;
}
enum FailurePhase {
  FAILURE_PHASE_UNSPECIFIED = 0;        // always rejected (NONE is explicit)
  FAILURE_PHASE_NONE = 1;
  FAILURE_PHASE_PREPARE = 2;
  FAILURE_PHASE_PRECONDITION = 3;
  FAILURE_PHASE_DISPATCH = 4;
  FAILURE_PHASE_TARGET = 5;
  FAILURE_PHASE_RECONCILIATION = 6;
}

// ---------------------------------------------------------------------------
// §3.1 External manual approval (p). Domain "PROVENANCE-RAZOR-APPROVAL-V0.3.0-Alpha"
// ---------------------------------------------------------------------------
message ExternalApproval {
  string domain = 1;
  string schema_ver = 2;
  string approval_id = 3;             // UUIDv7
  string approval_key_id = 4;         // key in the K_approval domain
  string approval_trust_epoch = 5;
  string authority_class = 6;
  string authority_scope = 7;
  Value x = 8;                        // CanonicalAction(x)
  Value a = 9;                        // CanonicalAssertion(a)
  string target = 10;                 // must equal d.target and CurrentTarget
  int64 state_ver = 11;
  bytes control_profile_hash = 12;    // exactly 32 bytes, SHA-256 (renamed from profile_hash)
  int64 tau_issued = 13;              // µs UTC; must satisfy tau_issued <= d.tau_eval
  int64 tau_expiry = 14;              // µs UTC
  DecisionOutcome decision = 15;      // must be ALLOW
  Value actor = 16;                   // must equal d.actor            (new in 0.2.0)
  string req_id = 17;                 // must equal d.req_id           (new in 0.2.0)
  string capability_id = 18;          // must equal d.capability_id    (new in 0.2.0; pre-allocated at request time)
  bytes scoring_profile_hash = 19;    // must equal d.scoring_profile_hash (new in 0.3.0)
  bytes trust_state_hash = 20;        // must equal d.trust_state_hash     (new in 0.3.0)
  AdverseDisposition adverse_disposition = 21;  // acknowledged disposition (new in 0.3.0)
  bytes containment_digest = 22;      // acknowledged adverse claims; exactly 32 bytes (new in 0.3.0)
}

// ---------------------------------------------------------------------------
// §3.3 Decision capability (d). Domain "PROVENANCE-RAZOR-DECISION-V0.3.0-Alpha"
// ---------------------------------------------------------------------------
message DecisionCapability {
  string domain = 1;
  string schema_ver = 2;
  string profile_version = 3;
  string lineage_schema_version = 4;
  string capability_id = 5;           // UUIDv7
  string req_id = 6;                  // UUIDv7
  string evidence_snapshot_id = 7;    // UUIDv7
  Value actor = 8;                    // ActorIdentity
  AuthorizationMode auth_mode = 9;
  Value x = 10;
  Value a = 11;
  string target = 12;
  int64 state_ver = 13;
  bytes control_profile_hash = 14;    // exactly 32 bytes (renamed from profile_hash)
  bytes evidence_hash = 15;           // exactly 32 bytes, H(F_eligible)
  bytes lineage_hash = 16;            // exactly 32 bytes, H(D_{x,a})
  bytes approval_hash = 17;           // exactly 32 bytes, H(p) or NULL_APPROVAL_DIGEST
  string decision_key_id = 18;        // key in the K_decision domain
  string decision_trust_epoch = 19;
  int64 tau_eval = 20;                // µs UTC
  int64 tau_expiry = 21;              // µs UTC
  DecisionOutcome decision = 22;      // must be ALLOW
  int64 reason_codes = 23;            // reason-code bitfield; non-negative, bit 63 reserved
  bytes scoring_profile_hash = 24;    // exactly 32 bytes (new in 0.3.0)
  bytes trust_state_hash = 25;        // exactly 32 bytes (new in 0.3.0)
  AdverseDisposition adverse_disposition = 26;  // CLEAR | CONTAIN (new in 0.3.0)
  bytes containment_digest = 27;      // exactly 32 bytes; NULL_CONTAINMENT_DIGEST when CLEAR (new in 0.3.0)
  string eval_clock_id = 28;          // registered clock authority of the evaluator (new in 0.3.0, C-1)
}

// ---------------------------------------------------------------------------
// §4.3 Execution receipt (r). Domain "PROVENANCE-RAZOR-RECEIPT-V0.3.0-Alpha"
// ---------------------------------------------------------------------------
message ExecutionReceipt {
  string domain = 1;
  string schema_ver = 2;
  string receipt_id = 3;
  string capability_id = 4;
  string req_id = 5;
  bytes decision_digest = 6;          // exactly 32 bytes, H(d)
  Value x = 7;
  string target = 8;
  bytes control_profile_hash = 9;     // exactly 32 bytes (renamed from profile_hash)
  // 10, 11 were pre_state_ver / post_state_ver in v1alpha1 and could contradict the digests.
  reserved 10, 11;
  reserved "pre_state_ver", "post_state_ver";
  ExecutionStatus execution_status = 12;
  FailurePhase failure_phase = 13;
  int64 reason_code = 14;
  bool reconciliation_required = 15;
  bytes pre_state_digest = 16;        // exactly 32 bytes, H(State_pre as observed)
  bytes post_state_digest = 17;       // exactly 32 bytes, H(State_post) if known, else NULL_STATE_DIGEST
  int64 tau_dispatch = 18;
  int64 tau_finalized = 19;
  string receipt_key_id = 20;         // key in the K_receipt domain
  string receipt_trust_epoch = 21;
  int64 expected_state_ver = 22;               // d.state_ver                       (R-1)
  int64 observed_pre_state_ver = 23;           // State_pre.version                 (R-1)
  optional int64 observed_post_state_ver = 24; // absent = unknown (UNRESOLVED)     (R-1)
}

// ---------------------------------------------------------------------------
// §2.4 Provenance attestation (P-1). Domain "PROVENANCE-RAZOR-PROVENANCE-V0.3.0-Alpha"
// ---------------------------------------------------------------------------
message ProvenanceAtom {
  string dependency_type = 1;         // SOURCE | DATABASE | COLLECTOR | RUNTIME | CLOCK | NETWORK_AS | TRUST_ROOT | API_ENDPOINT
  string dependency_id = 2;           // canonical id; canonical form is "TYPE:id"
}
message ProvenanceAttestation {
  string domain = 1;
  string schema_ver = 2;
  string attestation_id = 3;          // UUIDv7
  string provenance_key_id = 4;       // key in the K_provenance domain
  string provenance_trust_epoch = 5;
  string attester_id = 6;             // must be the key's registered owner; must differ from the evidence producer
  bytes evidence_digest = 7;          // exactly 32 bytes, H(EvidenceCore(e))
  repeated ProvenanceAtom dependencies = 8;  // canonical form sorts by "TYPE:id" (UTF-8) and forbids duplicates
  int64 tau_issued = 9;
  int64 tau_expiry = 10;
}

// ---------------------------------------------------------------------------
// §2.6 Family manifest (M-1). Domain "PROVENANCE-RAZOR-MEMBERSHIP-V0.3.0-Alpha"
// ---------------------------------------------------------------------------
message ManifestMember {
  string evidence_id = 1;
  bytes evidence_digest = 2;          // exactly 32 bytes, H(EvidenceCore(e))
  optional string parent_id = 3;      // absent = root
}
message FamilyManifest {
  string domain = 1;
  string schema_ver = 2;
  string manifest_id = 3;             // UUIDv7
  string membership_key_id = 4;       // key in the K_membership domain
  string membership_trust_epoch = 5;
  string authority_id = 6;            // the key's registered owner
  string family_id = 7;
  string ingress_channel = 8;
  repeated ManifestMember members = 9;  // canonical form sorts by evidence_id (UTF-8), unique
  int64 tau_issued = 10;
  int64 tau_expiry = 11;
}

// ---------------------------------------------------------------------------
// Detached signatures: Ed25519 over SHA-256(PRC-LP-0(canonical(payload))), 64 bytes.
// ---------------------------------------------------------------------------
message SignedApproval { ExternalApproval payload = 1; bytes signature = 2; }
message SignedDecision { DecisionCapability payload = 1; bytes signature = 2; }
message SignedReceipt { ExecutionReceipt payload = 1; bytes signature = 2; }
message SignedProvenanceAttestation { ProvenanceAttestation payload = 1; bytes signature = 2; }
message SignedFamilyManifest { FamilyManifest payload = 1; bytes signature = 2; }


### FlatBuffers transport schema (v1alpha3) — `schema/flatbuffers/provenance_razor_v1alpha3.fbs`

In [29]:
%%writefile schema/flatbuffers/provenance_razor_v1alpha3.fbs
// Provenance Razor Control Specification 0.3.0-alpha — TRANSPORT SCHEMA (FlatBuffers)
//
// STATUS: provisional, transport only. Compiled with flatc 24.3.25 and round-trip tested by
// tools/transport_roundtrip.py (canonical -> FlatBuffers JSON -> binary -> JSON -> canonical).
//
// FlatBuffers byte layout depends on builder order, alignment and vtable sharing, so buffers are
// never hashed or signed. Verifiers convert to the canonical object and hash PRC-LP-0, exactly as
// described in schema/protobuf/provenance_razor_v1alpha3.proto. Field meanings, units and
// constraints are identical to the Protobuf schema; see the comments there.
namespace Provenance.Razor.V1Alpha3;

enum AuthorizationMode : byte { UNSPECIFIED = 0, AUTOMATED = 1, MANUAL = 2 }
enum DecisionOutcome : byte { UNSPECIFIED = 0, ALLOW = 1, DENY = 2 }
enum AdverseDisposition : byte { UNSPECIFIED = 0, CLEAR = 1, CONTAIN = 2 }
enum ExecutionStatus : byte { UNSPECIFIED = 0, COMMITTED = 1, ROLLED_BACK = 2, REJECTED = 3, INDETERMINATE = 4, UNRESOLVED = 5 }
enum FailurePhase : byte { UNSPECIFIED = 0, NONE = 1, PREPARE = 2, PRECONDITION = 3, DISPATCH = 4, TARGET = 5, RECONCILIATION = 6 }

// Lossless canonical value (PRC-LP-0 type system)
table NullValue {}
table BoolValue { value: bool; }
table IntValue { value: long; }
table F64Bits { bits: ulong; }
table StringValue { value: string (required); }
table BytesValue { value: [ubyte] (required); }
table ValueList { items: [Value] (required); }
table MapEntry { key: string (required); value: Value (required); }
table ValueMap { entries: [MapEntry] (required); }
union ValueKind { NullValue, BoolValue, IntValue, F64Bits, StringValue, BytesValue, ValueList, ValueMap }
table Value { kind: ValueKind (required); }

table ExternalApproval {
  domain: string (required);
  schema_ver: string (required);
  approval_id: string (required);
  approval_key_id: string (required);
  approval_trust_epoch: string (required);
  authority_class: string (required);
  authority_scope: string (required);
  x: Value (required);
  a: Value (required);
  target: string (required);
  state_ver: long;
  control_profile_hash: [ubyte] (required);  // exactly 32 bytes
  tau_issued: long;
  tau_expiry: long;
  decision: DecisionOutcome;
  actor: Value (required);
  req_id: string (required);
  capability_id: string (required);
  scoring_profile_hash: [ubyte] (required);
  trust_state_hash: [ubyte] (required);
  adverse_disposition: AdverseDisposition;
  containment_digest: [ubyte] (required);
}

table DecisionCapability {
  domain: string (required);
  schema_ver: string (required);
  profile_version: string (required);
  lineage_schema_version: string (required);
  capability_id: string (required);
  req_id: string (required);
  evidence_snapshot_id: string (required);
  actor: Value (required);
  auth_mode: AuthorizationMode;
  x: Value (required);
  a: Value (required);
  target: string (required);
  state_ver: long;
  control_profile_hash: [ubyte] (required);  // exactly 32 bytes
  evidence_hash: [ubyte] (required);     // exactly 32 bytes
  lineage_hash: [ubyte] (required);      // exactly 32 bytes
  approval_hash: [ubyte] (required);     // exactly 32 bytes
  decision_key_id: string (required);
  decision_trust_epoch: string (required);
  tau_eval: long;
  tau_expiry: long;
  decision: DecisionOutcome;
  reason_codes: long;                    // non-negative; bit 63 reserved
  scoring_profile_hash: [ubyte] (required);
  trust_state_hash: [ubyte] (required);
  adverse_disposition: AdverseDisposition;
  containment_digest: [ubyte] (required);
  eval_clock_id: string (required);
}

table OptionalLong { value: long; }      // absent table = unknown

table ExecutionReceipt {
  domain: string (required);
  schema_ver: string (required);
  receipt_id: string (required);
  capability_id: string (required);
  req_id: string (required);
  decision_digest: [ubyte] (required);   // exactly 32 bytes
  x: Value (required);
  target: string (required);
  control_profile_hash: [ubyte] (required);  // exactly 32 bytes
  execution_status: ExecutionStatus;
  failure_phase: FailurePhase;
  reason_code: long;
  reconciliation_required: bool;
  pre_state_digest: [ubyte] (required);  // exactly 32 bytes
  post_state_digest: [ubyte] (required); // exactly 32 bytes
  tau_dispatch: long;
  tau_finalized: long;
  receipt_key_id: string (required);
  receipt_trust_epoch: string (required);
  expected_state_ver: long;
  observed_pre_state_ver: long;
  observed_post_state_ver: OptionalLong; // absent = unknown (UNRESOLVED)
}

table ProvenanceAtom { dependency_type: string (required); dependency_id: string (required); }

table ProvenanceAttestation {
  domain: string (required);
  schema_ver: string (required);
  attestation_id: string (required);
  provenance_key_id: string (required);
  provenance_trust_epoch: string (required);
  attester_id: string (required);
  evidence_digest: [ubyte] (required);   // exactly 32 bytes
  dependencies: [ProvenanceAtom] (required);
  tau_issued: long;
  tau_expiry: long;
}

table ManifestMember { evidence_id: string (required); evidence_digest: [ubyte] (required); parent_id: string; }

table FamilyManifest {
  domain: string (required);
  schema_ver: string (required);
  manifest_id: string (required);
  membership_key_id: string (required);
  membership_trust_epoch: string (required);
  authority_id: string (required);
  family_id: string (required);
  ingress_channel: string (required);
  members: [ManifestMember] (required);
  tau_issued: long;
  tau_expiry: long;
}

// Detached signatures
table SignedApproval { payload: ExternalApproval (required); signature: [ubyte] (required); }
table SignedDecision { payload: DecisionCapability (required); signature: [ubyte] (required); }
table SignedReceipt { payload: ExecutionReceipt (required); signature: [ubyte] (required); }
table SignedProvenanceAttestation { payload: ProvenanceAttestation (required); signature: [ubyte] (required); }
table SignedFamilyManifest { payload: FamilyManifest (required); signature: [ubyte] (required); }

root_type SignedDecision;


### README — `README.md`

In [30]:
%%writefile README.md
# Provenance Razor — specification and conformance suite (v0.3.0-alpha)

```
spec/PROVENANCE_RAZOR_SPEC.md normative text, 0.3.0-alpha (changelog in §6)
schema/protobuf/*.proto       TRANSPORT ONLY (v1alpha3; compiled + round-trip tested); never hashed or signed
schema/flatbuffers/*.fbs      TRANSPORT ONLY (v1alpha3; compiled with flatc + round-trip tested)
reference/prc_ref.py          Python reference model of the spec (pure, deterministic)
reference/prc_scoring.py      EXPERIMENTAL scoring profile jaccard-v0.1.0-alpha (superseded mass rule; kept as evidence)
reference/prc_coverage.py     scoring profile coverage-v0.2.0-alpha (superseded; atom-padding exploit reproduces by design)
reference/prc_provenance.py   P-1 provenance attestation + DependencyAdmissible
reference/prc_coverage_v021.py  scoring profile coverage-v0.2.1-alpha (superseded)
reference/prc_schema.py       CanonicalObjectValid (total) for d, p, r, attestations, manifests, evidence members
reference/prc_membership.py   M-1 family manifests: FamilyMembershipAdmissible
reference/prc_coverage_v030.py  EXPERIMENTAL scoring profile coverage-v0.3.0-alpha (A-1 containment + M-1)
reference/prc_engine.py       engine: non-finite guard (E-1), total boundary (V-1), profile dispatch
tools/generate_vectors.py     writes every artifact under tests/ (refuses if model ≠ declared intent)
tools/check_vectors.py        recomputes all expected values, checks MANIFEST, checks byte-identical regeneration
tools/scoring_cases.py        jaccard profile cases (imported by the generator)
tools/coverage_cases.py       coverage profile cases, engine-guard cases, property witnesses, known exploit
tools/coverage_v021_cases.py  v0.2.1 cases, exploit family x01–x14, defeaters d00–d03, witnesses
tools/coverage_v030_cases.py  v0.3.0 cases: disposition a00–a12, membership m01–m15, carry-overs, boundaries
tools/trust_fixture.py        the committed trust state and scoring profile used by 0.3.0 vectors
tools/property_tests.py       randomised property tests (P1–P6, P9–P10) and P11 totality fuzz
tools/transport_roundtrip.py  canonical -> Protobuf / FlatBuffers -> canonical -> PRC-LP-0 -> verify, + demos
tools/xcheck.mjs              independent JavaScript re-implementation; checks the same vectors
docs/CONFORMANCE_ASSUMPTIONS.md   every interpretive choice (A-xx) and spec gap (G-xx)
docs/SCORING_PROFILE_jaccard-v0.1.0-alpha.md   scoring definitions, calibration, findings F-1..F-5
docs/SCORING_PROFILE_coverage-v0.2.0-alpha.md  coverage mass, properties, proposed rules E-1 / N-1 / P-1
docs/SCORING_PROFILE_coverage-v0.2.1-alpha.md  P-1 / D-1 profile, exploit-family results
docs/SCORING_PROFILE_coverage-v0.3.0-alpha.md  A-1 containment, M-1 membership, properties
tests/
  MANIFEST.json               SHA-256 of every artifact
  reason_codes.json           provisional reason-code bit registry
  fixtures/golden_vectors.json            encoding primitives, must-reject bytes, signed d/p/r/profile
  fixtures/lineage_graphs/*.yaml          §2.1–2.2 consolidation cases
  fixtures/cas_reconciliation/*.yaml      §4.1–4.3 local CAS + distributed reconciliation, signed receipts
  vectors/auth_expiry_skew.json           §1.5, §3 DecisionBindingValid (71 cases: freshness, keys, schema, trust, containment, clocks)
  vectors/kahan_cordic_ieee754.bin/.json  §2.3 binary64 comparison + informative Kahan sums
  vectors/scoring_jaccard_v0.1.0-alpha.json   EXPERIMENTAL scoring profile: 26 cases + 32 prc-exp vectors
  vectors/scoring_coverage_v0.2.0-alpha.json  SUPERSEDED: 24 cases + 24 witnesses; x01 reproduces by design
  vectors/scoring_coverage_v0.2.1-alpha.json  SUPERSEDED: 32 cases + 18 witnesses
  vectors/scoring_coverage_v0.3.0-alpha.json  EXPERIMENTAL: 39 cases + 18 witnesses
```

Requirements: Python 3.9+ with `cryptography` and `PyYAML`. Optional: Node 18+ for the cross-check; `grpcio-tools` + `protobuf` and a `flatc` binary (on PATH or in `$FLATC`) for the transport round trips (each skipped when absent).

```bash
python tools/generate_vectors.py   # only when the model or the cases change
python tools/check_vectors.py      # includes 400 randomised property trials
python tools/property_tests.py     # full run: 1500 trials per property, coverage vs jaccard
node tools/xcheck.mjs
python tools/transport_roundtrip.py
```

The test keys are derived from public seeds (A-15). Never use them for anything except conformance testing.


### Line-ending policy (MANIFEST hashes exact bytes) — `.gitattributes`

In [31]:
%%writefile .gitattributes
# Keep vector bytes stable across platforms (MANIFEST.json hashes them)
tests/** -text
*.bin binary
*.py text eol=lf
*.mjs text eol=lf
*.md text eol=lf


## 3. Generate the vectors

In [32]:
!{sys.executable} tools/generate_vectors.py
!find tests -type f | sort

wrote 29 artifact files + MANIFEST.json to /tmp/claude-0/-home-claude/01bcb688-4405-528b-84a6-bca6c2d7c727/scratchpad/run/provenance-razor/tests


tests/MANIFEST.json
tests/fixtures/cas_reconciliation/01_local_commit.yaml
tests/fixtures/cas_reconciliation/02_local_cas_mismatch.yaml
tests/fixtures/cas_reconciliation/03_local_capability_consumed.yaml
tests/fixtures/cas_reconciliation/04_dist_ack.yaml
tests/fixtures/cas_reconciliation/05_dist_timeout_applied.yaml
tests/fixtures/cas_reconciliation/06_dist_timeout_not_applied.yaml
tests/fixtures/cas_reconciliation/07_dist_unresolved.yaml
tests/fixtures/cas_reconciliation/08_dist_target_reject.yaml
tests/fixtures/cas_reconciliation/09_dist_target_cas_fail.yaml
tests/fixtures/cas_reconciliation/10_dist_capability_reserved.yaml
tests/fixtures/golden_vectors.json
tests/fixtures/lineage_graphs/01_singleton_valid.yaml
tests/fixtures/lineage_graphs/02_degraded.yaml
tests/fixtures/lineage_graphs/03_contradictory.yaml
tests/fixtures/lineage_graphs/04_unavailable_ignored.yaml
tests/fixtures/lineage_graphs/05_invalid_family.yaml
tests/fixtures/lineage_graphs/06_empty_evidence.yaml
tests/fixtures

## 4. Check the vectors (Python reference, manifest, determinism)

In [33]:
!{sys.executable} tools/check_vectors.py

wrote 29 artifact files + MANIFEST.json to /tmp/tmplupdfxnu
vectors: golden=36, auth_expiry=71, numeric=23, lineage=10, cas=10, scoring=58, coverage_v0.2.0=48, coverage_v0.2.1=50, coverage_v0.3.0=57
transport round trip: 144 signed objects (approval=22, attestation=36, decision=58, manifest=17, receipt=11), 56 signatures re-verified, 0 failures
checks run: 1218, failures: 0
current profile coverage-v0.3.0-alpha: no known open exploit UNDER the declared trust state, atom registry/aliases, membership authorities, clock authorities and calibration
superseded coverage-v0.2.0-alpha still reproduces its atom-padding exploit by design: 1


## 5. Independent cross-check (JavaScript)

Colab ships with Node. The cell skips itself if Node isn't installed.

In [34]:
import shutil
if shutil.which("node"):
    !node tools/xcheck.mjs
else:
    print("node not found — skipping independent cross-check")

coverage-v0.2.0 (superseded): 24 cases, 24 witnesses, 1 exploit reproduced by design (closed in v0.2.1)


coverage-v0.2.1: 32 cases, 18 witnesses, provenance admissibility re-derived independently


coverage-v0.3.0: 39 cases, 18 witnesses; membership, containment and defeat re-derived independently
scoring: prc_exp matches published bits on every input; platform Math.exp differs from the published libm reference on 2 of 32 inputs
xcheck (JavaScript, independent): 1959 checks, 0 failures


## 6. What the vectors say

In [35]:
import json, yaml, glob
auth = json.load(open("tests/vectors/auth_expiry_skew.json"))
print(f"{'id':<20}{'valid':<7}reason codes")
for v in auth["vectors"]:
    e = v["expected_output"]
    print(f"{v['test_id']:<20}{str(e['DecisionBindingValid']):<7}{', '.join(e['reason_codes']) or '-'}")

id                  valid  reason codes
auth_expiry_001     True   -
auth_expiry_002     False  EXPIRED_CAPABILITY, EVAL_SKEW_EXCEEDED
auth_expiry_003     True   -
auth_expiry_004     False  EXPIRED_CAPABILITY
auth_expiry_005     False  NOT_YET_VALID
auth_expiry_006     False  EVAL_SKEW_EXCEEDED
auth_expiry_007     True   -
auth_expiry_008     True   -
auth_expiry_009     False  SIGNATURE_INVALID, DECISION_KEY_UNAUTHORIZED
auth_expiry_010     False  SIGNATURE_INVALID, DECISION_KEY_UNAUTHORIZED
auth_expiry_011     False  DECISION_KEY_UNAUTHORIZED
auth_expiry_012     False  SIGNATURE_INVALID
auth_expiry_013     False  DECODE_INVALID
auth_expiry_014     False  DOMAIN_MISMATCH
auth_expiry_015     False  STATE_VERSION_MISMATCH
auth_expiry_016     False  TARGET_MISMATCH
auth_expiry_017     False  PROFILE_HASH_MISMATCH
auth_expiry_018     False  EVIDENCE_HASH_MISMATCH, LINEAGE_HASH_MISMATCH
auth_expiry_019     False  DECISION_NOT_ALLOW
auth_expiry_020     False  CAPABILITY_UNAVAILABLE
auth_ex

In [36]:
num = json.load(open("tests/vectors/kahan_cordic_ieee754.json"))
print(f"{'case':<30}{'op':<11}{'result':<8}{'lock':<6}output")
for c in num["cases"]:
    e = c["expected"]
    print(f"{c['id']:<30}{c['op']:<11}{str(e.get('result','')):<8}{str(e['safe_state_lock']):<6}{e['output_repr']}")

case                          op         result  lock  output
above_threshold               AUTH_CHECK True    False 0.75
equal_threshold               AUTH_CHECK True    False 0.7
one_ulp_below                 AUTH_CHECK False   False 0.6999999999999998
sum_0.1_0.2_vs_0.3            AUTH_CHECK True    False 0.30000000000000004
0.3_vs_sum_0.1_0.2            AUTH_CHECK False   False 0.3
neg_zero_normalised           AUTH_CHECK True    False 0.0
threshold_neg_zero            AUTH_CHECK True    False 0.0
min_subnormal_preserved       AUTH_CHECK True    False 5e-324
subnormal_order               AUTH_CHECK False   False 5e-324
max_subnormal_vs_min_normal   AUTH_CHECK False   False 2.225073858507201e-308
quiet_nan                     AUTH_CHECK False   True  0.0
signaling_nan                 AUTH_CHECK False   True  0.0
negative_nan                  AUTH_CHECK False   True  0.0
pos_inf                       AUTH_CHECK False   True  0.0
neg_inf                       AUTH_CHECK False   True  

In [37]:
for f in sorted(glob.glob("tests/fixtures/lineage_graphs/*.yaml")):
    d = yaml.safe_load(open(f)); e = d["expected"]
    vals = {k: (c.get("V"), c.get("I")) for k, c in e["classes"].items() if c["eligible"]}
    print(f"{os.path.basename(f):<36} eligible={e['eligible_classes']} (V,I)={vals} C={e['C']} audit={[a['class'] for a in e['audit_events']]}")
print()
for f in sorted(glob.glob("tests/fixtures/cas_reconciliation/*.yaml")):
    d = yaml.safe_load(open(f)); e = d["expected"]
    print(f"{os.path.basename(f):<36} {e['execution_status']:<12}{e['failure_phase']:<15}"
          f"recon={str(e['reconciliation_required']):<6}consumed={e['capability_consumed']}  {', '.join(e['reason_codes'])}")

01_singleton_valid.yaml              eligible=['F1'] (V,I)={'F1': (0.8, 1.0)} C=UNDEFINED_BY_SPEC audit=[]
02_degraded.yaml                     eligible=['F1'] (V,I)={'F1': (0.45, 1.0)} C=UNDEFINED_BY_SPEC audit=[]
03_contradictory.yaml                eligible=['F1'] (V,I)={'F1': (0.0, 1.0)} C=UNDEFINED_BY_SPEC audit=[]
04_unavailable_ignored.yaml          eligible=['F1'] (V,I)={'F1': (0.7, 1.0)} C=UNDEFINED_BY_SPEC audit=[]
05_invalid_family.yaml               eligible=[] (V,I)={} C=0.0 audit=['F1']
06_empty_evidence.yaml               eligible=[] (V,I)={} C=0.0 audit=[]
07_pairwise_min.yaml                 eligible=['F1', 'F2', 'F3'] (V,I)={'F1': (0.8, 0.4), 'F2': (0.6, 0.7), 'F3': (0.5, 0.3)} C=UNDEFINED_BY_SPEC audit=[]
08_invalidation_shrinks_pairs.yaml   eligible=['F1', 'F2'] (V,I)={'F1': (0.8, 0.9), 'F2': (0.6, 0.7)} C=UNDEFINED_BY_SPEC audit=['F3']
09_invalidation_to_singleton.yaml    eligible=['F1'] (V,I)={'F1': (0.8, 1.0)} C=UNDEFINED_BY_SPEC audit=['F2']


10_mixed_states_min.yaml             eligible=['F1'] (V,I)={'F1': (0.2, 1.0)} C=UNDEFINED_BY_SPEC audit=[]

01_local_commit.yaml                 COMMITTED   NONE           recon=False consumed=True  
02_local_cas_mismatch.yaml           REJECTED    PRECONDITION   recon=False consumed=False  CAS_PRECONDITION_FAILED
03_local_capability_consumed.yaml    REJECTED    PRECONDITION   recon=False consumed=True  CAPABILITY_UNAVAILABLE
04_dist_ack.yaml                     COMMITTED   NONE           recon=False consumed=True  
05_dist_timeout_applied.yaml         COMMITTED   NONE           recon=False consumed=True  DISPATCH_TIMEOUT


06_dist_timeout_not_applied.yaml     ROLLED_BACK DISPATCH       recon=False consumed=True  DISPATCH_TIMEOUT
07_dist_unresolved.yaml              UNRESOLVED  RECONCILIATION recon=True  consumed=True  DISPATCH_TIMEOUT, RECONCILIATION_UNRESOLVED
08_dist_target_reject.yaml           REJECTED    TARGET         recon=False consumed=True  TARGET_REJECTED
09_dist_target_cas_fail.yaml         REJECTED    PRECONDITION   recon=False consumed=True  CAS_PRECONDITION_FAILED
10_dist_capability_reserved.yaml     REJECTED    PREPARE        recon=False consumed=True  CAPABILITY_UNAVAILABLE


## 6b. Scoring profile `jaccard-v0.1.0-alpha` (EXPERIMENTAL)

The success criterion: with identical apparent confirmations and identical authorization context, changing **only** the evidence-lineage graph produces a reproducible ALLOW → DENY transition at a consequence-calibrated boundary.

In [38]:
sc = json.load(open("tests/vectors/scoring_jaccard_v0.1.0-alpha.json"))
h = sc["headline"]
print(h["claim"]); print()
print("inputs identical except dependency atoms:", h["inputs_identical_except_dependencies"])
for k in ("allow", "deny"):
    s = h[k]; print(f"  {k.upper():<6} M={s['M']:<20} C={s['C']:<20} T={s['T']}  I={s['I']}")
print()
print(f"{'case':<36}{'dec':<6}{'M':<22}{'C':<22}T")
for c in sc["cases"]:
    s = c["summary"]
    print(f"{c['id']:<36}{s['decision']:<6}{s['M']:<22}{s['C']:<22}{s['T']}{'   <-- FINDING' if 'finding' in c else ''}")
print()
for c in sc["cases"]:
    if "finding" in c:
        print(f"[{c['id']}] {c['finding']}")

With identical apparent confirmations and identical authorization context, changing only the evidence-lineage graph produces a reproducible ALLOW -> DENY transition at a consequence-calibrated boundary.

inputs identical except dependency atoms: True
  ALLOW  M=2.77                 C=0.9373379952578469   T=0.9200014828863916  I={'F-vendor-a': '1.0', 'F-vendor-b': '1.0', 'F-webhook': '1.0'}
  DENY   M=0.9233333333333335   C=0.6028071441823737   T=0.9200014828863916  I={'F-vendor-a': '0.33333333333333337', 'F-vendor-b': '0.33333333333333337', 'F-webhook': '0.33333333333333337'}

case                                dec   M                     C                     T
01_no_eligible_evidence             DENY  0.0                   0.0                   0.9200014828863916
02_single_family                    ALLOW 0.8                   0.5506710358827784    0.5106646222689113
03_identical_dependencies           DENY  0.0                   0.0                   0.5510964300482651
04_disjoint_d

### Why the profile carries its own `exp` (finding F-1)

§2.3 requires bit-exact comparison. Platform `exp` implementations differ by 1 ulp on a noticeable share of inputs, and at the decision boundary that difference is an authorization difference. The cell below evaluates case 10 with platform `exp` in Python (glibc) and in Node (V8).

In [39]:
import math, struct
f = lambda h: struct.unpack(">d", bytes.fromhex(h))[0]
prof = sc["profiles"]["calibrated"]["declaration"]
g, b = f(prof["consequence_gamma_bits"]["payment.refund"]), f(prof["beta_bits"])
case = next(c for c in sc["cases"] if c["id"] == "10_boundary_one_ulp_below")
V = f(case["input"]["families"][0]["members"][0]["value_bits"])
T = 1 - 0.5 * math.exp(-(g * b)); C = 1 - math.exp(-V)
print("prc-exp-v1 (all runtimes):", case["summary"]["decision"])
print("Python + platform exp    :", "ALLOW" if C >= T else "DENY")
if shutil.which("node"):
    js = f"const T=1-0.5*Math.exp(-({g!r}*{b!r})), C=1-Math.exp(-{V!r}); console.log(C>=T?'ALLOW':'DENY')"
    print("Node   + platform exp    :", subprocess.run(["node", "-e", js], capture_output=True, text=True).stdout.strip())

prc-exp-v1 (all runtimes): DENY
Python + platform exp    : ALLOW
Node   + platform exp    : DENY


## 6c. Scoring profile `coverage-v0.2.0-alpha` (EXPERIMENTAL)

Coverage mass credits each material dependency once, at its best per-atom share, so adding evidence can never lower M. Unattributed evidence shares one `UNKNOWN:*` sink and carries no weight at or above the consequence gate. A non-finite guard runs in the engine before any profile code.

In [40]:
cv = json.load(open("tests/vectors/scoring_coverage_v0.2.0-alpha.json"))
print(f"{'case':<36}{'dec':<6}{'stage':<14}{'M':<22}C")
for c in cv["cases"]:
    s = c["summary"]
    print(f"{c['id']:<36}{s['decision']:<6}{s['stage']:<14}{s['M']:<22}{s['C']}{'   <-- KNOWN EXPLOIT (expected failure)' if 'known_exploit' in c else ''}")
x = next(c for c in cv["cases"] if "known_exploit" in c)
print()
print("open exploit:", x["known_exploit"]["why"])
print("security expectation:", x["known_exploit"]["security_expectation"], "| open until:", x["known_exploit"]["open_until"])

case                                dec   stage         M                     C
c01_no_eligible_evidence            DENY  PROFILE       0.0                   0.0
c02_single_family                   ALLOW PROFILE       0.8                   0.5506710358827784
c03_three_independent_pass          ALLOW PROFILE       2.7699999999999996    0.9373379952578468
c04_three_correlated_fail           DENY  PROFILE       1.3139999999999998    0.7312570681556056
c05_two_of_three_correlated         DENY  PROFILE       2.05                  0.8712650964121957
c06_two_independent_insufficient    DENY  PROFILE       1.8699999999999999    0.8458763381848685
c07_mirror_added_neutral            ALLOW PROFILE       2.7699999999999996    0.9373379952578468
c08_bridge_added_neutral            ALLOW PROFILE       2.7699999999999996    0.9373379952578468
c09_unknown_at_high_consequence     DENY  PROFILE       1.3139999999999998    0.7312570681556056
c10a_refund_known_only              DENY  PROFILE       0.6   

In [41]:
!{sys.executable} tools/property_tests.py --trials 1500

property tests, 1500 random trials per property
                  P1 mono  P2 idem  P3 order  P4 bridge  P5 decision
  coverage-v0.2         0        0         0          0            0
  jaccard-v0.1        287      720         0        463          245
  jaccard counterexample (P2): M 1.2587890625 -> 0.78515625


  coverage-v0.2.1-alpha over attested evidence, 500 trials: P1=0, P2=0, P3=0, P4=0, P5=0, P6=0, P7=0, P8=0
    (P6 defeater always DENY, P7 unattested neutral on a wire, P8 forged/lifted attestation neutral)


  coverage-v0.3.0-alpha with membership manifests, 500 trials: P1=0, P2=0, P3=0, P4=0, P5=0, P6=0, P9=0, P10=0
    (P9 unlisted injection changes nothing, P10 containment keeps M/C and forbids ALLOW)


  P11 totality fuzz (1500 mutations): decision_verifier_exceptions=0, bitflips_accepted=0, resigned_mutations_evaluated=1500, scoring_faults_converted_to_deny=0
coverage profiles: all properties hold


## 6d. `coverage-v0.2.1-alpha`: verified provenance (P-1) and defeaters (D-1)

Dependency atoms now count only when they arrive in an admitted provenance attestation. The v0.2.0 atom-padding exploit (x01) is a must-deny case, alongside thirteen variants of it. An authenticated contradiction vetoes a decision without touching M or C.

In [42]:
v21 = json.load(open("tests/vectors/scoring_coverage_v0.2.1-alpha.json"))
print(f"{'case':<38}{'dec':<6}{'M':<22}{'C':<21}provenance (non-admitted)")
for c in v21["cases"]:
    s, tr = c["summary"], c["expected_trace"]
    odd = sorted({f"{o['outcome']}: {o['detail']}" for f in tr.get("families", {}).values()
                  for o in f.get("provenance", {}).values() if o["outcome"] != "ADMITTED"})
    extra = ("  DEFEATER" if tr.get("defeaters") else "") + ("  " + odd[0] if odd else "")
    print(f"{c['id']:<38}{s['decision']:<6}{s['M']:<22}{s['C']:<21}{extra}")
old = next(c for c in cv["cases"] if c["id"] == "x01_atom_padding_exploit")["summary"]
new = next(c for c in v21["cases"] if c["id"] == "x01_self_declared_padding")["summary"]
print()
print(f"x01 atom padding: v0.2.0 {old['decision']} (C={old['C'][:6]})  ->  v0.2.1 {new['decision']} (C={new['C'][:6]})")

case                                  dec   M                     C                    provenance (non-admitted)
c03_three_independent_pass            ALLOW 2.7699999999999996    0.9373379952578468   
c04_three_correlated_fail             DENY  1.3139999999999998    0.7312570681556056   
c05_two_of_three_correlated           DENY  2.05                  0.8712650964121957   
c06_two_independent_insufficient      DENY  1.8699999999999999    0.8458763381848685   
c07_mirror_added_neutral              ALLOW 2.7699999999999996    0.9373379952578468   
c08_bridge_added_neutral              ALLOW 2.7699999999999996    0.9373379952578468   
c09_reattested_duplicate_idempotent   ALLOW 2.7699999999999996    0.9373379952578468   
c10a_refund_attested_only             DENY  0.6                   0.4511883639059735   
c10b_unattested_low_consequence       ALLOW 1.05                  0.6500622508888447     UNATTESTED: no provenance attestation
c11_unattested_spam_single_sink       ALLOW 1.05        

## 6e. `coverage-v0.3.0-alpha`: containment (A-1) and family membership (M-1)

An unattested contradiction no longer changes the evidence weight. Its presence removes eligibility for automatic authorization, so a request then needs a bound manual approval. An authenticated contradiction defeats the request, and a tampered one locks it. Only members listed in a signed family manifest take part at all, so anything injected is quarantined.

In [43]:
v30 = json.load(open("tests/vectors/scoring_coverage_v0.3.0-alpha.json"))
print(v30["exploit_statement"]); print()
print(f"{'case':<48}{'decision':<17}{'disposition':<12}{'M':<21}{'C':<20}quarantined")
for c in v30["cases"]:
    s, tr = c["summary"], c["expected_trace"]
    q = sum(e["event"] == "MEMBER_QUARANTINED" for e in tr.get("audit_events", []))
    print(f"{c['id']:<48}{s['decision']:<17}{str(s['disposition']):<12}{s['M']:<21}{s['C']:<20}{q or ''}")

No known open exploit UNDER the declared trust state, atom registry and alias authority, family-membership authorities, clock authorities and calibration. Each of those is an assumption, not a result (see spec §5 and docs/CONFORMANCE_ASSUMPTIONS.md Q-1..Q-6).

case                                            decision         disposition M                    C                   quarantined
c01_three_independent_pass                      ALLOW            CLEAR       2.7699999999999996   0.9373379952578468  
c02_three_correlated_fail                       DENY             CLEAR       1.3139999999999998   0.7312570681556056  
c03_two_independent_insufficient                DENY             CLEAR       1.8699999999999999   0.8458763381848685  
a00_four_independent_pass                       ALLOW            CLEAR       3.67                 0.9745235300533189  
a01_admitted_contradiction_defeats              DENY             DEFEAT      3.67                 0.9745235300533189  
a02_unattested

### Transport schemas are transport only

Every signed object in the suite goes canonical → Protobuf → wire → Protobuf → canonical → PRC-LP-0, and its hash and signature are re-checked. Two demos show why Protobuf bytes are never hashed. In the first, two different wire encodings parse to the same message. In the second, an unknown field is kept by a lenient parser and must be rejected by a strict verifier.

In [44]:
!{sys.executable} tools/transport_roundtrip.py

transport round trip: 144 signed objects (approval=22, attestation=36, decision=58, manifest=17, receipt=11), 56 signatures re-verified, 0 failures
  non-canonical wire demo: {'same_parsed_message': True, 'wire_sha256_differs': True, 'canonical_hash_identical': True}
  unknown-field demo:      {'lenient_parser_accepts': True, 'strict_verifier_rejects': True}


  FlatBuffers round trip: 144 signed objects, 56 signatures re-verified, 0 failures; demo {'field_order_changes_bytes': True, 'flatc_rejects_unknown_field': True}


## 7. Do the checks catch errors?

These cells work on a throwaway copy of the repo and break it on purpose. The checkers must fail.

- **Tampered expectation:** mark `auth_expiry_003` (execution exactly at `τ_expiry`) as invalid.
- **Model regression:** change `τ_exec + U ≤ τ_expiry` to `<` in the reference model. The generator must refuse to write vectors.

In [45]:
import tempfile, shutil, subprocess, json, sys
tmp = tempfile.mkdtemp()
shutil.copytree(".", tmp, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".git"))
p = os.path.join(tmp, "tests/vectors/auth_expiry_skew.json")
d = json.load(open(p)); d["vectors"][2]["expected_output"]["DecisionBindingValid"] = False
json.dump(d, open(p, "w"), indent=2)
for cmd in ([sys.executable, "tools/check_vectors.py"], ["node", "tools/xcheck.mjs"]):
    if cmd[0] == sys.executable or shutil.which(cmd[0]):
        r = subprocess.run(cmd, cwd=tmp, capture_output=True, text=True)
        print(os.path.basename(cmd[0]), cmd[1], "-> exit", r.returncode)
        print("\n".join(l for l in r.stdout.splitlines() if "FAIL" in l or "failures" in l))

shutil.rmtree(tmp); tmp = tempfile.mkdtemp()
shutil.copytree(".", tmp, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".git"))
m = os.path.join(tmp, "reference/prc_ref.py")
src = open(m).read(); assert "if not t_exec + U <= t_exp:" in src
open(m, "w").write(src.replace("if not t_exec + U <= t_exp:", "if not t_exec + U < t_exp:"))
r = subprocess.run([sys.executable, "tools/generate_vectors.py", "--out", os.path.join(tmp, "out")], cwd=tmp, capture_output=True, text=True)
print("generator with regressed model -> exit", r.returncode, "|", (r.stdout + r.stderr).strip())
shutil.rmtree(tmp)

python3 tools/check_vectors.py -> exit 1
transport round trip: 144 signed objects (approval=22, attestation=36, decision=58, manifest=17, receipt=11), 56 signatures re-verified, 0 failures
checks run: 1218, failures: 3
  FAIL manifest: vectors/auth_expiry_skew.json hash drift
  FAIL auth: auth_expiry_003 validity
  FAIL determinism: vectors/auth_expiry_skew.json differs


node tools/xcheck.mjs -> exit 1
xcheck (JavaScript, independent): 1959 checks, 1 failures
  FAIL auth auth_expiry_003 validity
generator with regressed model -> exit 1 | auth_expiry_003: reference model gives ['EXPIRED_CAPABILITY'], case declares []


## 8. Commit

Review `docs/CONFORMANCE_ASSUMPTIONS.md` before committing. The A-xx items are choices that need your sign-off, and the G-xx items are gaps in the normative text.

In [46]:
# !git add reference tools docs tests README.md .gitattributes
# !git commit -m "Add deterministic conformance vectors, reference model and cross-checks"
# !git push